# GC Analysis 60°×60° — Sanghwan Pipeline, Ported to 해밝 환경

**목표**: Sanghwan의 `GC_analysis-60x60-models_12yr.ipynb`를 해밝 서버에서 **원본 그대로** 재현.

**이전 시도와의 차이 (2026-04-20 Port v2)**:
- **데이터 생성 셀(Cell 12)을 활성화** 해서 `gtselect → gtmktime → gtbin → gtltcube → gtexpcube2`를 해밝 환경에서 **처음부터** 실행. 이전 포팅 시도에서 ~13% offset이 났던 근본 원인은 입력 데이터(CCUBE/LTCUBE/EXPCUBE)가 Sanghwan 원본과 달랐기 때문(REF_sanghwan_pipeline_port_SUMMARY.md §5.4 참고). Sanghwan 본인은 DMPNP2026 발표(슬라이드 23)에서 Cholis+2022와 2σ 이내 일치를 달성했음.
- Cell 13의 400×400 CCUBE 덮어쓰기 **비활성화** (600×600 유지 — `[100:500]` 슬라이싱 호환)
- Cell 34의 `Likelihood('I', 0)` 하드코딩 sanity test **제거** (모델 I 파일 미존재 시 `FileNotFoundError`)
- Cell 34의 `isotropic_flux_data[i]` → `isotropic_flux_data[self.energy_bin]` 수정
- Cell 34의 `GC_psc_model.xml` → `GC_psc_model_DR2.xml` 수정 (SourceList가 실제 생성한 파일명과 일치)
- Cell 61의 `/home/sanghwan/C.Weniger/covariance.dat` 하드코드 라인 비활성화 (탐색용 셀)

**작업 디렉토리**: `/home/haebarg/GCE-Chi-square-fitting/GCE_12yr_reproduce/`

## 사전 준비 (실행 전 체크)

다음 파일/디렉토리가 이미 준비되어 있어야 합니다:

```
./                                    (= GCE_12yr_reproduce/)
├── MapCubes/                         # 14 모델의 bremss/ics/pion MapCube (.fits)
├── GCE_template_NFW2.fits            # NFW² GCE 스페이셜 템플릿 (600×600)
├── Fermi_Bubbles_template.fits       # Fermi Bubble 스페이셜 템플릿
├── isotropic_spectrum_ff.txt         # IGRB 스펙트럼 FileFunction
├── fermi_bubble_spectrum.txt         # 버블 스펙트럼 FileFunction
├── GC_analysis_sanghwan/             # 중간 산출물 저장 디렉토리 (비어있으면 자동 생성)
│   └── Model/
│       ├── bubble_constraints.txt          # 1407.7905 Table 2
│       └── iso_constraints_full_err.txt    # 1410.3696 Table 3
```

상위 디렉토리:
```
../GCE_allsky_data/
├── lat_spacecraft_merged_12yr.fits
└── photon_files/lat_photon_weekly_*.fits

../GCE_12yr_data/
├── gll_psc_v23.xml, gll_psc_v23.fit
├── gll_iem_v07.fits
├── iso_P8R3_SOURCE_V3_v1.txt
└── Templates/                        # 확장소스 템플릿
```

## 실행 순서

1. **Cell 0**: imports (Fermi tools 포함)
2. **Cell 1–11**: energy bins 정의 (14 bins in `masking()` + 17 bins in `bin_definitions.fits`)
3. **Cell 12**: 데이터 선택 & CCUBE 생성 (이번 포팅의 핵심)
4. **Cell 13**: disabled
5. **Cell 14–17**: LTCUBE, EXPCUBE (center 600×600, edge 3600×1800)
6. **Cell 18–24**: SourceList & PSC 필터링 (source classification)
7. **Cell 25–30**: mask 생성 (PSC mask + disk mask)
8. **Cell 31–34**: **메인 루프** — 14 models × {XML gen, gtsrcmaps×2, gtmodel×N, emcee 14 bins}
9. **Cell 35–93**: 결과 분석 & 탐색 셀 (선택적)

## 주의사항

- **`LD_LIBRARY_PATH` 순서**: conda fermi env의 cfitsio/wcslib/healpix가 GALPROP 라이브러리보다 **앞**에 와야 함. 역순이면 gtsrcmaps 메모리 12+ GB 폭증.
- **VSCode Jupyter 커널 silent-kill 위험**: Cell 34가 매우 무겁습니다 (모델당 ~70분). 필요 시 `subprocess.run()` 또는 nohup + `jupyter nbconvert --to notebook --execute`로 백그라운드 실행하세요.
- **ChainConsumer API 호환성**: 최신 버전에서 `add_chain()` 시그니처가 변경됨. 원본 코드가 에러 내면 `try/except`로 감싸거나 `pip install chainconsumer==0.34.0`로 버전 고정.
- **emcee 속도**: 원본의 `log_factorial` Python 루프는 evaluation당 ~25초(매우 느림). 필요 시 `scipy.special.gammaln`으로 교체하면 ~88배 빨라집니다 (결과 동일). 일단 원본 그대로 유지.

## 검증 타겟

Sanghwan 본인의 DMPNP2026 발표 결과 (슬라이드 23):
- Model X, I에 대해 Cholis+2022 green band 내에 GCE flux가 들어감
- 1–10 GeV flux ratio ≈ 1.0 (2σ 이내)

이전 포팅 시도의 결과 (데이터 불일치 상태):
- 1–10 GeV ratio ≈ 0.87 (~13% deficit)

**이번 목표**: 데이터 생성을 처음부터 Sanghwan 코드대로 실행하여 ratio ≈ 1.0 달성.


---

## ⚡ v3 추가사항 (2026-04-20 update — kernel-crash safety)

이 노트북은 커널 충돌 후 안전하게 재실행할 수 있도록 다음과 같이 강화되었습니다:

### 1. 자동 skip 메커니즘

Cell 1 (imports)에 `needs_run()`, `skip_or_run()` 헬퍼가 추가되어 **이미 생성된 파일이 있으면 자동으로 그 셀/단계를 건너뜁니다**. 강제 재계산이 필요하면 노트북 어디서든:
```python
FORCE_RECOMPUTE = True
```
를 설정한 후 해당 셀부터 다시 실행하세요.

### 2. 가드가 추가된 셀

| Cell | 산출물 | 가드 동작 |
|---|---|---|
| 8 | `bin_definitions.fits` | 존재 시 gtbindef 스킵 |
| 13 | `Allsky_select`, `Allsky_gti`, `GC_ccube` | **3단계 각각** 별도 스킵 |
| 16 | `Allsky_ltcube` | 존재 시 gtltcube 스킵 |
| 17 | `GC_expcube_center` | 존재 시 스킵 |
| 18 | `Allsky_expcube_edge` | 존재 시 스킵 |
| 19 | `GC_model_DR2.xml` | 존재 시 SourceList 스킵 |
| 20 | `GC_psc_model_DR2.xml` | 존재 시 XML cleanup 스킵 |
| 29 | (in-memory `full_mask`) | **`.npy` 있으면 load**, 없으면 계산 (heavy step) |
| 30 | `GC_disk_mask_*.npy` | 존재 시 스킵 |
| 31 | `GC_mask_*_DR2.npy` | 존재 시 스킵 |
| 32 | `empty_model.xml` | 존재 시 스킵 |
| **35** | per model: `GCE_model_{M}_12yr_cholis.dat` | **모델 단위 + 단계 단위 다층 가드** |

### 3. Cell 35 (메인 루프) 동작

각 모델에 대해:
1. **모델 단위 가드**: `GCE_model_{M}_12yr_cholis.dat`가 이미 있으면 → `continue` (전체 모델 스킵)
2. **단계 단위 가드**: 각 `gtsrcmaps.run()` / `gtmodel.run()`이 자기 outfile이 이미 있으면 자동 스킵
3. **GCE/Fermi_bubble/isotropic gtmodel 추가**: 원본 노트북에서 빠져있던 부분 (E4 fix). emcee Likelihood 클래스가 이 컴포넌트 fits를 읽기 때문에 필수.
4. emcee 실행 후 .dat 저장

### 4. 비활성화된 셀 (36~94)

원본의 Cell 36 이후 모든 코드 셀은 탐색/legacy/디버깅용입니다. v3에서는 모두 주석 처리되어 "Run All" 시 자동 통과합니다. 결과 검증이 필요하면 해당 셀을 열어 수동으로 주석 해제하세요.

### 5. 권장 실행 방법

**시나리오 A — 깨끗한 상태 (모든 산출물 없음)**:
- Cell 1부터 순서대로 실행 → 모든 단계 진행

**시나리오 B — 커널 충돌 후 재시작**:
- 그냥 Cell 1부터 다시 "Run All" → 이미 만들어진 파일은 자동 스킵, 안 만들어진 것부터 이어서 진행

**시나리오 C — 특정 단계만 강제 재실행**:
- 해당 산출물을 `rm` 으로 삭제 후 그 셀부터 재실행
- 또는 셀 내부에서 일시적으로 `FORCE_RECOMPUTE = True` 설정

**시나리오 D — Cell 35의 특정 모델만 재실행**:
- `rm ./GCE_model_{M}_12yr_cholis.dat` → Cell 35 재실행 시 그 모델만 다시 돔
- 부분적으로 (e.g. emcee만) 재실행하려면 해당 모델의 srcmap/component fits도 함께 삭제


---

## 🩹 v3.1 hotfix (Cell 12 `E_bounds` NameError)

원본 노트북의 Cell 9, 10, 11 (v3 기준 Cell 10, 11, 12)과 22, 24, 26 (v3 기준 Cell 23, 25, 27)은 단순히 변수 이름 한 줄만 적혀있는 "값 표시" 셀이었음. Sanghwan이 인터랙티브 디버깅 시 변수 내용을 확인하던 흔적.

문제: **Cell 12 (`E_bounds`)는 변수가 Cell 26에서야 정의되므로 처음부터 Run All 시 NameError**로 실행이 중단됨.

해결: 이 6개 셀을 모두 `[DISABLED v3.1]` 배너와 함께 주석 처리. 파이프라인에 기여하지 않으므로 손실 없음. 필요시 해당 셀 열어서 수동 주석 해제하면 변수 값 확인 가능.


---

## 🩹 v3.2 hotfix (Cell 30 `w` NameError)

Cell 30 (disk mask 생성)에서 `NameError: name 'w' is not defined` 발생. 원인:

- Sanghwan 원본 Cell 29도 `w`를 정의 없이 사용함 (`w.wcs_pix2world(0, i, 0)[1]`)
- `w`는 원본 Cell 34(= v3 Cell 35, 메인 루프) L323에서야 정의됨: `w = WCS(raw_map[0].header).dropaxis(2)`
- Sanghwan이 인터랙티브 작업 중 Cell 34 일부를 먼저 실행한 뒤 Cell 29로 돌아와서 실행한 흔적 — 순서대로 Run All 시 실패.

해결: Cell 30 안에서 `w` 필요 직전에 `w = WCS(src[0].header).dropaxis(2)` 한 줄 추가. `src`는 같은 셀에서 이미 열려 있는 CCUBE fits 객체를 재사용.


---

## 🔧 v3.3 hotfix (gtsrcmaps energy-range error + subprocess separation)

### 문제 1: gtsrcmaps 에러
```
Caught St11range_error at the top level:
Requested energy, 556077, lies outside the range of the input file, 120, 344430
```

**원인**: CCUBE의 마지막 bin 상단(556 GeV = 556077 MeV)을 gtsrcmaps가 샘플하려는데, FileFunction 스펙트럼 파일(`isotropic_spectrum_ff.txt`, `fermi_bubble_spectrum.txt`)은 ~344 GeV에서 끝나서 범위 초과.

**해결**: 새로운 셀(원래 Cell 13 데이터 prep 다음)이 자동으로 `extend_spectrum_files.py`를 호출. Power-law 외삽으로 두 스펙트럼 파일을 1.5 TeV까지 늘립니다. Idempotent — 이미 늘어났으면 no-op.

### 문제 2: 메인 루프(원 Cell 35)가 너무 무거움
- 14 모델 × {gtsrcmaps × 2 + gtmodel × 12 + emcee × 14} = 16-20시간 wall time
- gtsrcmaps 한 번이 ~5-12 GB RAM 사용
- 노트북 안에서 돌리면 OOM/timeout으로 커널이 죽고 모든 in-memory 상태 손실

**해결**: 원래 Cell 35를 별도 Python 프로세스로 분리. 새 셀은 단순 subprocess wrapper로, 4가지 모드 지원:

| MODE | 용도 | 동작 |
|---|---|---|
| `"single"` | 단일 모델 검증 | `SINGLE_MODEL` 하나를 foreground subprocess로 실행, 출력 노트북에 stream |
| `"all"` | 전체 14 모델 (foreground) | 노트북 셀이 끝까지 대기 (~16-20시간). 권장 X |
| `"background"` | 전체 14 모델 (백그라운드) | `nohup` 형태로 detached subprocess 실행. 셀은 즉시 반환. 로그는 `./main_loop.log`, PID는 `./main_loop.pid` |
| `"monitor"` | 진행 상황 확인 | 백그라운드 프로세스 상태 + 완료된 .dat 파일 목록 + 로그 마지막 20줄 |

### 권장 워크플로우

**Step 1: 단일 모델 검증** (먼저 ~70-90분)
```python
# Cell 36 (이전 main loop 자리):
MODE = "single"
SINGLE_MODEL = "X"
```
실행 → `./GCE_model_X_12yr_cholis.dat` 검증 → 1-10 GeV ratio가 Sanghwan 결과(≈1.0)와 가까운지 확인.

**Step 2: 전체 14 모델 (백그라운드)**
```python
MODE = "background"
```
실행 → 즉시 반환, ~16-20시간 후 완료. 진행 상황은 다른 셀에서:
```python
MODE = "monitor"
```
또는 셸에서 `tail -f main_loop.log`.

### 새 파일

- `extend_spectrum_files.py`: 스펙트럼 확장 헬퍼 (자동 호출됨)
- `run_main_loop_subprocess.py`: 메인 루프 subprocess runner (Cell 36에서 호출됨)

두 파일을 작업 디렉토리에 복사한 뒤 노트북을 재실행하세요.


---

## 🚀 v3.3 release notes

### Two new fixes:

#### 1. Cell 35 (NEW): FileFunction spectrum extension to 1 TeV

**문제**: gtsrcmaps에서 다음과 같은 에러 발생
```
Caught St11range_error at the top level: Requested energy, 556077,
lies outside the range of the input file, 120, 344430
```
- gtsrcmaps의 `emapbnds=yes` + IRF energy dispersion 처리가 분석 bin보다 넓은 (~1 TeV) 에너지 범위의 FileFunction 스펙트럼을 요구함
- 해밝 환경의 `fermi_bubble_spectrum.txt`, `isotropic_spectrum_ff.txt`는 ~344 GeV에서 끝남 → range_error

**해결 (Cell 35)**: 두 스펙트럼 파일을 power-law 외삽으로 1.1 TeV까지 자동 연장
- 마지막 10 점에 PL fit → 30개 새 포인트 logspace로 추가
- 원본은 `.original` 백업
- 이미 1 TeV 이상이면 [SKIP]
- Cell 36 (model_list) 직전에 실행되어 메인 루프 시작 전에 보장

#### 2. Cell 37 (메인 루프): subprocess로 분리

**문제**: 메인 루프 (gtsrcmaps × 2 + gtmodel × 12 + emcee × 14 bins, per model × 14 models)가 너무 무거워서 ~16-20시간 소요. Jupyter 커널 안에서 돌리면 OOM, fork issue, VSCode timeout 등으로 죽을 위험 큼.

**해결**: Cell 37 = 외부 `run_main_loop_subprocess.py` 호출 wrapper
- 4가지 MODE 선택 가능:
  - `"single"`: 한 모델만 동기 실행 (검증용)
  - `"all"`: 14 모델 모두 foreground subprocess (notebook이 출력 relay)
  - `"background"`: nohup 비슷하게 detach 실행 (커널 죽어도 살아남음). PID 파일 저장
  - `"monitor"`: 백그라운드 작업 상태 + 로그 tail + 완료된 .dat 목록
- gammaln 최적화 적용됨 (Sanghwan 원본 Python `log_factorial` 대비 ~88× 속도 향상)
- 작업이 죽어도 다음 실행 시 미완성 모델만 자동으로 이어서 처리

**권장 사용 흐름**:
```
1. MODE="single", SINGLE_MODEL="X"로 셀 실행 → Model X만 ~70-90분
2. ./GCE_model_X_12yr_cholis.dat 검증 (Sanghwan ratio≈1.0과 비교)
3. 검증 OK 시 MODE="background"로 변경 → 14 모델 백그라운드 실행
4. 진행 상황은 MODE="monitor" 또는 `tail -f main_loop.log`로 확인
```

**필수 외부 파일**: `run_main_loop_subprocess.py` — 같은 디렉토리에 있어야 함


---

## 🛡️ v3.4 hotfix (integrity check in needs_run)

**문제 발견 (실행 중)**:
v3.3에서 첫 번째 gtsrcmaps가 RuntimeError로 죽으면서 부분적으로 만들어진 srcmap fits 파일이 디스크에 남음. 다음 실행 시 `needs_run()`이 그 파일이 "있다"고 판단해서 [SKIP]. 그 다음 gtmodel이 부분 srcmap을 읽으려고 시도 → 다음 에러:
```
Cannot read keyword "NDSKEYS" in file "..." (CFITSIO ERROR 202: keyword not found in header)
```

**해결 (v3.4)**: `needs_run()`을 단순 존재 체크에서 **무결성 체크**까지 하도록 강화.

새 동작:
- `.fits` 파일: astropy로 열어서 구조 확인. **srcmap 파일은 추가로 NDSKEYS 헤더 키워드 존재 여부 검증** (gtsrcmaps가 정상 종료해야만 적어주는 마커).
- `.npy` 파일: `np.load`로 열어서 검증
- `.xml` 파일: `ET.parse`로 파싱 검증
- 무결성 체크 실패 시: 자동으로 그 파일을 삭제하고 `[v3.4 GUARD] removing unhealthy file: ...` 메시지 출력 → 재생성 트리거
- `skip_or_run`이 실행 후에도 결과 무결성을 검증해서, 도구가 거짓 성공 후 corrupt 파일을 남기는 경우 명시적 RuntimeError로 알림

**부수 도구**: `diagnose_srcmaps.py`
실행 전후로 디스크 상태를 점검할 수 있는 진단 스크립트:
```
python diagnose_srcmaps.py            # 보고만
python diagnose_srcmaps.py --rm       # 불완전 파일 삭제
```


---

## 🔍 v3.5 additions: running diagnosis + post-run validation cells

**진단 도구**: `diagnose_running.md` — 메인 루프가 실행 중일 때 다른 터미널에서 돌릴 진단 명령 모음.

**새 검증 셀 4개** (notebook 맨 뒤, `## 🔍 Validation cells` 섹션):

| 셀 | 용도 |
|---|---|
| V1 | 단일 모델 결과 로드 (`VALIDATE_MODEL = "X"`로 선택). E, GCE flux, 5 컴포넌트 flux, CCUBE counts 모두 준비. 이후 셀에서 공유 변수로 사용. |
| V2 | **GCE SED decomposition plot** — 원본 Sanghwan Cell 42 스타일. 5 컴포넌트 + GCE fit + observed + summed 한 화면. `GCE_SED_plot_{M}_12yr.png` 저장. |
| V3 | **PSC + disk mask visualization** — 3개 에너지 bin에 대해 CCUBE와 CCUBE×mask 나란히 표시. 밝은 PSC mask (큰 원), 약한 PSC mask (작은 원), \|b\|<2° 디스크 마스크가 제대로 적용됐는지 즉시 판정 가능. `mask_verification.png` 저장. |
| V4 | **Multi-model envelope** — 여러 `.dat`가 있을 때 min-max 엔벨로프 + median 라인. Cholis Zenodo reference (`../GCE_TEMPLATES_FILES_v3/Figures_12_and_14_GCE_Spectra/`)가 있으면 자동으로 옆에 두고 1-10 GeV ratio 출력. Sanghwan DMPNP2026 목표값(≈1.0)과 비교. `GCE_multimodel_envelope.png` 저장. |

**누락된 검증 기능** (이번 포팅 범위 밖):

| 기능 | 현재 상태 | 대응 |
|---|---|---|
| **chi-square contour plot** (2D σv vs mass) | Sanghwan 노트북엔 없음. 이전 v2 port의 `plot_dm_contour_*.py`에 구현되어 있음 | 해밝님의 `claude_try/` 디렉토리에서 가져오기 |
| **80 models full envelope** | 이 노트북엔 없음. Cholis Fig. 12/14 재현은 별도 task | 14 모델 완주 후 추가 구현 필요 |
| **control-ROI covariance matrix** | 이 노트북엔 없음. Sanghwan의 16yr 노트북 covariance 셀에 구현됨 (REF_GCE_covariance_16yr_SUMMARY.md §2 참고) | 17yr 파이프라인 작업 시 포팅 |
| **DM mass/σv scan** | 이 노트북엔 없음 | 이전 세션의 `MG5Interpolator` 재활용 가능 |


---

## 🎯 v3.6: V4 hotfix + V5/V6 DM constraint cells

**V4 NameError 수정** (single-model case):
- `_E_ref`, `_env_med` 등이 `if len(all_flux) >= 2`에서만 정의되는데 아래쪽 비교 블록이 `>= 1` 조건으로 들어가 NameError 발생
- 고침: 변수 정의를 위로 옮기고 global-scope 이름(`E_ref`, `env_med`, `env_lo`, `env_hi`, `ref_E`, `ref_flux`)로 승격. 모델 1개일 때는 그 모델을 직접 Cholis reference와 비교
- 1-10 GeV ratio가 0.85-1.15 범위면 ✓, 30% 이상 빗나가면 ✗ 등 자동 판정 출력

**V5: PPPC4 DM spectrum 로더**
- `/home/haebarg/GCE-Chi-square-fitting/AtProduction_gammas.dat`에서 Cirelli+2011 표준 e⁺e⁻/γ 스펙트럼 로드
- `RegularGridInterpolator` 기반 `pppc4_channels['b']`, `pppc4_channels['τ']`, `pppc4_channels['μ']`, `pppc4_channels['W']` 4개 채널 interpolator 빌드
- 일반적인 `make_pppc4_interp(channel_name)`로 다른 채널도 쉽게 추가 가능

**V6: DM (m_χ, σv) contour plot**
- `plot_dm_contour_channels_batch.py`의 로직을 이식한 경량 버전
- 단일 `.dat` 파일만 있어도 동작 (14 모델 envelope 없음)
- 3채널 (bb̄, τ⁺τ⁻, μ⁺μ⁻) 동시 contour, 1σ/2σ 레벨 표시
- 공분산은 기본값 stat-only. 3개 이상 모델 있으면 자동으로 envelope systematic 추가
- J-factor와 solid angle은 Cholis+2022 40°×40° NFW γ=1.2 convention 하드코딩
- Thermal relic ⟨σv⟩=2.2×10⁻²⁶ 라인 자동 표시
- Sanghwan DMPNP2026 slide 21 결과 (bb̄ m≈60 GeV, ττ m≈11 GeV)와 자동 비교 출력

**주의**: V6는 **cascade 채널 (4b, 4τ)을 포함하지 않음**. 이는 MG5 spectra 디렉토리와 `MG5Interpolator` 의존성이 있어서 별도 포팅 필요. 14 모델 완주 + 1-10 GeV ratio 검증 후 추가.


---

## 🎯 v3.7: V4 per-model matching + V6 verbose diagnostics

**V4 수정 (Model X끼리 비교)**:
- 이전: Cholis Zenodo 디렉토리의 **첫 .dat 파일**을 로드해서 비교 (대부분 사용자 모델과 다름)
- 수정: `config.py`의 CHOLIS_REF_FILES 매핑을 따라 **같은 모델끼리** 비교
  - 사용자 Model X `.dat` → Cholis `GCE_ModelX_flux_Inner40x40_masked_disk.dat`와 비교
  - 여러 모델 있으면 각각 pairwise 비교 (색상 매칭)
- 1-10 GeV ratio 테이블 출력, 모델별 ✓/⚠/✗ 자동 판정
- 확장 변수 `cholis_ref_E`, `cholis_ref_flux`, `cholis_ref_lo`, `cholis_ref_hi` 제공

**V6 진단 강화 (χ²~200, 빈 contour 해결)**:
- 입력 데이터 표로 출력: E, flux, stat_err, rel_err — 눈으로 확인 가능
- **Stat error floor 적용** (default 5%): 관측값이 매우 큰 bin에서 rel_err이 지나치게 작으면 χ² 폭주 → `np.maximum(stat, 0.05*|flux|)`로 하한 설정
  - 이는 config.py의 `MIN_REL_ERROR_FLOOR=0.05`와 동일한 철학
- **스캔 범위 확장**: 10-500 GeV → **1-1000 GeV** (best-fit이 edge에 걸리지 않도록)
- **σv 범위 확장**: 1e-27 – 1e-24 → **1e-28 – 1e-23**
- **Grid edge 경고**: best-fit이 스캔 범위 경계에 있으면 "⚠ widen scan range!" 출력
- **Chi² grid 통계**: 각 채널의 χ²_min, χ²_max, median을 출력 → 현재 fit이 얼마나 나쁜지 눈으로 확인
- **Fit quality 자동 진단**:
  - χ²/dof < 3: ✓ acceptable DM fit
  - χ²/dof < 10: ⚠ marginal — cascade 채널 필요하거나 stat err 너무 작음
  - χ²/dof ≥ 10: ✗ 어떤 채널도 fit 안 됨 → stat err 재검토, FLOOR 올리기, flux 단위 재확인 권고

**예상 동작**:
- Model X 단일 모델 + 5% floor 적용 시, bb̄ 채널 best-fit이 m≈50-70 GeV, σv≈2×10⁻²⁶ 근처
- χ²/dof ~2-5 정도면 단일 채널 DM fit로 acceptable (GCE는 실제로 cascade/multi-component로도 잘 fit됨)
- 여전히 χ²/dof가 크다면 V6 진단 출력이 다음 단계 알려줌


---

## 🎛️ v3.8: centralized model selector (V0_SETUP)

**문제**: 특정 모델을 검증하고 싶을 때 V1, V6의 하드코딩된 `VALIDATE_MODEL = "X"`, `CONTOUR_MODEL = "X"`를 매번 수정해야 했음.

**해결**: notebook 뒷부분 `🔍 Validation cells` 섹션 맨 위에 **V0_SETUP** 셀 추가.
- `SELECTED_MODEL = "X"` 한 곳만 수정 → V1, V2, V3, V6 모두 자동 반영
- V4, V5는 모델 독립적 (V4는 multi-model, V5는 PPPC4 로더)
- `list_available_models()` 헬퍼: 현재 disk에 있는 완료된 .dat 파일 목록 (크기, 수정시각 포함)
- `SELECTED_MODEL`이 유효한 모델인지 자동 검증, 유효하지 않으면 사용 가능한 모델 제안 + 최신 모델로 fallback

**사용법**:
```
1. V0_SETUP 셀 실행 → 사용 가능한 모델 목록 출력
2. `SELECTED_MODEL = "..."` 수정 후 셀 재실행 → ✓ 확인
3. V1 → V2 → V3 → V6 순서대로 실행
4. 다른 모델로 바꾸려면 V0_SETUP으로 돌아가서 SELECTED_MODEL만 바꾸고 V1부터 다시 실행
```


---

## 🎯 v3.10: V2 fit coefficients + runner .npz output

**문제 발견** (사용자 diagnostic 결과):
- V2 GCE SED decomposition에서 π⁰+bremss와 ICS가 observed data보다 한참 아래에 그려짐
- 원인: V2가 `(pion+bremss) * E²/ΔE`만 그리고 **fit coefficient `c_pion_bremss`를 곱하지 않음** → 모든 component가 normalization=1인 raw template 상태
- Sanghwan 원본 Cell 42는 `fitted[n*0:n*1]*(pion+bremss)*E²/ΔE` 형태로 fit coeff 적용
- 이것이 Model X pairwise 비교에서 <1 GeV haebarg가 Cholis보다 낮은 것과 관련 있을 가능성 (저에너지 backgrounds over-fit → GCE under-fit)

**수정**:
1. Runner (`run_main_loop_subprocess.py`): `GCE_model_{M}_12yr_cholis_fit.npz` 추가 저장 — `fitted_params` 전체 (5*n array), template flux들 (pion, bremss, ics, GCE, bubble, isotropic) 포함
2. V2: .npz를 로드해서 각 component에 fit coefficient 적용. `c_pion_bremss`, `c_ics`, `c_gce`, `c_bubble`, `c_iso`의 per-bin 평균/범위 출력. Summed total 추가 (observed와 일치 확인). Raw template은 점선으로 참고용 표시.
3. V2 하단에 data/model ratio 표 + 자동 검증 (5%/15% 임계)

**Backward compatibility**: .npz 파일 없으면 raw template으로 plot + 경고. 이전에 완료한 Model X는 fit 결과 재생성하려면 `rm GCE_model_X_12yr_cholis.dat`로 guard 해제 후 재실행.


---

## 📊 v3.11: envelope fix + mask grid off

**문제 1 — V4 envelope이 바닥까지 늘어짐**:
80 모델 중 일부가 bin 0-1에서 near-zero flux solution으로 수렴 (MCMC가 c_gce≈0에 stuck). Min-max envelope이 이들을 포함해서 1e-8까지 떨어짐.

**근본 원인**: **Cholis 본인도 80 전체를 쓰지 않음**. Cholis+2022 Fig 15 범례에 명시적으로 "GCE 5 best models ±2σ"라고 적혀있음. 80 전체는 "other ROIs" (control regions) 시스템 체크용이고, 메인 GCE envelope은 **best 5 models** 만 사용. 이는 메모리 #10의 Cholis methodology 정확한 사양과 일치: "Covariance sample은 best-fit 5 models".

**수정 (V4 v3.11)**:
- `GCE_model_{M}_12yr_cholis_likelihood_value` 파일로 Σ log L 계산 → 상위 N개 선택 (default N=5)
- Min-max 대신 16-84 percentile (1σ) 사용
- All 80은 회색 faint band로 참고용 표시
- Best-N 각 모델을 Cholis Zenodo와 pairwise 비교 (색상 매칭)
- Outlier 모델 자동 감지 + 리스트 출력

**문제 2 — Mask plot에 grid 잔존**: matplotlib default rcParams의 grid가 imshow 뒤에도 보임.
**수정 (V3 v3.11)**: 각 imshow 뒤에 `ax.grid(False)` 명시.

**사용자 질문 — .npz 없이 재실행?**
**답**: ❌ 재실행 불필요. 이유:
- 80 모델 모두 .dat + likelihood_value 파일은 있음 → V4 best-N envelope 작동 가능
- V2 SED decomposition은 c=1 fallback 모드에서도 "유용한 근사"를 보여줌 (Cholis MapCubes는 이미 data scale로 normalized됨 → c ≈ 1이 기대값)
- 정확한 fit coefficient (c_pion_bremss, c_ics 등)가 꼭 필요하면 임의의 한 모델만 지우고 재실행 (~70분)
- Thesis 핵심 결과 (envelope vs Cholis ratio)는 .dat 만으로 완전 확보 가능


---

## 🔧 v3.12: V2/V7 errorbar yerr guard

**증상**: 일부 모델(Model I 등)에서 V2 실행 시 `ValueError: 'yerr' must not contain negative values`.

**원인**: runner가 .dat에
- col 1: `best_fit` = posterior argmax sample
- col 3: 16th percentile
- col 4: 84th percentile

을 저장하는데, MCMC posterior가 skewed되면 `best_fit`이 `[16th, 84th]` 밖으로 나올 수 있음 → `yerr = [flux - lo, hi - flux]`가 음수. 이건 **fitting bug가 아님** — posterior가 좁거나 비대칭인 bin에서 MCMC 통계상 가능한 현상.

특히 Model I (bs, SNR base) 같은 단순한 GDE 모델은 GCE residual이 커서 MCMC가 덜 안정적 → skewed posterior가 자주 발생.

**수정**: V2와 V7 모두에서 `yerr = np.maximum(..., 0)`으로 clip. 음수 yerr인 bin은 진단 출력으로 사용자에게 알림 (그 bin은 fit 수렴이 suspicious함을 의미). 플롯은 정상적으로 그려지되, 해당 bin의 error bar는 한쪽으로만 보임.


In [ ]:
from GtApp import GtApp
import matplotlib.pyplot as plt
from astropy.visualization import astropy_mpl_style, wcsaxes #wcsaxes manages coordinates of the plot.
plt.style.use(astropy_mpl_style)
from astropy.io import fits
from astropy.utils.data import get_pkg_data_filename
from astropy.wcs import WCS #This package expresses coordinates of the image file.
import numpy as np
from astropy.table import Table, hstack
import xml.etree.ElementTree as ET
from astropy.coordinates import SkyCoord
import astropy.units as u
from LATSourceModel import SourceList
from scipy.interpolate import interp1d
import warnings
import numpy as np
import emcee
from chainconsumer import ChainConsumer
import matplotlib.pyplot as plt
from multiprocessing import Pool
import time
from scipy.interpolate import CubicSpline
from scipy.integrate import dblquad
import gt_apps as gt_apps


# ============================================================================
# Skip-if-output-exists helpers (haebarg port v3.4 — with integrity check)
# ----------------------------------------------------------------------------
# v3.4 change: the existence check is no longer enough. A mid-run gtsrcmaps
# crash leaves a partial .fits file on disk that fools v3.3's needs_run().
# v3.4 also validates the file is readable and structurally complete.
# ============================================================================
import os as _os

FORCE_RECOMPUTE = False  # flip to True to force re-running every guarded cell


def _check_file_integrity(path):
    """Return (is_healthy, reason). False means the file should be regenerated.

    Validates by extension:
      - .fits  : astropy can open, primary HDU exists
                 special case: srcmap files must have NDSKEYS header keyword
      - .npy   : numpy can load
      - .xml   : ElementTree can parse
      - other  : just check file exists and is non-empty
    """
    if not _os.path.exists(path):
        return False, "missing"
    if _os.path.getsize(path) == 0:
        return False, "zero-byte file"

    lower = path.lower()
    try:
        if lower.endswith(".fits"):
            from astropy.io import fits as _fits
            with _fits.open(path) as hdul:
                # Must have at least the primary HDU
                if len(hdul) == 0:
                    return False, "empty FITS (no HDUs)"
                # gtsrcmaps outputs are recognized by filename pattern
                # ('Extended_srcmap' or '_srcmap_'). Those need NDSKEYS.
                base = _os.path.basename(path)
                if "Extended_srcmap" in base or "_srcmap_" in base:
                    if "NDSKEYS" not in hdul[0].header:
                        return False, "srcmap missing NDSKEYS keyword (gtsrcmaps crashed mid-write)"
        elif lower.endswith(".npy"):
            import numpy as _np
            _np.load(path, allow_pickle=False, mmap_mode='r')
        elif lower.endswith(".xml"):
            import xml.etree.ElementTree as _ET
            _ET.parse(path)
        # other extensions: existence + non-zero is enough
    except Exception as e:
        return False, f"unreadable ({type(e).__name__}: {e})"

    return True, "OK"


def needs_run(*output_paths):
    """True if any listed output is missing, corrupt, or FORCE_RECOMPUTE.

    Unhealthy files are deleted automatically so the producing tool will
    write fresh output without needing clobber=yes argument tweaks.
    """
    if FORCE_RECOMPUTE:
        return True
    for p in output_paths:
        ok, reason = _check_file_integrity(p)
        if not ok:
            if reason != "missing":
                print(f"[v3.4 GUARD] removing unhealthy file: {p}  ({reason})", flush=True)
                try:
                    _os.remove(p)
                except OSError as _e:
                    print(f"[v3.4 GUARD] WARNING could not remove {p}: {_e}", flush=True)
            return True
    return False


def skip_or_run(app, label=""):
    """Run a Fermi GtApp only when its `outfile` is missing or unhealthy."""
    try:
        out = app['outfile']
    except Exception:
        app.run()
        return
    name = label or getattr(app, 'appName', '') or 'GtApp'
    if needs_run(out):
        print(f"[RUN ]  {name:<10} -> {out}", flush=True)
        app.run()
        # Verify the output is actually healthy before declaring DONE
        ok, reason = _check_file_integrity(out)
        if not ok:
            raise RuntimeError(f"{name} appeared to succeed but produced unhealthy output: {reason} ({out})")
        print(f"[DONE]  {name}", flush=True)
    else:
        print(f"[SKIP]  {name:<10} -> {out} (already exists)", flush=True)


## Data preparation part

In [ ]:
#Space craft merged file can be downloaded from https://heasarc.gsfc.nasa.gov/FTP/fermi/data/lat/mission/spacecraft/
#Be advised that since SC file is large, one might experience getting a wrong file from the data base
#In such case, re-try downloading the file several times since by the time of update of the file might cause you the damaged file.
#For photon event weekly file, you can download from https://heasarc.gsfc.nasa.gov/db-perl/W3Browse/w3query.pl
#On the website, you can try "Create Download Script" to easilly download all-sky weekly files

In [ ]:
def galactic_to_equatorial(l, b):
    # Create a SkyCoord object with Galactic coordinates
    galactic_coord = SkyCoord(l=l*u.degree, b=b*u.degree, frame='galactic')

    # Convert to J2000 equatorial coordinates
    equatorial_coord = galactic_coord.icrs

    # Extract RA and Dec
    ra = equatorial_coord.ra.degree
    dec = equatorial_coord.dec.degree

    return ra, dec

In [ ]:
def equatorial_to_galactic(ra, dec):
    # Create a SkyCoord object with equatorial coordinates (J2000)
    equatorial_coord = SkyCoord(ra=ra*u.degree, dec=dec*u.degree, frame='icrs')

    # Convert to Galactic coordinates
    galactic_coord = equatorial_coord.galactic

    # Extract l and b
    l = galactic_coord.l.degree
    b = galactic_coord.b.degree

    return l, b

In [ ]:
!ls ../GCE_allsky_data/photon_files/lat_photon_weekly_* > photon_data.txt

In [ ]:
RA, DEC = galactic_to_equatorial(0, 0) #Coordinate for GC

In [ ]:
# Generate energy-bin definitions (skip if file exists)

# Define initial bins
first_bins = np.array([0.275, 0.357, 0.464, 0.603, 0.784, 1.02, 1.32, 1.72, 2.24, 2.91, 3.78, 4.91])
second_bins = np.array([4.91, 10.8, 23.7, 51.9])

# Calculate logarithmic widths
first_log_width = np.log10(first_bins[1]) - np.log10(first_bins[0])
second_log_width = np.log10(second_bins[1]) - np.log10(second_bins[0])

# Extend bins using the same log width as first_bins up to where it matches second_bins
extended_bins = list(first_bins)

# Use the wider log width to extend beyond the current range to 500 GeV
while extended_bins[-1] < 1000:
    next_bin = extended_bins[-1] * 10**second_log_width
    if next_bin > 1000:
        break
    extended_bins.append(next_bin)

# Convert to numpy array for easier manipulation
extended_bins = np.array(extended_bins)

_bindef_txt  = './GC_analysis_sanghwan/bin_definitions.txt'
_bindef_fits = './GC_analysis_sanghwan/bin_definitions.fits'

if needs_run(_bindef_fits):
    # Save the bin definitions to a text file
    with open(_bindef_txt, 'w') as f:
        for i in range(len(extended_bins) - 1):
            f.write(f"{extended_bins[i]:.3f} {extended_bins[i+1]:.3f}\n")
    print(f"Bin definitions saved to '{_bindef_txt}'.")

    bindef = GtApp('gtbindef')
    bindef['bintype']     = 'E'
    bindef['binfile']     = _bindef_txt
    bindef['outfile']     = _bindef_fits
    bindef['energyunits'] = 'GeV'
    print(f"[RUN ]  gtbindef -> {_bindef_fits}", flush=True)
    bindef.run()
    print(f"[DONE]  gtbindef", flush=True)
else:
    print(f"[SKIP]  gtbindef -> {_bindef_fits} (already exists)", flush=True)


In [ ]:
first_bins = np.array([0.275, 0.357, 0.464, 0.603, 0.784, 1.02, 1.32, 1.72, 2.24, 2.91, 3.78, 4.91])
second_bins = np.array([4.91, 10.8, 23.7, 51.9])

# Calculate logarithmic widths
first_log_width = np.log10(first_bins[1]) - np.log10(first_bins[0])
second_log_width = np.log10(second_bins[1]) - np.log10(second_bins[0])

# Extend bins using the same log width as first_bins up to where it matches second_bins
extended_bins = list(first_bins)

# Use the wider log width to extend beyond the current range to 500 GeV
while extended_bins[-1] < 1000:
    next_bin = extended_bins[-1] * 10**second_log_width
    if next_bin > 1000:
        break
    extended_bins.append(next_bin)

# Convert to numpy array for easier manipulation
extended_bins = np.array(extended_bins)

In [ ]:
# ==========================================================================
# [DISABLED v3.1] extended_bins (display only — Cell 9 already computed it)
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# extended_bins

In [ ]:
# ==========================================================================
# [DISABLED v3.1] second_log_width (display only — Cell 9 already computed it)
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# second_log_width

In [ ]:
# ==========================================================================
# [DISABLED v3.1] E_bounds (display only — variable only defined in Cell 26 → NameError on Run All)
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# E_bounds

In [ ]:
# ============================================================================
# Cell 12: data selection + GTI filtering + CCUBE creation
# ----------------------------------------------------------------------------
# Each of the three stages skips if its output already exists.
# This is the key fix for the ~13% offset (see PORT_v2_SUMMARY.md §1).
# ============================================================================

front = '_front'

# --- Stage 1: gtselect ---
_select_out = f'./GC_analysis_sanghwan/Allsky_select_12yr{front}_clean.fits'
if needs_run(_select_out):
    filters = GtApp('gtselect', 'dataSubselector')
    filters['evclass'] = 256       # CLEAN class
    filters['evtype']  = 1         # FRONT conversion only
    filters['ra']      = 'INDEF'
    filters['dec']     = 'INDEF'
    filters['rad']     = 'INDEF'
    filters['emin']    = 100       # MeV
    filters['emax']    = 1000000   # MeV (1 TeV)
    filters['zmax']    = 100       # deg, zenith cut
    # 12.5-year window (2008-08-04 to 2021-02-13). MET values match Cholis+2022.
    filters['tmin']    = 239557417   # 2008-08-04 00:00 UTC
    filters['tmax']    = 634953600   # 2021-02-13 00:00 UTC  (12.5 yr)
    filters['infile']  = 'photon_data.txt'
    filters['outfile'] = _select_out
    print(f"[RUN ]  gtselect -> {_select_out}", flush=True)
    filters.run()
    print(f"[DONE]  gtselect", flush=True)
else:
    print(f"[SKIP]  gtselect -> {_select_out} (already exists)", flush=True)

# --- Stage 2: gtmktime ---
_gti_out = f'./GC_analysis_sanghwan/Allsky_gti_12yr{front}_clean.fits'
if needs_run(_gti_out):
    space_craft = '../GCE_allsky_data/lat_spacecraft_merged_12yr.fits'
    maketime = GtApp('gtmktime', 'dataSubselector')
    maketime['scfile']  = space_craft
    maketime['filter']  = 'DATA_QUAL==1 && LAT_CONFIG==1 && ABS(ROCK_ANGLE) < 52'
    maketime['roicut']  = 'no'
    maketime['evfile']  = _select_out
    maketime['outfile'] = _gti_out
    print(f"[RUN ]  gtmktime -> {_gti_out}", flush=True)
    maketime.run()
    print(f"[DONE]  gtmktime", flush=True)
else:
    print(f"[SKIP]  gtmktime -> {_gti_out} (already exists)", flush=True)

# --- Stage 3: gtbin CCUBE 600x600 (REQUIRED for [100:500] slicing in main loop) ---
_ccube_out = f'./GC_analysis_sanghwan/GC_ccube_12yr{front}_clean.fits'
if needs_run(_ccube_out):
    counts_map = GtApp('gtbin', 'evtbin')
    counts_map['algorithm'] = 'CCUBE'
    counts_map['evfile']    = _gti_out
    counts_map['outfile']   = _ccube_out
    counts_map['nxpix']     = 600
    counts_map['nypix']     = 600
    counts_map['binsz']     = 0.1
    counts_map['coordsys']  = 'GAL'
    counts_map['xref']      = 0
    counts_map['yref']      = 0
    counts_map['axisrot']   = 0
    counts_map['proj']      = 'CAR'
    counts_map['ebinalg']   = 'FILE'
    counts_map['ebinfile']  = './GC_analysis_sanghwan/bin_definitions.fits'
    print(f"[RUN ]  gtbin CCUBE -> {_ccube_out}", flush=True)
    counts_map.run()
    print(f"[DONE]  gtbin CCUBE", flush=True)
else:
    print(f"[SKIP]  gtbin CCUBE -> {_ccube_out} (already exists)", flush=True)


In [ ]:
# ============================================================================
# [v3.3] Extend FileFunction spectra to cover full CCUBE energy range
# ----------------------------------------------------------------------------
# Sanghwan's CCUBE energy binning (Cell 8) reaches 556 GeV (= 556077 MeV).
# gtsrcmaps samples FileFunction values at the bin edges, so it would
# request flux at 556077 MeV. If the original isotropic_spectrum_ff.txt or
# fermi_bubble_spectrum.txt only define values up to ~344 GeV, gtsrcmaps
# crashes with:
#
#   Caught St11range_error at the top level:
#   Requested energy, 556077, lies outside the range of the input file,
#   120, 344430
#
# Fix: extend each spectrum file to 1.5 TeV (with margin) by power-law
# extrapolation in log-log space. Idempotent — does nothing if already
# extended.
# ============================================================================
import subprocess as _sp, sys as _sys, os as _os_ext

_extend_script = './extend_spectrum_files.py'
if not _os_ext.path.exists(_extend_script):
    print(f"  ⚠ {_extend_script} not found in working directory.")
    print(f"     Copy it from the v3.3 release before re-running this cell.")
else:
    print("  Running spectrum extension (idempotent — safe to re-run):")
    _r = _sp.run([_sys.executable, _extend_script],
                 capture_output=True, text=True)
    print(_r.stdout, end="")
    if _r.returncode != 0:
        print(_r.stderr, end="")
        raise RuntimeError("extend_spectrum_files.py failed (see above)")


In [ ]:
# ============================================================================
# Cell 13: disabled — the original notebook here re-ran gtbin with
# nxpix=400, nypix=400, overwriting the 600×600 CCUBE from Cell 12.
# But the main loop (Cell 34) uses `data[100:500, 100:500]` slicing which
# requires a 600×600 CCUBE. We therefore SKIP this cell.
# (See REF_sanghwan_pipeline_port_SUMMARY.md §2.2, item E1.)
# ============================================================================
pass


In [ ]:
#CCW binning makes each bin to contain comparable events to expect similar statistics at high energy
#-> Modification of this binning should be considered

In [ ]:
ltCube = GtApp('gtltcube', 'Likelihood')
ltCube['evfile']=f'./GC_analysis_sanghwan/Allsky_gti_12yr{front}{"_clean"}.fits'
ltCube['scfile']='../GCE_allsky_data/lat_spacecraft_merged_12yr.fits'
ltCube['outfile']=f'./GC_analysis_sanghwan/Allsky_ltcube_12yr{front}{"_clean"}.fits'
ltCube['dcostheta']=0.025
ltCube['binsz']=1
ltCube['zmax']=100
skip_or_run(ltCube)

In [ ]:
gtexpcube2 = GtApp('gtexpcube2', 'Likelihood')
gtexpcube2['infile']=f'./GC_analysis_sanghwan/Allsky_ltcube_12yr{front}{"_clean"}.fits'
gtexpcube2['cmap']='none'
gtexpcube2['outfile']=f'./GC_analysis_sanghwan/GC_expcube_center_12yr{front}{"_clean"}.fits'
gtexpcube2['evtype']=1
gtexpcube2['coordsys']='GAL'
gtexpcube2['xref']=0
gtexpcube2['yref']=0
gtexpcube2['nxpix']=600
gtexpcube2['nypix']=600
gtexpcube2['proj']='CAR'
gtexpcube2['binsz']=0.1
gtexpcube2['bincalc']='CENTER'
gtexpcube2['irfs']='P8R3_CLEAN_V3'
gtexpcube2['ebinalg']='FILE'
gtexpcube2['ebinfile']='./GC_analysis_sanghwan/bin_definitions.fits'
skip_or_run(gtexpcube2)

In [ ]:
gtexpcube2 = GtApp('gtexpcube2', 'Likelihood')
gtexpcube2['infile']=f'./GC_analysis_sanghwan/Allsky_ltcube_12yr{front}{"_clean"}.fits'
gtexpcube2['cmap']='none'
gtexpcube2['outfile']=f'./GC_analysis_sanghwan/Allsky_expcube_edge_12yr{front}{"_clean"}.fits'
gtexpcube2['evtype']=1
gtexpcube2['coordsys']='GAL'
gtexpcube2['xref']=0
gtexpcube2['yref']=0
gtexpcube2['nxpix']=3600
gtexpcube2['nypix']=1800
gtexpcube2['proj']='CAR'
gtexpcube2['binsz']=0.1
gtexpcube2['bincalc']='EDGE'
gtexpcube2['irfs']='P8R3_CLEAN_V3'
gtexpcube2['ebinalg']='FILE'
gtexpcube2['ebinfile']='./GC_analysis_sanghwan/bin_definitions.fits'
skip_or_run(gtexpcube2)

In [ ]:
_dr2_xml = './GC_analysis_sanghwan/Model/GC_model_DR2.xml'
if needs_run(_dr2_xml):
    print(f"[RUN ]  SourceList -> {_dr2_xml}", flush=True)
    source_list = SourceList(DR=2, catalog_file='../GCE_12yr_data/gll_psc_v23.xml', ROI=[266, -29, 35], output_name='GC_model_DR2.xml', write_directory='./GC_analysis_sanghwan/Model/')
    source_list.make_model(extended_catalog_names=True, norms_free_only=True, galactic_index_free=True, extra_radius=5, free_radius=28.28, max_free_radius=28.28, variable_free=True, sigma_to_free=49, galactic_name='gll_iem', galactic_file='../GCE_12yr_data/gll_iem_v07.fits', isotropic_file='../GCE_12yr_data/iso_P8R3_SOURCE_V3_v1.txt', isotropic_name='isotropic', extended_directory='../GCE_12yr_data/Templates/')
    print(f"[DONE]  SourceList -> {_dr2_xml}", flush=True)
else:
    print(f"[SKIP]  SourceList -> {_dr2_xml} (already exists)", flush=True)


In [ ]:

# Define the names of the sources you want to remove
_psc_xml = './GC_analysis_sanghwan/Model/GC_psc_model_DR2.xml'
if needs_run(_psc_xml):
    # Parse the existing XML file
    tree = ET.parse('./GC_analysis_sanghwan/Model/GC_model_DR2.xml')
    root = tree.getroot()

    # Find and remove the elements
    for source in root.findall(".//source"):
        if 'isotropic' in source.get('name', ''):
            root.remove(source)
        if 'gll_iem' in source.get('name', ''):
            root.remove(source)
    # Save the modified XML to a new file
    tree.write(_psc_xml, encoding='utf-8', xml_declaration=True)
    print(f"[DONE]  XML cleanup -> {_psc_xml}", flush=True)
else:
    print(f"[SKIP]  XML cleanup -> {_psc_xml} (already exists)", flush=True)


In [ ]:

# Define the names of the sources you want to remove

# Parse the existing XML file

ra_dec_values = []
spatial_ra_dec_values = []
tree = ET.parse('./GC_analysis_sanghwan/Model/GC_psc_model_DR2.xml')
root = tree.getroot()

# Find and remove the elements
for source in root.findall(".//source"):
    source_name = source.attrib.get('name', '')
    source_type = source.attrib.get('type', '')
    if source_type=='PointSource':
        #for param in source.findall(".//spectrum/parameter"):
        ra = float(source.find(".//spatialModel/parameter[@name='RA']").attrib['value'])
        dec = float(source.find(".//spatialModel/parameter[@name='DEC']").attrib['value'])
        l, b = equatorial_to_galactic(ra, dec)
        if True:#is_within_roi(l, b):
            #name = source.attrib['name']
            ra_dec_values.append([source_name, ra, dec])
        else:
            print(source_name, l, b)
            
    if source_type == "DiffuseSource":
        spatial_model = source.find(".//spatialModel")
        if spatial_model is not None:# and source.attrib['type']=='SpatialMap':
            if spatial_model.attrib.get('type') == 'SpatialMap':
                file_path = spatial_model.attrib.get('file')
                if file_path:
                    ra = fits.open(file_path)[0].header['CRVAL1']
                    dec = fits.open(file_path)[0].header['CRVAL2']
                    l, b = equatorial_to_galactic(ra, dec)
                    if True:#is_within_roi(l, b):
                    #spatial_ra_dec_values.append([source_name, ra, dec])
                    #ra_dec_values.append([source_name, ra, dec])
                        spatial_ra_dec_values.append([source_name, ra, dec])
            if spatial_model.attrib.get('type') != 'SpatialMap':
                ra = float(source.find(".//spatialModel/parameter[@name='RA']").attrib['value'])
                dec = float(source.find(".//spatialModel/parameter[@name='DEC']").attrib['value'])
                l, b = equatorial_to_galactic(ra, dec)
                if True:#is_within_roi(l, b):
                #spatial_ra_dec_values.append([source_name, ra, dec])
                #ra_dec_values.append([source_name, ra, dec])
                    spatial_ra_dec_values.append([source_name, ra, dec])

In [ ]:
# ==========================================================================
# [DISABLED v3.1] len(ra_dec_values) + len(spatial_ra_dec_values) (display only)
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# len(ra_dec_values) + len(spatial_ra_dec_values)

In [ ]:
#not_sig_ra_dec_values=ra_dec_values.copy()
not_sig_ra_dec_values = ra_dec_values + spatial_ra_dec_values
sig_ra_dec_values=[]
data=fits.open('../GCE_12yr_data/gll_psc_v23.fit')[1].data
for row in data:
    for lists in not_sig_ra_dec_values:
        if row['Source_Name'] == lists[0]:
            if row['Signif_Avg'] > 49:
                sig_ra_dec_values.append(lists)
                not_sig_ra_dec_values.remove(lists)
                print(row['Source_Name'], row['Signif_Avg'], lists)
                #sig_ra_dec_values.pop(index)


In [ ]:
# ==========================================================================
# [DISABLED v3.1] len(sig_ra_dec_values) (display only)
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# len(sig_ra_dec_values)

In [ ]:
E_bounds=fits.open('./GC_analysis_sanghwan/GC_ccube_12yr_front_clean.fits')[1].data
E=np.zeros(len(E_bounds))
for i in range(len(E_bounds)):
    E[i] = 1e-3*np.sqrt(E_bounds[i][2]*E_bounds[i][1]*1e-6)#Into GeV

In [ ]:
# ==========================================================================
# [DISABLED v3.1] len(E) (display only)
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# len(E)

In [ ]:
## Corrected code
def masking(significance, locations, energy, image_file): #Input GeV, location ra dec, radius theta
    warnings.filterwarnings("ignore", category=RuntimeWarning)
    warnings.filterwarnings("ignore", message="'datfix' made the change")

    data = [
    [0.275, 0.357, 1.125, 3.75],
    [0.357, 0.464, 0.975, 3.25],
    [0.464, 0.603, 0.788, 2.63],
    [0.603, 0.784, 0.600, 2.00],
    [0.784, 1.02, 0.450, 1.50],
    [1.02, 1.32, 0.375, 1.25],
    [1.32, 1.72, 0.300, 1.00],
    [1.72, 2.24, 0.225, 0.750],
    [2.24, 2.91, 0.188, 0.625],
    [2.91, 3.78, 0.162, 0.540],
    [3.78, 4.91, 0.125, 0.417],
    [4.91, 10.8, 0.100, 0.333],
    [10.8, 23.7, 0.060, 0.200],
    [23.7, 51.9, 0.053, 0.175]] #Mask definition is taken from 2112.09706v2, Table III
    data=np.asarray(data)
    
    mask_energy=np.sqrt(data[0:, 0]*data[0:, 1])
    mask_small=data[0:, 2]
    mask_large=data[0:, 3]
    

    mask_small_int=interp1d(mask_energy, mask_small, fill_value='extrapolate')
    mask_large_int=interp1d(mask_energy, mask_large, fill_value='extrapolate')
    
    
    mask_small_size=(mask_small_int(energy))
    mask_large_size=(mask_large_int(energy))

    mask_small_size[mask_small_size<=0] = 0
    mask_large_size[mask_large_size<=0] = 0

    mask_small_size = mask_small_size
    mask_large_size = mask_large_size
    
    print(energy, mask_small_size, mask_large_size)

    fits_file = image_file
    hdulist = fits.open(fits_file)
    data = hdulist[0].data[0]  # Assuming the image data is in the first extension
    header = hdulist[0].header

    masked_data = np.ones(np.shape(data))
    # Get WCS information from the header
    wcs = WCS(header).dropaxis(2)
    
    for points in locations:
        #print(points[0])
        l, b = equatorial_to_galactic(float(points[1]), float(points[2]))
        #print(l, b)
        # Define your point of interest in Galactic coordinates
        # Convert Galactic coordinates to Equatorial coordinates (RA, Dec)
        galactic_coords = SkyCoord(l=l*u.degree, b=b*u.degree, frame='galactic')
        #equatorial_coords = galactic_coords.icrs

        
        # Convert Equatorial coordinates to pixel coordinates
        #pixel_coords = wcs.world_to_pixel(equatorial_coords)
        ####
        pixel_coords = wcs.world_to_pixel(galactic_coords)
        ###
        # Extract pixel coordinates
        x_center = np.round(pixel_coords[0], 0)
        y_center = np.round(pixel_coords[1], 0)

        #print(f"Pixel Coordinates: x = {x_center}, y = {y_center}")

        # Calculate the pixel scale (degree per pixel)
        #if wcs.wcs.has_cd():
        #    cd = wcs.wcs.cd
        #    pixel_scale_x = np.sqrt(cd[0, 0]**2 + cd[1, 0]**2)
        #    pixel_scale_y = np.sqrt(cd[0, 1]**2 + cd[1, 1]**2)
        #else:
        #    pixel_scale_x = wcs.wcs.cdelt[0]
        #    pixel_scale_y = wcs.wcs.cdelt[1]
        pixel_scale_x = wcs.wcs.cdelt[0]
        pixel_scale_y = wcs.wcs.cdelt[1]

        # Calculate the radius in pixels corresponding to 1 degree
        if significance==1:
            radius_deg = mask_large_size
        if significance==0:
            radius_deg = mask_small_size
        
        radius_pix_x = radius_deg / pixel_scale_x
        radius_pix_y = radius_deg / pixel_scale_y
        radius_pix = min(radius_pix_x, radius_pix_y)
        if abs(radius_pix_x) != abs(radius_pix_y):
            print(radius_pix_x, radius_pix_y)
        
        #print(f"Radius in pixels: {radius_pix}, Radius in degree : {radius_deg}")

        # Create a circular mask
        y, x = np.ogrid[:data.shape[0], :data.shape[1]]
        mask = (x - x_center)**2 + (y - y_center)**2 < radius_pix**2
        
        #y, x = np.ogrid[:data.shape[0], :data.shape[1]]
        #distance_from_center = np.sqrt((x - x_center)**2 + (y - y_center)**2)
        #mask = distance_from_center <= radius_pix
        
        # Apply the mask to the image data
        masked_data[mask] = 0  # Set the masked region to NaN (or any other value that represents masking)

        #plt.imshow(masked_data, cmap='jet')
    
    return masked_data


In [ ]:
# Heavy mask computation (944 sources × 14 energy bins × per-source pixel calc).
# If the resulting full_mask .npy already exists, just load it instead.
_mask_npy = './GC_analysis_sanghwan/Model/GC_mask_60x60_definitions_DR2.npy'

if needs_run(_mask_npy):
    file_name = './GC_analysis_sanghwan/GC_ccube_12yr_front_clean.fits'
    raw_data_map = fits.open(file_name)[0].data
    mask_big = np.zeros(np.shape(raw_data_map))
    mask_small = np.zeros(np.shape(raw_data_map))
    mask_big_spatial = np.zeros(np.shape(raw_data_map))
    mask_small_spatial = np.zeros(np.shape(raw_data_map))

    for i in range(0, len(mask_big), 1):
        mask_big[i] = masking(1, sig_ra_dec_values, E[i], file_name)
        mask_small[i] = masking(0, not_sig_ra_dec_values, E[i], file_name)
        #mask_big_spatial[i] = masking(0, spatial_ra_dec_values, E[i], file_name)
        #mask_small_spatial[i] = masking(0, not_sig_spatial_ra_dec_values, E[i]*1e-3, file_name)

    full_mask = mask_big * mask_small
    print(f"[DONE]  mask compute (will be saved by Cell 31)", flush=True)
else:
    print(f"[SKIP]  mask compute -> loading from {_mask_npy}", flush=True)
    full_mask = np.load(_mask_npy)


In [ ]:
_disk_npy = './GC_analysis_sanghwan/Model/GC_disk_mask_60x60_definitions.npy'

if needs_run(_disk_npy):
    src = fits.open('./GC_analysis_sanghwan/GC_ccube_12yr_front_clean.fits')

    # [v3.2 fix] define w here — Sanghwan's original notebook relied on w
    # existing from an earlier interactive run of Cell 35. For top-to-bottom
    # execution we must define it locally. (dropaxis(2) because CCUBE is 3D:
    # the third axis is energy, and WCS should describe only the spatial 2D part.)
    w = WCS(src[0].header).dropaxis(2)

    x_dim = np.shape(src[0].data[0])[0]
    y_dim = np.shape(src[0].data[0])[1]
    b_list = np.zeros(x_dim)

    for i in range(0, x_dim, 1):
        if np.abs(w.wcs_pix2world(0, i, 0)[1]) <= 2:  # w.pixel_to_world takes index convention as fortran(start from 1)
            b_list[i] = i

    b_max = int(np.max(b_list))
    b_min = int(np.min(b_list[b_list != 0]))  # Specifying index interval for b<2 condition.
    masked_map = np.ones(np.shape(src[0].data[0]))
    masked_map[b_min:b_max+1, :] = 0
    np.save(_disk_npy, masked_map)
    print(f"[DONE]  disk_mask -> {_disk_npy}", flush=True)
else:
    print(f"[SKIP]  disk_mask -> {_disk_npy} (already exists)", flush=True)


In [ ]:
_mask_npy = './GC_analysis_sanghwan/Model/GC_mask_60x60_definitions_DR2.npy'
if needs_run(_mask_npy):
    np.save(_mask_npy, full_mask)
    print(f"[DONE]  full_mask -> {_mask_npy}", flush=True)
else:
    print(f"[SKIP]  full_mask -> {_mask_npy} (already exists)", flush=True)


In [ ]:
# Define the names of the sources you want to remove
_empty_xml = './GC_analysis_sanghwan/Model/empty_model.xml'

if needs_run(_empty_xml):
    # Parse the existing XML file
    tree = ET.parse('./GC_analysis_sanghwan/Model/GC_model_DR2.xml')
    root = tree.getroot()

    # Find and remove the elements
    for source in root.findall(".//source"):
        if 'quark' not in source.get('name', ''):
            root.remove(source)

    # Save the modified XML to a new file
    tree.write(_empty_xml, encoding='utf-8', xml_declaration=True)
    print(f"[DONE]  empty_model.xml -> {_empty_xml}", flush=True)
else:
    print(f"[SKIP]  empty_model.xml -> {_empty_xml} (already exists)", flush=True)


In [ ]:
def generate_roman_numerals(n):
    """
    Generate a list of Roman numerals from 1 to n.
    """
    roman_map = [
        (1000, 'M'), (900, 'CM'), (500, 'D'), (400, 'CD'),
        (100, 'C'), (90, 'XC'), (50, 'L'), (40, 'XL'),
        (10, 'X'), (9, 'IX'), (5, 'V'), (4, 'IV'), (1, 'I')
    ]

    def to_roman(num):
        roman = ""
        for value, symbol in roman_map:
            while num >= value:
                roman += symbol
                num -= value
        return roman

    return [to_roman(i) for i in range(1, n + 1)]

In [ ]:
# ============================================================================
# [v3.3 fix] Extend FileFunction spectrum files to 1 TeV
# ----------------------------------------------------------------------------
# gtsrcmaps with `emapbnds=yes` and IRF energy dispersion needs the
# FileFunction spectra (Fermi_bubble, isotropic) to span a wider energy range
# than the analysis bins themselves — typically up to ~1 TeV.
#
# Sanghwan's environment had pre-extended spectrum files. The default LAT
# distribution ones (`isotropic_spectrum_ff.txt`, `fermi_bubble_spectrum.txt`)
# may end at ~344 GeV, causing the runtime error:
#     Caught St11range_error at the top level: Requested energy, 556077,
#     lies outside the range of the input file, 120, 344430
#
# This cell:
#   1) Reads each spectrum file (2 columns: E [MeV], dN/dE [ph/cm²/s/MeV])
#   2) If the upper energy is below 1 TeV, fits a power-law on the last
#      ~10 points and extrapolates to 1.1 TeV
#   3) Backs up the original (.original) and writes the extended file in-place
#   4) Skips entirely if extension already done (last energy >= 1 TeV)
# ============================================================================
import shutil

def _extend_spectrum(path, target_emax_mev=1.1e6, n_extrap_points=10):
    """Power-law extrapolate a 2-col FileFunction spectrum to target_emax_mev."""
    import numpy as np

    if not _os.path.exists(path):
        print(f"  [WARN]  spectrum file missing: {path}")
        return False

    data = np.loadtxt(path)
    if data.ndim != 2 or data.shape[1] < 2:
        print(f"  [WARN]  unexpected format in {path}: shape {data.shape}")
        return False

    E   = data[:, 0]   # MeV
    dnde = data[:, 1]  # ph/cm²/s/MeV (or whatever; preserved)

    if E[-1] >= target_emax_mev * 0.99:
        print(f"  [SKIP]  {path} already extends to {E[-1]:.2e} MeV (>= {target_emax_mev:.2e})")
        return True

    # Fit power law to last n_extrap_points: log(dnde) = a + alpha * log(E)
    n = min(n_extrap_points, len(E))
    logE_tail   = np.log(E[-n:])
    logF_tail   = np.log(np.maximum(dnde[-n:], 1e-300))
    alpha, a = np.polyfit(logE_tail, logF_tail, 1)
    print(f"  [INFO]  {path}: extrapolating with PL index alpha={alpha:.3f} on last {n} points")

    # Generate ~30 new points logarithmically from current Emax to target
    n_new = 30
    new_E = np.logspace(np.log10(E[-1]), np.log10(target_emax_mev), n_new + 1)[1:]
    new_F = np.exp(a) * np.power(new_E, alpha)

    # Backup original
    backup = path + ".original"
    if not _os.path.exists(backup):
        shutil.copy(path, backup)
        print(f"  [INFO]  backed up original -> {backup}")

    # Write extended file
    extended = np.vstack([
        np.column_stack([E, dnde]),
        np.column_stack([new_E, new_F]),
    ])
    np.savetxt(path, extended, fmt="%.6e %.6e")
    print(f"  [DONE]  {path}: now spans {extended[0,0]:.2e} to {extended[-1,0]:.2e} MeV "
          f"({len(extended)} points)")
    return True

print("=== Extending FileFunction spectra to 1 TeV (if needed) ===")
for _spec_path in ["./fermi_bubble_spectrum.txt", "./isotropic_spectrum_ff.txt"]:
    _extend_spectrum(_spec_path, target_emax_mev=1.1e6)
print()


In [ ]:
model_list =['X', 'XLIX', 'I', 'IV', 'V', 'VI', 'VII', 'IX', 'XV', 'XLI', 'XLVII', 'XLVIII', 'L', 'LII']#['IV', 'V', 'VI', 'VII', 'IX', 'X', 'XV', 'XLI', 'XLVII', 'XLVIII', 'XLIX', 'L', 'LII']

In [ ]:
# ============================================================================
# Cell 35 [v3.3]: Main loop is now run as a SUBPROCESS
# ----------------------------------------------------------------------------
# Why subprocess?
#   The main loop (gtsrcmaps × 2  +  gtmodel × 12  +  emcee × 14 bins, all
#   per model × 14 models) is the heaviest part of the pipeline. It can take
#   16-20 hours total and a single gtsrcmaps call peaks at ~5-12 GB RAM.
#   When run inside the Jupyter kernel, any crash (OOM, fork-related Fermi
#   tools issue, VSCode kernel timeout) kills the entire notebook.
#
#   Running it as a separate process means:
#     - Notebook kernel stays alive even if the main loop crashes
#     - Output streams to a log file you can `tail -f`
#     - Progress is durable: re-running picks up from completed models
#       (per-model and per-step skip guards work the same as in-cell)
#
# Three usage modes:
#
#   1) RUN HERE (synchronous, output streams to notebook output):
#      Useful for a single model test (e.g. Model X first to validate).
#      Note: kernel still has to wait, but it just relays text output.
#
#   2) BACKGROUND (recommended for full 14-model run):
#      Launch from a shell:
#         nohup python run_main_loop_subprocess.py all > main_loop.log 2>&1 &
#      Then this cell can just verify it's running and tail the log.
#
#   3) SKIP (if you've already finished):
#      The model-level guard inside run_main_loop_subprocess.py will skip
#      every model whose .dat already exists, so re-running is cheap.
#
# Choose mode by editing MODE below:
# ============================================================================

import subprocess, sys, os, time

# ----- USER CHOICE -----
MODE = "single"        # one of: "single", "all", "background", "monitor"
SINGLE_MODEL = "X"     # only used when MODE == "single"
# -----------------------

_runner = "./run_main_loop_subprocess.py"
if not os.path.exists(_runner):
    raise FileNotFoundError(
        f"{_runner} not found. Copy it from the v3.3 release into the "
        f"working directory before running this cell."
    )

if MODE == "single":
    print(f"=== Running model {SINGLE_MODEL!r} as a subprocess ===")
    print(f"    log will stream below; this cell waits until done.")
    print(f"    (If the kernel dies, the subprocess keeps running — check\n"
          f"     ./GCE_model_{SINGLE_MODEL}_12yr_cholis.dat afterwards.)\n")
    p = subprocess.Popen(
        [sys.executable, _runner, SINGLE_MODEL],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        bufsize=1, text=True,
    )
    try:
        for line in p.stdout:
            print(line, end="")
        p.wait()
    except KeyboardInterrupt:
        print("\n[user interrupted — sending SIGTERM to subprocess]")
        p.terminate()
        p.wait()
    print(f"\n=== Subprocess exited with code {p.returncode} ===")

elif MODE == "all":
    print("=== Running ALL 14 models as a foreground subprocess ===")
    print("    This will take ~16-20 hours. Consider using MODE='background' instead.")
    print()
    p = subprocess.Popen(
        [sys.executable, _runner, "all"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        bufsize=1, text=True,
    )
    try:
        for line in p.stdout:
            print(line, end="")
        p.wait()
    except KeyboardInterrupt:
        print("\n[user interrupted — sending SIGTERM to subprocess]")
        p.terminate()
        p.wait()

elif MODE == "background":
    log_path = "./main_loop.log"
    pid_path = "./main_loop.pid"
    if os.path.exists(pid_path):
        with open(pid_path) as f:
            old_pid = f.read().strip()
        # Check if process is still alive
        alive = False
        try:
            os.kill(int(old_pid), 0)
            alive = True
        except (OSError, ValueError):
            pass
        if alive:
            print(f"  ⚠ Background process already running with PID {old_pid}")
            print(f"     - log:  tail -f {log_path}")
            print(f"     - kill: kill {old_pid}")
            print(f"     Skipping launch.")
        else:
            print(f"  Stale PID file ({old_pid}). Launching new background process.")
            os.remove(pid_path)

    if not os.path.exists(pid_path):
        # Launch in background, fully detached so closing the notebook
        # doesn't kill the subprocess.
        log = open(log_path, "ab")
        p = subprocess.Popen(
            [sys.executable, _runner, "all"],
            stdout=log, stderr=subprocess.STDOUT,
            start_new_session=True,
        )
        with open(pid_path, "w") as f:
            f.write(str(p.pid))
        print(f"  Launched in background.")
        print(f"     PID:  {p.pid}  (saved to {pid_path})")
        print(f"     log:  tail -f {log_path}")
        print(f"     stop: kill {p.pid}")
        print(f"\n  This cell returns immediately; the work continues.")

elif MODE == "monitor":
    log_path = "./main_loop.log"
    pid_path = "./main_loop.pid"
    print(f"=== Monitor: status of background main-loop run ===\n")
    if os.path.exists(pid_path):
        with open(pid_path) as f:
            pid = f.read().strip()
        try:
            os.kill(int(pid), 0)
            print(f"  ✓ Process PID {pid} is alive.")
        except (OSError, ValueError):
            print(f"  ✗ Process PID {pid} no longer exists (finished or crashed).")
    else:
        print(f"  No PID file. Either no background run is active, or it was launched manually.")

    # Count completed models
    import glob
    done_dats = sorted(glob.glob("./GCE_model_*_12yr_cholis.dat"))
    print(f"\n  Completed .dat files ({len(done_dats)}):")
    for d in done_dats:
        size = os.path.getsize(d)
        mtime = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(os.path.getmtime(d)))
        print(f"    {d}  ({size} B, modified {mtime})")

    if os.path.exists(log_path):
        print(f"\n  Last 20 lines of {log_path}:")
        with open(log_path) as f:
            tail = f.readlines()[-20:]
        for line in tail:
            print(f"    {line.rstrip()}")
    else:
        print(f"\n  No log file at {log_path}.")

else:
    raise ValueError(f"Unknown MODE: {MODE!r}. Choose one of "
                     "'single', 'all', 'background', 'monitor'.")


In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# np.loadtxt('./GCE_model_II_12yr_cholis_likelihood_value')

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# lhd_value = []
# for model_name in ['X', 'XV', 'XLVIII', 'XLIX' ,'LIII', 'II', 'LXIV', 'LXIX', 'LXX','LXXI']:
#     lhd_value.append([ np.sum(np.loadtxt(f'./GCE_model_{model_name}_12yr_cholis_likelihood_value')[0:17]), model_name])
#
# #lhd_value.append([np.loadtxt(f'./GCE_model_{"I"}_12yr_cholis_likelihood_value'), 'I'])
#

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# lhd_value

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# (-1875071.61082686  -  -1875036.53364852)/-1875071.61082686

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# (-350296.9117517157 -  -350618.8757585063)/-350296.9117517157

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# (-1875036.53364852 - -1876718.39873492)/-1876718.39873492

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# np.loadtxt(f'./GCE_model_{model_name}_12yr_cholis_likelihood_value')

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# import matplotlib.pyplot as plt
# from astropy.visualization import astropy_mpl_style, wcsaxes #wcsaxes manages coordinates of the plot.
# plt.style.use(astropy_mpl_style)
# from astropy.io import fits
# from astropy.utils.data import get_pkg_data_filename
# from astropy.wcs import WCS #This package expresses coordinates of the image file.
# import numpy as np
# from matplotlib.ticker import LogLocator, LogFormatter, AutoMinorLocator
# plt.style.use('default')
# ax=plt.subplot()
# n=len(E)
# ax.set_xscale('log')
# ax.set_yscale('log')
# ax.set_xlabel('E [GeV]')
# ax.set_ylabel(r'$E^2 \frac{dN}{dE}$[GeV$cm^{-2}$$s^{-1} sr^{-1}$]')
#
# ax.set_ylim(1e-8, 1e-4)
# ax.set_xlim(0.3, 500)
#
# ax.tick_params(axis='y', which='both', direction='in', left=True)
# ax.tick_params(axis='x', which='both', direction='in', bottom=True)
# ax.minorticks_on()
# ax.grid(True, which='Major', linestyle='-', linewidth=0.5)
#
# fitted=(fitted_params_median)
# fitted_errors=fitted_params_std
# sr=1#0.4288213187542626#0.214411*2
# #sr=0.4387
# #ax.errorbar(E, counts_per_exp*(E**2)/(delta_E*sr) , yerr=counts_per_exp_err*(E**2)/(delta_E*sr), linestyle='dotted', marker='.', elinewidth=2, capsize=4, capthick=2, label='Raw_data')
#
#
# #ax.errorbar(E, psc*(E**2)/(delta_E*sr), linestyle='', marker='.', elinewidth=2, capsize=4, capthick=2, label='psc', color='orange')
#
#
# #ax.errorbar(E, counts_per_exp*(E**2)/(delta_E*sr) , yerr=counts_per_exp_err*(E**2)/(delta_E*sr), linestyle='dotted', marker='.', elinewidth=2, capsize=4, capthick=2, label='Raw_data')
# ax.errorbar(E, counts_per_exp*(E**2)/(delta_E) , yerr=np.zeros(len(E)), linestyle='dotted', marker='.', elinewidth=2, capsize=4, capthick=2, label='Raw_data')
#
#
#
#
# ax.errorbar(E, fitted[n*0:n*1]*(pion+bremss)*(E**2)/(delta_E), yerr=fitted_errors[n*0:n*1]*(pion+bremss)*(E**2)/(delta_E), linestyle='dotted', marker='.', elinewidth=2, capsize=4, capthick=2, label='pion+bremss', color='red')
# ax.errorbar(E, fitted[n*1:n*2]*(ics)*(E**2)/(delta_E), yerr=fitted_errors[n*1:n*2]*(ics)*(E**2)/(delta_E), alpha=1, linestyle='dashdot', marker='.', elinewidth=2, capsize=4, capthick=2, label='ics', color='blue')
#
# ax.plot(E, (pion+bremss)*(E**2)/(delta_E), linestyle='solid', label='pion+bremss', color='red')
# ax.plot(E, (ics)*(E**2)/(delta_E), linestyle='solid', label='ics', color='blue')
#
#
#
# ax.errorbar(E,  fitted[n*2:n*3]*(GCE)*(E**2)/(delta_E), yerr=(fitted_errors[n*2:n*3]*GCE)*(E**2)/(delta_E), alpha=1, linestyle='dashed', marker='.', elinewidth=2, capsize=4, capthick=2, label='GCE', color='black')
# ax.errorbar(E,  fitted_params_median[n*2:n*3]*(GCE)*(E**2)/(delta_E), yerr=(fitted_errors[n*2:n*3]*GCE)*(E**2)/(delta_E), alpha=1, linestyle='dashed', marker='.', elinewidth=2, capsize=4, capthick=2, label='GCE', color='black')
#
#
#
# ax.errorbar(E, fitted[n*3:n*4]*(bubble)*(E**2)/(delta_E),yerr=fitted_errors[n*3:n*4]*(bubble)*(E**2)/(delta_E), linestyle='dashed', marker='.', elinewidth=2, capsize=4, capthick=2, label='bubble', color='purple')
#
# ax.errorbar(E, fitted[n*4:n*5]*(isotropic)*(E**2)/(delta_E),yerr=fitted_errors[n*4:n*5]*(isotropic)*(E**2)/(delta_E), linestyle='dashed', marker='.', elinewidth=2, capsize=4, capthick=2, label='isotropic', color='green')
# summed = fitted[n*0:n*1]*(pion+bremss) + fitted[n*1:n*2]*(ics) + fitted[n*2:n*3]*(GCE) + fitted[n*3:n*4]*(bubble) + fitted[n*4:n*5]*(isotropic)
#
# ax.plot(E, (E**2)*summed/(delta_E), label='summed')
#
#
# plt.show()
#
# #ax.errorbar(E, bubble_flux_data, [bubble_lower_error_data, bubble_upper_error_data])
# #ax.plot(E, bubble_flux_data)
# #ax.plot(E, iso_constraints_flux) 
# ax.legend()
# #plt.savefig('./GC_analysis_sanghwan/Model_I.png')

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# (fitted_params_upper[n*2:n*3] - fitted[n*2:n*3])*(GCE)*(E**2)/(delta_E)

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# ((fitted_params_upper[n*2:n*3] - fitted[n*2:n*3])*(GCE)*(E**2)/(delta_E) +  (fitted[n*2:n*3] - fitted_params_lower[n*2:n*3])*(GCE)*(E**2)/(delta_E))/2

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# fitted[n*2:n*3]*(GCE)*(E**2)/(delta_E)

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# fitted[n*2:n*3]*(GCE)*(E**2)/(delta_E)

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# (fitted_errors[n*2:n*3]*GCE)*(E**2)/(delta_E)

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# np.savetxt('./GCE_model_I_12yr_cholis.dat', np.vstack([E, fitted[n*2:n*3]*(GCE)*(E**2)/(delta_E), (fitted_errors[n*2:n*3]*GCE)*(E**2)/(delta_E), (fitted_params_lower[n*2:n*3])*(GCE)*(E**2)/(delta_E), (fitted_params_upper[n*2:n*3])*(GCE)*(E**2)/(delta_E)]).T)
#
#

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# np.savetxt('./GCE_model_I_test_test_test_test.dat', np.vstack([E, fitted[n*2:n*3]*(GCE)*(E**2)/(delta_E), (fitted_errors[n*2:n*3]*GCE)*(E**2)/(delta_E)]).T)

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# np.loadtxt('./GCE_model_I_test_test_test.dat')

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# E_bounds

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# import matplotlib.pyplot as plt
# from astropy.visualization import astropy_mpl_style, wcsaxes #wcsaxes manages coordinates of the plot.
# plt.style.use(astropy_mpl_style)
# from astropy.io import fits
# from astropy.utils.data import get_pkg_data_filename
# from astropy.wcs import WCS #This package expresses coordinates of the image file.
# import numpy as np
# from matplotlib.ticker import LogLocator, LogFormatter, AutoMinorLocator
# plt.style.use('default')
# ax=plt.subplot()
# n=14
# ax.set_xscale('log')
# ax.set_yscale('log')
# ax.set_xlabel('E [GeV]')
# ax.set_ylabel(r'$E^2 \frac{dN}{dE}$[GeV$cm^{-2}$$s^{-1} sr^{-1}$]')
#
# ax.set_ylim(1e-8, 1e-4)
# ax.set_xlim(0.3, 500)
#
# ax.tick_params(axis='y', which='both', direction='in', left=True)
# ax.tick_params(axis='x', which='both', direction='in', bottom=True)
# ax.minorticks_on()
# ax.grid(True, which='Major', linestyle='-', linewidth=0.5)
#
# fitted=np.ones(n*5)#(fitted_params)
# fitted_errors=np.zeros(n*5)#fitted_params_std
# sr=0.4288213187542626#0.214411*2
# #sr=0.4387
# #ax.errorbar(E, counts_per_exp*(E**2)/(delta_E*sr) , yerr=counts_per_exp_err*(E**2)/(delta_E*sr), linestyle='dotted', marker='.', elinewidth=2, capsize=4, capthick=2, label='Raw_data')
#
#
# #ax.errorbar(E, psc*(E**2)/(delta_E*sr), linestyle='', marker='.', elinewidth=2, capsize=4, capthick=2, label='psc', color='orange')
#
#
# #ax.errorbar(E, counts_per_exp*(E**2)/(delta_E*sr) , yerr=counts_per_exp_err*(E**2)/(delta_E*sr), linestyle='dotted', marker='.', elinewidth=2, capsize=4, capthick=2, label='Raw_data')
# ax.errorbar(E, counts_per_exp*(E**2)/(delta_E*sr) , yerr=np.zeros(14), linestyle='dotted', marker='.', elinewidth=2, capsize=4, capthick=2, label='Raw_data')
#
#
#
#
# ax.errorbar(E, fitted[n*0:n*1]*(pion+bremss)*(E**2)/(delta_E*sr), yerr=fitted_errors[n*0:n*1]*(pion+bremss)*(E**2)/(delta_E*sr), linestyle='dotted', marker='.', elinewidth=2, capsize=4, capthick=2, label='pion+bremss', color='red')
# ax.errorbar(E, fitted[n*1:n*2]*(ics)*(E**2)/(delta_E*sr), yerr=fitted_errors[n*1:n*2]*(ics)*(E**2)/(delta_E*sr), linestyle='dashdot', marker='.', elinewidth=2, capsize=4, capthick=2, label='ics', color='blue')
#
# ax.plot(E, (pion+bremss)*(E**2)/(delta_E*sr), linestyle='solid', label='pion+bremss', color='red')
# ax.plot(E, (ics)*(E**2)/(delta_E*sr), linestyle='solid', label='ics', color='blue')
#
#
#
# ax.errorbar(E,  fitted[n*2:n*3]*(GCE)*(E**2)/(delta_E*sr), yerr=np.sqrt((fitted_errors[n*2:n*3]*GCE)**2)*(E**2)/(delta_E*sr), alpha=1, linestyle='dashed', marker='.', elinewidth=2, capsize=4, capthick=2, label='GCE', color='black')
#
#
#
# ax.errorbar(E, fitted[n*3:n*4]*(bubble)*(E**2)/(delta_E*sr),yerr=fitted_errors[n*3:n*4]*(bubble)*(E**2)/(delta_E*sr), linestyle='dashed', marker='.', elinewidth=2, capsize=4, capthick=2, label='bubble', color='purple')
#
# ax.errorbar(E, fitted[n*4:n*5]*(isotropic)*(E**2)/(delta_E*sr),yerr=fitted_errors[n*4:n*5]*(isotropic)*(E**2)/(delta_E*sr), linestyle='dashed', marker='.', elinewidth=2, capsize=4, capthick=2, label='isotropic', color='green')
# summed = fitted[n*0:n*1]*(pion+bremss) + fitted[n*1:n*2]*(ics) + fitted[n*2:n*3]*(GCE) + fitted[n*3:n*4]*(bubble) + fitted[n*4:n*5]*(isotropic)
#
# ax.plot(E, (E**2)*summed/(delta_E*sr), label='summed')
#
#
# plt.show()
#
# #ax.errorbar(E, bubble_flux_data, [bubble_lower_error_data, bubble_upper_error_data])
# #ax.plot(E, bubble_flux_data)
# #ax.plot(E, iso_constraints_flux) 
# ax.legend()
# #plt.savefig('./GC_analysis_sanghwan/Model_I.png')

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# fitted[n*2:n*3]*(GCE)*(E**2)/(delta_E)

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# import matplotlib.pyplot as plt
# from astropy.visualization import astropy_mpl_style, wcsaxes #wcsaxes manages coordinates of the plot.
# plt.style.use(astropy_mpl_style)
# from astropy.io import fits
# from astropy.utils.data import get_pkg_data_filename
# from astropy.wcs import WCS #This package expresses coordinates of the image file.
# import numpy as np
# from matplotlib.ticker import LogLocator, LogFormatter, AutoMinorLocator
# plt.style.use('default')
# ax=plt.subplot()
#
# ax.set_xscale('log')
# ax.set_yscale('log')
# ax.set_xlabel('E [GeV]')
# ax.set_ylabel(r'$E^2 \frac{dN}{dE}$[GeV$cm^{-2}$$s^{-1} sr^{-1}$]')
#
# ax.set_ylim(1e-8, 1e-4)
# ax.set_xlim(0.3, 500)
#
# ax.tick_params(axis='y', which='both', direction='in', left=True)
# ax.tick_params(axis='x', which='both', direction='in', bottom=True)
# ax.minorticks_on()
# ax.grid(True, which='Major', linestyle='-', linewidth=0.5)
#
# fitted=(fitted_params)
# fitted_errors=fitted_params_std
# sr=1#0.214411*2
#
# #ax.errorbar(E, counts_per_exp*(E**2)/(delta_E*sr) , yerr=counts_per_exp_err*(E**2)/(delta_E*sr), linestyle='dotted', marker='.', elinewidth=2, capsize=4, capthick=2, label='Raw_data')
#
#
# #ax.errorbar(E, fitted[0:17]*(pion+bremss)*(E**2)/(delta_E*sr), yerr=fitted_errors[0:17]*(pion+bremss)*(E**2)/(delta_E*sr), linestyle='dotted', marker='.', elinewidth=2, capsize=4, capthick=2, label='pion+bremss', color='red')
# #ax.errorbar(E, fitted[17:34]*(ics)*(E**2)/(delta_E*sr), yerr=fitted_errors[17:34]*(ics)*(E**2)/(delta_E*sr), linestyle='dashdot', marker='.', elinewidth=2, capsize=4, capthick=2, label='ics', color='blue')
#
# #ax.plot(E, (pion+bremss)*(E**2)/(delta_E*sr), linestyle='solid', label='pion+bremss', color='red')
# #ax.plot(E, (ics)*(E**2)/(delta_E*sr), linestyle='solid', label='ics', color='blue')
#
#
#
# ax.errorbar(E,  fitted[2*n:3*n]*(GCE)*(E**2)/(delta_E*sr), yerr=np.sqrt((fitted_errors[2*n:3*n]*GCE)**2)*(E**2)/(delta_E*sr), alpha=0.5, linestyle='dashed', marker='.', elinewidth=2, capsize=4, capthick=2, label='GCE', color='purple')
#
# ax.errorbar(E,  fitted_params_median[2*n:3*n]*(GCE)*(E**2)/(delta_E*sr), yerr=np.sqrt((fitted_errors[2*n:3*n]*GCE)**2)*(E**2)/(delta_E*sr), alpha=0.5, linestyle='dashed', marker='.', elinewidth=2, capsize=4, capthick=2, label='GCE', color='purple')
#
#
# #ax.errorbar(E, fitted[51:68]*(bubble)*(E**2)/(delta_E*sr),yerr=fitted_errors[51:68]*(bubble)*(E**2)/(delta_E*sr), linestyle='dashed', marker='.', elinewidth=2, capsize=4, capthick=2, label='bubble', color='green')
#
# #ax.errorbar(E, fitted[68:85]*(isotropic)*(E**2)/(delta_E*sr),yerr=fitted_errors[68:85]*(isotropic)*(E**2)/(delta_E*sr), linestyle='dashed', marker='.', elinewidth=2, capsize=4, capthick=2, label='isotropic', color='skyblue')
#
# #summed = fitted[0:17]*(pion+bremss) + fitted[17:34]*ics + fitted[34:51]*GCE + fitted[51:68]*bubble + fitted[68:85]*isotropic
#
# #ax.errorbar(E, bubble_flux_data, [bubble_lower_error_data, bubble_upper_error_data])
# #ax.errorbar(E, isotropic_flux_data, [isotropic_lower_error_data, isotropic_upper_error_data])
# #ax.plot(E, bubble_flux_data)
# #ax.plot(E, iso_constraints_flux) 
# ax.legend()
# #plt.savefig('./GC_analysis_sanghwan/Model_I.png')

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# np.savetxt('./GC_analysis_sanghwan/GCE_model_X_12yr.dat', np.vstack([E, fitted[n*2:n*3]*(GCE)*(E**2)/(delta_E*sr), np.sqrt((fitted_errors[n*2:n*3]*GCE)**2)*(E**2)/(delta_E*sr)]).T)

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# a=np.loadtxt('./GC_analysis_sanghwan/GCE_model_I.dat')
# plt.loglog(a[0:, 0], a[0:, 1])

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# # Log-transform the x (Energy) and y (E^2dN/dE) values
# #clean_data = input data
# log_energy = np.log10(clean_data['Energy_GeV'])
# log_E2dN_dE = np.log10(clean_data['E2dN_dE_GeV_cm2s1sr1'])
#
# # Define the interpolation function in log-log space
# interp_func_log = interp1d(log_energy, log_E2dN_dE, kind='slinear', fill_value="extrapolate")
#
# # Generate new log-scale Energy values for extrapolation (log-scale)
# log_energy_extended = np.linspace(log_energy.min(), np.log10(370), 500)
#
#
# # Interpolate/extrapolate in log-log space
# log_E2dN_dE_extended = interp_func_log(log_energy_extended)
#
# # Convert back to linear scale
# energy_extended = 10**log_energy_extended
# E2dN_dE_extended = 10**log_E2dN_dE_extended
#
# # Plot the original data points and the extrapolated curve (log-log)
# plt.figure(figsize=(8, 6))
#
# # Plot in log-log scale
# plt.plot(energy_extended, E2dN_dE_extended, label='Extrapolated E^2dN/dE (Log-Log Scale)', color='red', linestyle='--')
# plt.errorbar(clean_data['Energy_GeV'], clean_data['E2dN_dE_GeV_cm2s1sr1'], 
#              yerr=[clean_data['E2dN_dE_GeV_cm2s1sr1'] - clean_data['sigma_lower'], 
#                    clean_data['sigma_upper'] - clean_data['E2dN_dE_GeV_cm2s1sr1']], 
#              fmt='o', label='Original Data', capsize=3)
#
# # Set the scale to logarithmic for both axes
# plt.xscale('log')
# plt.yscale('log')
#
# # Set labels and title
# plt.xlabel('Energy (GeV)')
# plt.ylabel(r'$E^2 \cdot dN/dE$ (GeV cm$^{-2}$ s$^{-1}$ sr$^{-1}$)')
# plt.title('Extrapolated Data using Log-Log Scale')
# plt.grid(True, which="both", ls="—")
# plt.legend()
#
# # Show the plot
# plt.show()

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# """
# model='A'
# component='psc'
# gtmodel = GtApp('gtmodel', 'Likelihood')
# gtmodel['irfs']='P8R3_CLEAN_V3'
# gtmodel['outtype']='ccube'
# gtmodel['srcmdl']=f'./GC_analysis_sanghwan/Model/GC_{component}_model.xml'
# gtmodel['outfile']=f'./GC_analysis_sanghwan/Model/GC_all_time_60x60_{component}_model.fits'
# gtmodel['expcube']='./GC_analysis_sanghwan/Allsky_all_time_ltcube.fits'
# gtmodel['bexpmap']='./GC_analysis_sanghwan/Allsky_all_time_60x60_expcube.fits'
# gtmodel['srcmaps']=f'./GC_analysis_sanghwan/GC_all_time_60x60_srcmap_model{model}_merged.fits'
# gtmodel.run()
# """

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# # Revision :: Aug 11, 2024
#
#
# model='A'
# E_bounds=fits.open('./GC_analysis_sanghwan/GC_all_time_60x60_ccube.fits')[1].data
#
# sr=0.214411*2
# E=np.zeros(len(E_bounds))
# for i in range(0, len(E_bounds), 1):
#     E[i] = np.sqrt(E_bounds[i][2]*E_bounds[i][1]*1e-6)*1e-3
#
# delta_E=np.zeros(len(E_bounds))
# for i in range(0, len(E_bounds), 1):
#     delta_E[i] = (E_bounds[i][2] - E_bounds[i][1])*1e-6
#
# exp_cube = np.zeros([len(E), 400, 400])
# #exp_cube=np.sqrt(fits.open('./GC_analysis_sanghwan/GC_all_time_60x60_expcube.fits')[0].data[:-1]*fits.open('./GC_analysis_sanghwan/GC_all_time_60x60_expcube.fits')[0].data[1:])
# for i in range(0, len(exp_cube), 1):
#     exp_cube[i] = (np.sqrt(fits.open('./GC_analysis_sanghwan/GC_all_time_60x60_expcube.fits')[0].data[i, 100:500, 100:500]*fits.open('./GC_analysis_sanghwan/GC_all_time_60x60_expcube.fits')[0].data[i+1, 100:500, 100:500]))   
#
#
#
# file_name='./GC_analysis_sanghwan/Model/GC_all_time_60x60_psc_model.fits'
# psc=np.zeros(len(fits.open(file_name)[0].data))
# for i in range(0, len(fits.open(file_name)[0].data), 1):
#     a=exp_cube[i]
#     psc[i] = np.sum((fits.open(file_name)[0].data[i][100:500, 100:500]/a))
#
# file_name=f'./GC_analysis_sanghwan/Model/GC_all_time_60x60_pion_model{model}.fits'
# pion=np.zeros(len(fits.open(file_name)[0].data))
# for i in range(0, len(fits.open(file_name)[0].data), 1):
#     a=exp_cube[i]
#     pion[i] = np.sum( (fits.open(file_name)[0].data[i][100:500, 100:500]/a) )
#
# file_name=f'./GC_analysis_sanghwan/Model/GC_all_time_60x60_bremss_model{model}.fits'
# bremss=np.zeros(len(fits.open(file_name)[0].data))
# for i in range(0, len(fits.open(file_name)[0].data), 1):
#     a=exp_cube[i]
#     bremss[i] = np.sum( (fits.open(file_name)[0].data[i][100:500, 100:500]/a) )
#
# file_name=f'./GC_analysis_sanghwan/Model/GC_all_time_60x60_ics_model{model}.fits'
# ics=np.zeros(len(fits.open(file_name)[0].data))
# for i in range(0, len(fits.open(file_name)[0].data), 1):
#     a=exp_cube[i]
#     ics[i] = np.sum( (fits.open(file_name)[0].data[i][100:500, 100:500]/a) )
#
# file_name=f'./GC_analysis_sanghwan/Model/GC_all_time_60x60_GCE_model.fits'
# GCE=np.zeros(len(fits.open(file_name)[0].data))
# for i in range(0, len(fits.open(file_name)[0].data), 1):
#     a=exp_cube[i]
#     GCE[i] = np.sum( (fits.open(file_name)[0].data[i][100:500, 100:500]/a) )
#
# file_name=f'./GC_analysis_sanghwan/Model/GC_all_time_60x60_fermi_bubble_model.fits'
# bubble=np.zeros(len(fits.open(file_name)[0].data))
# for i in range(0, len(fits.open(file_name)[0].data), 1):
#     a=exp_cube[i]
#     bubble[i] = np.sum( (fits.open(file_name)[0].data[i][100:500, 100:500]/a) )
#
# file_name=f'./GC_analysis_sanghwan/Model/GC_all_time_60x60_isotropic_model.fits'
# isotropic=np.zeros(len(fits.open(file_name)[0].data))
# for i in range(0, len(fits.open(file_name)[0].data), 1):
#     a=exp_cube[i]
#     isotropic[i] = np.sum( (fits.open(file_name)[0].data[i][100:500, 100:500]/a) )
#
# counts_per_exp=np.zeros(len(E))
# i=0
# for i in range(0, len(E), 1):
#     a=exp_cube[i]
#     counts_per_exp[i]=np.sum( ( (fits.open('./GC_analysis_sanghwan/GC_all_time_60x60_ccube.fits')[0].data[i][100:500, 100:500]) /a) )
#
# counts_per_exp_err=np.zeros(len(E))
# i=0
# for i in range(0, len(E), 1):
#     a=exp_cube[i]
#     counts_per_exp_err[i]=np.sqrt( np.sum( ( (fits.open('./GC_analysis_sanghwan/GC_all_time_60x60_ccube.fits')[0].data[i][100:500, 100:500]) /a)**2) )
#
#
#

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# plt.ylim(1e-8, 1e-4)
# plt.xlim(0.3, 500)
# plt.style.use(astropy_mpl_style)
# plt.loglog()
# plt.plot(E, (pion+bremss)*(E**2)/(delta_E*sr), linestyle='dotted', label='pion+bremss', color='red')
# plt.plot(E, (ics)*(E**2)/(delta_E*sr), linestyle='dashdot', label='ics', color='blue')
# #plt.plot(E, (pion+bremss)*(E**2)/(delta_E*sr), 'ro', label='pion+bremss', color='red')
# #plt.plot(E, (ics)*(E**2)/(delta_E*sr), 'bo', label='ics', color='blue')
# plt.legend()

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# import matplotlib.pyplot as plt
# from astropy.visualization import astropy_mpl_style, wcsaxes #wcsaxes manages coordinates of the plot.
# plt.style.use(astropy_mpl_style)
# from astropy.io import fits
# from astropy.utils.data import get_pkg_data_filename
# from astropy.wcs import WCS #This package expresses coordinates of the image file.
# import numpy as np
# from scipy.interpolate import interp1d
# from iminuit import Minuit
# #Constraints interpolated function
# #Contains constraints for bubble and isotropic as well
# #For isotropic, from https://arxiv.org/pdf/1410.3696.pdf Table 3
#
# #Testing exposure per pixel
#
# #Analysis for model F
# #Correcting bubble template given from https://arxiv.org/pdf/1407.7905, Table 2
#
#
# constraints=np.loadtxt('./GC_analysis_sanghwan/Model/bubble_constraints.txt')
# constraints_energy=constraints[0:, 0]
# constraints_flux=constraints[0:, 1]#/constraints[0:, 0]**2
# constraints_lower_error=constraints[0:, 2]#/constraints[0:, 0]**2
# constraints_upper_error=constraints[0:, 3]#/constraints[0:, 0]**2
#
# fluxint=interp1d(constraints_energy, constraints_flux, fill_value="extrapolate", kind='quadratic')
# upper_errint=interp1d(constraints_energy, constraints_upper_error, fill_value="extrapolate", kind='quadratic') 
# lower_errint=interp1d(constraints_energy, constraints_lower_error, fill_value="extrapolate", kind='quadratic') 
#
#
# bubble_flux_data=fluxint(E)
# bubble_lower_error_data=lower_errint(E)
# bubble_upper_error_data=upper_errint(E)
#
#
# iso_constraints=np.loadtxt('./GC_analysis_sanghwan/Model/egb_constraints_full_err.txt')
# iso_constraints_energy=iso_constraints[0:, 0]
# iso_constraints_flux=iso_constraints[0:, 1]
# iso_constraints_low_err=iso_constraints[0:, 2]
# iso_constraints_upp_err=iso_constraints[0:, 3]
#
#
# fluxint=interp1d(iso_constraints_energy, iso_constraints_flux, fill_value="extrapolate")#, kind='quadratic')
# low_errint=interp1d(iso_constraints_energy, iso_constraints_low_err, fill_value="extrapolate")#, kind='quadratic')    
# upp_errint=interp1d(iso_constraints_energy, iso_constraints_upp_err, fill_value="extrapolate")#, kind='quadratic')    
#
# iso_constraints_flux=((E)**2)*fluxint(E)
# iso_constraints_flux_low_err=((E)**2)*low_errint(E)
# iso_constraints_flux_upp_err=((E)**2)*upp_errint(E)
#
#
#
# class Likelihood:
#     def __init__(self, model, energy_bin):
#         self.model=model
#         self.energy_bin=energy_bin
#     def likelihood_constrained(self, parameter_set):
#         data=fits.open('./GC_analysis_sanghwan/GC_all_time_60x60_ccube.fits')
#         psc_map=fits.open('./GC_analysis_sanghwan/Model/GC_all_time_60x60_psc_model.fits')
#         pion=fits.open(f'./GC_analysis_sanghwan/Model/GC_all_time_60x60_pion_model{self.model}.fits')
#         ics=fits.open(f'./GC_analysis_sanghwan/Model/GC_all_time_60x60_ics_model{self.model}.fits')
#         bremss=fits.open(f'./GC_analysis_sanghwan/Model/GC_all_time_60x60_bremss_model{self.model}.fits')
#         GCE=fits.open(f'./GC_analysis_sanghwan/Model/GC_all_time_60x60_GCE_model.fits')
#         bubble=fits.open(f'./GC_analysis_sanghwan/Model/GC_all_time_60x60_fermi_bubble_model.fits')
#         isotropic=fits.open(f'./GC_analysis_sanghwan/Model/GC_all_time_60x60_isotropic_model.fits')
#         #####################################
#         pion_bremss_param=parameter_set[0]
#         ics_param=parameter_set[1]
#         GCE_param=parameter_set[2]
#         bubble_param=parameter_set[3]
#         isotropic_param=parameter_set[4]
#
#         ######################################
#         #Main body for calculating Poissonian log-likelihood
#         #for k in range(0, energy_length, 1):
#         observed_pixel=data[0].data[self.energy_bin]
#         observed_pixel=observed_pixel.astype('float32')
#         psc_pixel=psc_map[0].data[self.energy_bin]
#         pion_bremss_pixel=pion[0].data[self.energy_bin] + bremss[0].data[self.energy_bin]
#         ics_pixel=ics[0].data[self.energy_bin]
#         GCE_pixel=GCE[0].data[self.energy_bin]
#         iso_pixel=isotropic[0].data[self.energy_bin]
#         bubble_pixel=bubble[0].data[self.energy_bin]
#         #expected_pixel= pion_bremss_param*pion_bremss_pixel + ics_param*ics_pixel + psc_pixel + GCE_param*GCE_pixel + isotropic_param*iso_pixel + bubble_param*bubble_pixel
#         expected_pixel= (1+pion_bremss_param)*pion_bremss_pixel + (1+ics_param)*ics_pixel + (1+GCE_param)*GCE_pixel + (1+isotropic_param)*iso_pixel + (1+bubble_param)*bubble_pixel
#
#         if ( (expected_pixel) < 0.0 ).any():
#             return np.inf
#
#         psc_mask=np.load('./GC_analysis_sanghwan/Model/GC_mask_60x60_definitions.npy')[self.energy_bin]
#         disk_mask=np.load('./GC_analysis_sanghwan/Model/GC_disk_mask_60x60_definitions.npy')
#
#         full_mask=(psc_mask*disk_mask)[100:500, 100:500]
#
#         indices=np.where(full_mask==0.0)
#
#         observed_pixel = (observed_pixel)[100:500, 100:500]
#         expected_pixel = (expected_pixel)[100:500, 100:500]
#
#         E_bounds=fits.open('./GC_analysis_sanghwan/GC_all_time_60x60_ccube.fits')[1].data
#
#         E=np.zeros(len(E_bounds))
#         for i in range(0, len(E_bounds), 1):
#             E[i] = np.sqrt(E_bounds[i][2]*E_bounds[i][1]*1e-6)*1e-3
#
#         delta_E=np.zeros(len(E_bounds))
#         for i in range(0, len(E_bounds), 1):
#             delta_E[i] = (E_bounds[i][2] - E_bounds[i][1])*1e-6
#
#         exp_cube = np.zeros([len(E), 400, 400])
#         for i in range(0, len(exp_cube), 1):
#             exp_cube[i] = (np.sqrt(fits.open('./GC_analysis_sanghwan/GC_all_time_60x60_expcube.fits')[0].data[i, 100:500, 100:500]*fits.open('./GC_analysis_sanghwan/GC_all_time_60x60_expcube.fits')[0].data[i+1, 100:500, 100:500]))   
#
#         observed_pixel = np.delete(observed_pixel, indices)
#         expected_pixel = np.delete(expected_pixel, indices)
#
#         #observed_pixel[observed_pixel == 0.0] += 1e-20
#         expected_pixel[expected_pixel == 0.0] += 1e-20
#         lhd=2*( expected_pixel - observed_pixel*np.log(expected_pixel) )# + term )#np.log(factorial(observed_pixel)))
#
#         #zeros=1e-20*np.ones(np.shape(expected_pixel))
#
#         #base_value=2*( zeros - observed_pixel*np.log(zeros) )
#
#         E_bounds=fits.open('./GC_analysis_sanghwan/GC_all_time_60x60_ccube.fits')[1].data
#
#         sr=0.214411*2
#
#         exp_cube = np.zeros([len(E), 600, 600])
#         for i in range(0, len(exp_cube), 1):
#             exp_cube[i] = (np.sqrt(fits.open('./GC_analysis_sanghwan/GC_all_time_60x60_expcube.fits')[0].data[i]*fits.open('./GC_analysis_sanghwan/GC_all_time_60x60_expcube.fits')[0].data[i+1]))   
#
#
#
#         file_name='./GC_analysis_sanghwan/Model/GC_all_time_60x60_isotropic_model.fits'
#         isotropic=np.zeros(len(fits.open(file_name)[0].data))
#         for i in range(0, len(fits.open(file_name)[0].data), 1):
#             a=exp_cube[i]
#             isotropic[i] = np.sum( (fits.open(file_name)[0].data[i]/a) )#*isotropic_param
#
#         isotropic[self.energy_bin] *= (1+isotropic_param)
#
#         file_name='./GC_analysis_sanghwan/Model/GC_all_time_60x60_fermi_bubble_model.fits'
#         bubble=np.zeros(len(fits.open(file_name)[0].data))
#         for i in range(0, len(fits.open(file_name)[0].data), 1):
#             a=exp_cube[i]
#             bubble[i] = np.sum( fits.open(file_name)[0].data[i]/a )#*bubble_param
#
#         bubble[self.energy_bin] *= (1+bubble_param)
#
#         sr_60=1.0471975511965974#0.9773843811168244
#
#
#         bubble_sed=(E**2)*bubble/(delta_E*sr_60)
#
#
#
#         chi2=0
#         i=self.energy_bin
#         larger_error=max([bubble_upper_error_data[i], bubble_lower_error_data[i]])
#         if bubble_flux_data[i] < bubble_sed[i]:
#             chi2 = ((bubble_sed[i] - bubble_flux_data[i])/bubble_upper_error_data[i])**2
#         if bubble_flux_data[i] > bubble_sed[i]:
#             chi2 = ((bubble_sed[i] - bubble_flux_data[i])/bubble_lower_error_data[i])**2
#         if bubble_flux_data[i] == bubble_sed[i]:
#             chi2 = ((bubble_sed[i] - bubble_flux_data[i])/larger_error)**2
#
#         iso_flux=(E**2)*isotropic/(sr_60*delta_E)
#
#         chi2_iso=0
#         i=self.energy_bin
#         iso_larger_error=max([iso_constraints_flux_low_err[i], iso_constraints_flux_upp_err[i]])
#         if iso_flux[i] > iso_constraints_flux[i]:
#             chi2_iso = ((iso_constraints_flux[i] - iso_flux[i])/iso_constraints_flux_upp_err[i])**2
#         if iso_flux[i] < iso_constraints_flux[i]:
#             chi2_iso = ((iso_constraints_flux[i] - iso_flux[i])/iso_constraints_flux_low_err[i])**2
#         if iso_flux[i] == iso_constraints_flux[i]:
#             chi2_iso = ((iso_constraints_flux[i] - iso_flux[i])/iso_larger_error[i])**2
#
#         #return (np.sum(lhd) - np.sum(base_value)) + chi2 + chi2_iso
#         return np.sum(lhd) + chi2 + chi2_iso
#
#     def minuit_module(self):
#         tol=1e-4
#         tol1=tol
#         #m=Minuit(self.likelihood_constrained, 0*np.ones(5))
#         #m.errordef=0.5
#         while True:
#             if True:
#                 m=Minuit(self.likelihood_constrained, 0.0*np.ones(5))#np.random.rand(5))
#                 m.errordef=0.5
#                 m.limits[:]=(-5, 5)
#                 m.limits[2]=(-5, 5)
#                 m.tol=tol1
#                 print(f'tol1 = {m.tol}')
#                 m.strategy=0
#                 m.simplex()
#                 m.strategy=1
#                 m.simplex()
#                 m.strategy=2
#                 m.simplex()
#                 #m.migrad()#.migrad()
#                 if m.valid==True:
#                     print(m)
#                     break;
#                 else:
#                     tol1*=1e+1
#         return 1+np.array(m.values[:]), 1+np.array(m.errors[:])
# class Parameter_estimation():
#     def __init__(self, model):
#         self.model=model
#     def minuit_run(self):
#         pion_bremss_fitted=list(np.ones(17))
#         ics_fitted=list(np.ones(17))
#         GCE_fitted=list(np.ones(17))
#         bubble_fitted=list(np.ones(17))
#         isotropic_fitted=list(np.ones(17))
#
#         pion_bremss_fitted_errors=list(np.zeros(17))
#         ics_fitted_errors=list(np.zeros(17))
#         GCE_fitted_errors=list(np.zeros(17))
#         bubble_fitted_errors=list(np.zeros(17))
#         isotropic_fitted_errors=list(np.zeros(17))   
#
#         for i in range(0, 17, 1):
#         #for i in range(0, 14, 1):
#             results=Likelihood(self.model, i).minuit_module()
#             #results=np.ones([5, 5])
#             pion_bremss_fitted[i] = (results[0][0])
#             ics_fitted[i] = (results[0][1])
#             GCE_fitted[i] = (results[0][2])
#             bubble_fitted[i] = (results[0][3])
#             isotropic_fitted[i] = (results[0][4])
#
#             pion_bremss_fitted_errors[i] = (results[1][0])
#             ics_fitted_errors[i] = (results[1][1])
#             GCE_fitted_errors[i] = (results[1][2])
#             bubble_fitted_errors[i] = (results[1][3])
#             isotropic_fitted_errors[i] = (results[1][4])
#
#             fitted=np.hstack([pion_bremss_fitted, ics_fitted, GCE_fitted, bubble_fitted, isotropic_fitted])
#             fitted_errors=np.zeros(17*5)
#             # [Disabled in haebarg port — this file was Sanghwan-private]
#             # data = np.loadtxt('/home/sanghwan/C.Weniger/covariance.dat')
#             data = None  # placeholder; ensure this branch is not hit
#             ebins = data[:,0:2]  # Energy bins [GeV]
#             emeans = ebins.prod(axis=1)**0.5  # Geometric mean energy [GeV]
#             de = ebins[:,1] - ebins[:,0]  # Energy bin width [GeV]
#             flux = data[:,2]  # (Average flux)*E^2 in energy bin [GeV/cm2/s/sr]
#             flux_err = data[:,3]  # Flux error [GeV/cm2/s/sr]
#
#
#             plt.figure()
#             print(i)
#             ax=plt.subplot()
#
#             ax.set_xscale('log')
#             ax.set_yscale('log')
#             ax.set_xlabel('E [GeV]')
#             ax.set_ylabel(r'$E^2 \frac{dN}{dE}$[GeV$cm^{-2}$$s^{-1} sr^{-1}$]')
#
#             ax.set_ylim(1e-8, 1e-4)
#             ax.set_xlim(0.3, 500)
#
#             ax.tick_params(axis='y', which='both', direction='in', left=True)
#             ax.tick_params(axis='x', which='both', direction='in', bottom=True)
#             ax.minorticks_on()
#             ax.grid(True, which='Major', linestyle='-', linewidth=0.5)
#
#             sr=0.214411*2
#             ax.errorbar(emeans, flux, flux_err)
#             ax.errorbar(E, counts_per_exp*(E**2)/(delta_E*sr) , yerr=counts_per_exp_err*(E**2)/(delta_E*sr), linestyle='dotted', marker='.', elinewidth=2, capsize=4, capthick=2, label='Raw_data')
#
#
#             ax.errorbar(E, psc*(E**2)/(delta_E*sr), linestyle='', marker='.', elinewidth=2, capsize=4, capthick=2, label='psc', color='orange')
#
#
#             ax.errorbar(E, fitted[0:17]*(pion+bremss)*(E**2)/(delta_E*sr), yerr=fitted_errors[0:17]*(pion+bremss)*(E**2)/(delta_E*sr), linestyle='dotted', marker='.', elinewidth=2, capsize=4, capthick=2, label='pion+bremss', color='red')
#             ax.errorbar(E, fitted[17:34]*(ics)*(E**2)/(delta_E*sr), yerr=fitted_errors[17:34]*(ics)*(E**2)/(delta_E*sr), linestyle='dashdot', marker='.', elinewidth=2, capsize=4, capthick=2, label='ics', color='blue')
#
#             ax.plot(E, (pion+bremss)*(E**2)/(delta_E*sr), linestyle='dotted', label='pion+bremss', color='red')
#             ax.plot(E, (ics)*(E**2)/(delta_E*sr), linestyle='dashdot', label='ics', color='blue')
#
#
#
#             ax.errorbar(E,  fitted[34:51]*(GCE)*(E**2)/(delta_E*sr), yerr=np.sqrt((fitted_errors[34:51]*GCE)**2)*(E**2)/(delta_E*sr), linestyle='dashed', marker='.', elinewidth=2, capsize=4, capthick=2, label='GCE', color='black')
#
#
#
#             ax.errorbar(E, fitted[51:68]*(bubble)*(E**2)/(delta_E*sr),yerr=fitted_errors[51:68]*(bubble)*(E**2)/(delta_E*sr), linestyle='dashed', marker='.', elinewidth=2, capsize=4, capthick=2, label='bubble', color='purple')
#
#             ax.errorbar(E, fitted[68:85]*(isotropic)*(E**2)/(delta_E*sr),yerr=fitted_errors[68:85]*(isotropic)*(E**2)/(delta_E*sr), linestyle='dashed', marker='.', elinewidth=2, capsize=4, capthick=2, label='isotropic', color='green')
#
#             plt.show()
#
#
#
#
#         return np.hstack([pion_bremss_fitted, ics_fitted, GCE_fitted, bubble_fitted, isotropic_fitted]), np.hstack([pion_bremss_fitted_errors, ics_fitted_errors, GCE_fitted_errors, bubble_fitted_errors, isotropic_fitted_errors])
#

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# Likelihood('A', 0).likelihood_constrained(-5*np.ones(5))

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# %%time
#
# import time
# start_time = time.time()
#
# params, param_errors=Parameter_estimation('A').minuit_run()
#
# end_time = time.time()
# execution_time = end_time - start_time
# print(execution_time)

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# params

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# import matplotlib.pyplot as plt
# from astropy.visualization import astropy_mpl_style, wcsaxes #wcsaxes manages coordinates of the plot.
# plt.style.use(astropy_mpl_style)
# from astropy.io import fits
# from astropy.utils.data import get_pkg_data_filename
# from astropy.wcs import WCS #This package expresses coordinates of the image file.
# import numpy as np
# from matplotlib.ticker import LogLocator, LogFormatter, AutoMinorLocator
# plt.style.use('default')
# ax=plt.subplot()
#
# ax.set_xscale('log')
# ax.set_yscale('log')
# ax.set_xlabel('E [GeV]')
# ax.set_ylabel(r'$E^2 \frac{dN}{dE}$[GeV$cm^{-2}$$s^{-1} sr^{-1}$]')
#
# ax.set_ylim(1e-8, 1e-4)
# ax.set_xlim(0.3, 500)
#
# ax.tick_params(axis='y', which='both', direction='in', left=True)
# ax.tick_params(axis='x', which='both', direction='in', bottom=True)
# ax.minorticks_on()
# ax.grid(True, which='Major', linestyle='-', linewidth=0.5)
#
# #m=minuit4
# #fitted=params
# #fitted_errors=param_errors
#
# fitted_errors=param_errors
# fitted=params
# #fitted=np.ones(17*5)
# fitted_errors=np.zeros(17*5)
# sr=0.214411*2
#
# ax.errorbar(E, counts_per_exp*(E**2)/(delta_E*sr) , yerr=counts_per_exp_err*(E**2)/(delta_E*sr), linestyle='dotted', marker='.', elinewidth=2, capsize=4, capthick=2, label='Raw_data')
#
#
# ax.errorbar(E, psc*(E**2)/(delta_E*sr), linestyle='', marker='.', elinewidth=2, capsize=4, capthick=2, label='psc', color='orange')
#
#
# ax.errorbar(E, fitted[0:17]*(pion+bremss)*(E**2)/(delta_E*sr), yerr=fitted_errors[0:17]*(pion+bremss)*(E**2)/(delta_E*sr), linestyle='dotted', marker='.', elinewidth=2, capsize=4, capthick=2, label='pion+bremss', color='red')
# ax.errorbar(E, fitted[17:34]*(ics)*(E**2)/(delta_E*sr), yerr=fitted_errors[17:34]*(ics)*(E**2)/(delta_E*sr), linestyle='dashdot', marker='.', elinewidth=2, capsize=4, capthick=2, label='ics', color='blue')
#
# ax.plot(E, (pion+bremss)*(E**2)/(delta_E*sr), linestyle='dotted', label='pion+bremss', color='red')
# ax.plot(E, (ics)*(E**2)/(delta_E*sr), linestyle='dashdot', label='ics', color='blue')
#
#
#
# ax.errorbar(E,  fitted[34:51]*(GCE)*(E**2)/(delta_E*sr), yerr=np.sqrt((fitted_errors[34:51]*GCE)**2)*(E**2)/(delta_E*sr), linestyle='dashed', marker='.', elinewidth=2, capsize=4, capthick=2, label='GCE', color='black')
#
#
#
# ax.errorbar(E, fitted[51:68]*(bubble)*(E**2)/(delta_E*sr),yerr=fitted_errors[51:68]*(bubble)*(E**2)/(delta_E*sr), linestyle='dashed', marker='.', elinewidth=2, capsize=4, capthick=2, label='bubble', color='purple')
#
# ax.errorbar(E, fitted[68:85]*(isotropic)*(E**2)/(delta_E*sr),yerr=fitted_errors[68:85]*(isotropic)*(E**2)/(delta_E*sr), linestyle='dashed', marker='.', elinewidth=2, capsize=4, capthick=2, label='isotropic', color='green')
#
# ax.legend()

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# plt.ylim(1e-8, 1e-4)
# plt.xlim(0.3, 500)
# plt.loglog()
# plt.errorbar(E,  fitted[34:51]*(GCE)*(E**2)/(delta_E*sr), yerr=np.sqrt((fitted_errors[34:51]*GCE)**2)*(E**2)/(delta_E*sr), linestyle='dashed', marker='.', elinewidth=2, capsize=4, capthick=2, label='GCE', color='black')
#

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# np.savetxt('./GCE_model_F_p_p_p.dat', np.vstack([E*1e-3, fitted[48:72]*(GCE)*(E**2)/(1000*delta_E*sr), counts_per_exp_err*(E**2)/(1000*delta_E*sr)]).T)

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# import matplotlib.pyplot as plt
# from astropy.visualization import astropy_mpl_style, wcsaxes #wcsaxes manages coordinates of the plot.
# plt.style.use(astropy_mpl_style)
# from astropy.io import fits
# from astropy.utils.data import get_pkg_data_filename
# from astropy.wcs import WCS #This package expresses coordinates of the image file.
# import numpy as np
# from scipy.interpolate import interp1d
# from iminuit import Minuit
# #Constraints interpolated function
# #Contains constraints for bubble and isotropic as well
# #For isotropic, from https://arxiv.org/pdf/1410.3696.pdf Table 3
#
# #Testing exposure per pixel
#
# #Analysis for model F
# #Correcting bubble template given from https://arxiv.org/pdf/1407.7905, Table 2
#
# class Likelihood:
#     def __init__(self, model, energy_bin):
#         self.model=model
#         self.energy_bin=energy_bin
#     def likelihood_constrained(self, parameter_set):
#         data=fits.open('./GC_analysis_sanghwan/GC_all_time_ccube_masked.fits')
#         psc_map=fits.open('./GC_analysis_sanghwan/Model/GC_all_time_psc_model.fits')
#         pion=fits.open(f'./GC_analysis_sanghwan/Model/GC_all_time_pion_model{self.model}.fits')
#         ics=fits.open(f'./GC_analysis_sanghwan/Model/GC_all_time_ics_model{self.model}.fits')
#         bremss=fits.open(f'./GC_analysis_sanghwan/Model/GC_all_time_bremss_model{self.model}.fits')
#         GCE=fits.open(f'./GC_analysis_sanghwan/Model/GC_all_time_GCE_model.fits')
#         bubble=fits.open(f'./GC_analysis_sanghwan/Model/GC_all_time_fermi_bubble_model_no_convol.fits')
#         isotropic=fits.open(f'./GC_analysis_sanghwan/Model/GC_all_time_isotropic_model.fits')
#         #####################################
#         pion_bremss_param=parameter_set[0]
#         ics_param=parameter_set[1]
#         GCE_param=parameter_set[2]
#         bubble_param=parameter_set[3]
#         isotropic_param=parameter_set[4]
#         #alpha_psc=5
#         alpha_psc=5
#         f_psc=0.1
#         ######################################
#         #Main body for calculating Poissonian log-likelihood
#         #for k in range(0, energy_length, 1):
#         observed_pixel=data[0].data[self.energy_bin]
#         observed_pixel=observed_pixel.astype('float32')
#         psc_pixel=psc_map[0].data[self.energy_bin]
#         pion_bremss_pixel=pion[0].data[self.energy_bin] + bremss[0].data[self.energy_bin]
#         ics_pixel=ics[0].data[self.energy_bin]
#         GCE_pixel=GCE[0].data[self.energy_bin]
#         iso_pixel=isotropic[0].data[self.energy_bin]
#         bubble_pixel=bubble[0].data[self.energy_bin]
#         expected_pixel= pion_bremss_param*pion_bremss_pixel + ics_param*ics_pixel + psc_pixel + GCE_param*GCE_pixel + isotropic_param*iso_pixel + bubble_param*bubble_pixel
#         #expected_pixel= pion_bremss_param*pion_bremss_pixel + ics_param*ics_pixel + GCE_param*GCE_pixel + isotropic_param*iso_pixel + bubble_param*bubble_pixel
#         #expected_pixel[expected_pixel==0] = 1e-20
#
#
#         #bgr_pixel=pion_bremss_pixel + ics_pixel
#         bgr_pixel = fits.open('./GC_analysis_sanghwan/Model/GC_all_time_bremss_modelP.fits')[0].data[self.energy_bin] + fits.open('./GC_analysis_sanghwan/Model/GC_all_time_ics_modelP.fits')[0].data[self.energy_bin] + fits.open('./GC_analysis_sanghwan/Model/GC_all_time_pion_modelP.fits')[0].data[self.energy_bin]
#
#
#         #bgr_pixel[bgr_pixel==0] = 1e-20
#         #psc_pixel[psc_pixel==0] = 1e-20
#
#         w=1/( ( (psc_pixel)/(f_psc*bgr_pixel) )**alpha_psc +1 )
#         #w[np.isnan(w)]=0
#         lhd=2*w*( expected_pixel - observed_pixel*np.log((expected_pixel)) )
#         lhd[np.isnan(lhd)]=0
#
#         #At the end of the likelihood expression, there's term with ln(observed!) but is omitted since it does not affect overall analysis because it is constant term.
#
#         # Preparing for the constraints
#
#         E_bounds=fits.open('./GC_analysis_sanghwan/GC_all_time_ccube_masked.fits')[1].data
#
#
#         E=np.zeros(len(E_bounds))
#         for i in range(0, 4, 1):
#             E[i] = 1e-3*(E_bounds[i][2] + E_bounds[i][1])/2
#         for i in range(4, len(E_bounds), 1):
#             E[i] = np.sqrt(E_bounds[i][2]*E_bounds[i][1]*1e-6)
#
#         delta_E=np.zeros(len(E_bounds))
#         for i in range(0, len(E_bounds), 1):
#             delta_E[i] = (E_bounds[i][2] - E_bounds[i][1])*1e-3
#
#         sr=0.214411*2
#
#         exp_cube=np.sqrt(fits.open('./GC_analysis_sanghwan/GC_all_time_expcube.fits')[0].data[:1]*fits.open('./GC_analysis_sanghwan/GC_all_time_expcube.fits')[0].data[1:])
#         for i in range(0, 4, 1):
#             exp_cube[i] = ( fits.open('./GC_analysis_sanghwan/GC_all_time_expcube.fits')[0].data[i] + fits.open('./GC_analysis_sanghwan/GC_all_time_expcube.fits')[0].data[i+1] )/2
#         for i in range(4, len(exp_cube), 1):
#             exp_cube[i] = np.sqrt(fits.open('./GC_analysis_sanghwan/GC_all_time_expcube.fits')[0].data[i]*fits.open('./GC_analysis_sanghwan/GC_all_time_expcube.fits')[0].data[i+1])
#
#
#         #Flux assiciated with constraints are considered with it's exposure per pixel values
#
#         file_name='./GC_analysis_sanghwan/Model/GC_all_time_isotropic_model.fits'
#         isotropic=np.zeros(len(fits.open(file_name)[0].data))
#         for i in range(0, len(fits.open(file_name)[0].data), 1):
#             a=exp_cube[i]
#             isotropic[i] = np.sum( fits.open(file_name)[0].data[i]/a )#*isotropic_param
#
#         isotropic[self.energy_bin] *= isotropic_param
#
#         file_name='./GC_analysis_sanghwan/Model/GC_all_time_fermi_bubble_model_no_convol.fits'
#         bubble=np.zeros(len(fits.open(file_name)[0].data))
#         for i in range(0, len(fits.open(file_name)[0].data), 1):
#             a=exp_cube[i]
#             bubble[i] = np.sum( fits.open(file_name)[0].data[i]/a )#*bubble_param
#
#         bubble[self.energy_bin] *= bubble_param
#
#         bubble_sed=bubble/(1e-3*delta_E*sr)
#
#         constraints=np.loadtxt('./GC_analysis_sanghwan/Model/bubble_constraints_asymmetric.txt')
#         constraints_energy=constraints[0:, 0]
#         constraints_flux=constraints[0:, 1]/constraints[0:, 0]**2
#         constraints_lower_error=constraints[0:, 2]/constraints[0:, 0]**2
#         constraints_upper_error=constraints[0:, 3]/constraints[0:, 0]**2
#
#         flux_data=constraints_flux
#         lower_error_data=constraints_lower_error
#         upper_error_data=constraints_upper_error
#         #chi2=np.zeros(len(bubble_sed))
#
#         #for i in range(len(bubble_sed)):
#         chi2=0
#         i=self.energy_bin
#         larger_error=max([upper_error_data[i], lower_error_data[i]])
#         if flux_data[i] < bubble_sed[i]:
#             chi2 = ((bubble_sed[i] - flux_data[i])/upper_error_data[i])**2
#         if flux_data[i] > bubble_sed[i]:
#             chi2 = ((bubble_sed[i] - flux_data[i])/lower_error_data[i])**2
#         if flux_data[i] == bubble_sed[i]:
#             chi2 = ((bubble_sed[i] - flux_data[i])/larger_error)**2
#         #print(chi2)
#         iso_flux=isotropic/(sr*delta_E*1e-3)
#
#         iso_constraints=np.loadtxt('./GC_analysis_sanghwan/Model/iso_constraints_conservative_sys.txt')
#         iso_constraints_energy=iso_constraints[0:, 0]
#         iso_constraints_flux=iso_constraints[0:, 1]
#         iso_constraints_full_error=iso_constraints[0:, 2]
#
#
#         fluxint=interp1d(iso_constraints_energy, iso_constraints_flux, fill_value="extrapolate", kind='quadratic')
#         errint=interp1d(iso_constraints_energy, iso_constraints_full_error, fill_value="extrapolate", kind='quadratic')    
#
#         iso_constraints_flux=fluxint(E*1e-3)
#         iso_constraints_flux_err=errint(E*1e-3)
#
#         iso_constraints_flux[iso_constraints_flux <= 0] = 1e-20
#         iso_constraints_flux_err[iso_constraints_flux_err <= 0] = 1e-20
#
#         chi2_iso=((iso_flux[self.energy_bin] - iso_constraints_flux[self.energy_bin])**2/iso_constraints_flux_err[self.energy_bin]**2) 
#
#         return np.sum(lhd) + chi2 + chi2_iso
#         #return np.sum(lhd) + chi2[self.energy_bin] + chi2_iso[self.energy_bin]
#
#     def likelihood_module(self):
#         return self.likelihood_constrained(np.ones(5))
#
#     def minuit_module(self):
#         tol=1e-2
#         tol1=tol
#         m=Minuit(self.likelihood_constrained, np.ones(5))
#         m.errors=np.ones(5)
#         m.errordef=0.5
#         m.limits=(-1, 4)
#         m.strategy=2
#         while True:
#             if True:
#                 m.tol=tol1
#                 print(f'tol1 = {m.tol}')
#                 m.migrad()
#                 if m.valid==True:
#                     print(m)
#                     break;
#                 else:
#                     tol1*=1e+1
#         return m.values[:], m.errors[:]
#
# class Parameter_estimation():
#     def __init__(self, model):
#         self.model=model
#     def minuit_run(self):
#         pion_bremss_fitted=[]
#         ics_fitted=[]
#         GCE_fitted=[]
#         bubble_fitted=[]
#         isotropic_fitted=[]
#
#         pion_bremss_fitted_errors=[]
#         ics_fitted_errors=[]
#         GCE_fitted_errors=[]
#         bubble_fitted_errors=[]
#         isotropic_fitted_errors=[]     
#         for i in range(0, 24, 1):
#             results=Likelihood(self.model, i).minuit_module()
#             pion_bremss_fitted.append(results[0][0])
#             ics_fitted.append(results[0][1])
#             GCE_fitted.append(results[0][2])
#             bubble_fitted.append(results[0][3])
#             isotropic_fitted.append(results[0][4])
#
#             pion_bremss_fitted_errors.append(results[1][0])
#             ics_fitted_errors.append(results[1][1])
#             GCE_fitted_errors.append(results[1][2])
#             bubble_fitted_errors.append(results[1][3])
#             isotropic_fitted_errors.append(results[1][4])
#         return np.hstack([pion_bremss_fitted, ics_fitted, GCE_fitted, bubble_fitted, isotropic_fitted]), np.hstack([pion_bremss_fitted_errors, ics_fitted_errors, GCE_fitted_errors, bubble_fitted_errors, isotropic_fitted_errors])
#

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# #no mask test

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# %%time
#
# import time
# start_time = time.time()
#
# params, param_errors=Parameter_estimation('F').minuit_run()
#
# end_time = time.time()
# execution_time = end_time - start_time
# print(execution_time)

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# #Fermi bubble -> current : Systematics only Next : Statistical error within

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# import matplotlib.pyplot as plt
# from astropy.visualization import astropy_mpl_style, wcsaxes #wcsaxes manages coordinates of the plot.
# plt.style.use(astropy_mpl_style)
# from astropy.io import fits
# from astropy.utils.data import get_pkg_data_filename
# from astropy.wcs import WCS #This package expresses coordinates of the image file.
# import numpy as np
# from matplotlib.ticker import LogLocator, LogFormatter, AutoMinorLocator
# plt.style.use('default')
# ax=plt.subplot()
#
# ax.set_xscale('log')
# ax.set_yscale('log')
# ax.set_xlabel('E [GeV]')
# ax.set_ylabel(r'$E^2 \frac{dN}{dE}$[GeV$cm^{-2}$$s^{-1} sr^{-1}$]')
#
# ax.set_ylim(1e-8, 1e-4)
# ax.set_xlim(0.3, 500)
#
# ax.tick_params(axis='y', which='both', direction='in', left=True)
# ax.tick_params(axis='x', which='both', direction='in', bottom=True)
# ax.minorticks_on()
# ax.grid(True, which='Major', linestyle='-', linewidth=0.5)
#
# #m=minuit4
# #fitted=minuit4.values[:]
# #fitted_errors=minuit4.errors[:]
#
# fitted_errors=param_errors
# fitted=params
# #fitted=np.ones(120)
# sr=0.214411*2
#
# ax.errorbar(E*1e-3, counts_per_exp*(E**2)/(1000*delta_E*sr) , yerr=counts_per_exp_err*(E**2)/(1000*delta_E*sr), linestyle='dotted', marker='.', elinewidth=2, capsize=4, capthick=2, label='Raw_data')
#
# #ax.plot(E*1e-3, psc*(E**2)/(1000*delta_E*sr))
#
# ax.errorbar(E*1e-3, psc*(E**2)/(1000*delta_E*sr), linestyle='', marker='.', elinewidth=2, capsize=4, capthick=2, label='psc', color='orange')
#
#
# ax.errorbar(E*1e-3, fitted[0:24]*(pion+bremss)*(E**2)/(1000*delta_E*sr), yerr=fitted_errors[0:24]*(pion+bremss)*(E**2)/(1000*delta_E*sr), linestyle='dotted', marker='.', elinewidth=2, capsize=4, capthick=2, label='pion+bremss', color='red')
# ax.errorbar(E*1e-3, fitted[24:48]*(ics)*(E**2)/(1000*delta_E*sr), yerr=fitted_errors[24:48]*(ics)*(E**2)/(1000*delta_E*sr), linestyle='dashdot', marker='.', elinewidth=2, capsize=4, capthick=2, label='ics', color='blue')
#
# ax.plot(E*1e-3, (pion+bremss)*(E**2)/(1000*delta_E*sr), linestyle='dotted', label='pion+bremss', color='red')
# ax.plot(E*1e-3, (ics)*(E**2)/(1000*delta_E*sr), linestyle='dashdot', label='ics', color='blue')
#
#
#
# ax.errorbar(E*1e-3,  fitted[48:72]*(GCE)*(E**2)/(1000*delta_E*sr), yerr=np.sqrt((fitted_errors[48:72]*GCE)**2)*(E**2)/(1000*delta_E*sr), linestyle='dashed', marker='.', elinewidth=2, capsize=4, capthick=2, label='GCE', color='black')
#
#
#
# ax.errorbar(E*1e-3, fitted[72:96]*(bubble)*(E**2)/(1000*delta_E*sr),yerr=fitted_errors[72:96]*(bubble)*(E**2)/(1000*delta_E*sr), linestyle='dashed', marker='.', elinewidth=2, capsize=4, capthick=2, label='bubble', color='purple')
#
# ax.errorbar(E*1e-3, fitted[96:120]*(isotropic)*(E**2)/(1000*delta_E*sr),yerr=fitted_errors[96:120]*(isotropic)*(E**2)/(1000*delta_E*sr), linestyle='dashed', marker='.', elinewidth=2, capsize=4, capthick=2, label='isotropic', color='green')
#
#
# summed=fitted[0:24]*(pion+bremss) + fitted[24:48]*(ics) + fitted[48:72]*(GCE) + fitted[72:96]*(bubble) + fitted[96:120]*(isotropic) + psc
#
#
# #summed=fitted[0:24]*(pion+bremss) + fitted[24:48]*(ics) + 0*(GCE) + fitted[72:96]*(bubble) + fitted[96:120]*(isotropic) + psc
#
# summed_err=np.sqrt((fitted_errors[0:24]*(pion+bremss)*(E**2)/(1000*delta_E*sr))**2 + (fitted_errors[24:48]*(ics)*(E**2)/(1000*delta_E*sr))**2 + (fitted_errors[48:72]*(0.1*GCE)*(E**2)/(1000*delta_E*sr))**2 + (fitted_errors[72:96]*(bubble)*(E**2)/(1000*delta_E*sr))**2 + (fitted_errors[96:120]*(isotropic)*(E**2)/(1000*delta_E*sr))**2)
#
# ax.errorbar(E*1e-3, summed*(E**2)/(1000*delta_E*sr), yerr=summed_err, linestyle='dotted', marker='.', elinewidth=2, capsize=4, capthick=2, label='Model', color='black')
#
# #ax.errorbar(E*1e-3, (counts_per_exp-summed_test)*(E**2)/(1000*delta_E*sr), yerr=summed_err, linestyle='', marker='.', elinewidth=2, capsize=4, capthick=2, label='Model', color='black')
#
#
#
# """
# ax.plot(E*1e-3, (isotropic)*(E**2)/(1000*exposure(E*1e-3)*delta_E*sr), label='iso', color='green')
# ax.plot(E*1e-3, (model)*(E**2)/(1000*exposure(E*1e-3)*delta_E*sr), label='Total model', color='cyan')
# ax.plot(E*1e-3, psc*(E**2)/(1000*exposure(E*1e-3)*delta_E*sr), label='Total point sources', color='yellow')
# #ax.plot(E*1e-3, bubble.T[1, 0:], label='Bubble', color='purple') """
# ax.legend()

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# np.savetxt('./GCE_model_F_p_p.dat', np.vstack([E*1e-3, fitted[48:72]*(GCE)*(E**2)/(1000*delta_E*sr), counts_per_exp_err*(E**2)/(1000*delta_E*sr)]).T)

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# import matplotlib.pyplot as plt
# from astropy.visualization import astropy_mpl_style, wcsaxes #wcsaxes manages coordinates of the plot.
# plt.style.use(astropy_mpl_style)
# from astropy.io import fits
# from astropy.utils.data import get_pkg_data_filename
# from astropy.wcs import WCS #This package expresses coordinates of the image file.
# import numpy as np
# from scipy.interpolate import interp1d
# from iminuit import Minuit
# #Constraints interpolated function
# #Contains constraints for bubble and isotropic as well
# #For isotropic, from https://arxiv.org/pdf/1410.3696.pdf Table 3
#
# #Testing exposure per pixel
#
# #Analysis for model F
# #Correcting bubble template given from https://arxiv.org/pdf/1407.7905, Table 2
#
# model='F'
# def likelihood_constrained(parameter_set):
#     data=fits.open('./GC_analysis_sanghwan/GC_all_time_ccube_masked.fits')
#     psc_map=fits.open('./GC_analysis_sanghwan/Model/GC_all_time_psc_model.fits')
#     pion=fits.open(f'./GC_analysis_sanghwan/Model/GC_all_time_pion_model{model}.fits')
#     ics=fits.open(f'./GC_analysis_sanghwan/Model/GC_all_time_ics_model{model}.fits')
#     bremss=fits.open(f'./GC_analysis_sanghwan/Model/GC_all_time_bremss_model{model}.fits')
#     GCE=fits.open(f'./GC_analysis_sanghwan/Model/GC_all_time_GCE_model.fits')
#     bubble=fits.open(f'./GC_analysis_sanghwan/Model/GC_all_time_fermi_bubble_model.fits')
#     isotropic=fits.open(f'./GC_analysis_sanghwan/Model/GC_all_time_isotropic_model.fits')
#     #####################################
#     pion_bremss_param=parameter_set[0:24]
#     ics_param=parameter_set[24:48]
#     GCE_param=parameter_set[48:72]
#     bubble_param=parameter_set[72:96]
#     isotropic_param=parameter_set[96:120]
#     alpha_psc=5
#     f_psc=0.1
#     ######################################
#     #Main body for calculating Poissonian log-likelihood
#     #for k in range(0, energy_length, 1):
#     observed_pixel=data[0].data
#     observed_pixel=observed_pixel.astype('float32')
#     psc_pixel=psc_map[0].data
#     pion_bremss_pixel=pion[0].data + bremss[0].data
#     ics_pixel=ics[0].data
#     GCE_pixel=GCE[0].data
#     iso_pixel=isotropic[0].data
#     bubble_pixel=bubble[0].data
#     expected_pixel=np.zeros(np.shape(observed_pixel))
#     for i in range(0, len(expected_pixel), 1):
#         expected_pixel[i]= pion_bremss_param[i]*pion_bremss_pixel[i] + ics_param[i]*ics_pixel[i] + psc_pixel[i] + GCE_param[i]*GCE_pixel[i] + isotropic_param[i]*iso_pixel[i] + bubble_param[i]*bubble_pixel[i]
#     #expected_pixel= pion_bremss_param*pion_bremss_pixel + ics_param*ics_pixel + GCE_param*GCE_pixel + isotropic_param*iso_pixel + bubble_param*bubble_pixel
#     expected_pixel[expected_pixel==0] = 1e-10
#
#
#     #bgr_pixel=pion_bremss_pixel + ics_pixel
#     bgr_pixel = fits.open('./GC_analysis_sanghwan/Model/GC_all_time_bremss_modelP.fits')[0].data + fits.open('./GC_analysis_sanghwan/Model/GC_all_time_ics_modelP.fits')[0].data + fits.open('./GC_analysis_sanghwan/Model/GC_all_time_pion_modelP.fits')[0].data
#
#
#     bgr_pixel[bgr_pixel==0] = 1e-10
#     psc_pixel[psc_pixel==0] = 1e-10
#
#     w=1/( ( (psc_pixel)/(f_psc*bgr_pixel) )**alpha_psc +1 )
#     lhd=2*w*( expected_pixel - observed_pixel*np.log((expected_pixel)) )
#
#     #At the end of the likelihood expression, there's term with ln(observed!) but is omitted since it does not affect overall analysis because it is constant term.
#
#     # Preparing for the constraints
#
#     E_bounds=fits.open('./GC_analysis_sanghwan/GC_all_time_ccube_masked.fits')[1].data
#
#
#     E=np.zeros(len(E_bounds))
#     for i in range(0, 4, 1):
#         E[i] = 1e-3*(E_bounds[i][2] + E_bounds[i][1])/2
#     for i in range(4, len(E_bounds), 1):
#         E[i] = np.sqrt(E_bounds[i][2]*E_bounds[i][1]*1e-6)
#
#     delta_E=np.zeros(len(E_bounds))
#     for i in range(0, len(E_bounds), 1):
#         delta_E[i] = (E_bounds[i][2] - E_bounds[i][1])*1e-3
#
#     sr=0.214411*2
#
#     exp_cube=np.sqrt(fits.open('./GC_analysis_sanghwan/GC_all_time_expcube.fits')[0].data[:1]*fits.open('./GC_analysis_sanghwan/GC_all_time_expcube.fits')[0].data[1:])
#     for i in range(0, 4, 1):
#         exp_cube[i] = ( fits.open('./GC_analysis_sanghwan/GC_all_time_expcube.fits')[0].data[i] + fits.open('./GC_analysis_sanghwan/GC_all_time_expcube.fits')[0].data[i+1] )/2
#     for i in range(4, len(exp_cube), 1):
#         exp_cube[i] = np.sqrt(fits.open('./GC_analysis_sanghwan/GC_all_time_expcube.fits')[0].data[i]*fits.open('./GC_analysis_sanghwan/GC_all_time_expcube.fits')[0].data[i+1])
#
#
#     #Flux assiciated with constraints are considered with it's exposure per pixel values
#
#     file_name='./GC_analysis_sanghwan/Model/GC_all_time_isotropic_model.fits'
#     isotropic=np.zeros(len(fits.open(file_name)[0].data))
#     for i in range(0, len(fits.open(file_name)[0].data), 1):
#         a=exp_cube[i]
#         isotropic[i] = np.sum( fits.open(file_name)[0].data[i]/a )*isotropic_param[i]
#
#     file_name='./GC_analysis_sanghwan/Model/GC_all_time_fermi_bubble_model.fits'
#     bubble=np.zeros(len(fits.open(file_name)[0].data))
#     for i in range(0, len(fits.open(file_name)[0].data), 1):
#         a=exp_cube[i]
#         bubble[i] = np.sum( fits.open(file_name)[0].data[i]/a )*bubble_param[i]
#
#
#     #bubble_sed=bubble/(1e-3*delta_E*sr)
#
#     constraints=np.loadtxt('./GC_analysis_sanghwan/Model/bubble_constraints_asymmetric.txt')
#     constraints_energy=constraints[0:, 0]
#     constraints_flux=constraints[0:, 1]
#     constraints_lower_error=constraints[0:, 2]
#     constraints_upper_error=constraints[0:, 3]
#
#
#     fluxint=interp1d(constraints_energy, constraints_flux, fill_value='extrapolate',  kind='cubic')
#     lower_errint=interp1d(constraints_energy, constraints_lower_error, fill_value='extrapolate', kind='cubic')
#     upper_errint=interp1d(constraints_energy, constraints_upper_error, fill_value='extrapolate', kind='cubic')
#
#     flux_data=fluxint(E*1e-3)/((E*1e-3)**2)
#     lower_error_data=lower_errint(E*1e-3)/((E*1e-3)**2)
#     upper_error_data=upper_errint(E*1e-3)/((E*1e-3)**2)
#
#     flux_data[flux_data<=0] = 1e-20
#     lower_error_data[lower_error_data<=0] = 1e-20
#     upper_error_data[upper_error_data<=0] = 1e-20
#
#
#     for i in range(len(chi2)):
#         larger_error=max([upper_error_data[i], lower_error_data[i]])
#         if flux_data[i] < bubble_sed[i]:
#             chi2[i] = ((bubble_sed[i] - flux_data[i])/upper_error_data[i])**2
#         if flux_data[i] > bubble_sed[i]:
#             chi2[i] = ((bubble_sed[i] - flux_data[i])/lower_error_data[i])**2
#         if flux_data[i] == bubble_sed[i]:
#             chi2[i] = ((bubble_sed[i] - flux_data[i])/larger_error)**2
#         #chi2[i] = ((bubble_sed[i] - flux_data[i])/larger_error)**2
#
#     iso_flux=isotropic/(sr*delta_E*1e-3)
#
#     iso_constraints=np.loadtxt('./GC_analysis_sanghwan/Model/iso_constraints_conservative_sys.txt')
#     iso_constraints_energy=iso_constraints[0:, 0]
#     iso_constraints_flux=iso_constraints[0:, 1]
#     iso_constraints_full_error=iso_constraints[0:, 2]
#
#
#     fluxint=interp1d(iso_constraints_energy, iso_constraints_flux, fill_value="extrapolate")
#     errint=interp1d(iso_constraints_energy, iso_constraints_full_error, fill_value="extrapolate")    
#
#     iso_constraints_flux=fluxint(E*1e-3)
#     iso_constraints_flux_err=errint(E*1e-3)
#
#     iso_constraints_flux[iso_constraints_flux <= 0] = 1e-20
#     iso_constraints_flux_err[iso_constraints_flux_err <= 0] = 1e-20
#
#     chi2_iso=((iso_flux - iso_constraints_flux)**2/iso_constraints_flux_err**2) 
#
#     return np.sum(lhd) + np.sum(chi2) + np.sum(chi2_iso)

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# import time
# start_time = time.time()
# end_time = time.time()
# execution_time=end_time-start_time
# print(execution_time)

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# likelihood_constrained(params)

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# %%time
# import time
# start_time = time.time()
#
# execution_time = end_time - start_time
# m=Minuit(likelihood_constrained, params)
# m.errordef=0.5
# m.limits=(-4, 4)
# m.strategy=1
# m.tol=1e-1
# m.simplex()
# end_time = time.time()
# execution_time = end_time - start_time
# print(execution_time)
# print(m)

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# import matplotlib.pyplot as plt
# from astropy.visualization import astropy_mpl_style, wcsaxes #wcsaxes manages coordinates of the plot.
# plt.style.use(astropy_mpl_style)
# from astropy.io import fits
# from astropy.utils.data import get_pkg_data_filename
# from astropy.wcs import WCS #This package expresses coordinates of the image file.
# import numpy as np
# from matplotlib.ticker import LogLocator, LogFormatter, AutoMinorLocator
# plt.style.use('default')
# ax=plt.subplot()
#
# ax.set_xscale('log')
# ax.set_yscale('log')
# ax.set_xlabel('E [GeV]')
# ax.set_ylabel(r'$E^2 \frac{dN}{dE}$[GeV$cm^{-2}$$s^{-1} sr^{-1}$]')
#
# ax.set_ylim(1e-8, 1e-4)
# ax.set_xlim(0.3, 500)
#
# ax.tick_params(axis='y', which='both', direction='in', left=True)
# ax.tick_params(axis='x', which='both', direction='in', bottom=True)
# ax.minorticks_on()
# ax.grid(True, which='Major', linestyle='-', linewidth=0.5)
#
# #m=minuit4
# fitted=m.values[:]
# fitted_errors=m.errors[:]
# fitted=params
#
# sr=0.214411*2
#
# ax.errorbar(E*1e-3, counts_per_exp*(E**2)/(1000*delta_E*sr) , yerr=counts_per_exp_err*(E**2)/(1000*delta_E*sr), linestyle='dotted', marker='.', elinewidth=2, capsize=4, capthick=2, label='Raw_data')
#
# #ax.plot(E*1e-3, psc*(E**2)/(1000*delta_E*sr))
#
# ax.errorbar(E*1e-3, psc*(E**2)/(1000*delta_E*sr), linestyle='', marker='.', elinewidth=2, capsize=4, capthick=2, label='psc', color='orange')
#
#
# ax.errorbar(E*1e-3, fitted[0:24]*(pion+bremss)*(E**2)/(1000*delta_E*sr), yerr=fitted_errors[0:24]*(pion+bremss)*(E**2)/(1000*delta_E*sr), linestyle='dotted', marker='.', elinewidth=2, capsize=4, capthick=2, label='pion+bremss', color='red')
# ax.errorbar(E*1e-3, fitted[24:48]*(ics)*(E**2)/(1000*delta_E*sr), yerr=fitted_errors[24:48]*(ics)*(E**2)/(1000*delta_E*sr), linestyle='dashdot', marker='.', elinewidth=2, capsize=4, capthick=2, label='ics', color='blue')
#
# ax.plot(E*1e-3, (pion+bremss)*(E**2)/(1000*delta_E*sr), linestyle='dotted', label='pion+bremss', color='red')
#
# ax.plot(E*1e-3, (ics)*(E**2)/(1000*delta_E*sr), linestyle='dashdot', label='ics', color='blue')
#
#
#
# ax.errorbar(E*1e-3,  fitted[48:72]*(GCE)*(E**2)/(1000*delta_E*sr), yerr=np.sqrt((fitted_errors[48:72]*GCE)**2)*(E**2)/(1000*delta_E*sr), linestyle='dashed', marker='.', elinewidth=2, capsize=4, capthick=2, label='GCE', color='black')
#
#
#
# ax.errorbar(E*1e-3, fitted[72:96]*(bubble)*(E**2)/(1000*delta_E*sr),yerr=fitted_errors[72:96]*(bubble)*(E**2)/(1000*delta_E*sr), linestyle='dashed', marker='.', elinewidth=2, capsize=4, capthick=2, label='bubble', color='purple')
# ax.plot(E*1e-3, bubble*(E**2)/(1000*delta_E*sr) )
#
#
# ax.errorbar(E*1e-3, fitted[96:120]*(isotropic)*(E**2)/(1000*delta_E*sr),yerr=fitted_errors[96:120]*(isotropic)*(E**2)/(1000*delta_E*sr), linestyle='dashed', marker='.', elinewidth=2, capsize=4, capthick=2, label='isotropic', color='green')
#
#
# summed=fitted[0:24]*(pion+bremss) + fitted[24:48]*(ics) + fitted[48:72]*(GCE) + fitted[72:96]*(bubble) + fitted[96:120]*(isotropic) + psc
#
# ax.errorbar(E*1e-3, summed*(E**2)/(1000*delta_E*sr), linestyle='dotted', marker='.', elinewidth=2, capsize=4, capthick=2, label='Model', color='black')
#
# ax.legend()

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# import matplotlib.pyplot as plt
# from astropy.visualization import astropy_mpl_style, wcsaxes #wcsaxes manages coordinates of the plot.
# plt.style.use(astropy_mpl_style)
# from astropy.io import fits
# from astropy.utils.data import get_pkg_data_filename
# from astropy.wcs import WCS #This package expresses coordinates of the image file.
# import numpy as np
# from scipy.interpolate import interp1d
# from iminuit import Minuit
# #Constraints interpolated function
# #Contains constraints for bubble and isotropic as well
# #For isotropic, from https://arxiv.org/pdf/1410.3696.pdf Table 3
#
# #Testing exposure per pixel
#
# #Analysis for model F
# #Correcting bubble template given from https://arxiv.org/pdf/1407.7905, Table 2
#
# class Likelihood:
#     def __init__(self, model, energy_bin):
#         self.model=model
#         self.energy_bin=energy_bin
#     def likelihood_constrained(self, parameter_set):
#         data=fits.open('./GC_analysis_sanghwan/GC_ccw_exposure_ccube_masked.fits')
#         psc_map=fits.open('./GC_analysis_sanghwan/Model/GC_psc_model.fits')
#         pion=fits.open(f'./GC_analysis_sanghwan/Model/GC_pion_model{self.model}.fits')
#         ics=fits.open(f'./GC_analysis_sanghwan/Model/GC_ics_model{self.model}.fits')
#         bremss=fits.open(f'./GC_analysis_sanghwan/Model/GC_bremss_model{self.model}.fits')
#         GCE=fits.open(f'./GC_analysis_sanghwan/Model/GC_GCE_model.fits')
#         bubble=fits.open(f'./GC_analysis_sanghwan/Model/GC_fermi_bubble_model.fits')
#         isotropic=fits.open(f'./GC_analysis_sanghwan/Model/GC_isotropic_model.fits')
#         #####################################
#         pion_bremss_param=parameter_set[0]
#         ics_param=parameter_set[1]
#         GCE_param=parameter_set[2]
#         bubble_param=parameter_set[3]
#         isotropic_param=parameter_set[4]
#         alpha_psc=5
#         f_psc=0.1
#         ######################################
#         #Main body for calculating Poissonian log-likelihood
#         #for k in range(0, energy_length, 1):
#         observed_pixel=data[0].data[self.energy_bin]
#         observed_pixel=observed_pixel.astype('float32')
#         psc_pixel=psc_map[0].data[self.energy_bin]
#         pion_bremss_pixel=pion[0].data[self.energy_bin] + bremss[0].data[self.energy_bin]
#         ics_pixel=ics[0].data[self.energy_bin]
#         GCE_pixel=GCE[0].data[self.energy_bin]
#         iso_pixel=isotropic[0].data[self.energy_bin]
#         bubble_pixel=bubble[0].data[self.energy_bin]
#         expected_pixel= pion_bremss_param*pion_bremss_pixel + ics_param*ics_pixel + psc_pixel + GCE_param*GCE_pixel + isotropic_param*iso_pixel + bubble_param*bubble_pixel
#         #expected_pixel= pion_bremss_param*pion_bremss_pixel + ics_param*ics_pixel + GCE_param*GCE_pixel + isotropic_param*iso_pixel + bubble_param*bubble_pixel
#         expected_pixel[expected_pixel==0] = 1e-10
#
#
#         #bgr_pixel=pion_bremss_pixel + ics_pixel
#         bgr_pixel = fits.open('./GC_analysis_sanghwan/Model/GC_bremss_modelP.fits')[0].data[self.energy_bin] + fits.open('./GC_analysis_sanghwan/Model/GC_ics_modelP.fits')[0].data[self.energy_bin] + fits.open('./GC_analysis_sanghwan/Model/GC_pion_modelP.fits')[0].data[self.energy_bin]
#
#
#         bgr_pixel[bgr_pixel==0] = 1e-10
#         psc_pixel[psc_pixel==0] = 1e-10
#
#         w=1/( ( (psc_pixel)/(f_psc*bgr_pixel) )**alpha_psc +1 )
#         lhd=2*w*( expected_pixel - observed_pixel*np.log((expected_pixel)) )
#
#         #At the end of the likelihood expression, there's term with ln(observed!) but is omitted since it does not affect overall analysis because it is constant term.
#
#         # Preparing for the constraints
#
#         E_bounds=fits.open('./GC_analysis_sanghwan/GC_ccw_exposure_ccube_masked.fits')[1].data
#
#
#         E=np.zeros(len(E_bounds))
#         for i in range(0, 4, 1):
#             E[i] = 1e-3*(E_bounds[i][2] + E_bounds[i][1])/2
#         for i in range(4, len(E_bounds), 1):
#             E[i] = np.sqrt(E_bounds[i][2]*E_bounds[i][1]*1e-6)
#
#         delta_E=np.zeros(len(E_bounds))
#         for i in range(0, len(E_bounds), 1):
#             delta_E[i] = (E_bounds[i][2] - E_bounds[i][1])*1e-3
#
#         sr=0.214411*2
#
#         exp_cube=np.sqrt(fits.open('./GC_analysis_sanghwan/GC_ccw_exposure_expcube.fits')[0].data[:1]*fits.open('./GC_analysis_sanghwan/GC_ccw_exposure_expcube.fits')[0].data[1:])
#         for i in range(0, 4, 1):
#             exp_cube[i] = ( fits.open('./GC_analysis_sanghwan/GC_ccw_exposure_expcube.fits')[0].data[i] + fits.open('./GC_analysis_sanghwan/GC_ccw_exposure_expcube.fits')[0].data[i+1] )/2
#         for i in range(4, len(exp_cube), 1):
#             exp_cube[i] = np.sqrt(fits.open('./GC_analysis_sanghwan/GC_ccw_exposure_expcube.fits')[0].data[i]*fits.open('./GC_analysis_sanghwan/GC_ccw_exposure_expcube.fits')[0].data[i+1])
#
#
#         #Flux assiciated with constraints are considered with it's exposure per pixel values
#
#         file_name='./GC_analysis_sanghwan/Model/GC_isotropic_model.fits'
#         isotropic=np.zeros(len(fits.open(file_name)[0].data))
#         for i in range(0, len(fits.open(file_name)[0].data), 1):
#             a=exp_cube[i]
#             isotropic[i] = np.sum( fits.open(file_name)[0].data[i]/a )*isotropic_param
#
#         file_name='./GC_analysis_sanghwan/Model/GC_fermi_bubble_model.fits'
#         bubble=np.zeros(len(fits.open(file_name)[0].data))
#         for i in range(0, len(fits.open(file_name)[0].data), 1):
#             a=exp_cube[i]
#             bubble[i] = np.sum( fits.open(file_name)[0].data[i]/a )*bubble_param
#
#
#
#         #constraints=np.loadtxt('./GC_analysis_sanghwan/Model/bubble_constraints.txt')
#         bubble_sed=(E**2)*bubble/(1000*delta_E*sr)
#         #errint=interp1d(constraints[0:, 0], (constraints[0:, 6]*1e-7),kind='cubic', fill_value="extrapolate")
#         #fluxint=interp1d(constraints[0:, 0], (constraints[0:, 3]*1e-7),kind='cubic', fill_value="extrapolate")
#         #constraints=np.loadtxt('./GC_analysis_sanghwan/Model/bubble_constraints_conservative_sys.txt')
#         constraints=np.loadtxt('./GC_analysis_sanghwan/Model/bubble_constraints_asymmetric.txt')
#         constraints_energy=constraints[0:, 0]
#         constraints_flux=constraints[0:, 1]
#         constraints_lower_error=constraints[0:, 2]
#         constraints_upper_error=constraints[0:, 3]
#
#
#         fluxint=interp1d(constraints_energy, constraints_flux, fill_value='extrapolate',  kind='cubic')
#         lower_errint=interp1d(constraints_energy, constraints_lower_error, fill_value='extrapolate', kind='cubic')
#         upper_errint=interp1d(constraints_energy, constraints_upper_error, fill_value='extrapolate', kind='cubic')
#
#         flux_data=fluxint(E*1e-3)
#         lower_error_data=lower_errint(E*1e-3)
#         upper_error_data=upper_errint(E*1e-3)
#
#         flux_data[flux_data<=0] = 1e-20
#         lower_error_data[lower_error_data<=0] = 1e-20
#         upper_error_data[upper_error_data<=0] = 1e-20
#
#         chi2=np.zeros(len(bubble_sed))
#
#         for i in range(len(chi2)):
#             larger_error=max([upper_error_data[i], lower_error_data[i]])
#             if flux_data[i] > bubble_sed[i]:
#                 chi2[i] = ((bubble_sed[i] - flux_data[i])/upper_error_data[i])**2
#             if flux_data[i] < bubble_sed[i]:
#                 chi2[i] = ((bubble_sed[i] - flux_data[i])/lower_error_data[i])**2
#             if flux_data[i] == bubble_sed[i]:
#                 chi2[i] = ((bubble_sed[i] - flux_data[i])/larger_error)**2
#             #chi2[i] = ((bubble_sed[i] - flux_data[i])/larger_error)**2
#
#         #chi2=((bubble_sed - flux_data)/error_data)**2
#         #errint=interp1d(constraints[0:, 0], constraints[0:, 6]*1e-7,kind='cubic', fill_value="extrapolate")
#         #fluxint=interp1d(constraints[0:, 0], constraints[0:, 3]*1e-7,kind='cubic', fill_value="extrapolate")
#         #chi2 = ( (bubble_sed - fluxint(E*1e-3))/(errint(E*1e-3)) )**2
#
#
#         ####
#         # Preparing isotropic component constraints
#         #iso_constraints=np.loadtxt('./GC_analysis_sanghwan/Model/iso_constraints.txt')
#         iso_flux=isotropic/(sr*delta_E*1e-3)
#         #iso_constraints_energy=np.sqrt(iso_constraints[0:, 0]*iso_constraints[0:, 1]) #GeV
#         #iso_constraints_delta=iso_constraints[0:, 1] - iso_constraints[0:, 0]
#         #iso_constraints_data=(iso_constraints[0:, 2])/iso_constraints_delta #Will give GeV^-1 cm^-2 s^-1 sr^-1
#         #iso_constraints_err=(iso_constraints[0:, 3])/iso_constraints_delta
#         iso_constraints=np.loadtxt('./GC_analysis_sanghwan/Model/iso_constraints_conservative_sys.txt')
#         iso_constraints_energy=iso_constraints[0:, 0]
#         iso_constraints_flux=iso_constraints[0:, 1]
#         iso_constraints_full_error=iso_constraints[0:, 2]
#
#         #fluxint=interp1d(iso_constraints_energy, iso_constraints_data, fill_value="extrapolate")
#         #errint=interp1d(iso_constraints_energy, iso_constraints_err, fill_value="extrapolate")
#
#         fluxint=interp1d(iso_constraints_energy, iso_constraints_flux, fill_value="extrapolate")
#         errint=interp1d(iso_constraints_energy, iso_constraints_full_error, fill_value="extrapolate")    
#
#         iso_constraints_flux=fluxint(E*1e-3)
#         iso_constraints_flux_err=errint(E*1e-3)
#
#         iso_constraints_flux[iso_constraints_flux <= 0] = 1e-20
#         iso_constraints_flux_err[iso_constraints_flux_err <= 0] = 1e-20
#
#         chi2_iso=((iso_flux - iso_constraints_flux)**2/iso_constraints_flux_err**2) 
#
#         #return np.sum(lhd) + np.sum(chi2) + np.sum(chi2_iso)
#         return np.sum(lhd) + chi2[self.energy_bin] + chi2_iso[self.energy_bin]
#         #return lhd, w   
#
#     def likelihood_module(self, parameters):
#         m=Minuit(self.likelihood_constrained, parameters)
#         m.migrad()
#         return m
#
#     def minuit_module(self):
#         tol=1e-1
#         tol1=tol
#         tol2=tol
#         tol3=tol
#         m=Minuit(self.likelihood_constrained,np.ones(5))
#         m.limits=(-10, 10)
#         m.strategy=0
#         while True:
#             if True:
#                 #m=Minuit(self.likelihood_constrained, np.ones(5))
#                 m.limits=(-10, 10)
#                 m.strategy=0
#                 #m.limits=(0, None)
#                 m.tol=tol1
#                 print(f'tol1 = {m.tol}')
#                 #m.migrad()
#                 m.migrad()
#                 if m.valid==True:
#                     print(m)
#                     break;
#                 else:
#                     tol1*=1e+1
#             #if tol1>=1e+15:
#             #    m=Minuit(self.likelihood_constrained, np.ones(5))
#             #    m.limits=(-10, 10)
#             #    m.strategy=1
#             #    m.tol=tol2
#             #    print(f'tol2 = {m.tol}')
#             #    m.migrad()
#             #    if m.valid==True:
#             #        break;
#             #    else:
#             #        tol2*=1e+1
#             #if tol1>=1e+5 and tol2>=1e+10:
#             #    print(tol2)
#             #if tol1>=1e+5 and tol2>=1e+5:
#             #    m.strategy=0
#             #    m.tol=tol3
#             #    m.migrad()
#             #    if m.valie==True:
#             #        print(m.strategy)
#             #        break;
#             #    else:
#             #        tol3*=1e+1
#         return m.values[:]
#

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# Likelihood('F', 0).likelihood_constrained(np.ones(5))

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# plt.imshow(fits.open(f'./GC_analysis_sanghwan/Model/GC_ics_model{"F"}.fits')[0].data[0])

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# # 1) asymmetric error bar for bubble test 2) no PSC template for fitting test

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# params=np.hstack([pion_bremss_fitted, ics_fitted, GCE_fitted, bubble_fitted, isotropic_fitted])

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# import matplotlib.pyplot as plt
# from astropy.visualization import astropy_mpl_style, wcsaxes #wcsaxes manages coordinates of the plot.
# plt.style.use(astropy_mpl_style)
# from astropy.io import fits
# from astropy.utils.data import get_pkg_data_filename
# from astropy.wcs import WCS #This package expresses coordinates of the image file.
# import numpy as np
# from matplotlib.ticker import LogLocator, LogFormatter, AutoMinorLocator
# plt.style.use('default')
# ax=plt.subplot()
#
# ax.set_xscale('log')
# ax.set_yscale('log')
# ax.set_xlabel('E [GeV]')
# ax.set_ylabel(r'$E^2 \frac{dN}{dE}$[GeV$cm^{-2}$$s^{-1} sr^{-1}$]')
#
# ax.set_ylim(1e-8, 1e-4)
# ax.set_xlim(0.3, 500)
#
# ax.tick_params(axis='y', which='both', direction='in', left=True)
# ax.tick_params(axis='x', which='both', direction='in', bottom=True)
# ax.minorticks_on()
# ax.grid(True, which='Major', linestyle='-', linewidth=0.5)
#
# #m=minuit4
# #fitted=m.values[:]
# fitted_errors=np.zeros(120)
# fitted=params
#
# sr=0.214411*2
#
# ax.errorbar(E*1e-3, counts_per_exp*(E**2)/(1000*delta_E*sr) , yerr=counts_per_exp_err*(E**2)/(1000*delta_E*sr), linestyle='dotted', marker='.', elinewidth=2, capsize=4, capthick=2, label='Raw_data')
#
# #ax.plot(E*1e-3, psc*(E**2)/(1000*delta_E*sr))
#
# ax.errorbar(E*1e-3, psc*(E**2)/(1000*delta_E*sr), linestyle='', marker='.', elinewidth=2, capsize=4, capthick=2, label='psc', color='orange')
#
#
# ax.errorbar(E*1e-3, fitted[0:24]*(pion+bremss)*(E**2)/(1000*delta_E*sr), yerr=fitted_errors[0:24]*(pion+bremss)*(E**2)/(1000*delta_E*sr), linestyle='dotted', marker='.', elinewidth=2, capsize=4, capthick=2, label='pion+bremss', color='red')
# ax.errorbar(E*1e-3, fitted[24:48]*(ics)*(E**2)/(1000*delta_E*sr), yerr=fitted_errors[24:48]*(ics)*(E**2)/(1000*delta_E*sr), linestyle='dashdot', marker='.', elinewidth=2, capsize=4, capthick=2, label='ics', color='blue')
#
# ax.plot(E*1e-3, (pion+bremss)*(E**2)/(1000*delta_E*sr), linestyle='dotted', label='pion+bremss', color='red')
#
# ax.plot(E*1e-3, (ics)*(E**2)/(1000*delta_E*sr), linestyle='dashdot', label='ics', color='blue')
#
#
#
# ax.errorbar(E*1e-3,  fitted[48:72]*(GCE)*(E**2)/(1000*delta_E*sr), yerr=np.sqrt((fitted_errors[48:72]*GCE)**2)*(E**2)/(1000*delta_E*sr), linestyle='dashed', marker='.', elinewidth=2, capsize=4, capthick=2, label='GCE', color='black')
#
#
#
# ax.errorbar(E*1e-3, fitted[72:96]*(bubble)*(E**2)/(1000*delta_E*sr),yerr=fitted_errors[72:96]*(bubble)*(E**2)/(1000*delta_E*sr), linestyle='dashed', marker='.', elinewidth=2, capsize=4, capthick=2, label='bubble', color='purple')
# ax.plot(E*1e-3, bubble*(E**2)/(1000*delta_E*sr) )
#
#
# ax.errorbar(E*1e-3, fitted[96:120]*(isotropic)*(E**2)/(1000*delta_E*sr),yerr=fitted_errors[96:120]*(isotropic)*(E**2)/(1000*delta_E*sr), linestyle='dashed', marker='.', elinewidth=2, capsize=4, capthick=2, label='isotropic', color='green')
#
#
# summed=fitted[0:24]*(pion+bremss) + fitted[24:48]*(ics) + fitted[48:72]*(GCE) + fitted[72:96]*(bubble) + fitted[96:120]*(isotropic) + psc
#
# ax.errorbar(E*1e-3, summed*(E**2)/(1000*delta_E*sr), linestyle='dotted', marker='.', elinewidth=2, capsize=4, capthick=2, label='Model', color='black')
#
# ax.legend()

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# import matplotlib.pyplot as plt
# from astropy.visualization import astropy_mpl_style, wcsaxes #wcsaxes manages coordinates of the plot.
# plt.style.use(astropy_mpl_style)
# from astropy.io import fits
# from astropy.utils.data import get_pkg_data_filename
# from astropy.wcs import WCS #This package expresses coordinates of the image file.
# import numpy as np
# from scipy.interpolate import interp1d
# from iminuit import Minuit
# #Constraints interpolated function
# #Contains constraints for bubble and isotropic as well
# #For isotropic, from https://arxiv.org/pdf/1410.3696.pdf Table 3
#
# #Testing exposure per pixel
#
# #Analysis for model F
# #Correcting bubble template given from https://arxiv.org/pdf/1407.7905, Table 2
#
# model='F'
# def likelihood_constrained(parameter_set):
#     data=fits.open('./GC_analysis_sanghwan/GC_ccw_exposure_ccube_masked.fits')
#     psc_map=fits.open('./GC_analysis_sanghwan/Model/GC_psc_model.fits')
#     pion=fits.open(f'./GC_analysis_sanghwan/Model/GC_pion_model{model}.fits')
#     ics=fits.open(f'./GC_analysis_sanghwan/Model/GC_ics_model{model}.fits')
#     bremss=fits.open(f'./GC_analysis_sanghwan/Model/GC_bremss_model{model}.fits')
#     GCE=fits.open(f'./GC_analysis_sanghwan/Model/GC_GCE_model.fits')
#     bubble=fits.open(f'./GC_analysis_sanghwan/Model/GC_fermi_bubble_model.fits')
#     isotropic=fits.open(f'./GC_analysis_sanghwan/Model/GC_isotropic_model.fits')
#     #####################################
#     pion_bremss_param=parameter_set[0:24]
#     ics_param=parameter_set[24:48]
#     GCE_param=parameter_set[48:72]
#     bubble_param=parameter_set[72:96]
#     isotropic_param=parameter_set[96:120]
#     alpha_psc=5
#     f_psc=0.1
#     ######################################
#     #Main body for calculating Poissonian log-likelihood
#     #for k in range(0, energy_length, 1):
#     observed_pixel=data[0].data
#     observed_pixel=observed_pixel.astype('float32')
#     psc_pixel=psc_map[0].data
#     pion_bremss_pixel=pion[0].data + bremss[0].data
#     ics_pixel=ics[0].data
#     GCE_pixel=GCE[0].data
#     iso_pixel=isotropic[0].data
#     bubble_pixel=bubble[0].data
#
#     expected_pixel=np.zeros(np.shape(observed_pixel))
#     for i in range(0, len(observed_pixel), 1):
#         expected_pixel[i]= pion_bremss_param[i]*pion_bremss_pixel[i] + ics_param[i]*ics_pixel[i] + psc_pixel[i] + GCE_param[i]*GCE_pixel[i] + isotropic_param[i]*iso_pixel[i] + bubble_param[i]*bubble_pixel[i]
#     #expected_pixel= pion_bremss_param*pion_bremss_pixel + ics_param*ics_pixel + GCE_param*GCE_pixel + isotropic_param*iso_pixel + bubble_param*bubble_pixel
#     expected_pixel[expected_pixel==0] = 1e-10
#
#
#     #bgr_pixel=pion_bremss_pixel + ics_pixel
#     bgr_pixel = fits.open('./GC_analysis_sanghwan/Model/GC_bremss_modelP.fits')[0].data + fits.open('./GC_analysis_sanghwan/Model/GC_ics_modelP.fits')[0].data + fits.open('./GC_analysis_sanghwan/Model/GC_pion_modelP.fits')[0].data
#
#
#     bgr_pixel[bgr_pixel==0] = 1e-10
#     psc_pixel[psc_pixel==0] = 1e-10
#
#     w=1/( ( (psc_pixel)/(f_psc*bgr_pixel) )**alpha_psc +1 )
#     lhd=2*w*( expected_pixel - observed_pixel*np.log((expected_pixel)) )
#
#     #At the end of the likelihood expression, there's term with ln(observed!) but is omitted since it does not affect overall analysis because it is constant term.
#
#     # Preparing for the constraints
#
#     E_bounds=fits.open('./GC_analysis_sanghwan/GC_ccw_exposure_ccube_masked.fits')[1].data
#
#
#     E=np.zeros(len(E_bounds))
#     for i in range(0, 4, 1):
#         E[i] = 1e-3*(E_bounds[i][2] + E_bounds[i][1])/2
#     for i in range(4, len(E_bounds), 1):
#         E[i] = np.sqrt(E_bounds[i][2]*E_bounds[i][1]*1e-6)
#
#     delta_E=np.zeros(len(E_bounds))
#     for i in range(0, len(E_bounds), 1):
#         delta_E[i] = (E_bounds[i][2] - E_bounds[i][1])*1e-3
#
#     sr=0.214411*2
#
#     exp_cube=np.sqrt(fits.open('./GC_analysis_sanghwan/GC_ccw_exposure_expcube.fits')[0].data[:1]*fits.open('./GC_analysis_sanghwan/GC_ccw_exposure_expcube.fits')[0].data[1:])
#     for i in range(0, 4, 1):
#         exp_cube[i] = ( fits.open('./GC_analysis_sanghwan/GC_ccw_exposure_expcube.fits')[0].data[i] + fits.open('./GC_analysis_sanghwan/GC_ccw_exposure_expcube.fits')[0].data[i+1] )/2
#     for i in range(4, len(exp_cube), 1):
#         exp_cube[i] = np.sqrt(fits.open('./GC_analysis_sanghwan/GC_ccw_exposure_expcube.fits')[0].data[i]*fits.open('./GC_analysis_sanghwan/GC_ccw_exposure_expcube.fits')[0].data[i+1])
#
#
#     #Flux assiciated with constraints are considered with it's exposure per pixel values
#
#     file_name='./GC_analysis_sanghwan/Model/GC_isotropic_model.fits'
#     isotropic=np.zeros(len(fits.open(file_name)[0].data))
#     for i in range(0, len(fits.open(file_name)[0].data), 1):
#         a=exp_cube[i]
#         isotropic[i] = np.sum( fits.open(file_name)[0].data[i]/a )*isotropic_param[i]
#
#     file_name='./GC_analysis_sanghwan/Model/GC_fermi_bubble_model_no_convol.fits'
#     bubble=np.zeros(len(fits.open(file_name)[0].data))
#     for i in range(0, len(fits.open(file_name)[0].data), 1):
#         a=exp_cube[i]
#         bubble[i] = np.sum( fits.open(file_name)[0].data[i]/a )*bubble_param[i]
#
#
#     bubble_sed=(E**2)*bubble/(1000*delta_E*sr)
#
#     constraints=np.loadtxt('./GC_analysis_sanghwan/Model/bubble_constraints_asymmetric.txt')
#     constraints_energy=constraints[0:, 0]
#     constraints_flux=constraints[0:, 1]
#     constraints_lower_error=constraints[0:, 2]
#     constraints_upper_error=constraints[0:, 3]
#
#
#     fluxint=interp1d(constraints_energy, constraints_flux, fill_value='extrapolate',  kind='cubic')
#     lower_errint=interp1d(constraints_energy, constraints_lower_error, fill_value='extrapolate', kind='cubic')
#     upper_errint=interp1d(constraints_energy, constraints_upper_error, fill_value='extrapolate', kind='cubic')
#
#     flux_data=fluxint(E*1e-3)
#     lower_error_data=lower_errint(E*1e-3)
#     upper_error_data=upper_errint(E*1e-3)
#
#     flux_data[flux_data<=0] = 1e-20
#     lower_error_data[lower_error_data<=0] = 1e-20
#     upper_error_data[upper_error_data<=0] = 1e-20
#
#     chi2=np.zeros(len(bubble_sed))
#     for i in range(len(chi2)):
#         larger_error=max([upper_error_data[i], lower_error_data[i]])
#         if flux_data[i] > bubble_sed[i]:
#             chi2[i] = ((bubble_sed[i] - flux_data[i])/upper_error_data[i])**2
#         if flux_data[i] < bubble_sed[i]:
#             chi2[i] = ((bubble_sed[i] - flux_data[i])/lower_error_data[i])**2
#         if flux_data[i] == bubble_sed[i]:
#             chi2[i] = ((bubble_sed[i] - flux_data[i])/larger_error)**2
#         #chi2[i]=((bubble_sed[i] - flux_data[i])/larger_error)**2
#
#
#
#     iso_flux=isotropic/(sr*delta_E*1e-3)
#
#     iso_constraints=np.loadtxt('./GC_analysis_sanghwan/Model/iso_constraints_conservative_sys.txt')
#     iso_constraints_energy=iso_constraints[0:, 0]
#     iso_constraints_flux=iso_constraints[0:, 1]
#     iso_constraints_full_error=iso_constraints[0:, 2]
#
#     fluxint=interp1d(iso_constraints_energy, iso_constraints_flux, fill_value="extrapolate")
#     errint=interp1d(iso_constraints_energy, iso_constraints_full_error, fill_value="extrapolate")    
#
#     iso_constraints_flux=fluxint(E*1e-3)
#     iso_constraints_flux_err=errint(E*1e-3)
#
#     iso_constraints_flux[iso_constraints_flux <= 0] = 1e-20
#     iso_constraints_flux_err[iso_constraints_flux_err <= 0] = 1e-20
#
#     chi2_iso=((iso_flux - iso_constraints_flux)**2/iso_constraints_flux_err**2) 
#
#     return np.sum(lhd) + np.sum(chi2) + np.sum(chi2_iso)
#

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# likelihood_constrained(params)

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# m.fval

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# m=Minuit(likelihood_constrained, params)
# m.limits=(-10, 10)
# m.migrad()

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# import matplotlib.pyplot as plt
# from astropy.visualization import astropy_mpl_style, wcsaxes #wcsaxes manages coordinates of the plot.
# plt.style.use(astropy_mpl_style)
# from astropy.io import fits
# from astropy.utils.data import get_pkg_data_filename
# from astropy.wcs import WCS #This package expresses coordinates of the image file.
# import numpy as np
# from matplotlib.ticker import LogLocator, LogFormatter, AutoMinorLocator
# plt.style.use('default')
# ax=plt.subplot()
#
# ax.set_xscale('log')
# ax.set_yscale('log')
# ax.set_xlabel('E [GeV]')
# ax.set_ylabel(r'$E^2 \frac{dN}{dE}$[GeV$cm^{-2}$$s^{-1} sr^{-1}$]')
#
# ax.set_ylim(1e-8, 1e-4)
# ax.set_xlim(0.3, 500)
#
# ax.tick_params(axis='y', which='both', direction='in', left=True)
# ax.tick_params(axis='x', which='both', direction='in', bottom=True)
# ax.minorticks_on()
# ax.grid(True, which='Major', linestyle='-', linewidth=0.5)
#
# #m=minuit4
# fitted=m.values[:]
# fitted_errors=m.errors[:]
# fitted=params
#
# sr=0.214411*2
#
# ax.errorbar(E*1e-3, counts_per_exp*(E**2)/(1000*delta_E*sr) , yerr=counts_per_exp_err*(E**2)/(1000*delta_E*sr), linestyle='dotted', marker='.', elinewidth=2, capsize=4, capthick=2, label='Raw_data')
#
# #ax.plot(E*1e-3, psc*(E**2)/(1000*delta_E*sr))
#
# ax.errorbar(E*1e-3, psc*(E**2)/(1000*delta_E*sr), linestyle='', marker='.', elinewidth=2, capsize=4, capthick=2, label='psc', color='orange')
#
#
# ax.errorbar(E*1e-3, fitted[0:24]*(pion+bremss)*(E**2)/(1000*delta_E*sr), yerr=fitted_errors[0:24]*(pion+bremss)*(E**2)/(1000*delta_E*sr), linestyle='dotted', marker='.', elinewidth=2, capsize=4, capthick=2, label='pion+bremss', color='red')
# ax.errorbar(E*1e-3, fitted[24:48]*(ics)*(E**2)/(1000*delta_E*sr), yerr=fitted_errors[24:48]*(ics)*(E**2)/(1000*delta_E*sr), linestyle='dashdot', marker='.', elinewidth=2, capsize=4, capthick=2, label='ics', color='blue')
#
# ax.plot(E*1e-3, (pion+bremss)*(E**2)/(1000*delta_E*sr), linestyle='dotted', label='pion+bremss', color='red')
#
# ax.plot(E*1e-3, (ics)*(E**2)/(1000*delta_E*sr), linestyle='dashdot', label='ics', color='blue')
#
#
#
# ax.errorbar(E*1e-3,  fitted[48:72]*(GCE)*(E**2)/(1000*delta_E*sr), yerr=np.sqrt((fitted_errors[48:72]*GCE)**2)*(E**2)/(1000*delta_E*sr), linestyle='dashed', marker='.', elinewidth=2, capsize=4, capthick=2, label='GCE', color='black')
#
#
#
# ax.errorbar(E*1e-3, fitted[72:96]*(bubble)*(E**2)/(1000*delta_E*sr),yerr=fitted_errors[72:96]*(bubble)*(E**2)/(1000*delta_E*sr), linestyle='dashed', marker='.', elinewidth=2, capsize=4, capthick=2, label='bubble', color='purple')
# ax.plot(E*1e-3, bubble*(E**2)/(1000*delta_E*sr) )
#
#
# ax.errorbar(E*1e-3, fitted[96:120]*(isotropic)*(E**2)/(1000*delta_E*sr),yerr=fitted_errors[96:120]*(isotropic)*(E**2)/(1000*delta_E*sr), linestyle='dashed', marker='.', elinewidth=2, capsize=4, capthick=2, label='isotropic', color='green')
#
#
# summed=fitted[0:24]*(pion+bremss) + fitted[24:48]*(ics) + fitted[48:72]*(GCE) + fitted[72:96]*(bubble) + fitted[96:120]*(isotropic) + psc
#
# ax.errorbar(E*1e-3, summed*(E**2)/(1000*delta_E*sr), linestyle='dotted', marker='.', elinewidth=2, capsize=4, capthick=2, label='Model', color='black')
#
# ax.legend()

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# import matplotlib.pyplot as plt
# from astropy.visualization import astropy_mpl_style, wcsaxes #wcsaxes manages coordinates of the plot.
# plt.style.use(astropy_mpl_style)
# from astropy.io import fits
# from astropy.utils.data import get_pkg_data_filename
# from astropy.wcs import WCS #This package expresses coordinates of the image file.
# import numpy as np
# from scipy.interpolate import interp1d
# from iminuit import Minuit
# #Constraints interpolated function
# #Contains constraints for bubble and isotropic as well
# #For isotropic, from https://arxiv.org/pdf/1410.3696.pdf Table 3
#
# #Testing exposure per pixel
#
# #Analysis for model F
# #Correcting bubble template given from https://arxiv.org/pdf/1407.7905, Table 2
#
# class Likelihood:
#     def __init__(self, model, energy_bin):
#         self.model=model
#         self.energy_bin=energy_bin
#     def likelihood_constrained(self, parameter_set):
#         data=fits.open('./GC_analysis_sanghwan/GC_all_time_ccube_masked.fits')
#         psc_map=fits.open('./GC_analysis_sanghwan/Model/GC_all_time_psc_model.fits')
#         pion=fits.open(f'./GC_analysis_sanghwan/Model/GC_all_time_pion_model{self.model}.fits')
#         ics=fits.open(f'./GC_analysis_sanghwan/Model/GC_all_time_ics_model{self.model}.fits')
#         bremss=fits.open(f'./GC_analysis_sanghwan/Model/GC_all_time_bremss_model{self.model}.fits')
#         GCE=fits.open(f'./GC_analysis_sanghwan/Model/GC_all_time_GCE_model.fits')
#         bubble=fits.open(f'./GC_analysis_sanghwan/Model/GC_all_time_fermi_bubble_model.fits')
#         isotropic=fits.open(f'./GC_analysis_sanghwan/Model/GC_all_time_isotropic_model.fits')
#         #####################################
#         pion_bremss_param=parameter_set[0]
#         ics_param=parameter_set[1]
#         GCE_param=parameter_set[2]
#         bubble_param=parameter_set[3]
#         isotropic_param=parameter_set[4]
#         alpha_psc=5
#         #alpha_psc=0.5
#         f_psc=0.2
#         #f_psc=0.1
#
#         ######################################
#         #Main body for calculating Poissonian log-likelihood
#         #for k in range(0, energy_length, 1):
#         observed_pixel=data[0].data[self.energy_bin]
#         observed_pixel=observed_pixel.astype('float32')
#         psc_pixel=psc_map[0].data[self.energy_bin]
#         pion_bremss_pixel=pion[0].data[self.energy_bin] + bremss[0].data[self.energy_bin]
#         ics_pixel=ics[0].data[self.energy_bin]
#         GCE_pixel=GCE[0].data[self.energy_bin]
#         iso_pixel=isotropic[0].data[self.energy_bin]
#         bubble_pixel=bubble[0].data[self.energy_bin]
#         expected_pixel= pion_bremss_param*pion_bremss_pixel + ics_param*ics_pixel + psc_pixel + GCE_param*GCE_pixel + isotropic_param*iso_pixel + bubble_param*bubble_pixel
#         #expected_pixel= pion_bremss_param*pion_bremss_pixel + ics_param*ics_pixel + GCE_param*GCE_pixel + isotropic_param*iso_pixel + bubble_param*bubble_pixel
#         #expected_pixel[expected_pixel==0] = 1e-20
#
#
#         #bgr_pixel=pion_bremss_pixel + ics_pixel
#         bgr_pixel = fits.open('./GC_analysis_sanghwan/Model/GC_all_time_bremss_modelP.fits')[0].data[self.energy_bin] + fits.open('./GC_analysis_sanghwan/Model/GC_all_time_ics_modelP.fits')[0].data[self.energy_bin] + fits.open('./GC_analysis_sanghwan/Model/GC_all_time_pion_modelP.fits')[0].data[self.energy_bin]
#
#
#         #bgr_pixel[bgr_pixel==0] = 1e-20
#         #psc_pixel[psc_pixel==0] = 1e-20
#
#         w=1/( ( (psc_pixel)/(f_psc*bgr_pixel) )**alpha_psc +1 )
#         #w[np.isnan(w)]=0
#         lhd=2*w*( expected_pixel - observed_pixel*np.log((expected_pixel)) )
#         lhd[np.isnan(lhd)]=0
#
#         #At the end of the likelihood expression, there's term with ln(observed!) but is omitted since it does not affect overall analysis because it is constant term.
#
#         # Preparing for the constraints
#
#         E_bounds=fits.open('./GC_analysis_sanghwan/GC_all_time_ccube_masked.fits')[1].data
#
#
#         E=np.zeros(len(E_bounds))
#         for i in range(0, 4, 1):
#             E[i] = 1e-3*(E_bounds[i][2] + E_bounds[i][1])/2
#         for i in range(4, len(E_bounds), 1):
#             E[i] = np.sqrt(E_bounds[i][2]*E_bounds[i][1]*1e-6)
#
#         delta_E=np.zeros(len(E_bounds))
#         for i in range(0, len(E_bounds), 1):
#             delta_E[i] = (E_bounds[i][2] - E_bounds[i][1])*1e-3
#
#         sr=0.214411*2
#
#         exp_cube=np.sqrt(fits.open('./GC_analysis_sanghwan/GC_all_time_expcube.fits')[0].data[:1]*fits.open('./GC_analysis_sanghwan/GC_all_time_expcube.fits')[0].data[1:])
#         for i in range(0, 4, 1):
#             exp_cube[i] = ( fits.open('./GC_analysis_sanghwan/GC_all_time_expcube.fits')[0].data[i] + fits.open('./GC_analysis_sanghwan/GC_all_time_expcube.fits')[0].data[i+1] )/2
#         for i in range(4, len(exp_cube), 1):
#             exp_cube[i] = np.sqrt(fits.open('./GC_analysis_sanghwan/GC_all_time_expcube.fits')[0].data[i]*fits.open('./GC_analysis_sanghwan/GC_all_time_expcube.fits')[0].data[i+1])
#
#
#         #Flux assiciated with constraints are considered with it's exposure per pixel values
#
#         file_name='./GC_analysis_sanghwan/Model/GC_all_time_isotropic_model.fits'
#         isotropic=np.zeros(len(fits.open(file_name)[0].data))
#         for i in range(0, len(fits.open(file_name)[0].data), 1):
#             a=exp_cube[i]
#             isotropic[i] = np.sum( fits.open(file_name)[0].data[i]/a )#*isotropic_param
#
#         isotropic[self.energy_bin] *= isotropic_param
#
#         file_name='./GC_analysis_sanghwan/Model/GC_all_time_fermi_bubble_model.fits'
#         bubble=np.zeros(len(fits.open(file_name)[0].data))
#         for i in range(0, len(fits.open(file_name)[0].data), 1):
#             a=exp_cube[i]
#             bubble[i] = np.sum( fits.open(file_name)[0].data[i]/a )#*bubble_param
#
#         bubble[self.energy_bin] *= bubble_param
#
#         bubble_sed=bubble/(1e-3*delta_E*sr)
#
#         constraints=np.loadtxt('./GC_analysis_sanghwan/Model/bubble_constraints_asymmetric.txt')
#         constraints_energy=constraints[0:, 0]
#         constraints_flux=constraints[0:, 1]/constraints[0:, 0]**2
#         constraints_lower_error=constraints[0:, 2]/constraints[0:, 0]**2
#         constraints_upper_error=constraints[0:, 3]/constraints[0:, 0]**2
#
#         flux_data=constraints_flux
#         lower_error_data=constraints_lower_error
#         upper_error_data=constraints_upper_error
#         #chi2=np.zeros(len(bubble_sed))
#
#         #for i in range(len(bubble_sed)):
#         chi2=0
#         i=self.energy_bin
#         larger_error=max([upper_error_data[i], lower_error_data[i]])
#         if flux_data[i] < bubble_sed[i]:
#             chi2 = ((bubble_sed[i] - flux_data[i])/upper_error_data[i])**2
#         if flux_data[i] > bubble_sed[i]:
#             chi2 = ((bubble_sed[i] - flux_data[i])/lower_error_data[i])**2
#         if flux_data[i] == bubble_sed[i]:
#             chi2 = ((bubble_sed[i] - flux_data[i])/larger_error)**2
#         #print(chi2)
#         iso_flux=isotropic/(sr*delta_E*1e-3)
#
#         iso_constraints=np.loadtxt('./GC_analysis_sanghwan/Model/iso_constraints_conservative_sys.txt')
#         iso_constraints_energy=iso_constraints[0:, 0]
#         iso_constraints_flux=iso_constraints[0:, 1]
#         iso_constraints_full_error=iso_constraints[0:, 2]
#
#
#         fluxint=interp1d(iso_constraints_energy, iso_constraints_flux, fill_value="extrapolate", kind='quadratic')
#         errint=interp1d(iso_constraints_energy, iso_constraints_full_error, fill_value="extrapolate", kind='quadratic')    
#
#         iso_constraints_flux=fluxint(E*1e-3)
#         iso_constraints_flux_err=errint(E*1e-3)
#
#         iso_constraints_flux[iso_constraints_flux <= 0] = 1e-20
#         iso_constraints_flux_err[iso_constraints_flux_err <= 0] = 1e-20
#
#         chi2_iso=((iso_flux[self.energy_bin] - iso_constraints_flux[self.energy_bin])**2/iso_constraints_flux_err[self.energy_bin]**2) 
#
#         return np.sum(lhd) + chi2 + chi2_iso
#
#     def likelihood_module(self):
#         return self.likelihood_constrained(np.ones(5))
#
#     def minuit_module(self):
#         tol=1e+1
#         tol1=tol
#         while True:
#             if True:
#                 m=Minuit(self.likelihood_constrained, np.ones(5))
#                 m.errors[:]=np.ones(5)
#                 m.errordef=0.5
#                 m.limits=(-5, 5)
#                 m.strategy=2
#                 m.tol=tol1
#                 print(f'tol1 = {m.tol}')
#                 m.migrad()
#                 #m.simplex()
#                 if m.valid==True:
#                     print(m)
#                     break;
#                 else:
#                     tol1*=1e+1
#         return m.values[:], m.errors[:]
#
# class Parameter_estimation():
#     def __init__(self, model):
#         self.model=model
#     def minuit_run(self):
#         pion_bremss_fitted=[]
#         ics_fitted=[]
#         GCE_fitted=[]
#         bubble_fitted=[]
#         isotropic_fitted=[]
#
#         pion_bremss_fitted_errors=[]
#         ics_fitted_errors=[]
#         GCE_fitted_errors=[]
#         bubble_fitted_errors=[]
#         isotropic_fitted_errors=[]     
#         for i in range(0, 24, 1):
#             results=Likelihood(self.model, i).minuit_module()
#             pion_bremss_fitted.append(results[0][0])
#             ics_fitted.append(results[0][1])
#             GCE_fitted.append(results[0][2])
#             bubble_fitted.append(results[0][3])
#             isotropic_fitted.append(results[0][4])
#
#             pion_bremss_fitted_errors.append(results[1][0])
#             ics_fitted_errors.append(results[1][1])
#             GCE_fitted_errors.append(results[1][2])
#             bubble_fitted_errors.append(results[1][3])
#             isotropic_fitted_errors.append(results[1][4])
#         return np.hstack([pion_bremss_fitted, ics_fitted, GCE_fitted, bubble_fitted, isotropic_fitted]), np.hstack([pion_bremss_fitted_errors, ics_fitted_errors, GCE_fitted_errors, bubble_fitted_errors, isotropic_fitted_errors])

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# %%time
# params, param_errors=Parameter_estimation('F').minuit_run()

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# import matplotlib.pyplot as plt
# from astropy.visualization import astropy_mpl_style, wcsaxes #wcsaxes manages coordinates of the plot.
# plt.style.use(astropy_mpl_style)
# from astropy.io import fits
# from astropy.utils.data import get_pkg_data_filename
# from astropy.wcs import WCS #This package expresses coordinates of the image file.
# import numpy as np
# from matplotlib.ticker import LogLocator, LogFormatter, AutoMinorLocator
# plt.style.use('default')
# ax=plt.subplot()
#
# ax.set_xscale('log')
# ax.set_yscale('log')
# ax.set_xlabel('E [GeV]')
# ax.set_ylabel(r'$E^2 \frac{dN}{dE}$[GeV$cm^{-2}$$s^{-1} sr^{-1}$]')
#
# ax.set_ylim(1e-8, 1e-4)
# ax.set_xlim(0.3, 500)
#
# ax.tick_params(axis='y', which='both', direction='in', left=True)
# ax.tick_params(axis='x', which='both', direction='in', bottom=True)
# ax.minorticks_on()
# ax.grid(True, which='Major', linestyle='-', linewidth=0.5)
#
# #m=minuit4
# fitted=m.values[:]
# fitted_errors=m.errors[:]
# fitted=params
#
# sr=0.214411*2
#
# ax.errorbar(E*1e-3, counts_per_exp*(E**2)/(1000*delta_E*sr) , yerr=counts_per_exp_err*(E**2)/(1000*delta_E*sr), linestyle='dotted', marker='.', elinewidth=2, capsize=4, capthick=2, label='Raw_data')
#
# #ax.plot(E*1e-3, psc*(E**2)/(1000*delta_E*sr))
#
# ax.errorbar(E*1e-3, psc*(E**2)/(1000*delta_E*sr), linestyle='', marker='.', elinewidth=2, capsize=4, capthick=2, label='psc', color='orange')
#
#
# ax.errorbar(E*1e-3, fitted[0:24]*(pion+bremss)*(E**2)/(1000*delta_E*sr), yerr=fitted_errors[0:24]*(pion+bremss)*(E**2)/(1000*delta_E*sr), linestyle='dotted', marker='.', elinewidth=2, capsize=4, capthick=2, label='pion+bremss', color='red')
# ax.errorbar(E*1e-3, fitted[24:48]*(ics)*(E**2)/(1000*delta_E*sr), yerr=fitted_errors[24:48]*(ics)*(E**2)/(1000*delta_E*sr), linestyle='dashdot', marker='.', elinewidth=2, capsize=4, capthick=2, label='ics', color='blue')
#
# ax.plot(E*1e-3, (pion+bremss)*(E**2)/(1000*delta_E*sr), linestyle='dotted', label='pion+bremss', color='red')
#
# ax.plot(E*1e-3, (ics)*(E**2)/(1000*delta_E*sr), linestyle='dashdot', label='ics', color='blue')
#
#
#
# ax.errorbar(E*1e-3,  fitted[48:72]*(GCE)*(E**2)/(1000*delta_E*sr), yerr=np.sqrt((fitted_errors[48:72]*GCE)**2)*(E**2)/(1000*delta_E*sr), linestyle='dashed', marker='.', elinewidth=2, capsize=4, capthick=2, label='GCE', color='black')
#
#
#
# ax.errorbar(E*1e-3, fitted[72:96]*(bubble)*(E**2)/(1000*delta_E*sr),yerr=fitted_errors[72:96]*(bubble)*(E**2)/(1000*delta_E*sr), linestyle='dashed', marker='.', elinewidth=2, capsize=4, capthick=2, label='bubble', color='purple')
# ax.plot(E*1e-3, bubble*(E**2)/(1000*delta_E*sr) )
#
#
# ax.errorbar(E*1e-3, fitted[96:120]*(isotropic)*(E**2)/(1000*delta_E*sr),yerr=fitted_errors[96:120]*(isotropic)*(E**2)/(1000*delta_E*sr), linestyle='dashed', marker='.', elinewidth=2, capsize=4, capthick=2, label='isotropic', color='green')
#
#
# summed=fitted[0:24]*(pion+bremss) + fitted[24:48]*(ics) + fitted[48:72]*(GCE) + fitted[72:96]*(bubble) + fitted[96:120]*(isotropic) + psc
#
# ax.errorbar(E*1e-3, summed*(E**2)/(1000*delta_E*sr), linestyle='dotted', marker='.', elinewidth=2, capsize=4, capthick=2, label='Model', color='black')
#
# ax.legend()

In [ ]:
# ==========================================================================
# [DISABLED v3] exploratory/legacy — not part of main pipeline
# Original content preserved below; uncomment manually to inspect.
# ==========================================================================
# #Weak masking test

## 🔍 Validation cells (after main loop completes)

Run these after at least one `GCE_model_{M}_12yr_cholis.dat` file exists.
Each cell is independent — you can run them in any order.

The cells below are adapted from the exploratory cells in Sanghwan's original
notebook (formerly Cells 42-95 before v3 disabled them), cleaned up to work
with haebarg's environment and with v3 gtmodel output layout.


## 🎛️ Validation setup

**모델 선택은 여기 한 곳에서만 바꾸세요** — 아래 V1/V2/V3/V6이 모두 이 값을 따라갑니다.
(V4는 multi-model envelope이라 모든 `.dat`을 자동으로 로드합니다.)


In [ ]:
# ============================================================================
# V0_SETUP. Central model selection for the validation cells
# ----------------------------------------------------------------------------
# Change SELECTED_MODEL below, then re-run V1 -> V2 -> V3 -> V6.
# V4 and V5 are model-independent (V4 uses all .dat, V5 loads PPPC4 once).
# ============================================================================
import os as _os
import glob as _glob
import time as _time

SELECTED_MODEL = "X"     # ← change this to inspect a different model

def list_available_models(verbose=True):
    """Return a list of model labels that have a completed .dat on disk.

    Sorted by modification time (newest first). Useful for picking a
    freshly-finished model to validate.
    """
    dats = _glob.glob('./GCE_model_*_12yr_cholis.dat')
    info = []
    for p in dats:
        lab = _os.path.basename(p).replace('GCE_model_', '').replace('_12yr_cholis.dat', '')
        st = _os.stat(p)
        info.append((lab, p, st.st_size, st.st_mtime))
    info.sort(key=lambda r: -r[3])     # newest first
    if verbose:
        if not info:
            print("[info] No completed .dat files in current directory.")
            print("       Run the main loop first (Cell 37 subprocess wrapper).")
        else:
            print(f"{'model':<10} {'size':>8}  {'modified':<20}  path")
            print('-' * 78)
            for lab, p, size, mtime in info:
                ts = _time.strftime('%Y-%m-%d %H:%M', _time.localtime(mtime))
                print(f"{lab:<10} {size:>8} B  {ts:<20}  {p}")
            print()
            print(f"  Set SELECTED_MODEL above to one of: "
                  + ", ".join(repr(l) for l, *_ in info[:20]))
    return [lab for lab, *_ in info]

# Validate user's selection and list available options
_avail = list_available_models(verbose=True)

if SELECTED_MODEL not in _avail:
    if not _avail:
        print(f"\n⚠ No models available yet. SELECTED_MODEL = {SELECTED_MODEL!r} will fail.")
        print(f"  Run the main loop (Cell 37) to produce at least one .dat file first.")
    else:
        print(f"\n⚠ SELECTED_MODEL = {SELECTED_MODEL!r} but no such .dat file exists.")
        print(f"  Pick from available: {_avail}")
        print(f"  Falling back to SELECTED_MODEL = {_avail[0]!r} (newest)")
        SELECTED_MODEL = _avail[0]
else:
    print(f"\n✓ SELECTED_MODEL = {SELECTED_MODEL!r} -> will be used by V1, V2, V3, V6")


In [ ]:
# ============================================================================
# V1. Load single-model results + fit parameters from disk
# ----------------------------------------------------------------------------
# Reads SELECTED_MODEL from V0_SETUP. Defaults to "X" if V0_SETUP not run.
# Output variables used by V2, V3, V6:
#   _model, E, delta_E, model_GCE_flux, model_GCE_std, model_GCE_lo, model_GCE_hi
#   pion, bremss, ics, GCE, bubble, isotropic, counts_per_exp, counts_per_exp_err
# ============================================================================
VALIDATE_MODEL = SELECTED_MODEL if 'SELECTED_MODEL' in dir() else "X"

import numpy as _np
from astropy.io import fits as _fits
from astropy.wcs import WCS as _WCS

front = '_front'
_model = VALIDATE_MODEL
print(f"=== V1: loading Model {_model!r} ===\n")

# Load E bins from CCUBE (same as Cell 26 does)
_E_bounds = _fits.open(f'./GC_analysis_sanghwan/GC_ccube_12yr{front}_clean.fits')[1].data
E       = _np.array([_np.sqrt(b[2]*b[1]*1e-6)*1e-3 for b in _E_bounds])      # GeV
delta_E = _np.array([(b[2] - b[1])*1e-6 for b in _E_bounds])                 # GeV

# Load the final .dat (5 columns: E, flux, std, lower_1sigma, upper_1sigma)
_dat_path = f'./GCE_model_{_model}_12yr_cholis.dat'
if not _os.path.exists(_dat_path):
    print(f"[ERROR] {_dat_path} not found.")
    print(f"        Available models: {_avail if '_avail' in dir() else '(run V0_SETUP to list)'}")
else:
    _dat = _np.loadtxt(_dat_path)
    print(f"Loaded {_dat_path}")
    print(f"  shape: {_dat.shape}  (expect (14, 5))")
    print(f"  E range:    {_dat[:,0].min():.3f} — {_dat[:,0].max():.3f} GeV")
    print(f"  Flux range: {_dat[:,1].min():.2e} — {_dat[:,1].max():.2e} GeV/cm²/s/sr")
    model_dat = _dat
    model_E        = _dat[:, 0]
    model_GCE_flux = _dat[:, 1]
    model_GCE_std  = _dat[:, 2]
    model_GCE_lo   = _dat[:, 3]
    model_GCE_hi   = _dat[:, 4]

    # Load CCUBE, build exp cube and flux-per-bin arrays for all components
    _raw_map = _fits.open(f'./GC_analysis_sanghwan/GC_ccube_12yr{front}_clean.fits')
    _w = _WCS(_raw_map[0].header).dropaxis(2)
    _W, _H = _np.shape(_raw_map[0].data[0])
    _srp = _np.zeros([_W, _H])
    for _i in range(_H):
        for _j in range(_W):
            _l, _b = _w.wcs_pix2world(_j, _i, 0)
            _srp[_i, _j] = _np.radians(0.1) * _np.radians(0.1) * _np.cos(_np.radians(_b))
    _exp = _fits.open(f'./GC_analysis_sanghwan/GC_expcube_center_12yr{front}_clean.fits')[0].data[:, 100:500, 100:500] * _srp[100:500, 100:500]
    _disk = _np.load('./GC_analysis_sanghwan/Model/GC_disk_mask_60x60_definitions.npy')[100:500, 100:500]

    def _comp_flux(fname):
        out = _np.zeros(len(E))
        if not _os.path.exists(fname):
            print(f"  [warn] missing {fname}; returning zeros")
            return out
        d = _fits.open(fname)[0].data
        for i in range(len(E)):
            out[i] = _np.sum(_disk * (d[i][100:500, 100:500] / _exp[i])) / _np.sum(_disk)
        return out

    pion      = _comp_flux(f'./GC_analysis_sanghwan/GC_pion_model{_model}_12yr{front}_clean_no_convol.fits')
    bremss    = _comp_flux(f'./GC_analysis_sanghwan/GC_bremss_model{_model}_12yr{front}_clean_no_convol.fits')
    ics       = _comp_flux(f'./GC_analysis_sanghwan/GC_ics_model{_model}_12yr{front}_clean_no_convol.fits')
    GCE       = _comp_flux(f'./GC_analysis_sanghwan/GC_GCE_model_12yr{front}_clean_no_convol.fits')
    bubble    = _comp_flux(f'./GC_analysis_sanghwan/GC_fermi_bubble_model_12yr{front}_clean_no_convol.fits')
    isotropic = _comp_flux(f'./GC_analysis_sanghwan/GC_isotropic_model_12yr{front}_clean_no_convol.fits')

    _ccube = _fits.open(f'./GC_analysis_sanghwan/GC_ccube_12yr{front}_clean.fits')[0].data
    counts_per_exp     = _np.zeros(len(E))
    counts_per_exp_err = _np.zeros(len(E))
    for i in range(len(E)):
        counts_per_exp[i]     = _np.sum(_disk * (_ccube[i][100:500, 100:500] / _exp[i])) / _np.sum(_disk)
        counts_per_exp_err[i] = _np.sqrt(_np.sum((_np.sqrt(_disk*_ccube[i][100:500, 100:500])/_exp[i])**2)) / _np.sum(_disk)

    n = len(E)
    fitted_GCE = model_GCE_flux * delta_E / (GCE * E**2)
    print(f"\n[OK] V1 finished for Model {_model}")
    print(f"  Ready variables: model_GCE_*, pion, bremss, ics, GCE, bubble, isotropic,")
    print(f"                   counts_per_exp, counts_per_exp_err")


In [ ]:
# ============================================================================
# V2. Single-model GCE SED decomposition (v3.13)
# ----------------------------------------------------------------------------
# Shows all 5 background components (π⁰+bremss, ICS, Fermi bubble, isotropic,
# GCE) per energy bin for the selected model, fitted or raw.
#
# Enhancements in v3.13:
#   - Per-bin fit coefficient table (all 5 c values, 14 bins)
#   - Coefficient-coefficient correlation within each bin
#   - Skewed posterior diagnostic (which bins have best-fit outside 16-84%)
#   - Scenario hint: based on c distributions, suggest interpretation
#
# Requires:
#   V0_SETUP: SELECTED_MODEL loaded
#   V1:       E, delta_E, pion, bremss, ics, model_GCE_flux etc. loaded
# ============================================================================
import os as _os
import numpy as _np
import matplotlib.pyplot as plt

_npz_path = f'./GCE_model_{_model}_12yr_cholis_fit.npz'
_have_fits = _os.path.exists(_npz_path)
_n = len(E)

if _have_fits:
    _npz = _np.load(_npz_path)
    _fp = _npz['fitted_params']
    # flat array layout: [c_pion_bremss, c_ics, c_gce, c_bubble, c_iso] × n_bins
    c_pion  = _fp[0*_n:1*_n]
    c_ics   = _fp[1*_n:2*_n]
    c_gce   = _fp[2*_n:3*_n]
    c_bub   = _fp[3*_n:4*_n]
    c_iso   = _fp[4*_n:5*_n]
    # Lower/upper 1σ if present
    _fp_lo = _npz.get('fitted_params_lower')
    _fp_hi = _npz.get('fitted_params_upper')
    if _fp_lo is not None and _fp_hi is not None:
        _have_bands = True
        c_pion_lo, c_pion_hi = _fp_lo[0*_n:1*_n], _fp_hi[0*_n:1*_n]
        c_ics_lo,  c_ics_hi  = _fp_lo[1*_n:2*_n], _fp_hi[1*_n:2*_n]
        c_gce_lo,  c_gce_hi  = _fp_lo[2*_n:3*_n], _fp_hi[2*_n:3*_n]
        c_bub_lo,  c_bub_hi  = _fp_lo[3*_n:4*_n], _fp_hi[3*_n:4*_n]
        c_iso_lo,  c_iso_hi  = _fp_lo[4*_n:5*_n], _fp_hi[4*_n:5*_n]
    else:
        _have_bands = False
    print(f"Loaded fit coefficients from {_npz_path}")
    print(f"  c_pion_bremss: mean={c_pion.mean():.3f}  range={c_pion.min():.3f}–{c_pion.max():.3f}  std={c_pion.std():.3f}")
    print(f"  c_ics:         mean={c_ics.mean():.3f}   range={c_ics.min():.3f}–{c_ics.max():.3f}  std={c_ics.std():.3f}")
    print(f"  c_gce:         mean={c_gce.mean():.3f}   range={c_gce.min():.3f}–{c_gce.max():.3f}  std={c_gce.std():.3f}")
    print(f"  c_bubble:      mean={c_bub.mean():.3f}   range={c_bub.min():.3f}–{c_bub.max():.3f}  std={c_bub.std():.3f}")
    print(f"  c_isotropic:   mean={c_iso.mean():.3f}   range={c_iso.min():.3f}–{c_iso.max():.3f}  std={c_iso.std():.3f}")

    # ---- Per-bin table ----
    print()
    print(f"  {'bin':>3} {'E[GeV]':>8} {'c_π+br':>8} {'c_ics':>8} {'c_gce':>8} {'c_bub':>8} {'c_iso':>8}")
    for _i in range(_n):
        print(f"  {_i:>3} {E[_i]:>8.3f} "
              f"{c_pion[_i]:>8.3f} {c_ics[_i]:>8.3f} {c_gce[_i]:>8.3f} "
              f"{c_bub[_i]:>8.3f} {c_iso[_i]:>8.3f}")

    # ---- Bin-by-bin correlation (among c values): are large c_gas matched by small c_gce? ----
    print()
    print(f"  Bin-wise correlation between coefficients (across 14 bins):")
    _corr_pair = [
        ('c_π+br vs c_gce', c_pion, c_gce),
        ('c_ics vs c_gce',  c_ics,  c_gce),
        ('c_π+br vs c_ics', c_pion, c_ics),
        ('c_gce vs c_bub',  c_gce,  c_bub),
    ]
    for _nm, _a, _b in _corr_pair:
        if _a.std() > 0 and _b.std() > 0:
            _r = _np.corrcoef(_a, _b)[0, 1]
            _tag = "strong" if abs(_r) > 0.7 else ("moderate" if abs(_r) > 0.4 else "weak")
            print(f"    {_nm:<22} r = {_r:+.3f} ({_tag})")

    # ---- Scenario hint (single model) ----
    print()
    print(f"  Quick scenario hint (1 model only, use V10 for multi-model comparison):")
    _mean_gas = (c_pion.mean() + c_ics.mean()) / 2
    if 0.8 < _mean_gas < 1.2 and 0.8 < c_gce.mean() < 1.2:
        print(f"    ✓ GDE and GCE coefficients both near 1.0 — fit looks healthy")
    elif _mean_gas > 1.15 and c_gce.mean() < 0.85:
        print(f"    ⚠ GDE ({_mean_gas:.2f}) compensates, GCE ({c_gce.mean():.2f}) weak — "
              f"possible degeneracy")
    elif c_bub.mean() < 0.1:
        print(f"    → bubble coefficient near 0: chi² constraint pushing it out "
              f"(physical in inner ROI)")
    else:
        print(f"    → mixed signature; check V10 across models for pattern")

else:
    _have_bands = False
    c_pion = c_ics = c_gce = c_bub = c_iso = _np.ones(_n)
    print(f"[V2] .npz not found at {_npz_path}")
    print(f"     Falling back to c=1 for all components (raw templates)")
    print(f"     To get actual fit coefficients: re-run runner for Model {_model}")

# ============================================================================
# Plot: SED decomposition
# ============================================================================
_E2_dE = E**2 / delta_E

_pb_sed  = c_pion * (pion + bremss) * _E2_dE
_ics_sed = c_ics  *  ics            * _E2_dE
_bub_sed = c_bub  *  bubble_flux    * _E2_dE if 'bubble_flux' in dir() else _np.zeros(_n)
_iso_sed = c_iso  *  isotropic_flux * _E2_dE if 'isotropic_flux' in dir() else _np.zeros(_n)

# Observed data (CCUBE masked mean)
if 'obs_flux_mean' in dir():
    _obs_E2dN = obs_flux_mean * _E2_dE
else:
    _obs_E2dN = _np.full(_n, _np.nan)

fig, ax = plt.subplots(figsize=(10, 7))
ax.plot(E, _pb_sed,  color='red',     linestyle='-', linewidth=1.5,
        label=r'$\pi^0$+bremss (fitted)')
ax.plot(E, _ics_sed, color='blue',    linestyle='-', linewidth=1.5,
        label='ICS (fitted)')
ax.plot(E, _bub_sed, color='purple',  linestyle='-', linewidth=1.5,
        label='Fermi bubble (fitted)')
ax.plot(E, _iso_sed, color='green',   linestyle='-', linewidth=1.5,
        label='Isotropic (fitted)')
ax.scatter(E, _obs_E2dN, color='k', marker='o', s=28, zorder=10,
           label='Observed (CCUBE)')
# v3.12 yerr guard
_yerr_lo = _np.maximum(model_GCE_flux - model_GCE_lo, 0)
_yerr_hi = _np.maximum(model_GCE_hi - model_GCE_flux, 0)
_skew_bins = [(i, E[i]) for i in range(_n)
              if (model_GCE_flux[i] - model_GCE_lo[i]) < 0
              or (model_GCE_hi[i] - model_GCE_flux[i]) < 0]
if _skew_bins:
    print(f"\n  [info] {len(_skew_bins)}/{_n} bins have best-fit outside [16th, 84th] percentile:")
    for _i, _e in _skew_bins:
        print(f"    bin {_i:2d} E={_e:.3f} GeV: "
              f"flux={model_GCE_flux[_i]:.3e} [lo,hi]=[{model_GCE_lo[_i]:.3e}, {model_GCE_hi[_i]:.3e}]")
ax.errorbar(E, model_GCE_flux, yerr=[_yerr_lo, _yerr_hi],
            linestyle='-', marker='s', color='orange', markersize=7,
            linewidth=2, elinewidth=2, capsize=4, capthick=2,
            label=f'GCE (Model {_model})')

if _have_fits:
    _sum = _pb_sed + _ics_sed + _bub_sed + _iso_sed + model_GCE_flux
    ax.plot(E, _sum, color='gray', linestyle='--', linewidth=1.2, alpha=0.7,
            label='Sum of fitted components')

ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlim(0.3, 500); ax.set_ylim(1e-8, 3e-5)
ax.set_xlabel('E [GeV]')
ax.set_ylabel(r'$E^2\,dN/dE$  [GeV cm$^{-2}$ s$^{-1}$ sr$^{-1}$]')
_title_tag = "fitted coefficients" if _have_fits else "raw templates (c=1 fallback)"
ax.set_title(f'GCE SED decomposition — Model {_model}   (haebarg v3 12yr, {_title_tag})')
ax.legend(loc='upper right', ncol=2, fontsize=9)
ax.grid(True, which='major', alpha=0.3)
fig.tight_layout()
_out = f'./GCE_SED_plot_{_model}_12yr.png'
fig.savefig(_out, dpi=130)
plt.show()
print(f"\n[OK] Saved {_out}")

# ---- Quality check: data vs sum of fitted components ----
if _have_fits and not _np.isnan(_obs_E2dN).all():
    _ratio = (_pb_sed + _ics_sed + _bub_sed + _iso_sed + model_GCE_flux) / _obs_E2dN
    print(f"\n  === data/model total ratio (should be ≈ 1 if fit is good) ===")
    print(f"  {'E[GeV]':>8} {'obs':>12} {'model':>12} {'ratio':>8}")
    for _i in range(_n):
        _mo = _pb_sed[_i] + _ics_sed[_i] + _bub_sed[_i] + _iso_sed[_i] + model_GCE_flux[_i]
        print(f"  {E[_i]:>8.3f} {_obs_E2dN[_i]:>12.3e} {_mo:>12.3e} {_ratio[_i]:>8.3f}")
    _mean_r = _np.nanmean(_ratio[(E >= 1) & (E <= 10)])
    print(f"  Mean ratio 1-10 GeV: {_mean_r:.3f}")

# ============================================================================
# V2 ADDENDUM (v3.15 fix): template magnitude diagnostic
# ----------------------------------------------------------------------------
# Shows raw vs fitted template values per bin to help interpret the fit
# coefficients. Uses correct variable names from V1.
# ============================================================================
if _have_fits:
    print()
    print("=" * 92)
    print(f"TEMPLATE MAGNITUDES  —  Model {_model}")
    print("=" * 92)
    # Raw templates (c=1) in E² dN/dE units [GeV/cm²/s/sr]
    _pb_raw   = (pion + bremss) * _E2_dE
    _ics_raw  = ics * _E2_dE
    _bub_raw  = bubble * _E2_dE
    _iso_raw  = isotropic * _E2_dE
    _gce_raw  = GCE * _E2_dE
    # Observed: counts_per_exp is in ph/cm²/s/sr per bin; × E²/dE → E²·dN/dE
    _obs_E2   = counts_per_exp * _E2_dE

    # Fitted (c × raw)
    _pb_fit   = c_pion * _pb_raw
    _ics_fit  = c_ics  * _ics_raw
    _bub_fit  = c_bub  * _bub_raw
    _iso_fit  = c_iso  * _iso_raw
    _gce_fit  = c_gce  * _gce_raw

    print()
    print(f"  Raw template values (c=1) + observed data, in E²dN/dE [GeV/cm²/s/sr]:")
    print(f"  {'bin':>3} {'E':>6} {'obs':>10} {'pi+br_raw':>11} {'ICS_raw':>11} "
          f"{'GCE_raw':>11} {'bub_raw':>11} {'iso_raw':>11}")
    for _i in range(_n):
        print(f"  {_i:>3} {E[_i]:>6.2f} {_obs_E2[_i]:>10.3e} "
              f"{_pb_raw[_i]:>11.3e} {_ics_raw[_i]:>11.3e} "
              f"{_gce_raw[_i]:>11.3e} {_bub_raw[_i]:>11.3e} {_iso_raw[_i]:>11.3e}")

    print()
    print(f"  Raw template / Observed ratio (how big each template is vs data, c=1):")
    print(f"  {'bin':>3} {'E':>6} {'pi+br':>8} {'ICS':>8} {'GCE':>8} {'bubble':>10} {'iso':>8}")
    for _i in range(_n):
        _o = _obs_E2[_i] if _obs_E2[_i] > 0 else _np.nan
        print(f"  {_i:>3} {E[_i]:>6.2f} "
              f"{_pb_raw[_i]/_o:>8.3f} {_ics_raw[_i]/_o:>8.3f} "
              f"{_gce_raw[_i]/_o:>8.3f} {_bub_raw[_i]/_o:>10.3e} {_iso_raw[_i]/_o:>8.3f}")

    print()
    print(f"  Fitted contributions (c × raw): fraction of observed explained by each:")
    print(f"  {'bin':>3} {'E':>6} {'pi+br':>8} {'ICS':>8} {'GCE':>8} {'bubble':>8} {'iso':>8} {'sum':>8}")
    for _i in range(_n):
        _o = _obs_E2[_i] if _obs_E2[_i] > 0 else _np.nan
        _sum = (_pb_fit[_i] + _ics_fit[_i] + _gce_fit[_i] + _bub_fit[_i] + _iso_fit[_i])
        print(f"  {_i:>3} {E[_i]:>6.2f} "
              f"{_pb_fit[_i]/_o:>8.3f} {_ics_fit[_i]/_o:>8.3f} "
              f"{_gce_fit[_i]/_o:>8.3f} {_bub_fit[_i]/_o:>8.3f} "
              f"{_iso_fit[_i]/_o:>8.3f} {_sum/_o:>8.3f}")

    print()
    print(f"  KEY DIAGNOSTIC:")
    print(f"  ---------------")
    # Bin closest to 1 GeV
    _i1 = int(_np.argmin(_np.abs(E - 1.0)))
    # Sum of gas (pi+br) and ICS raw vs observed
    _gde_ratio = (_pb_raw[_i1] + _ics_raw[_i1]) / _obs_E2[_i1]
    _sum_ratio = (_pb_fit[_i1] + _ics_fit[_i1] + _gce_fit[_i1]
                  + _bub_fit[_i1] + _iso_fit[_i1]) / _obs_E2[_i1]
    print(f"  At E ≈ {E[_i1]:.2f} GeV:")
    print(f"    Observed:         {_obs_E2[_i1]:.3e} GeV/cm²/s/sr")
    print(f"    (π+br + ICS)_raw: {_pb_raw[_i1] + _ics_raw[_i1]:.3e}  "
          f"({_gde_ratio*100:.0f}% of obs)")
    if _gde_ratio > 1.3:
        print(f"    → Raw GDE alone is {_gde_ratio:.2f}x observed.")
        print(f"      Fit will reduce c_gas to compensate. This explains c_π+br < 1.")
    elif _gde_ratio < 0.7:
        print(f"    → Raw GDE is only {_gde_ratio:.2f}x observed. Fit amplifies c_gas > 1.")
    else:
        print(f"    → Raw GDE naturally covers {_gde_ratio*100:.0f}% of obs.")
    print(f"    Sum of fitted components / obs = {_sum_ratio:.3f}")
    if 0.95 < _sum_ratio < 1.05:
        print(f"    ✓ Fit converged (all components explain data to within 5%)")
    else:
        print(f"    ⚠ Fit residual is {(_sum_ratio-1)*100:+.0f}%")

    # Bubble-specific check
    print()
    _bub_avg_ratio = _np.nanmean([_bub_raw[i]/_obs_E2[i] for i in range(_n) if _obs_E2[i] > 0])
    print(f"  Bubble template fraction of observed (avg): {_bub_avg_ratio:.3e}")
    if _bub_avg_ratio < 1e-5:
        print(f"    → Bubble template is ~{1/_bub_avg_ratio:.0f}× smaller than observed.")
        print(f"      Fit's c_bubble ~ 100+ is fit amplifying it to match chi² constraint.")
        print(f"      This is likely a template scaling issue, not a fit bug.")
    elif _bub_avg_ratio < 1e-2:
        print(f"    → Bubble is small but plausibly reflects inner ROI "
              f"(bubbles dominate |b|>10°).")

    # Isotropic-specific check
    _iso_avg_ratio = _np.nanmean([_iso_raw[i]/_obs_E2[i] for i in range(_n) if _obs_E2[i] > 0])
    print(f"  Isotropic template fraction of observed (avg): {_iso_avg_ratio:.3e}")
    if _iso_avg_ratio < 1e-3:
        print(f"    → Isotropic is very small vs data, c_iso is fit-dependent")
    elif _iso_avg_ratio > 0.1:
        print(f"    → Isotropic is substantial vs data, expect c_iso ≈ 0.1-1")


In [ ]:
# ============================================================================
# V3. PSC + disk mask visualization (v3.11 — grid off)
# ----------------------------------------------------------------------------
# Quickly verify that the full_mask (Cell 31) is correctly masking:
#   - Bright point sources (TS > 49 → large mask radius)
#   - All other catalog sources (small mask radius)
#   - The galactic disk |b| < 2°
# Shows 3 energy bins (low / mid / high) side by side.
# ============================================================================
import matplotlib.pyplot as plt
import numpy as _np
from astropy.io import fits as _fits
from astropy.visualization import LogStretch, ImageNormalize

_ccube_path = './GC_analysis_sanghwan/GC_ccube_12yr_front_clean.fits'
_full_mask_path = './GC_analysis_sanghwan/Model/GC_mask_60x60_definitions_DR2.npy'
_disk_mask_path = './GC_analysis_sanghwan/Model/GC_disk_mask_60x60_definitions.npy'

if not all(_os.path.exists(p) for p in [_ccube_path, _full_mask_path, _disk_mask_path]):
    print(f"[ERROR] Missing one of:")
    for p in [_ccube_path, _full_mask_path, _disk_mask_path]:
        print(f"  {'✓' if _os.path.exists(p) else '✗'}  {p}")
else:
    _ccube = _fits.open(_ccube_path)[0].data
    _psc_mask = _np.load(_full_mask_path)
    _disk_mask = _np.load(_disk_mask_path)

    _ccube_roi = _ccube[:, 100:500, 100:500]
    _psc_roi   = _psc_mask[:, 100:500, 100:500]
    _disk_roi  = _disk_mask[100:500, 100:500]
    _full_roi  = _psc_roi * _disk_roi

    _bins_to_show = [0, 6, 13]
    fig, axes = plt.subplots(2, len(_bins_to_show), figsize=(5*len(_bins_to_show), 9))
    for col, ebin in enumerate(_bins_to_show):
        E_gev = _np.sqrt(_E_bounds[ebin][2]*_E_bounds[ebin][1]*1e-6)*1e-3

        # Top row: CCUBE
        ax = axes[0, col]
        _norm = ImageNormalize(_ccube_roi[ebin], vmin=0.1, stretch=LogStretch())
        im = ax.imshow(_ccube_roi[ebin], origin='lower', cmap='inferno', norm=_norm)
        ax.set_title(f'CCUBE  bin {ebin}  E≈{E_gev:.2f} GeV\n'
                     f'({_np.sum(_ccube_roi[ebin]):.0f} counts in ROI)')
        ax.set_xlabel('pixel x (l)'); ax.set_ylabel('pixel y (b)')
        ax.grid(False)  # v3.11: turn off matplotlib grid overlay
        _cb = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        # v3.16: clean up cluttered tick labels from LogStretch
        from matplotlib.ticker import FixedLocator, FormatStrFormatter
        _vmax_int = max(1, int(_ccube_roi[ebin].max()))
        _ticks = [t for t in [1, 3, 10, 30, 100, 300, 1000, 3000] if t <= _vmax_int]
        _cb.set_ticks(_ticks)
        _cb.ax.yaxis.set_minor_locator(FixedLocator([]))
        _cb.ax.yaxis.set_major_formatter(FormatStrFormatter('%d'))

        # Bottom row: masked
        ax = axes[1, col]
        masked_ccube = _ccube_roi[ebin] * _full_roi[ebin]
        frac_kept = _np.sum(_full_roi[ebin]) / _full_roi[ebin].size
        im = ax.imshow(masked_ccube, origin='lower', cmap='inferno', norm=_norm)
        ax.set_title(f'CCUBE × mask  bin {ebin}\n({frac_kept*100:.1f}% pixels kept)')
        ax.set_xlabel('pixel x (l)'); ax.set_ylabel('pixel y (b)')
        ax.grid(False)  # v3.11: turn off matplotlib grid overlay
        _cb = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        # v3.16: clean up cluttered tick labels
        _cb.set_ticks(_ticks)
        _cb.ax.yaxis.set_minor_locator(FixedLocator([]))
        _cb.ax.yaxis.set_major_formatter(FormatStrFormatter('%d'))

    fig.suptitle('PSC + disk mask verification (low/mid/high energy)', fontsize=14)
    fig.tight_layout()
    _out = './mask_verification.png'
    fig.savefig(_out, dpi=120)
    plt.show()
    print(f"[OK] Saved {_out}")


In [ ]:
# ============================================================================
# V4. Best-N model GCE envelope + Cholis Zenodo comparison (v3.11)
# ----------------------------------------------------------------------------
# WHY best-N and not all 80?
#   Cholis+2022 Fig 12 (Fig 15 for comparison) labels the main GCE
#   envelope as "GCE 5 best models ±2σ", NOT all 80. The full 80 are only
#   used for "other ROIs" systematic checks. Using all 80 for the main
#   envelope gives a band dragged down by poorly-converged outlier fits
#   with near-zero GCE solutions. Using top-N by log-likelihood gives a
#   tighter band that matches Cholis's published style.
#
# Tuning knob: N_BEST. Set to 5 to match Cholis Fig 12 exactly.
#             Set to 80 to see the full (wide, messy) distribution.
#
# Also:
#   - Switched from min-max to 16-84 percentile for robustness
#   - Flags outlier models that fall below 10x the median at low E
# ============================================================================
import os as _os
import glob
import matplotlib.pyplot as plt
import numpy as _np

# ---- User knobs ----
N_BEST      = 5        # number of top-likelihood models for envelope (Cholis uses 5)
PCT_LOW     = 16       # lower percentile for envelope (16 = 1σ)
PCT_HIGH    = 84       # upper percentile
OUTLIER_FRAC = 0.1     # flag models with low-E flux < this × median as outliers
# ---------------------

CHOLIS_ZENODO_DIR = '../GCE_TEMPLATES_FILES_v3/Figures_12_and_14_GCE_Spectra/'

def _cholis_path(model):
    fname = f'GCE_Model{model}_flux_Inner40x40_masked_disk.dat'
    full = _os.path.join(CHOLIS_ZENODO_DIR, fname)
    return full if _os.path.exists(full) else None

_dat_files = sorted(glob.glob('./GCE_model_*_12yr_cholis.dat'))
if not _dat_files:
    print("[ERROR] No ./GCE_model_*_12yr_cholis.dat files found.")
else:
    print(f"Found {len(_dat_files)} model result files.")

    # --- Load all pipeline fits + likelihood values ---
    all_flux, all_stat, all_labels, all_loglike = [], [], [], []
    for f in _dat_files:
        d = _np.loadtxt(f)
        if d.ndim == 1:
            continue
        lab = _os.path.basename(f).replace('GCE_model_', '').replace('_12yr_cholis.dat', '')
        all_flux.append(d[:, 1])
        all_stat.append(d[:, 2] if d.shape[1] > 2 else d[:, 1] * 0.1)
        all_labels.append(lab)
        # likelihood per bin, sum for total
        lh_path = f'./GCE_model_{lab}_12yr_cholis_likelihood_value'
        if _os.path.exists(lh_path):
            lh_arr = _np.loadtxt(lh_path)
            all_loglike.append(float(_np.sum(lh_arr)))
        else:
            all_loglike.append(_np.nan)

    E_ref = _np.loadtxt(_dat_files[0])[:, 0]
    all_flux_arr = _np.array(all_flux)
    all_loglike_arr = _np.array(all_loglike)

    # --- Select best-N by total log-likelihood (higher = better) ---
    if _np.all(_np.isnan(all_loglike_arr)):
        print("  [warn] no likelihood_value files found; using ALL models for envelope")
        best_idx = _np.arange(len(all_flux))
    else:
        # Rank by descending loglike (NaNs sorted last)
        valid = ~_np.isnan(all_loglike_arr)
        ranked = _np.argsort(-all_loglike_arr)
        # Skip NaNs
        ranked = [i for i in ranked if valid[i]]
        best_idx = _np.array(ranked[:N_BEST])
        print(f"  Top {N_BEST} models by Σ log L:")
        for rank, idx in enumerate(best_idx, 1):
            print(f"    {rank}. Model {all_labels[idx]:<8} Σ log L = {all_loglike_arr[idx]:.1f}")

    # --- Compute envelopes ---
    best_flux = all_flux_arr[best_idx]
    env_lo_best  = _np.percentile(best_flux, PCT_LOW, axis=0)
    env_hi_best  = _np.percentile(best_flux, PCT_HIGH, axis=0)
    env_med_best = _np.median(best_flux, axis=0)

    env_lo_all  = _np.percentile(all_flux_arr, PCT_LOW, axis=0)
    env_hi_all  = _np.percentile(all_flux_arr, PCT_HIGH, axis=0)
    env_med_all = _np.median(all_flux_arr, axis=0)

    # --- Outlier detection (low-E flux) ---
    _bin0_med = env_med_best[0]
    outliers = [all_labels[i] for i in range(len(all_flux))
                if all_flux_arr[i, 0] < OUTLIER_FRAC * _bin0_med]
    if outliers:
        print(f"\n  Outlier models (bin 0 flux < {OUTLIER_FRAC:.0%} of best-N median):")
        print(f"    {outliers}")
        print(f"    (these are excluded from best-N if below median cut)")

    # --- Load matching Cholis Zenodo refs for the best-N ---
    cholis_refs = {}
    for idx in best_idx:
        lab = all_labels[idx]
        p = _cholis_path(lab)
        if p is not None:
            r = _np.loadtxt(p)
            if r.ndim == 2 and r.shape[1] >= 4:
                cholis_refs[lab] = dict(E=r[:, 0], flux=r[:, 1], lo=r[:, 2], hi=r[:, 3])

    # ========== PLOT ==========
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6.5), sharey=True)

    # Left panel: all 80 (faint) + best-N envelope + median
    for i, (lab, flux) in enumerate(zip(all_labels, all_flux)):
        color = 'lightgray' if i not in best_idx else None
        alpha = 0.25 if i not in best_idx else 0.5
        ax1.plot(E_ref, flux, linestyle='-', alpha=alpha, linewidth=0.7,
                 color=color, zorder=1)
    # best-N envelope on top
    ax1.fill_between(E_ref, env_lo_best, env_hi_best, alpha=0.3, color='orange',
                     label=f'Best {len(best_idx)} models  ({PCT_LOW}–{PCT_HIGH}th %)', zorder=5)
    ax1.plot(E_ref, env_med_best, 'r-', linewidth=2.5, label='Best-N median', zorder=6)
    # full 80 as faint gray band for reference
    ax1.fill_between(E_ref, env_lo_all, env_hi_all, alpha=0.12, color='gray',
                     label=f'All {len(all_flux)} ({PCT_LOW}–{PCT_HIGH}th %)', zorder=2)

    ax1.set_xscale('log'); ax1.set_yscale('log')
    ax1.set_xlim(0.3, 100); ax1.set_ylim(1e-8, 1e-5)
    ax1.set_xlabel('E [GeV]')
    ax1.set_ylabel(r'$E^2\,dN/dE$  [GeV cm$^{-2}$ s$^{-1}$ sr$^{-1}$]')
    ax1.set_title(f'GCE SED — haebarg v3 pipeline  '
                  f'(best {len(best_idx)} of {len(all_flux)} models)')
    ax1.grid(True, which='major', linestyle='-', linewidth=0.4, alpha=0.3)
    ax1.legend(loc='best', fontsize=9)

    # Right panel: best-N pairwise vs Cholis (same model, same color)
    _colors = plt.cm.tab10.colors
    for i, idx in enumerate(best_idx):
        c = _colors[i % 10]
        lab = all_labels[idx]
        ax2.plot(E_ref, all_flux_arr[idx], linestyle='-', color=c, marker='s',
                 markersize=4, linewidth=1.2, alpha=0.9,
                 label=f'haebarg {lab}')
        if lab in cholis_refs:
            r = cholis_refs[lab]
            ax2.plot(r['E'], r['flux'], linestyle='--', color=c, linewidth=1.3, alpha=0.8)
            ax2.fill_between(r['E'], r['lo'], r['hi'], alpha=0.12, color=c)
    ax2.plot([], [], linestyle='--', color='black', alpha=0.5, label='Cholis Zenodo (dashed)')
    # Show all-80 envelope on right too for context
    ax2.fill_between(E_ref, env_lo_all, env_hi_all, alpha=0.08, color='gray', zorder=0)

    ax2.set_xscale('log'); ax2.set_yscale('log')
    ax2.set_xlim(0.3, 100); ax2.set_ylim(1e-8, 1e-5)
    ax2.set_xlabel('E [GeV]')
    ax2.set_title(f'Best {len(best_idx)} models: pipeline vs Cholis Zenodo')
    ax2.grid(True, which='major', linestyle='-', linewidth=0.4, alpha=0.3)
    ax2.legend(loc='best', fontsize=8, ncol=2)

    fig.tight_layout()
    _out = './GCE_multimodel_envelope.png'
    fig.savefig(_out, dpi=150)
    plt.show()
    print(f"\n[OK] Saved {_out}")

    # --- Per-model ratio table (best-N only) ---
    print(f"\n{'='*72}")
    print(f"Best-{len(best_idx)} pairwise ratios (1–10 GeV)")
    print('-' * 72)
    print(f"{'rank':<5} {'Model':<8} {'Σ log L':<12} {'mean ratio':<12} {'verdict'}")
    for rank, idx in enumerate(best_idx, 1):
        lab = all_labels[idx]
        if lab not in cholis_refs:
            print(f"{rank:<5} {lab:<8} {all_loglike_arr[idx]:<12.1f} (no Cholis ref)")
            continue
        r = cholis_refs[lab]
        interp = _np.interp(E_ref, r['E'], r['flux'])
        mask_110 = (E_ref >= 1.0) & (E_ref <= 10.0)
        mean_r = (all_flux_arr[idx, mask_110] / interp[mask_110]).mean()
        verdict = "✓ GOOD" if abs(mean_r-1)<0.15 else ("⚠ OK" if abs(mean_r-1)<0.30 else "✗ OFF")
        print(f"{rank:<5} {lab:<8} {all_loglike_arr[idx]:<12.1f} {mean_r:<12.3f} {verdict}")
    print('=' * 72)


In [ ]:
# ============================================================================
# V5. Load PPPC4 DM spectra (e+e-/gamma tables from Cirelli+2011)
# ----------------------------------------------------------------------------
# Standard PPPC4 distribution file: `AtProduction_gammas.dat` (one file for
# all channels). Columns are
#   log10(m_chi/GeV)  log10(x=E/m_chi)  dN/dlog10x  for each channel
#
# This cell builds an interpolator `pppc4_dnde(m_chi, E)` usable by V6.
# ============================================================================
import numpy as _np
from scipy.interpolate import RegularGridInterpolator

_PPPC4_CANDIDATES = [
    '/home/haebarg/GCE-Chi-square-fitting/AtProduction_gammas.dat',
    '../AtProduction_gammas.dat',
    './AtProduction_gammas.dat',
    '/home/haebarg/GCE-Chi-square-fitting/PPPC4/AtProduction_gammas.dat',
]
_pppc4_path = None
for _p in _PPPC4_CANDIDATES:
    if _os.path.exists(_p):
        _pppc4_path = _p
        break

if _pppc4_path is None:
    print("[ERROR] PPPC4 file AtProduction_gammas.dat not found in:")
    for _p in _PPPC4_CANDIDATES:
        print(f"  - {_p}")
    print("  V6 will not be able to run. Provide the file or edit _PPPC4_CANDIDATES.")
else:
    print(f"Loading PPPC4 spectra from: {_pppc4_path}")
    # Parse PPPC4 header to find available channels
    with open(_pppc4_path) as _f:
        _hdr = _f.readline().strip().split()
    print(f"  Channels available: {_hdr[2:]}  (first two columns are mass and log10x)")
    _pppc4_data = _np.loadtxt(_pppc4_path, skiprows=1)
    _col_map = {name: i for i, name in enumerate(_hdr)}

    _masses   = _np.unique(_pppc4_data[:, 0])   # log10(m_chi/GeV)
    _log10x   = _np.unique(_pppc4_data[:, 1])   # log10(E/m_chi)
    print(f"  mass grid: {len(_masses)} points, {10**_masses.min():.1f} – {10**_masses.max():.1f} GeV")
    print(f"  log10(x) grid: {len(_log10x)} points, {_log10x.min():.2f} – {_log10x.max():.2f}")

    def make_pppc4_interp(channel_name):
        """Return a function dnde(m_chi_GeV, E_GeV) for a given channel."""
        if channel_name not in _col_map:
            raise KeyError(f"Channel {channel_name!r} not in PPPC4 (available: {_hdr[2:]})")
        _col = _col_map[channel_name]
        # Reshape into a 2D grid (mass × log10x)
        _grid = _np.zeros((len(_masses), len(_log10x)))
        for row in _pppc4_data:
            mi = _np.searchsorted(_masses, row[0])
            xi = _np.searchsorted(_log10x, row[1])
            _grid[mi, xi] = row[_col]
        _rgi = RegularGridInterpolator(
            (_masses, _log10x), _grid,
            bounds_error=False, fill_value=0.0, method='linear',
        )

        def _dnde(m_chi, E):
            """Return dN/dE [1/GeV] at energies E [GeV] for WIMP mass m_chi [GeV]."""
            E = _np.atleast_1d(E).astype(float)
            x = E / m_chi
            # PPPC4 has dN/dlog10(x). Convert to dN/dE = dN/dlog10x / (E * ln(10))
            valid = (E > 0) & (E < m_chi)
            out = _np.zeros_like(E)
            if valid.any():
                pts = _np.column_stack([_np.full(valid.sum(), _np.log10(m_chi)),
                                        _np.log10(x[valid])])
                dn_dlogx = _rgi(pts)
                out[valid] = dn_dlogx / (E[valid] * _np.log(10))
            return out
        return _dnde

    # Pre-build interpolators for the main channels
    pppc4_channels = {}
    for _ch in ['b', 'τ', 'μ', 'W']:
        # PPPC4 column names use Greek letters in some distributions;
        # standard names are: 'b', 'tau', 'mu', 'W'
        _alt_names = {'b': ['b'], 'τ': ['tau'], 'μ': ['mu'], 'W': ['W']}.get(_ch, [_ch])
        for _nm in _alt_names:
            if _nm in _col_map:
                pppc4_channels[_ch] = make_pppc4_interp(_nm)
                break
    print(f"  Built interpolators for: {list(pppc4_channels.keys())}")
    print(f"  (Cascade channels 4b / 4τ via MG5 are not included — port after 14-model run.)")


In [ ]:
# ============================================================================
# V6. DM contour: chi^2 scan in (m_chi, sigma_v)   (v3.7 — verbose diagnostics)
# ----------------------------------------------------------------------------
# v3.7 fixes a problem reported by user: empty contour, chi^2 ≈ 200. Causes:
#   (a) scan range misses the best-fit
#   (b) fit data are dominated by very small stat errors (chi2 blows up even
#       for tiny model-data differences)
#   (c) model-data scale mismatch
# This version:
#   - prints the 2D chi^2 grid statistics so you can see what's happening
#   - widens default mass range to 1-1000 GeV
#   - applies a minimum relative error floor (default 5%) to stat errors
#   - warns if best-fit is on the grid edge (means scan range is wrong)
# ============================================================================
import os as _os
import numpy as _np
import matplotlib.pyplot as plt

# --- User knobs ---
CONTOUR_MODEL    = VALIDATE_MODEL if 'VALIDATE_MODEL' in dir() else "X"
CONTOUR_CHANNELS = ['b', 'τ', 'μ']
CONTOUR_N_GRID   = 50
CONTOUR_MASS_MIN, CONTOUR_MASS_MAX = 1.0, 1000.0     # widened
CONTOUR_SV_MIN,   CONTOUR_SV_MAX   = 1e-28, 1e-23    # widened
MIN_REL_ERR_FLOOR = 0.05      # 5% minimum relative error (stabilizes chi^2)
# -------------------

# Physical constants (Cholis+2022 convention, 40°×40° NFW γ=1.2, |b|>2°)
J_FACTOR_40 = 8.53e22       # GeV^2 cm^-5 sr
SR_ROI_40   = 0.4288213187542626

_CH_LABEL = {'b': r'$b\bar{b}$', 'τ': r'$\tau^{+}\tau^{-}$',
             'μ': r'$\mu^{+}\mu^{-}$', 'W': r'$W^{+}W^{-}$'}
_CH_COLOR = {'b': 'crimson', 'τ': 'royalblue', 'μ': 'seagreen', 'W': 'purple'}

if 'pppc4_channels' not in dir():
    print("[ERROR] V5 (PPPC4 loader) must be run first.")
elif 'E' not in dir() or 'delta_E' not in dir():
    print("[ERROR] V1 (data loader) must be run first.")
else:
    _dat_path = f'./GCE_model_{CONTOUR_MODEL}_12yr_cholis.dat'
    if not _os.path.exists(_dat_path):
        print(f"[ERROR] {_dat_path} not found.")
    else:
        _dat = _np.loadtxt(_dat_path)
        _data_E    = _dat[:, 0]
        _data_flux = _dat[:, 1]
        _data_stat = _dat[:, 2] if _dat.shape[1] >= 3 else _data_flux * 0.1

        # --- Diagnostic 1: show input data ---
        print(f"=== Input data (Model {CONTOUR_MODEL}) ===")
        print(f"{'E[GeV]':>10}  {'flux':>12}  {'stat_err':>12}  {'rel_err':>8}")
        for _e, _f, _s in zip(_data_E, _data_flux, _data_stat):
            _rel = _s / _f if _f != 0 else float('nan')
            print(f"{_e:>10.3f}  {_f:>12.3e}  {_s:>12.3e}  {_rel*100:>7.1f}%")

        # --- Apply error floor: many bins have very small stat errors
        #     (especially at low E where counts are high). Without a floor,
        #     small model-data differences produce chi2 ~ O(100+). ---
        _stat_floor = _np.maximum(_data_stat, MIN_REL_ERR_FLOOR * _np.abs(_data_flux))
        print(f"\nApplied {MIN_REL_ERR_FLOOR*100:.0f}% relative error floor.")
        _n_floored = int(_np.sum(_stat_floor > _data_stat))
        print(f"  {_n_floored}/{len(_data_stat)} bins had their error inflated by the floor.")

        _cov = _np.diag(_stat_floor ** 2)
        _cov_tag = f'stat (with {int(MIN_REL_ERR_FLOOR*100)}% floor)'
        try:
            if ('env_lo' in dir() and 'env_hi' in dir() and 
                'all_flux' in dir() and len(all_flux) >= 3):
                _env_sigma = 0.5 * (env_hi - env_lo)
                _cov = _np.diag(_stat_floor ** 2 + _env_sigma ** 2)
                _cov_tag = f'stat + {len(all_flux)}-model envelope'
        except Exception:
            pass

        _inv_cov = _np.linalg.inv(_cov)
        _dof = len(_data_E) - 2

        print(f"\n=== Scan setup ===")
        print(f"  Channels: {CONTOUR_CHANNELS}")
        print(f"  Mass range: {CONTOUR_MASS_MIN}–{CONTOUR_MASS_MAX} GeV, {CONTOUR_N_GRID} pts")
        print(f"  σv range:   {CONTOUR_SV_MIN:.0e}–{CONTOUR_SV_MAX:.0e} cm³/s, {CONTOUR_N_GRID} pts")
        print(f"  Covariance: {_cov_tag}")
        print(f"  dof = {_dof}")

        _mass_grid = _np.logspace(_np.log10(CONTOUR_MASS_MIN), _np.log10(CONTOUR_MASS_MAX), CONTOUR_N_GRID)
        _sv_grid   = _np.logspace(_np.log10(CONTOUR_SV_MIN),   _np.log10(CONTOUR_SV_MAX),   CONTOUR_N_GRID)
        _DM, _SIG  = _np.meshgrid(_mass_grid, _sv_grid)

        _results = {}
        for _ch in CONTOUR_CHANNELS:
            if _ch not in pppc4_channels:
                print(f"  SKIP {_ch} (not in pppc4_channels)")
                continue
            _provider = pppc4_channels[_ch]
            _chi2 = _np.zeros_like(_DM)
            for i in range(CONTOUR_N_GRID):
                for j in range(CONTOUR_N_GRID):
                    _dnde = _provider(_DM[i, j], _data_E)
                    _model = (_data_E ** 2) * _dnde * (_SIG[i, j] / _DM[i, j] ** 2) * J_FACTOR_40 / SR_ROI_40
                    _d = _model - _data_flux
                    _chi2[i, j] = float(_d @ _inv_cov @ _d)
            _ibf = _np.unravel_index(_np.argmin(_chi2), _chi2.shape)
            _results[_ch] = dict(DM=_DM, SIG=_SIG, chi2=_chi2, ibf=_ibf,
                                 bf_m=_DM[_ibf], bf_s=_SIG[_ibf], chi2_min=_chi2[_ibf])

        if not _results:
            print("[ERROR] No successful scans.")
        else:
            # --- Diagnostic 2: per-channel summary ---
            print(f"\n=== Per-channel chi^2 scan summary ===")
            print(f"{'channel':<10} {'bf_m[GeV]':>12} {'bf_σv':>16} "
                  f"{'χ²_min':>10} {'χ²/dof':>10} {'on edge?':>10}")
            print("-" * 78)
            for _ch, _r in _results.items():
                i0, j0 = _r['ibf']
                on_edge = (i0 in (0, CONTOUR_N_GRID-1)) or (j0 in (0, CONTOUR_N_GRID-1))
                edge_str = "⚠ YES" if on_edge else "no"
                print(f"{_ch:<10} {_r['bf_m']:>12.2f} {_r['bf_s']:>16.3e} "
                      f"{_r['chi2_min']:>10.2f} {_r['chi2_min']/_dof:>10.3f} {edge_str:>10}")
                if on_edge:
                    print(f"  ⚠ {_ch}: best-fit at grid edge (i={i0}, j={j0}). "
                          f"Widen scan range!")

            # --- Diagnostic 3: show chi^2 grid stats ---
            print(f"\n=== Chi² grid statistics (per channel) ===")
            for _ch, _r in _results.items():
                _c = _r['chi2']
                print(f"  {_ch:<5}  min={_c.min():.2f}  max={_c.max():.2e}  "
                      f"median={_np.median(_c):.2f}")

            # --- Plot ---
            fig, ax = plt.subplots(figsize=(10, 7.5))
            for _ch, _r in _results.items():
                _color = _CH_COLOR.get(_ch, 'black')
                _levels = [_r['chi2_min'] + 2.30, _r['chi2_min'] + 6.18]
                _cs = ax.contour(_r['DM'], _r['SIG'], _r['chi2'], levels=_levels,
                                 colors=_color, linestyles=['-', ':'], linewidths=[2.5, 2.0])
                ax.plot(_r['bf_m'], _r['bf_s'], marker='*', color=_color,
                        markersize=18, markeredgecolor='black', zorder=10,
                        label=(f"{_CH_LABEL.get(_ch, _ch)}  "
                               f"$m_\\chi$={_r['bf_m']:.0f} GeV, "
                               f"$\\chi^2/\\mathrm{{dof}}={_r['chi2_min']/_dof:.2f}$"))

            ax.axhline(2.2e-26, color='black', linestyle='--', alpha=0.5,
                       label=r'Thermal relic $\langle\sigma v\rangle = 2.2\times10^{-26}$')

            ax.set_xscale('log'); ax.set_yscale('log')
            ax.set_xlim(CONTOUR_MASS_MIN, CONTOUR_MASS_MAX)
            ax.set_ylim(CONTOUR_SV_MIN, CONTOUR_SV_MAX)
            ax.set_xlabel(r'$m_\chi$ [GeV]', fontsize=13)
            ax.set_ylabel(r'$\langle\sigma v\rangle$ [cm$^3$ s$^{-1}$]', fontsize=13)
            ax.grid(True, which='major', linestyle='--', alpha=0.4)
            ax.grid(True, which='minor', linestyle=':',  alpha=0.2)
            ax.set_title(f'DM channel constraints — Model {CONTOUR_MODEL}  ({_cov_tag})\n'
                         f'1$\\sigma$ solid, 2$\\sigma$ dotted', fontsize=12)
            ax.legend(loc='upper right', fontsize=10, framealpha=0.9)

            _outpath = f'./GCE_dm_contour_{CONTOUR_MODEL}.png'
            fig.tight_layout()
            fig.savefig(_outpath, dpi=150, bbox_inches='tight')
            plt.show()
            print(f"\n[OK] Saved {_outpath}")

            # --- Diagnostic 4: if chi^2_min is still huge, print interpretation ---
            _best_chi2 = min(r['chi2_min'] for r in _results.values())
            _best_chi2_dof = _best_chi2 / _dof
            print(f"\n=== Fit quality ===")
            if _best_chi2_dof < 3:
                print(f"  ✓ Best χ²/dof = {_best_chi2_dof:.2f} — acceptable DM fit.")
            elif _best_chi2_dof < 10:
                print(f"  ⚠ Best χ²/dof = {_best_chi2_dof:.2f} — marginal.")
                print(f"    Possible causes: (a) single-channel DM doesn't fit the")
                print(f"    full GCE shape (cascade channels may be needed);")
                print(f"    (b) your stat errors might be too small.")
            else:
                print(f"  ✗ Best χ²/dof = {_best_chi2_dof:.2f} — no channel fits well.")
                print(f"    Most likely cause: stat errors in your .dat are too small.")
                print(f"    Options: (a) raise MIN_REL_ERR_FLOOR above (try 0.10 or 0.15);")
                print(f"    (b) add a systematic error budget from covariance matrix;")
                print(f"    (c) verify that the data flux is really in GeV/cm²/s/sr and")
                print(f"        not accidentally 4π-times-larger or per-cm² (not per-sr).")


In [ ]:
# ============================================================================
# V7. GCE vs GDE balance diagnostic (works without .npz)
# ----------------------------------------------------------------------------
# Purpose: answer the question "why is my low-E GCE lower than Cholis?"
# Two competing hypotheses:
#   (A) Cholis Model X itself is that shape at low E   → not a problem
#   (B) GDE (π⁰+bremss+ICS) templates overshoot low-E  → GCE residual is squeezed
#
# We compute residual = CCUBE_observed - (π⁰+bremss + ICS)_template.
# If residual >> model_GCE_flux at low E: hypothesis A (Cholis also low)
# If residual ≈ model_GCE_flux: hypothesis B (GCE is the only thing absorbing
#                                              what GDE can't explain)
# ============================================================================
import numpy as _np
import matplotlib.pyplot as plt

if 'pion' not in dir() or 'model_GCE_flux' not in dir():
    print("[ERROR] Run V1 first.")
else:
    _E2_dE = (E ** 2) / delta_E
    _obs        = counts_per_exp * _E2_dE
    _gde_raw    = (pion + bremss + ics) * _E2_dE
    _gde_all    = (pion + bremss + ics + bubble + isotropic) * _E2_dE
    _residual_gde = _obs - _gde_raw      # what GCE + bubble + iso need to explain
    _residual_all = _obs - _gde_all      # what GCE alone needs to explain

    print("=" * 80)
    print("GDE / GCE BALANCE CHECK  (no fit coefficients applied — raw templates)")
    print("=" * 80)
    print(f"{'E[GeV]':>8} {'obs':>10} {'π+b+ICS':>10} {'all_bkg':>10} "
          f"{'GCE_fit':>10} {'res/GCE':>8}")
    print('-' * 80)
    for i, _e in enumerate(E):
        _rat = _residual_all[i] / max(model_GCE_flux[i], 1e-30)
        print(f"{_e:>8.3f} {_obs[i]:>10.3e} {_gde_raw[i]:>10.3e} "
              f"{_gde_all[i]:>10.3e} {model_GCE_flux[i]:>10.3e} {_rat:>8.2f}")

    # --- Interpretation ---
    print()
    _low = E <= 1.0    # low-E bins (where your plot shows discrepancy)
    _gde_dominance = (_gde_raw[_low] / _obs[_low]).mean()
    _residual_to_gce = (_residual_all[_low] / _np.maximum(model_GCE_flux[_low], 1e-30)).mean()
    print(f"At E <= 1 GeV ({_low.sum()} bins):")
    print(f"  Mean (GDE_template / obs):       {_gde_dominance:.3f}")
    print(f"    If ≈ 1.0: raw GDE templates already account for ALL the data")
    print(f"              → MCMC must scale GDE DOWN (c_GDE < 1) to leave")
    print(f"              room for GCE.")
    print(f"    If < 0.9: templates explain < 90% of data → residual is large")
    print(f"              → GCE gets a big chunk → model_GCE_flux is big")
    print(f"    If > 1.0: templates over-predict data → MCMC scales DOWN")
    print()
    print(f"  Mean (residual_after_all_bkg / model_GCE_flux): {_residual_to_gce:.3f}")
    print(f"    If ≈ 1: GCE is absorbing the full residual (healthy fit).")
    print(f"    If << 1: residual >> fitted GCE — MCMC chose NOT to put it")
    print(f"             all in GCE (covariance pulls another component).")
    print(f"    If negative: residual is below zero — bkg over-predicts at low E.")

    # --- Plot ---
    fig, ax = plt.subplots(figsize=(9, 6.5))
    ax.errorbar(E, _obs, yerr=counts_per_exp_err * _E2_dE,
                linestyle='', marker='o', color='black', markersize=5,
                elinewidth=1.5, capsize=3, label='Observed (CCUBE)')
    ax.plot(E, _gde_raw, color='red', linestyle='-', linewidth=2,
            label=r'raw π$^0$+bremss+ICS template')
    ax.plot(E, _gde_all, color='purple', linestyle='-', linewidth=1.5, alpha=0.7,
            label='raw GDE + bubble + iso template')
    ax.plot(E, _residual_all, color='orange', linestyle='--', linewidth=2,
            label='obs − all_bkg_templates (what GCE should fit)')
    # v3.12: clip yerr to non-negative (skewed posterior handling)
    _yerr_lo_v7 = _np.maximum(model_GCE_flux - model_GCE_lo, 0)
    _yerr_hi_v7 = _np.maximum(model_GCE_hi - model_GCE_flux, 0)
    ax.errorbar(E, model_GCE_flux,
                yerr=[_yerr_lo_v7, _yerr_hi_v7],
                linestyle='-', marker='s', color='darkgreen', markersize=6,
                elinewidth=1.5, capsize=3,
                label=f'actual fitted GCE (Model {_model})')

    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlim(0.3, 100); ax.set_ylim(1e-9, 3e-5)
    ax.set_xlabel('E [GeV]')
    ax.set_ylabel(r'$E^2\,dN/dE$  [GeV cm$^{-2}$ s$^{-1}$ sr$^{-1}$]')
    ax.set_title(f'GDE/GCE balance diagnostic — Model {_model}')
    ax.grid(True, which='major', linestyle='-', linewidth=0.5, alpha=0.4)
    ax.legend(loc='best', fontsize=9)
    fig.tight_layout()
    _out = f'./GCE_gde_balance_{_model}.png'
    fig.savefig(_out, dpi=150)
    plt.show()
    print(f"\n[OK] Saved {_out}")


In [ ]:
# ============================================================================
# V8. External constraint diagnostic (Ackermann+2015 IGRB, Fermi bubble)
# ----------------------------------------------------------------------------
# Purpose: reconcile V2 plot with Cholis Fig "Model I, GCE with 4FGL-DR2 Mask":
#   - User observed: haebarg V2 shows much higher iso and almost-zero bubble
#   - Question: is iso_constraints_full_err.txt in the right format?
#
# This cell loads the actual constraint files used by the runner and compares
# them with Ackermann+2015 Table 3 values. If the interpolated flux_data at
# our energy bins differs from Table 3 by > 2x, there's a unit/format bug.
# ============================================================================
import os as _os
import numpy as _np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d as _interp1d

# -------- Ackermann+2015 Table 3 hardcoded (IGRB, Foreground Model A) --------
# Columns: E_mid [GeV], integrated_intensity [cm⁻² s⁻¹ sr⁻¹]
# Source: arXiv:1410.3696 Table 3, pp. 32
_ACK_TABLE3 = [
    # (E_low, E_high, intensity_integrated)  in GeV and cm⁻² s⁻¹ sr⁻¹
    (0.10, 0.14, 2.8e-6),  (0.14, 0.20, 1.7e-6),  (0.20, 0.28, 1.1e-6),
    (0.28, 0.40, 6.7e-7),  (0.40, 0.57, 4.5e-7),  (0.57, 0.80, 3.3e-7),
    (0.80, 1.1,  1.9e-7),  (1.1,  1.6,  1.1e-7),  (1.6,  2.3,  6.0e-8),
    (2.3,  3.2,  3.9e-8),  (3.2,  4.5,  2.3e-8),  (4.5,  6.4,  1.5e-8),
    (6.4,  9.1,  9.6e-9),  (9.1,  13,   7.6e-9),  (13,   18,   4.0e-9),
    (18,   26,   2.6e-9),  (26,   36,   1.6e-9),  (36,   51,   1.1e-9),
    (51,   72,   6.3e-10), (72,   100,  3.6e-10), (100,  140,  1.5e-10),
    (140,  200,  9.8e-11), (200,  290,  4.7e-11), (290,  410,  3.2e-11),
    (410,  580,  7.3e-12),
]
# Convert to differential: dN/dE ≈ intensity / dE, evaluate at geometric mean
_ack_E  = _np.array([ _np.sqrt(lo*hi) for lo,hi,_ in _ACK_TABLE3 ])       # GeV
_ack_dE = _np.array([ hi-lo for lo,hi,_ in _ACK_TABLE3 ])                 # GeV
_ack_intensity_per_bin = _np.array([x for _,_,x in _ACK_TABLE3])          # cm⁻²s⁻¹sr⁻¹
_ack_dNdE  = _ack_intensity_per_bin / _ack_dE                             # cm⁻²s⁻¹sr⁻¹ GeV⁻¹
_ack_E2dN  = _ack_E**2 * _ack_dNdE                                        # GeV cm⁻²s⁻¹sr⁻¹

print("=" * 78)
print("Ackermann+2015 Table 3 (Foreground Model A) - computed E² dN/dE")
print("=" * 78)
print(f"{'E[GeV]':>10} {'intensity':>12} {'dE[GeV]':>10} {'dN/dE':>12} {'E²dN/dE':>12}")
for i in range(len(_ack_E)):
    print(f"{_ack_E[i]:>10.3f} {_ack_intensity_per_bin[i]:>12.3e} "
          f"{_ack_dE[i]:>10.3f} {_ack_dNdE[i]:>12.3e} {_ack_E2dN[i]:>12.3e}")
print(f"\nExpected E²dN/dE at 1 GeV: ~1-2e-7 GeV cm⁻²s⁻¹sr⁻¹")
print(f"Cholis Fig for Model I shows iso ~1e-7 at 0.3-1 GeV.")

# -------- Load what runner actually uses --------
print()
print("=" * 78)
print("Loading actual constraint files used by runner")
print("=" * 78)

_iso_path = './GC_analysis_sanghwan/Model/iso_constraints_full_err.txt'
_bub_path = './GC_analysis_sanghwan/Model/bubble_constraints.txt'

if _os.path.exists(_iso_path):
    _iso_raw = _np.loadtxt(_iso_path)
    print(f"iso_constraints_full_err.txt shape: {_iso_raw.shape}")
    print(f"First 5 rows (columns = E, flux, lo_err, hi_err):")
    for row in _iso_raw[:5]:
        print(f"  {row}")
    print(f"Last 3 rows:")
    for row in _iso_raw[-3:]:
        print(f"  {row}")
    print(f"\nColumn 1 (E) range: {_iso_raw[:,0].min():.3e} - {_iso_raw[:,0].max():.3e}")
    print(f"Column 2 (flux) range: {_iso_raw[:,1].min():.3e} - {_iso_raw[:,1].max():.3e}")
    print()

    # Interpolate at our E grid to see what runner computes
    _iso_int = _interp1d(_iso_raw[:,0], _iso_raw[:,1], fill_value='extrapolate',
                         kind='quadratic')
    _iso_at_E = _iso_int(E)
    # Runner multiplies by E²:
    _iso_flux_data = E**2 * _iso_at_E
    print(f"Runner computation: isotropic_flux_data = E² × iso_fluxint(E)")
    print(f"{'E[GeV]':>10} {'iso(E)':>14} {'E² × iso(E)':>16} {'Ack Table3':>14} {'ratio':>8}")
    for i, _e in enumerate(E):
        # Find nearest Ackermann bin
        _j = int(_np.argmin(_np.abs(_ack_E - _e)))
        _ack_ref = _ack_E2dN[_j]
        _ratio = _iso_flux_data[i] / _ack_ref if _ack_ref > 0 else _np.nan
        print(f"{_e:>10.3f} {_iso_at_E[i]:>14.3e} {_iso_flux_data[i]:>16.3e} "
              f"{_ack_ref:>14.3e} {_ratio:>8.2f}")

    _mean_ratio = _np.mean([
        E[i]**2 * _iso_at_E[i] / _ack_E2dN[int(_np.argmin(_np.abs(_ack_E - E[i])))]
        for i in range(len(E))
        if _ack_E2dN[int(_np.argmin(_np.abs(_ack_E - E[i])))] > 0
    ])
    print(f"\nMean ratio (runner iso / Ackermann Table 3): {_mean_ratio:.2f}")
    if 0.8 < _mean_ratio < 1.2:
        print(f"  ✓ Within 20% of Ackermann. Unit handling OK.")
    elif 0.1 < _mean_ratio < 10:
        print(f"  ⚠ {_mean_ratio:.1f}x off from Ackermann.")
        print(f"    Possible cause: file column 2 has different unit than expected.")
    else:
        print(f"  ✗ Order-of-magnitude mismatch. Likely unit/format bug.")
else:
    print(f"  ✗ {_iso_path} not found")

print()
if _os.path.exists(_bub_path):
    _bub_raw = _np.loadtxt(_bub_path)
    print(f"bubble_constraints.txt shape: {_bub_raw.shape}")
    print(f"First 5 rows:")
    for row in _bub_raw[:5]:
        print(f"  {row}")
    print(f"Column 2 range: {_bub_raw[:,1].min():.3e} - {_bub_raw[:,1].max():.3e}")
    _bub_int = _interp1d(_bub_raw[:,0], _bub_raw[:,1], fill_value='extrapolate',
                         kind='quadratic')
    _bub_at_E = _bub_int(E)
    # Runner does NOT multiply by E² (asymmetric handling)
    print(f"\nRunner: bubble_flux_data = bubble_fluxint(E)  [NO E² multiplication]")
    print(f"{'E[GeV]':>10} {'bubble_flux_data':>20}  (expected ~1e-7 GeV cm⁻²s⁻¹sr⁻¹)")
    for i, _e in enumerate(E):
        print(f"{_e:>10.3f} {_bub_at_E[i]:>20.3e}")
    print(f"\nIf magnitudes are ~1e-8–1e-6: file already in E²·dN/dE units, runner OK.")
    print(f"If magnitudes are ~1e-10 or smaller: file is dN/dE, runner is wrong.")
else:
    print(f"  ✗ {_bub_path} not found")

# -------- Compare with actual fit coefficients if .npz exists --------
print()
print("=" * 78)
print("Actual fit coefficients (if .npz exists)")
print("=" * 78)
_npz_path = f'./GCE_model_{_model}_12yr_cholis_fit.npz'
if _os.path.exists(_npz_path):
    _npz = _np.load(_npz_path)
    _fp = _npz['fitted_params']
    _n = len(E)
    _c_pion = _fp[0*_n:1*_n]
    _c_ics  = _fp[1*_n:2*_n]
    _c_gce  = _fp[2*_n:3*_n]
    _c_bub  = _fp[3*_n:4*_n]
    _c_iso  = _fp[4*_n:5*_n]
    print(f"{'E[GeV]':>10} {'c_pion+b':>10} {'c_ics':>10} {'c_gce':>10} "
          f"{'c_bubble':>10} {'c_iso':>10}")
    for i, _e in enumerate(E):
        print(f"{_e:>10.3f} {_c_pion[i]:>10.3f} {_c_ics[i]:>10.3f} "
              f"{_c_gce[i]:>10.3f} {_c_bub[i]:>10.3f} {_c_iso[i]:>10.3f}")
    print()
    print(f"Interpretation:")
    print(f"  c_bubble mean = {_c_bub.mean():.3f}   → if ≈ 0, bubble is pushed to zero by fit")
    print(f"  c_iso    mean = {_c_iso.mean():.3f}   → if ≠ 1, iso template norm differs from expected")
    if _c_bub.mean() < 0.1:
        print(f"  ⚠ c_bubble ≈ 0: bubble template not absorbed by fit. Possible causes:")
        print(f"    (a) Cholis chi² constraint for bubble is too restrictive here")
        print(f"    (b) bubble template in 60°×60° ROI is too small (bubble is mostly |b|>10°)")
    if _c_iso.mean() > 3 or _c_iso.mean() < 0.3:
        print(f"  ⚠ c_iso mean {_c_iso.mean():.2f} is far from 1. Either iso template is")
        print(f"    badly normalized, or the iso_constraints file units don't match runner.")
else:
    print(f"  .npz not found for Model {_model}. Re-run this model with v3.10+ runner to get it.")

npz = np.load("./GCE_model_I_12yr_cholis_fit.npz")
fp = npz['fitted_params']
print(f"c_bub mean = {fp[3*17:4*17].mean():.3f}")  # 17 bin 기준
print(f"c_iso mean = {fp[4*17:5*17].mean():.3f}")


In [ ]:
# ============================================================================
# V9. Template audit v3 — fixed Zenodo cube unit interpretation
# ----------------------------------------------------------------------------
# BUG FIX from v2: Zenodo MapCube stores dN/dE per MeV (Fermi standard),
# NOT E²·dN/dE directly. v2 treated the cube values as E²·dN/dE which caused
# ratio to appear ~E²×1000 too large. This version converts correctly:
#   cube value [dN/dE per MeV]  →  E²·dN/dE [GeV/cm²/s/sr] = E_GeV² × value × 1000
#
# Expected behavior after fix (user's Model I data, post-hoc verified):
#   1-10 GeV ratios: 0.99-1.02 (GDE templates match Cholis within 1-2%)
#   >10 GeV ratios:  1.15-1.18 (high-E overshoot to investigate)
# ============================================================================
import os as _os
import glob as _glob
import numpy as _np
import matplotlib.pyplot as plt
from astropy.io import fits as _fits
from scipy.interpolate import interp1d as _i1d

if 'SELECTED_MODEL' in dir() and isinstance(SELECTED_MODEL, str):
    _M = SELECTED_MODEL
elif 'VALIDATE_MODEL' in dir() and isinstance(VALIDATE_MODEL, str):
    _M = VALIDATE_MODEL
else:
    _M = 'X'

print(f"\n{'='*80}")
print(f"TEMPLATE AUDIT v3  —  Model {_M!r}")
print(f"{'='*80}")

# ------------- Rebuild E/delta_E if needed (defensive) -------------
_have_E = isinstance(E, _np.ndarray) and hasattr(E, 'shape') and E.shape == (14,) \
          if 'E' in dir() else False
if not _have_E:
    _cc = _fits.open('./GC_analysis_sanghwan/GC_ccube_12yr_front_clean.fits')
    _Eb = _cc[1].data
    E = _np.array([_np.sqrt(_b[2]*_b[1]*1e-6)*1e-3 for _b in _Eb])
    delta_E = _np.array([(_b[2]-_b[1])*1e-6 for _b in _Eb])
    _cc.close()

_E2_dE = E**2 / delta_E

# ============================================================================
# CHECK 1,2,3 — already validated in v2 run; keep short summary
# ============================================================================
print("\nCHECK 1-3 (from v2 run):")
print("  1. NFW² integral = 0.7154 (normalized over 60x60, absorbed by map_based_integral)")
print("  2. Bubble area = 0.555 sr (consistent with Su+2010)")
print("  3. Isotropic spectrum = Ackermann Table 3 (1.000x, perfect)")
print("  → non-GDE templates are consistent with their ground truths")

# ============================================================================
# CHECK 4: ⭐ GDE templates vs Cholis Zenodo — CORRECTED UNIT HANDLING
# ============================================================================
print()
print("=" * 80)
print("CHECK 4: ⭐ GDE templates — our map vs Cholis Zenodo (fixed units)")
print("=" * 80)

_disk = _np.load('./GC_analysis_sanghwan/Model/GC_disk_mask_60x60_definitions.npy')[100:500, 100:500]
_exp_hdu = _fits.open('./GC_analysis_sanghwan/GC_expcube_center_12yr_front_clean.fits')
_exp = _exp_hdu[0].data[:, 100:500, 100:500]
_exp_hdu.close()
_raw = _fits.open('./GC_analysis_sanghwan/GC_ccube_12yr_front_clean.fits')
from astropy.wcs import WCS as _WCS
_wcs = _WCS(_raw[0].header).dropaxis(2)
_sr_pix = _np.zeros((600, 600))
for _i in range(600):
    for _j in range(600):
        _lon, _lat = _wcs.wcs_pix2world(_j, _i, 0)
        _sr_pix[_i, _j] = _np.radians(0.1)**2 * _np.cos(_np.radians(_lat))
_sr_roi = _sr_pix[100:500, 100:500]
_exp_sr = _exp * _sr_roi
_raw.close()

def _our_flux(fp):
    """Returns E²·dN/dE per bin for our pipeline component (disk-masked mean)."""
    if not _os.path.exists(fp):
        return None
    _d = _fits.open(fp)[0].data
    _f = _np.zeros(len(E))
    for _i in range(len(E)):
        _px = _d[_i][100:500, 100:500]
        _f[_i] = _np.sum(_disk * (_px / _exp_sr[_i])) / _np.sum(_disk)
    return _f * _E2_dE   # convert to E²·dN/dE [GeV/cm²/s/sr]

def _zen_flux(fp):
    """Returns Zenodo E²·dN/dE interpolated to our E grid.
    FIXED: cube stores dN/dE per MeV; convert via × E² × 1000."""
    if not _os.path.exists(fp):
        return None
    _hdul = _fits.open(fp)
    _cube = _hdul[0].data
    if 'ENERGIES' in [h.name for h in _hdul]:
        _emev = _hdul['ENERGIES'].data['Energy']
    else:
        _hdul.close()
        return None
    _egev = _emev / 1000
    _ny, _nx = _cube.shape[-2:]
    # Figure out inner 40x40 region with |b|<2° disk mask
    _cd = 0.25 if _nx == 240 else 0.1
    _cx, _cy = _nx // 2, _ny // 2
    _half40 = int(20 / _cd)
    _half2b = int(2 / _cd)
    _x0, _x1 = max(0, _cx - _half40), min(_nx, _cx + _half40)
    _y0, _y1 = max(0, _cy - _half40), min(_ny, _cy + _half40)
    _native_dNdE_perMeV = _np.zeros(len(_egev))
    for _i in range(len(_egev)):
        _slab = _cube[_i, _y0:_y1, _x0:_x1].copy()
        _yc = _slab.shape[0] // 2
        _slab[max(0, _yc - _half2b):min(_slab.shape[0], _yc + _half2b), :] = 0
        _w = _slab != 0
        _native_dNdE_perMeV[_i] = _slab[_w].mean() if _w.any() else 0.0
    _hdul.close()
    # ⭐ CORRECT CONVERSION: cube unit = dN/dE per MeV
    # E²·dN/dE [GeV/cm²/s/sr] = E_GeV² × (dN/dE per MeV) × 1000
    _native_E2dN = _egev**2 * _native_dNdE_perMeV * 1000
    # Interpolate (log-log) onto our 14-bin grid
    _safe = _np.maximum(_native_E2dN, 1e-30)
    _interp = _i1d(_np.log(_egev), _np.log(_safe),
                   fill_value='extrapolate', kind='linear')
    return _np.exp(_interp(_np.log(E)))

_ratios = {}
for _comp, _our_p, _zen_p in [
    ('pion',   f'./GC_analysis_sanghwan/GC_pion_model{_M}_12yr_front_clean_no_convol.fits',
               f'./MapCubes/pion_mapcube_model{_M}.fits'),
    ('bremss', f'./GC_analysis_sanghwan/GC_bremss_model{_M}_12yr_front_clean_no_convol.fits',
               f'./MapCubes/bremss_mapcube_model{_M}.fits'),
    ('ics',    f'./GC_analysis_sanghwan/GC_ics_model{_M}_12yr_front_clean_no_convol.fits',
               f'./MapCubes/ics_mapcube_model{_M}.fits'),
]:
    print(f"\n  Component: {_comp}")
    _our = _our_flux(_our_p)
    _zen = _zen_flux(_zen_p)
    if _our is None or _zen is None:
        print(f"    ✗ missing file(s)")
        continue
    _rs = _our / _np.maximum(_zen, 1e-30)
    print(f"    {'E[GeV]':>10} {'our E²dN/dE':>14} {'Zen E²dN/dE':>14} {'ratio':>8}")
    for _i, _e in enumerate(E):
        print(f"    {_e:>10.3f} {_our[_i]:>14.3e} {_zen[_i]:>14.3e} {_rs[_i]:>8.3f}")
    _m110 = (E >= 1.0) & (E <= 10.0)
    _mhi  = E > 10.0
    _mlo  = E < 1.0
    _r110 = _np.nanmean(_rs[_m110])
    _rhi  = _np.nanmean(_rs[_mhi])
    _rlo  = _np.nanmean(_rs[_mlo])
    _ratios[_comp] = dict(r_1_10=_r110, r_lo=_rlo, r_hi=_rhi)
    print(f"    Mean ratio 0.3-1 GeV   = {_rlo:.3f}")
    print(f"    Mean ratio 1-10 GeV    = {_r110:.3f}")
    print(f"    Mean ratio >10 GeV     = {_rhi:.3f}")
    if 0.9 < _r110 < 1.1:
        print(f"    ✓ 1-10 GeV: PASS (within 10%)")
    else:
        print(f"    ⚠ 1-10 GeV: {_r110*100-100:+.0f}% deviation")

# ============================================================================
# SUMMARY and INTERPRETATION
# ============================================================================
print()
print("=" * 80)
print("FINAL SUMMARY & INTERPRETATION")
print("=" * 80)
if _ratios:
    print(f"\n  {'Component':<10} {'0.3-1 GeV':>12} {'1-10 GeV':>12} {'>10 GeV':>12}")
    for _c, _r in _ratios.items():
        print(f"  {_c:<10} {_r['r_lo']:>12.3f} {_r['r_1_10']:>12.3f} {_r['r_hi']:>12.3f}")

    print()
    _all_1_10 = [r['r_1_10'] for r in _ratios.values()]
    _all_hi   = [r['r_hi']   for r in _ratios.values()]

    if all(0.95 < r < 1.05 for r in _all_1_10):
        print(f"  ✓ 1-10 GeV: all GDE templates match Cholis within 5%")
        print(f"    → GDE template processing is CORRECT")
        print(f"    → ~13% pipeline offset is NOT from template normalization")
        print(f"    → Root cause must be in fit stage or input data (CCUBE/LTCUBE)")
    elif all(0.9 < r < 1.1 for r in _all_1_10):
        print(f"  ✓ 1-10 GeV: all within 10% — templates essentially correct")
    else:
        print(f"  ⚠ 1-10 GeV: some deviations (see individual ratios above)")

    if all(r > 1.10 for r in _all_hi):
        print(f"\n  ⚠ >10 GeV: all components show ~15-20% overshoot vs Cholis")
        print(f"    → Separate issue (likely spectrum extrapolation at high E)")
        print(f"    → Cell 35 extended spectrum to 1 TeV via PL extrapolation")
        print(f"    → Cholis Zenodo cubes have native grid up to ~820 GeV")
        print(f"    → Our extrapolation may slightly overshoot true high-E behavior")
        print(f"    → Does NOT affect 1-10 GeV GCE science")
else:
    print(f"  [no ratios computed — check file paths]")

print()
print("  Conclusion so far:")
print("  - NFW² template: normalization convention difference, absorbed by XML")
print("  - Bubble template: OK")
print("  - Isotropic spectrum: perfect Ackermann match")
print("  - GDE templates: 1-10 GeV match Cholis within ~1-5%")
print("  → Templates are NOT the source of ~13% pipeline offset.")
print("  Next diagnostic: fit-stage analysis (c_coefficient distribution across 80 models)")


In [ ]:
# ============================================================================
# V10. Multi-model fit coefficient comparison
# ----------------------------------------------------------------------------
# Loads fit coefficients from .npz files across multiple models and compares:
#   - Per-model mean/std of each coefficient
#   - Bin-by-bin pattern: do all models have the same GDE-GCE trade-off?
#   - Scenario judgment:
#       1. All c ≈ 1.0 → fit is healthy, offset is input-data
#       2. c_gas > 1.15 + c_gce < 0.85 consistently → degeneracy (memory #8)
#       3. Model-by-model heterogeneous → GDE-specific issue
#
# Add/remove models in MODELS_TO_COMPARE as needed.
# ============================================================================
import os as _os
import numpy as _np
import matplotlib.pyplot as plt

# ---- User knobs ----
MODELS_TO_COMPARE = ['X', 'I', 'XLIX']  # add more after re-running
# ---------------------

_n = len(E)

# Load all available .npz files for requested models
_loaded = {}
for _M in MODELS_TO_COMPARE:
    _p = f'./GCE_model_{_M}_12yr_cholis_fit.npz'
    if not _os.path.exists(_p):
        print(f"  [skip] Model {_M}: .npz not found")
        continue
    _npz = _np.load(_p)
    _fp = _npz['fitted_params']
    _loaded[_M] = dict(
        c_pion=_fp[0*_n:1*_n],
        c_ics =_fp[1*_n:2*_n],
        c_gce =_fp[2*_n:3*_n],
        c_bub =_fp[3*_n:4*_n],
        c_iso =_fp[4*_n:5*_n],
    )

if not _loaded:
    print("[V10] No .npz files found. Re-run at least one model with v3.10+ runner.")
    print("      Example: python run_main_loop_subprocess.py X,I")
else:
    print(f"Comparing {len(_loaded)} models: {list(_loaded.keys())}")
    print()

    # ---- Summary table ----
    print(f"  Per-model coefficient means (14-bin average):")
    print(f"  {'Model':>8}  {'c_π+br':>8}  {'c_ics':>8}  {'c_gce':>8}  "
          f"{'c_bub':>8}  {'c_iso':>8}")
    for _M, _d in _loaded.items():
        print(f"  {_M:>8}  "
              f"{_d['c_pion'].mean():>8.3f}  {_d['c_ics'].mean():>8.3f}  "
              f"{_d['c_gce'].mean():>8.3f}  {_d['c_bub'].mean():>8.3f}  "
              f"{_d['c_iso'].mean():>8.3f}")

    # ---- Per-bin detail ----
    print()
    print(f"  === Per-bin c_gce (GCE coefficient) across models ===")
    print(f"  {'bin':>3} {'E[GeV]':>8}  " + "  ".join(f"{M:>8}" for M in _loaded))
    for _i in range(_n):
        _row = f"  {_i:>3} {E[_i]:>8.3f}  "
        _row += "  ".join(f"{_d['c_gce'][_i]:>8.3f}" for _d in _loaded.values())
        print(_row)

    print()
    print(f"  === Per-bin c_pion_bremss (gas coefficient) across models ===")
    print(f"  {'bin':>3} {'E[GeV]':>8}  " + "  ".join(f"{M:>8}" for M in _loaded))
    for _i in range(_n):
        _row = f"  {_i:>3} {E[_i]:>8.3f}  "
        _row += "  ".join(f"{_d['c_pion'][_i]:>8.3f}" for _d in _loaded.values())
        print(_row)

    # ---- Scenario judgment ----
    print()
    print(f"  {'='*76}")
    print(f"  SCENARIO ASSESSMENT")
    print(f"  {'='*76}")
    _c_gas_means = [ (_d['c_pion'].mean() + _d['c_ics'].mean()) / 2 for _d in _loaded.values() ]
    _c_gce_means = [  _d['c_gce'].mean() for _d in _loaded.values() ]
    _c_bub_means = [  _d['c_bub'].mean() for _d in _loaded.values() ]
    _c_iso_means = [  _d['c_iso'].mean() for _d in _loaded.values() ]

    _all_gas_healthy = all(0.85 < _g < 1.15 for _g in _c_gas_means)
    _all_gce_healthy = all(0.85 < _g < 1.15 for _g in _c_gce_means)
    _consistent_gas_hi = all(_g > 1.15 for _g in _c_gas_means)
    _consistent_gce_lo = all(_g < 0.85 for _g in _c_gce_means)
    _diverge = (max(_c_gce_means) - min(_c_gce_means)) > 0.3

    print(f"  Gas (π+br + ICS) means:    {_c_gas_means}")
    print(f"  GCE means:                 {_c_gce_means}")
    print(f"  Bubble means:              {_c_bub_means}")
    print(f"  Iso means:                 {_c_iso_means}")
    print()

    if _all_gas_healthy and _all_gce_healthy:
        print(f"  → SCENARIO 1: Fit is healthy across all models (c ≈ 1.0)")
        print(f"    → 13% offset is NOT from fit stage")
        print(f"    → Look at: input data (CCUBE/LTCUBE/EXPCUBE) vs Sanghwan")
    elif _consistent_gas_hi and _consistent_gce_lo:
        print(f"  → SCENARIO 2: GDE-GCE degeneracy confirmed (memory #8)")
        print(f"    Gas absorbs ~{(_np.mean(_c_gas_means)-1)*100:.0f}% more,"
              f"  GCE loses ~{(1-_np.mean(_c_gce_means))*100:.0f}%")
        print(f"    → 13% offset has fit-stage contribution")
        print(f"    → Fix options: (a) Fix c_iso or c_bubble prior strength")
        print(f"                   (b) Re-weight chi² for gas components")
    elif _diverge:
        print(f"  → SCENARIO 3: Model-specific heterogeneity")
        print(f"    GCE coefficient varies by {(max(_c_gce_means)-min(_c_gce_means)):.2f}")
        print(f"    → Different GDE models give different GCE normalizations")
        print(f"    → Normal physical expectation (Cholis+2022 shows 10-50% variation)")
    else:
        print(f"  → MIXED: inspect individual coefficients")

    # ---- Correlation across models ----
    print()
    if len(_loaded) >= 2:
        print(f"  Model-to-model correlation of c_gce (does GCE shape agree between models?)")
        _Mlist = list(_loaded.keys())
        for _i in range(len(_Mlist)):
            for _j in range(_i+1, len(_Mlist)):
                _M1, _M2 = _Mlist[_i], _Mlist[_j]
                _r = _np.corrcoef(_loaded[_M1]['c_gce'], _loaded[_M2]['c_gce'])[0, 1]
                print(f"    c_gce[{_M1}] vs c_gce[{_M2}]: r = {_r:+.3f}")

    # ---- Plot: side-by-side coefficient comparison ----
    fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True)
    _coef_data = [
        ('c_π+br',  'c_pion', axes[0, 0]),
        ('c_ICS',   'c_ics',  axes[0, 1]),
        ('c_GCE',   'c_gce',  axes[0, 2]),
        ('c_bubble','c_bub',  axes[1, 0]),
        ('c_iso',   'c_iso',  axes[1, 1]),
    ]
    _colors = plt.cm.tab10.colors
    for _i, (_label, _key, _ax) in enumerate(_coef_data):
        for _j, (_M, _d) in enumerate(_loaded.items()):
            _ax.plot(E, _d[_key], marker='o', linewidth=1.5,
                     color=_colors[_j % 10], label=f'Model {_M}')
        _ax.axhline(1.0, color='gray', linestyle='--', linewidth=0.8)
        _ax.set_xscale('log')
        _ax.set_xlabel('E [GeV]')
        _ax.set_ylabel(_label)
        _ax.set_title(_label)
        _ax.legend(fontsize=8)
        _ax.grid(True, alpha=0.3)
    # Hide last subplot
    axes[1, 2].axis('off')
    axes[1, 2].text(0.1, 0.9, f"Models compared:\n{list(_loaded.keys())}",
                    transform=axes[1, 2].transAxes, fontsize=10, va='top')
    axes[1, 2].text(0.1, 0.5,
                    "Interpretation guide:\n\n"
                    "All c ≈ 1.0:  fit healthy\n"
                    "c_gas > 1.2 + c_gce < 0.8:\n  GDE-GCE degeneracy\n\n"
                    "Different patterns per model:\n  model-specific variation",
                    transform=axes[1, 2].transAxes, fontsize=9, va='top',
                    family='monospace')
    fig.tight_layout()
    _out = './fit_coefficient_comparison.png'
    fig.savefig(_out, dpi=130)
    plt.show()
    print(f"\n[OK] Saved {_out}")


In [ ]:
# ============================================================================
# V11. Spatial residual map  (data - fitted_model) / data per pixel
# ----------------------------------------------------------------------------
# Goal: identify which sky region the fit fails to explain.
#
# Reconstructs the runner's per-pixel model:
#   expected = c_pb*pion+bremss + c_ics*ics + c_gce*GCE + c_iso*iso + c_bub*bubble
# using the PSF-convolved component maps (the runner uses these).
#
# Output: 4 energy bins × (data | model | residual fraction)
#       + b-profile and l-profile of residual.
#
# Requires: V0_SETUP, V1, .npz fit file present
# ============================================================================
import os as _os
import numpy as _np
import matplotlib.pyplot as plt
from astropy.io import fits as _fits
from astropy.wcs import WCS as _WCS
from matplotlib.colors import SymLogNorm as _SymLog

# v3.17: use SELECTED_MODEL to avoid _model pollution
if 'SELECTED_MODEL' in dir() and isinstance(SELECTED_MODEL, str):
    _M = SELECTED_MODEL
elif 'VALIDATE_MODEL' in dir() and isinstance(VALIDATE_MODEL, str):
    _M = VALIDATE_MODEL
elif '_model' in dir() and isinstance(_model, str):
    _M = _model
else:
    _M = 'X'
    print(f"[V11] [warn] no valid model string found; defaulting to '{_M}'")

# Also ensure `front` is defined (may be polluted)
if 'front' not in dir() or not isinstance(front, str):
    front = '_front'   # default

# Rebuild E/delta_E if polluted
if 'E' not in dir() or not hasattr(E, 'shape') or E.shape != (14,):
    print(f"[V11] [info] rebuilding E/delta_E from CCUBE")
    _cc = _fits.open(f'./GC_analysis_sanghwan/GC_ccube_12yr{front}_clean.fits')
    _Eb = _cc[1].data
    E = _np.array([_np.sqrt(_b[2]*_b[1]*1e-6)*1e-3 for _b in _Eb])
    delta_E = _np.array([(_b[2]-_b[1])*1e-6 for _b in _Eb])
    _cc.close()

_npz_path = f'./GCE_model_{_M}_12yr_cholis_fit.npz'
print(f"[V11] using Model {_M!r}, npz={_npz_path}")
if not _os.path.exists(_npz_path):
    print(f"[V11] {_npz_path} not found. Run V2 first to ensure .npz exists.")
    raise SystemExit

_npz = _np.load(_npz_path)
_fp  = _npz['fitted_params']
_n   = len(E)
_c_pb  = _fp[0*_n:1*_n]
_c_ics = _fp[1*_n:2*_n]
_c_gce = _fp[2*_n:3*_n]
_c_bub = _fp[3*_n:4*_n]
_c_iso = _fp[4*_n:5*_n]

# Load convolved (PSF-applied) component maps (_clean.fits, NOT _no_convol)
print(f"Loading convolved component maps for Model {_M}...")
_pb_cube  = (_fits.open(f'./GC_analysis_sanghwan/GC_pion_model{_M}_12yr{front}_clean.fits')[0].data
             + _fits.open(f'./GC_analysis_sanghwan/GC_bremss_model{_M}_12yr{front}_clean.fits')[0].data)
_ics_cube = _fits.open(f'./GC_analysis_sanghwan/GC_ics_model{_M}_12yr{front}_clean.fits')[0].data
_gce_cube = _fits.open(f'./GC_analysis_sanghwan/GC_GCE_model_12yr{front}_clean.fits')[0].data
_bub_cube = _fits.open(f'./GC_analysis_sanghwan/GC_fermi_bubble_model_12yr{front}_clean.fits')[0].data
_iso_cube = _fits.open(f'./GC_analysis_sanghwan/GC_isotropic_model_12yr{front}_clean.fits')[0].data

# Data (CCUBE counts)
_ccube_h = _fits.open(f'./GC_analysis_sanghwan/GC_ccube_12yr{front}_clean.fits')
_data_cube = _ccube_h[0].data
_wcs = _WCS(_ccube_h[0].header).dropaxis(2)

# Mask (PSC + disk)
_psc_mask  = _np.load('./GC_analysis_sanghwan/Model/GC_mask_60x60_definitions_DR2.npy')
_disk_mask = _np.load('./GC_analysis_sanghwan/Model/GC_disk_mask_60x60_definitions.npy')
_full_mask = _psc_mask * _disk_mask  # (n_e, 600, 600) × (600, 600) broadcast

print(f"  Loaded all cubes; shape={_data_cube.shape}")

# Build per-pixel model: c × component (counts/pixel)
_n_e = _data_cube.shape[0]
_model_cube = _np.zeros_like(_data_cube, dtype=float)
for _i in range(_n_e):
    _model_cube[_i] = (_c_pb[_i]  * _pb_cube[_i]  +
                       _c_ics[_i] * _ics_cube[_i] +
                       _c_gce[_i] * _gce_cube[_i] +
                       _c_iso[_i] * _iso_cube[_i] +
                       _c_bub[_i] * _bub_cube[_i])

# ROI = 60×60 inner region (pixels 100:500 of 600×600)
_roi = slice(100, 500)
_data_roi  = _data_cube[:, _roi, _roi]
_model_roi = _model_cube[:, _roi, _roi]
_mask_roi  = (_psc_mask[:, _roi, _roi] * _disk_mask[_roi, _roi])

# Residual fraction per pixel (only where mask>0)
_resid_cube = _np.zeros_like(_data_roi, dtype=float)
_resid_cube[:] = _np.nan
_pos = (_data_roi > 0) & (_mask_roi > 0)
_resid_cube[_pos] = (_data_roi[_pos] - _model_roi[_pos]) / _data_roi[_pos]

# ---- Plot: 4 bins × 3 panels ----
_bins_to_show = [0, 5, 10, 13]  # 0.31, 1.16, 4.31, 35 GeV
fig, axes = plt.subplots(len(_bins_to_show), 3, figsize=(15, 4.5*len(_bins_to_show)))

for _row, _eb in enumerate(_bins_to_show):
    _E_GeV = E[_eb]
    _d = _data_roi[_eb].astype(float)
    _m = _model_roi[_eb]
    _r = _resid_cube[_eb]
    _msk = _mask_roi[_eb] > 0
    _d_masked = _np.where(_msk, _d, _np.nan)
    _m_masked = _np.where(_msk, _m, _np.nan)
    _r_masked = _np.where(_msk, _r, _np.nan)

    # Per-row vmax for data/model
    _vmax = _np.nanmax(_d_masked)
    _vmin = max(_np.nanmin(_d_masked[_d_masked > 0]) if _np.any(_d_masked > 0) else 0.1, 0.1)

    # Data
    _ax = axes[_row, 0]
    _im = _ax.imshow(_d_masked, origin='lower', cmap='inferno',
                     norm=_SymLog(linthresh=1, vmin=_vmin, vmax=_vmax))
    _ax.set_title(f'data (counts) — bin {_eb} E≈{_E_GeV:.2f} GeV')
    _ax.set_xlabel('pixel x (l)'); _ax.set_ylabel('pixel y (b)')
    _ax.grid(False)
    plt.colorbar(_im, ax=_ax, fraction=0.046, pad=0.04)

    # Model
    _ax = axes[_row, 1]
    _im = _ax.imshow(_m_masked, origin='lower', cmap='inferno',
                     norm=_SymLog(linthresh=1, vmin=_vmin, vmax=_vmax))
    _ax.set_title(f'fitted model (counts)')
    _ax.set_xlabel('pixel x (l)'); _ax.set_ylabel('pixel y (b)')
    _ax.grid(False)
    plt.colorbar(_im, ax=_ax, fraction=0.046, pad=0.04)

    # Residual
    _ax = axes[_row, 2]
    _im = _ax.imshow(_r_masked, origin='lower', cmap='RdBu_r',
                     vmin=-0.3, vmax=0.3)
    _frac_pos = _np.nansum(_r_masked > 0.05) / _np.sum(_msk)
    _frac_neg = _np.nansum(_r_masked < -0.05) / _np.sum(_msk)
    _mean_r = _np.nanmean(_r_masked)
    _ax.set_title(f'(data−model)/data    mean={_mean_r:+.3f}\n'
                  f'    >+5%: {_frac_pos*100:.1f}%, <−5%: {_frac_neg*100:.1f}%')
    _ax.set_xlabel('pixel x (l)'); _ax.set_ylabel('pixel y (b)')
    _ax.grid(False)
    plt.colorbar(_im, ax=_ax, fraction=0.046, pad=0.04, label='residual fraction')

fig.suptitle(f'Spatial residual maps  —  Model {_M}  '
             f'(red=data exceeds model, blue=model exceeds data)', fontsize=14)
fig.tight_layout()
_out1 = f'./spatial_residual_maps_{_M}.png'
fig.savefig(_out1, dpi=120)
plt.show()
print(f"\n[OK] Saved {_out1}")

# ---- Profile plots ----
fig2, axes2 = plt.subplots(1, 2, figsize=(14, 5))

# Compute galactic l, b for each pixel of the ROI
_ny_roi, _nx_roi = _resid_cube.shape[-2:]
# Map ROI pixel (j_roi, i_roi) → original full pixel (j_full, i_full=j_roi+100)
_j_idx = _np.arange(_ny_roi) + 100  # rows (b)
_i_idx = _np.arange(_nx_roi) + 100  # cols (l)
# Convert center column (j=300, i=300) to (l, b) → just check
_l_axis = _np.zeros(_nx_roi)
_b_axis = _np.zeros(_ny_roi)
for _ii, _ix in enumerate(_i_idx):
    _l_axis[_ii], _ = _wcs.wcs_pix2world(_ix, 300, 0)
for _jj, _jy in enumerate(_j_idx):
    _, _b_axis[_jj] = _wcs.wcs_pix2world(300, _jy, 0)

# Wrap longitudes to [-30, 30]
_l_axis = ((_l_axis + 180) % 360) - 180

_colors = plt.cm.viridis(_np.linspace(0, 1, len(_bins_to_show)))
# Latitude profile: average over l for each b
for _ic, _eb in enumerate(_bins_to_show):
    _r = _resid_cube[_eb]
    _b_prof = _np.nanmean(_r, axis=1)  # mean across l
    axes2[0].plot(_b_axis, _b_prof, color=_colors[_ic], linewidth=1.5,
                  label=f'bin {_eb}: {E[_eb]:.2f} GeV')
axes2[0].axhline(0, color='k', linestyle='--', linewidth=0.5)
axes2[0].set_xlabel('b [deg]')
axes2[0].set_ylabel('mean residual fraction (over l)')
axes2[0].set_title('Latitude profile of residual')
axes2[0].set_ylim(-0.3, 0.3)
axes2[0].legend(fontsize=9)
axes2[0].grid(True, alpha=0.3)

# Longitude profile: average over b
for _ic, _eb in enumerate(_bins_to_show):
    _r = _resid_cube[_eb]
    _l_prof = _np.nanmean(_r, axis=0)  # mean across b
    axes2[1].plot(_l_axis, _l_prof, color=_colors[_ic], linewidth=1.5,
                  label=f'bin {_eb}: {E[_eb]:.2f} GeV')
axes2[1].axhline(0, color='k', linestyle='--', linewidth=0.5)
axes2[1].set_xlabel('l [deg]')
axes2[1].set_ylabel('mean residual fraction (over b)')
axes2[1].set_title('Longitude profile of residual')
axes2[1].set_ylim(-0.3, 0.3)
axes2[1].invert_xaxis()  # GLON-CAR convention: positive l goes left
axes2[1].legend(fontsize=9)
axes2[1].grid(True, alpha=0.3)

fig2.suptitle(f'Residual profiles  —  Model {_M}', fontsize=14)
fig2.tight_layout()
_out2 = f'./spatial_residual_profiles_{_M}.png'
fig2.savefig(_out2, dpi=120)
plt.show()
print(f"[OK] Saved {_out2}")

# ---- Summary stats ----
print()
print("=" * 72)
print(f"Spatial residual summary — Model {_M}")
print("=" * 72)
print(f"  {'bin':>3} {'E[GeV]':>8} {'mean_res':>10} {'std_res':>10} "
      f"{'%>+5':>8} {'%<-5':>8}")
for _eb in range(_n_e):
    _r = _resid_cube[_eb]
    _msk = _mask_roi[_eb] > 0
    _r_v = _r[_msk]
    _r_v = _r_v[~_np.isnan(_r_v)]
    if len(_r_v) == 0:
        continue
    _frac_pos = _np.sum(_r_v > 0.05) / len(_r_v)
    _frac_neg = _np.sum(_r_v < -0.05) / len(_r_v)
    print(f"  {_eb:>3} {E[_eb]:>8.3f} {_r_v.mean():>+10.4f} "
          f"{_r_v.std():>10.4f} {_frac_pos*100:>7.1f}% {_frac_neg*100:>7.1f}%")

print()
print("  Reading the plot:")
print("    RED pixels: data > model (model is missing flux there)")
print("    BLUE pixels: data < model (model is over-predicting there)")
print("    All gray/black: masked out")
print("    Look for spatial coherence: disk band, central blob, edge effects")
print("    Latitude profile: peak at b≈0 → galactic disk under-fit")
print("    Longitude profile: peak at l≈0 → galactic center under-fit")


In [ ]:
# ============================================================================
# V12. Direct comparison with Sanghwan's 12yr input files
# ----------------------------------------------------------------------------
# Ground truth: /home/sanghwan/FermiLAT/Fermi-LAT-GCE/Analysis_data/12yr/
#                 evtype_front_evclass_clean/
# Confirms (or refutes) the "input data mismatch" hypothesis from memory #2.
#
# If totals match within 1% → inputs are consistent, offset is in processing
# If totals differ by ~10% → found the root cause of ~13% pipeline offset
# ============================================================================
import os as _os
import numpy as _np
import matplotlib.pyplot as plt
from astropy.io import fits as _fits
from astropy.wcs import WCS as _WCS

_SANG_DIR = '/home/sanghwan/FermiLAT/Fermi-LAT-GCE/Analysis_data/12yr/evtype_front_evclass_clean'
_SANG_CCUBE  = f'{_SANG_DIR}/GC_ccube_60x60_GAL_CAR.fits'
_SANG_LTCUBE = f'{_SANG_DIR}/Allsky_ltcube.fits'
_SANG_BINDEF = f'{_SANG_DIR}/bin_definitions.fits'
_SANG_SELECT = f'{_SANG_DIR}/Allsky_select.fits'
_SANG_GTI    = f'{_SANG_DIR}/Allsky_gti.fits'

_OUR_CCUBE  = './GC_analysis_sanghwan/GC_ccube_12yr_front_clean.fits'
_OUR_LTCUBE = './GC_analysis_sanghwan/GC_ltcube_12yr_front_clean.fits'

print("=" * 80)
print("V12: Direct comparison with Sanghwan's input files")
print("=" * 80)

# ============================================================================
# CHECK 1: File existence and basic accessibility
# ============================================================================
print()
print("Accessibility check:")
for _tag, _p in [
    ('SANG CCUBE',  _SANG_CCUBE),
    ('SANG LTCUBE', _SANG_LTCUBE),
    ('SANG BIN_DEF', _SANG_BINDEF),
    ('SANG SELECT', _SANG_SELECT),
    ('SANG GTI',    _SANG_GTI),
    ('OUR CCUBE',   _OUR_CCUBE),
    ('OUR LTCUBE',  _OUR_LTCUBE),
]:
    if _os.path.exists(_p):
        _s = _os.path.getsize(_p)
        print(f"  ✓ {_tag:<12}: {_s:>12,} bytes  {_p}")
    else:
        print(f"  ✗ {_tag:<12}: NOT FOUND  {_p}")

# ============================================================================
# CHECK 2: Energy bin definitions — ARE WE USING THE SAME BINNING?
# ============================================================================
print()
print("-" * 80)
print("CHECK 2: Energy bin definitions")
print("-" * 80)

if _os.path.exists(_SANG_BINDEF):
    _sb = _fits.open(_SANG_BINDEF)
    print(f"  Sanghwan bin_definitions.fits:")
    for _h in _sb:
        if hasattr(_h, 'name') and _h.name:
            print(f"    HDU: {_h.name}  nrows={_h.data.shape if _h.data is not None else 'None'}")
    # Usually bin_definitions is in extension 1 (or 'EBOUNDS')
    for _hdu in _sb[1:]:
        if _hdu.data is not None and 'E_MIN' in _hdu.data.dtype.names:
            _sang_emin = _hdu.data['E_MIN'] / 1000   # MeV → GeV
            _sang_emax = _hdu.data['E_MAX'] / 1000
            print(f"\n  Sanghwan's {len(_sang_emin)} energy bins [GeV]:")
            for _i in range(len(_sang_emin)):
                print(f"    bin {_i:>2}: {_sang_emin[_i]:>10.4f} – {_sang_emax[_i]:>10.4f}")
            break
    else:
        print(f"    [warn] could not parse E_MIN/E_MAX")
        _sang_emin = None
    _sb.close()

    # Compare with our E array
    if _sang_emin is not None:
        print(f"\n  Our 14-bin E array (from V1):")
        for _i, _e in enumerate(E):
            print(f"    bin {_i:>2}: center E = {_e:.4f} GeV,  dE = {delta_E[_i]:.4f} GeV")
        # Key check: do our bin centers match √(E_min*E_max) of Sanghwan's?
        if len(_sang_emin) == len(E):
            print(f"\n  Bin-by-bin comparison:")
            for _i, _e in enumerate(E):
                _sang_ctr = _np.sqrt(_sang_emin[_i] * _sang_emax[_i])
                _ratio = _e / _sang_ctr
                print(f"    bin {_i:>2}: ours E={_e:.4f}, Sang ctr={_sang_ctr:.4f}, "
                      f"ratio={_ratio:.4f}")
        else:
            print(f"\n  ⚠ Different number of bins! ours={len(E)}, Sang={len(_sang_emin)}")

# ============================================================================
# CHECK 3: CCUBE — shape, total counts, per-bin counts
# ============================================================================
print()
print("-" * 80)
print("CHECK 3: CCUBE comparison (the decisive check)")
print("-" * 80)

if _os.path.exists(_SANG_CCUBE) and _os.path.exists(_OUR_CCUBE):
    _sh = _fits.open(_SANG_CCUBE)
    _oh = _fits.open(_OUR_CCUBE)
    _sdata = _sh[0].data
    _odata = _oh[0].data
    _shdr = _sh[0].header
    _ohdr = _oh[0].header

    print(f"  Shapes: ours={_odata.shape},  Sanghwan={_sdata.shape}")
    print(f"  WCS key comparison:")
    for _k in ['NAXIS1', 'NAXIS2', 'NAXIS3',
              'CDELT1', 'CDELT2', 'CRVAL1', 'CRVAL2',
              'CRPIX1', 'CRPIX2', 'CTYPE1', 'CTYPE2']:
        _o = _ohdr.get(_k, '?')
        _s = _shdr.get(_k, '?')
        _match = "✓" if _o == _s else "✗"
        print(f"    {_k:>8} {_match}  ours={_o}  Sang={_s}")

    # Total counts
    print(f"\n  Total counts:")
    print(f"    ours:     {_odata.sum():>14,.0f}")
    print(f"    Sanghwan: {_sdata.sum():>14,.0f}")
    _total_ratio = _odata.sum() / _sdata.sum()
    print(f"    ratio:    {_total_ratio:>14.4f}")
    if 0.98 < _total_ratio < 1.02:
        print(f"    ✓ Total counts match within 2%. CCUBE level consistent.")
    elif 0.9 < _total_ratio < 1.1:
        print(f"    ⚠ {(_total_ratio-1)*100:+.1f}% total count difference — "
              f"possible gtmktime/GTI difference.")
    else:
        print(f"    ✗ Large difference {(_total_ratio-1)*100:+.0f}% — "
              f"check gtselect/gtmktime cuts.")

    # Per-bin
    if _odata.shape[0] == _sdata.shape[0]:
        print(f"\n  Per-bin total counts:")
        print(f"    {'bin':>3} {'ours':>14} {'Sanghwan':>14} {'ratio':>8}")
        for _i in range(_odata.shape[0]):
            _o_sum = float(_odata[_i].sum())
            _s_sum = float(_sdata[_i].sum())
            _r = _o_sum / _s_sum if _s_sum > 0 else _np.nan
            print(f"    {_i:>3} {_o_sum:>14,.0f} {_s_sum:>14,.0f} {_r:>8.4f}")

    # Spatial pixel-by-pixel ratio for a mid-energy bin
    print(f"\n  Pixel-by-pixel ratio at bin 5 (mid-energy):")
    if _odata.shape[0] > 5 and _odata.shape[1:] == _sdata.shape[1:]:
        _b = 5
        # Avoid div-by-zero: only compare where both > 0
        _om = _odata[_b]
        _sm = _sdata[_b]
        _pos = (_om > 0) & (_sm > 0)
        if _pos.sum() > 0:
            _pixel_ratio = _om[_pos] / _sm[_pos]
            print(f"    median ratio: {_np.median(_pixel_ratio):.4f}")
            print(f"    mean ratio:   {_np.mean(_pixel_ratio):.4f}")
            print(f"    std ratio:    {_np.std(_pixel_ratio):.4f}")
            print(f"    (pixels with both counts > 0: {_pos.sum()} / {_om.size})")
            # Plot ratio map
            fig, ax = plt.subplots(figsize=(8, 7))
            _ratio_map = _np.ones_like(_om, dtype=float)
            _ratio_map[_pos] = _om[_pos] / _sm[_pos]
            _im = ax.imshow(_ratio_map, origin='lower', cmap='RdBu_r',
                            vmin=0.5, vmax=1.5)
            ax.set_title(f'Pixel ratio ours/Sanghwan at bin {_b}\n'
                         f'median={_np.median(_pixel_ratio):.3f}')
            plt.colorbar(_im, ax=ax, label='ours / Sanghwan')
            ax.set_xlabel('pixel x'); ax.set_ylabel('pixel y')
            _out = './ccube_ratio_map_b5.png'
            fig.savefig(_out, dpi=120)
            plt.show()
            print(f"    [OK] Saved {_out}")
    _sh.close()
    _oh.close()
else:
    if not _os.path.exists(_SANG_CCUBE):
        print(f"  ✗ Sanghwan CCUBE not found at {_SANG_CCUBE}")
    if not _os.path.exists(_OUR_CCUBE):
        print(f"  ✗ Our CCUBE not found at {_OUR_CCUBE}")

# ============================================================================
# CHECK 4: LTCUBE comparison
# ============================================================================
print()
print("-" * 80)
print("CHECK 4: LTCUBE comparison")
print("-" * 80)

if _os.path.exists(_SANG_LTCUBE) and _os.path.exists(_OUR_LTCUBE):
    _sh = _fits.open(_SANG_LTCUBE)
    _oh = _fits.open(_OUR_LTCUBE)

    # LTCUBE is HEALPix; compare TSTART/TSTOP and total livetime
    print(f"  Our LTCUBE:")
    print(f"    TSTART: {_oh[0].header.get('TSTART', '?')}")
    print(f"    TSTOP:  {_oh[0].header.get('TSTOP',  '?')}")
    print(f"    TELAPSE: {_oh[0].header.get('TELAPSE', '?')}")

    print(f"\n  Sanghwan LTCUBE:")
    print(f"    TSTART: {_sh[0].header.get('TSTART', '?')}")
    print(f"    TSTOP:  {_sh[0].header.get('TSTOP',  '?')}")
    print(f"    TELAPSE: {_sh[0].header.get('TELAPSE', '?')}")

    # Mission elapsed time in seconds:
    # Fermi MET: seconds since 2001-01-01 00:00:00 UTC
    _our_tel = _oh[0].header.get('TELAPSE', 0)
    _sang_tel = _sh[0].header.get('TELAPSE', 0)
    if _our_tel and _sang_tel:
        _tel_ratio = _our_tel / _sang_tel
        _our_years = _our_tel / 3.156e7
        _sang_years = _sang_tel / 3.156e7
        print(f"\n  Elapsed time:")
        print(f"    ours:     {_our_tel:,.0f} s  ({_our_years:.3f} yr)")
        print(f"    Sanghwan: {_sang_tel:,.0f} s  ({_sang_years:.3f} yr)")
        print(f"    ratio:    {_tel_ratio:.4f}")
        if abs(_tel_ratio - 1) < 0.01:
            print(f"    ✓ Same time range to within 1%")
        elif abs(_tel_ratio - 1) < 0.05:
            print(f"    ⚠ {(_tel_ratio-1)*100:+.1f}% time-range difference")
        else:
            print(f"    ✗ Very different time ranges — different gtmktime cuts")

    # Try to compare the livetime data itself (binned HEALPix)
    try:
        _ol = _oh[1].data['COSBINS']
        _sl = _sh[1].data['COSBINS']
        _ol_sum = _np.sum(_ol)
        _sl_sum = _np.sum(_sl)
        print(f"\n  Total livetime sum (all pixels × all cosθ bins):")
        print(f"    ours:     {_ol_sum:.3e}")
        print(f"    Sanghwan: {_sl_sum:.3e}")
        print(f"    ratio:    {_ol_sum/_sl_sum:.4f}")
    except Exception as _e:
        print(f"  [info] COSBINS comparison skipped: {_e}")

    _sh.close(); _oh.close()
else:
    if not _os.path.exists(_SANG_LTCUBE):
        print(f"  ✗ Sanghwan LTCUBE not found")

# ============================================================================
# SUMMARY
# ============================================================================
print()
print("=" * 80)
print("INTERPRETATION")
print("=" * 80)
print("""
  If CCUBE total counts match within 1%:
    → Memory #2 hypothesis ('input data mismatch') is REFUTED.
    → Offset is somewhere in processing: exposure, component maps, or fit.

  If CCUBE counts differ by ~10%:
    → Memory #2 hypothesis CONFIRMED.
    → Root cause: different gtselect/gtmktime/photon file versions.
    → Fix: rerun our pipeline with Sanghwan's input files or match his cuts.

  If per-bin counts vary (some bins match, some don't):
    → Different energy binning or different event class/type cuts.

  If pixel-ratio map is spatially structured (e.g., ring pattern):
    → Different exposure computation; check LTCUBE closely.
""")


In [ ]:
# ============================================================================
# V12a. Sanghwan file inventory + CCUBE comparison (updated paths)
# ----------------------------------------------------------------------------
# Confirmed Sanghwan data locations (from user's ls):
#   CCUBE:    /home/sanghwan/FermiLAT/Fermi-LAT-GCE/Analysis_data/12yr/
#             evtype_front_evclass_clean/GC_ccube_60x60_GAL_CAR.fits
#   LTCUBE:   same dir, Allsky_ltcube.fits
#   Results:  /home/sanghwan/FermiLAT/Sanghwan/
#             GCE_model_{M}_12yr_cholis.dat  (for Models X, II, XLIX, etc.)
#             GCE_model_I_12yr.dat           (I uses different pattern)
#
# This cell:
#   1. Inventory Sanghwan/Model/, Templates/, GCE_references/
#   2. Direct CCUBE comparison (ours vs Sanghwan)
# ============================================================================
import os as _os
import subprocess as _sp
import numpy as _np
import matplotlib.pyplot as plt
from astropy.io import fits as _fits

SANGHWAN_BASE    = '/home/sanghwan/FermiLAT'
SANGHWAN_RESULTS = f'{SANGHWAN_BASE}/Sanghwan'
SANGHWAN_DATA    = f'{SANGHWAN_BASE}/Fermi-LAT-GCE/Analysis_data/12yr/evtype_front_evclass_clean'

_OURS_CCUBE = './GC_analysis_sanghwan/GC_ccube_12yr_front_clean.fits'
_SANG_CCUBE = f'{SANGHWAN_DATA}/GC_ccube_60x60_GAL_CAR.fits'

# ----------------------------------------------------------------------------
# Part 1: Inventory key sub-directories in Sanghwan/
# ----------------------------------------------------------------------------
print("=" * 78)
print("PART 1: Inventory of Sanghwan/Model, Templates, GCE_references")
print("=" * 78)

for _d_name in ['Model', 'Templates', 'GCE_references']:
    _d = f'{SANGHWAN_RESULTS}/{_d_name}'
    if _os.path.isdir(_d):
        _files = sorted(_os.listdir(_d))
        print(f"\n  {_d}  ({len(_files)} entries):")
        for _f in _files[:30]:
            _p = _os.path.join(_d, _f)
            if _os.path.isfile(_p):
                _sz = _os.path.getsize(_p)
                _sz_str = f"{_sz/1e6:.1f}M" if _sz > 1e6 else f"{_sz/1e3:.0f}K" if _sz > 1e3 else f"{_sz}B"
                print(f"    [F] {_f}   ({_sz_str})")
            else:
                print(f"    [D] {_f}/")
        if len(_files) > 30:
            print(f"    ... and {len(_files)-30} more")
    else:
        print(f"\n  {_d}: not found")

# ----------------------------------------------------------------------------
# Part 2: Find all 12yr .dat files available
# ----------------------------------------------------------------------------
print()
print("=" * 78)
print("PART 2: Available 12yr GCE .dat files in Sanghwan/")
print("=" * 78)

_dat_12yr = []
for _f in sorted(_os.listdir(SANGHWAN_RESULTS)):
    if '12yr' in _f and _f.endswith('.dat'):
        _dat_12yr.append(_f)
print(f"  Found {len(_dat_12yr)} 12yr .dat files:")
for _f in _dat_12yr:
    _sz = _os.path.getsize(f'{SANGHWAN_RESULTS}/{_f}')
    print(f"    {_f}   ({_sz}B)")

# Inventory which Roman-numeral models have 12yr results
_models_with_12yr = []
for _f in _dat_12yr:
    if 'cholis' in _f:
        # e.g. GCE_model_X_12yr_cholis.dat
        _parts = _f.replace('.dat', '').split('_')
        if len(_parts) >= 4 and _parts[0] == 'GCE':
            _models_with_12yr.append(_parts[2])
print(f"\n  Models with GCE_model_<M>_12yr_cholis.dat format:")
print(f"    {_models_with_12yr}")

# Model I has different name
if any('GCE_model_I_12yr.dat' in _f for _f in _dat_12yr):
    print(f"  Model I:  GCE_model_I_12yr.dat  (no '_cholis' suffix)")

# ----------------------------------------------------------------------------
# Part 3: ⭐ DIRECT CCUBE COMPARISON
# ----------------------------------------------------------------------------
print()
print("=" * 78)
print("PART 3: ⭐ CCUBE comparison — ours vs Sanghwan")
print("=" * 78)

if not _os.path.exists(_SANG_CCUBE):
    print(f"  ✗ {_SANG_CCUBE} not found")
elif not _os.path.exists(_OURS_CCUBE):
    print(f"  ✗ {_OURS_CCUBE} not found")
else:
    _ours_hdu = _fits.open(_OURS_CCUBE)
    _sang_hdu = _fits.open(_SANG_CCUBE)
    _ours = _ours_hdu[0].data
    _sang = _sang_hdu[0].data
    print(f"  OURS     shape = {_ours.shape}")
    print(f"  SANGHWAN shape = {_sang.shape}")

    if _ours.shape != _sang.shape:
        print(f"  ⚠ Shape mismatch — cannot compare pixel-by-pixel")
        # Still compare totals per bin if at least n_e matches
        if _ours.shape[0] == _sang.shape[0]:
            print(f"\n  Per-bin total counts (full map):")
            print(f"  {'bin':>3} {'ours':>12} {'sanghwan':>12} {'ratio':>8}")
            for _i in range(_ours.shape[0]):
                _o = _np.sum(_ours[_i])
                _s = _np.sum(_sang[_i])
                _r = _o / _s if _s > 0 else _np.nan
                print(f"  {_i:>3} {_o:>12,.0f} {_s:>12,.0f} {_r:>8.4f}")
    else:
        _tot_o = _np.sum(_ours)
        _tot_s = _np.sum(_sang)
        print(f"\n  Total counts:")
        print(f"    OURS     = {_tot_o:,.0f}")
        print(f"    SANGHWAN = {_tot_s:,.0f}")
        print(f"    ratio    = {_tot_o/_tot_s:.4f}")

        _roi = slice(100, 500)
        print(f"\n  Per-bin counts in 60×60 ROI:")
        print(f"  {'bin':>3} {'ours':>12} {'sanghwan':>12} {'ratio':>8} {'status':>8}")
        _ratios = []
        for _i in range(_ours.shape[0]):
            _o = _np.sum(_ours[_i, _roi, _roi])
            _s = _np.sum(_sang[_i, _roi, _roi])
            _r = _o / _s if _s > 0 else _np.nan
            _ratios.append(_r)
            _status = "✓" if 0.95 < _r < 1.05 else "⚠"
            print(f"  {_i:>3} {_o:>12,.0f} {_s:>12,.0f} {_r:>8.4f} {_status:>8}")
        _ratios = _np.array(_ratios)

        # WCS comparison
        _ho, _hs = _ours_hdu[0].header, _sang_hdu[0].header
        print(f"\n  WCS header comparison:")
        for _k in ['CDELT1', 'CDELT2', 'CRVAL1', 'CRVAL2', 'CRPIX1', 'CRPIX2',
                  'CTYPE1', 'CTYPE2']:
            _o_val = _ho.get(_k); _s_val = _hs.get(_k)
            _eq = "=" if _o_val == _s_val else "≠"
            print(f"    {_k:>8}: ours={_o_val}  {_eq}  sanghwan={_s_val}")

        # Spatial difference plot
        fig, axes = plt.subplots(2, 3, figsize=(15, 8))
        for _c, _b in enumerate([0, 5, 10]):
            _o_img = _ours[_b, _roi, _roi]
            _s_img = _sang[_b, _roi, _roi]
            _r_img = (_o_img.astype(float) - _s_img.astype(float)) / _np.maximum(_s_img, 1)
            _vmax = max(_o_img.max(), _s_img.max())
            axes[0, _c].imshow(_o_img, origin='lower', cmap='inferno', vmin=0, vmax=_vmax)
            axes[0, _c].set_title(f'OURS bin {_b}\n({_np.sum(_o_img):,.0f} counts)')
            axes[0, _c].grid(False)
            axes[1, _c].imshow(_r_img, origin='lower', cmap='RdBu_r', vmin=-0.1, vmax=0.1)
            axes[1, _c].set_title(f'(ours-sang)/sang bin {_b}\n'
                                  f'mean={_np.nanmean(_r_img):+.3f}')
            axes[1, _c].grid(False)
        fig.suptitle('CCUBE: ours vs Sanghwan', fontsize=13)
        fig.tight_layout()
        _out = './ccube_ours_vs_sanghwan.png'
        fig.savefig(_out, dpi=120)
        plt.show()
        print(f"\n  [OK] Saved {_out}")

        # Verdict
        _mean_ratio = _np.nanmean(_ratios)
        print()
        print(f"  VERDICT:")
        if 0.98 < _mean_ratio < 1.02:
            print(f"    ✓ CCUBE counts agree within 2% (mean ratio {_mean_ratio:.4f})")
            print(f"      → Data is essentially identical.")
            print(f"      → 13% pipeline offset is NOT caused by CCUBE differences.")
            print(f"      → Focus must shift to fit logic, component maps, or mask.")
        elif 0.90 < _mean_ratio < 1.10:
            print(f"    ⚠ CCUBE differs by {(_mean_ratio-1)*100:+.1f}% on average")
            print(f"      → Partial explanation of 13% offset possible.")
        else:
            print(f"    ✗ Large CCUBE difference: ratio = {_mean_ratio:.3f}")

    _ours_hdu.close()
    _sang_hdu.close()


In [ ]:
# ============================================================================
# V12b. GCE flux comparison: ours vs Sanghwan vs Cholis Zenodo (v2, clean layout)
# ----------------------------------------------------------------------------
# Layout fix: title previously dumped a Python dict (overlapping text). Now
# each panel shows:  "Model X:  ours/sang=X.XXX, ours/cholis=X.XXX"
# in two clean lines with smaller font.
#
# Panels are sorted by |ours/sang - 1| ascending (best agreement first).
# ============================================================================
import os as _os
import numpy as _np
import matplotlib.pyplot as plt

SANGHWAN_RESULTS = '/home/sanghwan/FermiLAT/Sanghwan'
CHOLIS_ZEN = '../GCE_TEMPLATES_FILES_v3/Figures_12_and_14_GCE_Spectra'

MODELS_TO_COMPARE = ['X', 'I', 'XLIX', 'II', 'XV', 'XLVIII', 'LIII']

print("=" * 78)
print("V12b: GCE flux — ours vs Sanghwan vs Cholis Zenodo (3-way)")
print("=" * 78)

def _sang_path(m):
    if m == 'I':
        p = f'{SANGHWAN_RESULTS}/GCE_model_I_12yr.dat'
    else:
        p = f'{SANGHWAN_RESULTS}/GCE_model_{m}_12yr_cholis.dat'
    return p if _os.path.exists(p) else None

def _ours_path(m):
    p = f'./GCE_model_{m}_12yr_cholis.dat'
    return p if _os.path.exists(p) else None

def _cholis_path(m):
    p = f'{CHOLIS_ZEN}/GCE_Model{m}_flux_Inner40x40_masked_disk.dat'
    return p if _os.path.exists(p) else None

_triples = []
for _m in MODELS_TO_COMPARE:
    _triples.append((_m, _ours_path(_m), _sang_path(_m), _cholis_path(_m)))

# First pass: compute all ratios + data for sorting
_data_all = []
for _m, _op, _sp, _cp in _triples:
    if _op is None or _sp is None:
        continue
    _ours_d = _np.loadtxt(_op)
    _sang_d = _np.loadtxt(_sp)
    _E = _ours_d[:, 0]
    _Es = _sang_d[:, 0]
    _m110 = (_E >= 1.0) & (_E <= 10.0)
    _m110_s = (_Es >= 1.0) & (_Es <= 10.0)
    _sang_interp = _np.interp(_E, _Es, _sang_d[:, 1])
    _r_os = _np.nanmean((_ours_d[:, 1] / _sang_interp)[_m110])
    _r_oc = _np.nan
    _r_sc = _np.nan
    _cholis_d = None
    if _cp:
        _cholis_d = _np.loadtxt(_cp)
        if _cholis_d.ndim == 2 and _cholis_d.shape[1] >= 4:
            _cholis_interp_o = _np.interp(_E, _cholis_d[:, 0], _cholis_d[:, 1])
            _cholis_interp_s = _np.interp(_Es, _cholis_d[:, 0], _cholis_d[:, 1])
            _r_oc = _np.nanmean((_ours_d[:, 1] / _cholis_interp_o)[_m110])
            _r_sc = _np.nanmean((_sang_d[:, 1] / _cholis_interp_s)[_m110_s])
    _data_all.append({
        'model': _m, 'ours': _ours_d, 'sang': _sang_d, 'cholis': _cholis_d,
        'r_os': _r_os, 'r_oc': _r_oc, 'r_sc': _r_sc,
        'quality': abs(_r_os - 1.0),  # closer to 1 = better agreement
    })

# Sort by agreement quality (best first)
_data_all.sort(key=lambda d: d['quality'])

if not _data_all:
    print("  No model has both ours + sanghwan .dat — nothing to plot.")
else:
    _nplot = len(_data_all)
    _ncols = min(3, _nplot)
    _nrows = (_nplot + _ncols - 1) // _ncols
    fig, axes = plt.subplots(_nrows, _ncols, figsize=(5.5*_ncols, 5.0*_nrows),
                             squeeze=False)
    axes_flat = axes.flat

    for _idx, _d in enumerate(_data_all):
        _ax = axes_flat[_idx]
        _m = _d['model']
        _ours_d = _d['ours']; _sang_d = _d['sang']; _cholis_d = _d['cholis']
        _E = _ours_d[:, 0]; _Es = _sang_d[:, 0]

        # haebarg
        _yerr_o = [_np.maximum(_ours_d[:, 1] - _ours_d[:, 3], 0),
                   _np.maximum(_ours_d[:, 4] - _ours_d[:, 1], 0)]
        _ax.errorbar(_E, _ours_d[:, 1], yerr=_yerr_o,
                     marker='s', color='crimson', markersize=5, linewidth=1.5,
                     label='haebarg', capsize=3)

        # sanghwan
        if _sang_d.shape[1] >= 5:
            _yerr_s = [_np.maximum(_sang_d[:, 1] - _sang_d[:, 3], 0),
                       _np.maximum(_sang_d[:, 4] - _sang_d[:, 1], 0)]
            _ax.errorbar(_Es, _sang_d[:, 1], yerr=_yerr_s,
                         marker='o', color='royalblue', markersize=5, linewidth=1.5,
                         label='sanghwan', capsize=3, alpha=0.85)
        else:
            _ax.plot(_Es, _sang_d[:, 1], 'o-', color='royalblue', markersize=5,
                     label='sanghwan', alpha=0.85)

        # cholis
        if _cholis_d is not None and _cholis_d.shape[1] >= 4:
            _ax.fill_between(_cholis_d[:, 0], _cholis_d[:, 2], _cholis_d[:, 3],
                             alpha=0.2, color='gray', label='cholis 1σ band')
            _ax.plot(_cholis_d[:, 0], _cholis_d[:, 1], '--', color='black',
                     linewidth=1, label='cholis best-fit')

        # ---- Clean title: two lines, small font ----
        _r_os  = _d['r_os']
        _r_oc  = _d['r_oc']
        _r_sc  = _d['r_sc']
        _verdict = ("✓" if abs(_r_os - 1.0) < 0.05
                    else "⚠" if abs(_r_os - 1.0) < 0.15
                    else "✗")
        _title_l1 = f"Model {_m}  {_verdict}"
        if not _np.isnan(_r_oc):
            _title_l2 = (f"ours/sang = {_r_os:.3f}   "
                         f"ours/cholis = {_r_oc:.3f}   "
                         f"sang/cholis = {_r_sc:.3f}")
        else:
            _title_l2 = f"ours/sang = {_r_os:.3f}"
        _ax.set_title(f"{_title_l1}\n{_title_l2}", fontsize=10)

        _ax.set_xscale('log'); _ax.set_yscale('log')
        _ax.set_xlim(0.3, 100); _ax.set_ylim(1e-8, 3e-6)
        _ax.set_xlabel('E [GeV]')
        _ax.set_ylabel(r'$E^2\,dN/dE$  [GeV cm$^{-2}$ s$^{-1}$ sr$^{-1}$]')
        _ax.legend(fontsize=8, loc='lower left')
        _ax.grid(True, which='major', alpha=0.3)

    # Hide unused subplots
    for _i in range(_nplot, len(axes_flat)):
        axes_flat[_i].axis('off')

    fig.suptitle('GCE flux comparison: haebarg vs Sanghwan vs Cholis  (12yr)\n'
                 'panels sorted by |ours/sang − 1| (best agreement first)',
                 fontsize=13, y=0.995)
    fig.tight_layout()
    _out = './gce_flux_3way_comparison.png'
    fig.savefig(_out, dpi=120)
    plt.show()
    print(f"\n  [OK] Saved {_out}")

    # ---- Text summary table (sorted) ----
    print()
    print(f"  Summary table — mean ratio 1-10 GeV, sorted by agreement with Sanghwan:")
    print(f"  {'Model':<8} {'ours/sang':>12} {'ours/cholis':>14} "
          f"{'sang/cholis':>14} {'verdict':>10}")
    for _d in _data_all:
        _m = _d['model']; _r_os = _d['r_os']; _r_oc = _d['r_oc']; _r_sc = _d['r_sc']
        _v = ("✓ excellent" if abs(_r_os - 1.0) < 0.05
              else "⚠ marginal"  if abs(_r_os - 1.0) < 0.15
              else "✗ poor")
        _oc_str = f"{_r_oc:.3f}" if not _np.isnan(_r_oc) else "-"
        _sc_str = f"{_r_sc:.3f}" if not _np.isnan(_r_sc) else "-"
        print(f"  {_m:<8} {_r_os:>12.3f} {_oc_str:>14} {_sc_str:>14} {_v:>12}")

    # ---- Group-based interpretation ----
    _good = [d for d in _data_all if abs(d['r_os'] - 1.0) < 0.05]
    _marg = [d for d in _data_all if 0.05 <= abs(d['r_os'] - 1.0) < 0.15]
    _poor = [d for d in _data_all if abs(d['r_os'] - 1.0) >= 0.15]

    print()
    print(f"  Groups by |ours/sang − 1|:")
    print(f"    ✓ excellent (<5%):  {[d['model'] for d in _good]}")
    print(f"    ⚠ marginal  (5-15%): {[d['model'] for d in _marg]}")
    print(f"    ✗ poor      (>15%): {[d['model'] for d in _poor]}")
    print()
    if len(_good) > 0 and len(_marg) + len(_poor) > 0:
        print(f"  OBSERVATION: agreement is model-dependent.")
        print(f"  Some models agree with Sanghwan to within 5% (pipeline CAN work),")
        print(f"  but others (like {[d['model'] for d in _poor + _marg]}) diverge.")
        print(f"  → The ~13% offset is NOT a global systematic — it's specific to")
        print(f"    certain GDE models. This suggests the issue is in how those")
        print(f"    particular galprop models get processed (MapCube conversion,")
        print(f"    or the XML model specification for those models).")


In [ ]:
# ============================================================================
# V12d v2. Sanghwan MapCube content comparison (fixed matching + gzip)
# ----------------------------------------------------------------------------
# Sanghwan stores MapCubes as:
#   /Pion/pion_mapcube_model<ROMAN>.gz      (16 MB)
#   /Bremss/bremss_mapcube_model<ROMAN>.gz  (16 MB)
#   /ICs/ics_mapcube_model<ROMAN>.gz        (10 MB)
# Our MapCubes:
#   ./MapCubes/{pion,bremss,ics}_mapcube_model<ROMAN>.fits  (8.76 MB each)
#
# Size difference suggests different structure. Compare:
#   - HDU structure (ENERGIES extension? Shape?)
#   - Spatial shape: Sanghwan may be larger (all-sky or bigger ROI) vs ours (60x60)
#   - Pixel values in a common sub-region, with proper WCS alignment
# ============================================================================
import os as _os
import gzip as _gzip
import tempfile as _tf
import numpy as _np
from astropy.io import fits as _fits

SANGHWAN_MAPCUBES = '/home/sanghwan/FermiLAT/Fermi-LAT-GCE-Analysis/Fermi-LAT-Galactic-Center-Excess-analysis/Model_maps/GDE_maps/GALPROP_Mapcubes/Mapcubes'

GROUP_A = ['XLIX', 'II', 'XV']
GROUP_B = ['X', 'XLVIII', 'LIII']
GROUP_C = ['I']

# Exact filename map (no substring match!)
def _sang_path(comp, roman):
    """Returns Sanghwan's gz path for (component, Roman numeral)."""
    _subdir = {'pion': 'Pion', 'bremss': 'Bremss', 'ics': 'ICs'}[comp]
    _p = f'{SANGHWAN_MAPCUBES}/{_subdir}/{comp}_mapcube_model{roman}.gz'
    return _p if _os.path.exists(_p) else None

def _our_path(comp, roman):
    _p = f'./MapCubes/{comp}_mapcube_model{roman}.fits'
    return _p if _os.path.exists(_p) else None

def _load_sang(path):
    """Decompress .gz to a tempfile then open as FITS."""
    with _gzip.open(path, 'rb') as _fin:
        _raw = _fin.read()
    _tmp = _tf.NamedTemporaryFile(suffix='.fits', delete=False)
    _tmp.write(_raw); _tmp.close()
    _hdul = _fits.open(_tmp.name)
    return _hdul, _tmp.name

# ----------------------------------------------------------------------------
# Part 1: HDU structure comparison (1 model, 1 component)
# ----------------------------------------------------------------------------
print("=" * 80)
print("V12d v2 — Part 1: HDU / structure comparison")
print("=" * 80)

_refmodel = 'XLIX'   # Group A reference
_refcomp  = 'pion'
_our_p = _our_path(_refcomp, _refmodel)
_sang_p = _sang_path(_refcomp, _refmodel)

print(f"\n  Reference: Model {_refmodel}, {_refcomp}")
print(f"    Our:  {_our_p}  ({_os.path.getsize(_our_p)/1e6:.2f} MB)")
print(f"    Sang: {_sang_p}  ({_os.path.getsize(_sang_p)/1e6:.2f} MB gz)")

# Load ours
_our_hdul = _fits.open(_our_p)
print(f"\n  OURS HDU list:")
for _i, _h in enumerate(_our_hdul):
    _shape = _h.data.shape if _h.data is not None else "None"
    print(f"    [{_i}] {_h.name:<12} shape={_shape}")
    if _i == 0 and _h.data is not None:
        print(f"         dtype={_h.data.dtype}")
        for _k in ['NAXIS1', 'NAXIS2', 'NAXIS3', 'CDELT1', 'CDELT2',
                  'CRVAL1', 'CRVAL2', 'CTYPE1', 'CTYPE2']:
            print(f"         {_k} = {_h.header.get(_k)}")

# Load Sanghwan's (decompress first)
_sang_hdul, _tmp_path = _load_sang(_sang_p)
print(f"\n  SANGHWAN HDU list (after gunzip):")
for _i, _h in enumerate(_sang_hdul):
    _shape = _h.data.shape if _h.data is not None else "None"
    print(f"    [{_i}] {_h.name:<12} shape={_shape}")
    if _i == 0 and _h.data is not None:
        print(f"         dtype={_h.data.dtype}")
        for _k in ['NAXIS1', 'NAXIS2', 'NAXIS3', 'CDELT1', 'CDELT2',
                  'CRVAL1', 'CRVAL2', 'CTYPE1', 'CTYPE2']:
            print(f"         {_k} = {_h.header.get(_k)}")

_our_cube = _our_hdul[0].data
_sang_cube = _sang_hdul[0].data

print(f"\n  Comparing data cubes:")
print(f"    OURS  shape={_our_cube.shape}  dtype={_our_cube.dtype}")
print(f"    SANG  shape={_sang_cube.shape}  dtype={_sang_cube.dtype}")

# Interpret shape
if _our_cube.shape == _sang_cube.shape:
    print(f"\n  ✓ Shapes match → direct pixel comparison possible")
    _direct = True
else:
    print(f"\n  ⚠ Shapes differ — likely different ROI or binning")
    print(f"    Sanghwan appears to be native Zenodo format")
    print(f"    Our MapCube is post-processed (60x60 crop, 17 bin)")
    _direct = False

# Try ENERGIES HDU
_our_e = None; _sang_e = None
for _h in _our_hdul:
    if _h.name == 'ENERGIES':
        _our_e = _h.data['Energy']
        break
for _h in _sang_hdul:
    if _h.name == 'ENERGIES':
        _sang_e = _h.data['Energy']
        break

if _our_e is not None and _sang_e is not None:
    print(f"\n  Energy bins:")
    print(f"    OURS  n={len(_our_e)}  range=[{_our_e.min():.1f}, {_our_e.max():.1f}]")
    print(f"    SANG  n={len(_sang_e)}  range=[{_sang_e.min():.1f}, {_sang_e.max():.1f}]")
    if len(_our_e) == len(_sang_e) and _np.allclose(_our_e, _sang_e):
        print(f"    ✓ Energy grids identical")
    else:
        print(f"    ⚠ Energy grids differ")

_sang_hdul.close()
_our_hdul.close()
_os.unlink(_tmp_path)

# ----------------------------------------------------------------------------
# Part 2: per-model file matching with EXACT Roman numeral
# ----------------------------------------------------------------------------
print()
print("=" * 80)
print("V12d v2 — Part 2: per-model file presence (exact match)")
print("=" * 80)

_all_tested = [(m, 'A') for m in GROUP_A] + [(m, 'B') for m in GROUP_B] + [(m, 'C') for m in GROUP_C]
print(f"\n  {'Group':<5} {'Model':<8} {'pion_our':>10} {'pion_sang':>12} "
      f"{'bremss_sang':>13} {'ics_sang':>10}")
_present_models = []
for _m, _g in _all_tested:
    _sizes = {}
    for _c in ['pion', 'bremss', 'ics']:
        _sp = _sang_path(_c, _m)
        _sizes[f'sang_{_c}'] = _os.path.getsize(_sp)/1e6 if _sp else 0
    _op = _our_path('pion', _m)
    _our_sz = _os.path.getsize(_op)/1e6 if _op else 0
    _print_sang = lambda k: f"{_sizes[k]:.2f}M" if _sizes[k] else "-"
    print(f"  {_g:<5} {_m:<8} {_our_sz:>8.2f}M  "
          f"{_print_sang('sang_pion'):>12} {_print_sang('sang_bremss'):>13} "
          f"{_print_sang('sang_ics'):>10}")
    if all(_sizes[k] > 0 for k in _sizes) and _our_sz > 0:
        _present_models.append((_m, _g))

print(f"\n  {len(_present_models)} models have both ours + Sanghwan (all 3 components)")

# ----------------------------------------------------------------------------
# Part 3: actual pixel content comparison — Group A vs Group B
# ----------------------------------------------------------------------------
print()
print("=" * 80)
print("V12d v2 — Part 3: Content comparison (pixel-level)")
print("=" * 80)

def _compare_cubes_properly(our_path, sang_path, label, our_E_bins=None):
    """Compare two MapCubes accounting for possible shape differences.
    Our cube is likely 60x60 (600x600 at 0.1°) or 240x240 (60x60 at 0.25°).
    Sanghwan may be native Zenodo format — need to find common ROI.
    """
    _our_hdu = _fits.open(our_path)
    _our_d = _our_hdu[0].data
    _our_hdr = _our_hdu[0].header
    _sang_hdul_l, _tmp = _load_sang(sang_path)
    _sang_d = _sang_hdul_l[0].data
    _sang_hdr = _sang_hdul_l[0].header

    print(f"\n  [{label}]")
    print(f"    ours  shape={_our_d.shape} dtype={_our_d.dtype}")
    print(f"    sang  shape={_sang_d.shape} dtype={_sang_d.dtype}")

    # Check if ENERGIES HDU exists in both
    _our_e = None; _sang_e = None
    for _h in _our_hdu:
        if _h.name == 'ENERGIES':
            _our_e = _h.data['Energy']
    for _h in _sang_hdul_l:
        if _h.name == 'ENERGIES':
            _sang_e = _h.data['Energy']

    # Case A: same shape → direct compare
    if _our_d.shape == _sang_d.shape:
        _r_total = _np.sum(_our_d) / _np.sum(_sang_d) if _np.sum(_sang_d) > 0 else _np.nan
        print(f"    ✓ shapes identical → direct ratio")
        print(f"      total sum ratio = {_r_total:.4f}")
        _nz = (_sang_d > 0) & (_our_d > 0)
        if _nz.any():
            _rp = _our_d[_nz] / _sang_d[_nz]
            print(f"      pixel ratio: median={_np.median(_rp):.4f}, "
                  f"mean={_rp.mean():.4f}, std={_rp.std():.4f}")
        for _i in [0, _our_d.shape[0]//2, _our_d.shape[0]-1]:
            _rbin = _np.sum(_our_d[_i]) / _np.sum(_sang_d[_i]) if _np.sum(_sang_d[_i]) > 0 else _np.nan
            print(f"      bin {_i:>3}: ratio = {_rbin:.4f}")
    else:
        # Case B: shapes differ, try to find common sub-region
        print(f"    shapes differ; attempting alignment")
        # Extract center slice from Sanghwan matching our spatial shape
        _our_ny, _our_nx = _our_d.shape[-2:]
        _sang_ny, _sang_nx = _sang_d.shape[-2:]
        # Assume both centered at galactic center (CRVAL1=CRVAL2=0)
        # Center pixel of Sanghwan:
        _cy_s = _sang_ny // 2
        _cx_s = _sang_nx // 2
        _y0 = _cy_s - _our_ny // 2
        _x0 = _cx_s - _our_nx // 2
        if _y0 >= 0 and _x0 >= 0 and _y0 + _our_ny <= _sang_ny and _x0 + _our_nx <= _sang_nx:
            # Compare Sanghwan's crop with ours at matching energy bins
            if _our_e is not None and _sang_e is not None and _our_d.shape[0] == _sang_d.shape[0]:
                _sang_crop = _sang_d[:, _y0:_y0+_our_ny, _x0:_x0+_our_nx]
                _r_total = _np.sum(_our_d) / _np.sum(_sang_crop) if _np.sum(_sang_crop) > 0 else _np.nan
                print(f"    aligned crop: ours vs sang[{_y0}:{_y0+_our_ny}, {_x0}:{_x0+_our_nx}]")
                print(f"      total sum ratio = {_r_total:.4f}")
            elif _our_d.shape[0] != _sang_d.shape[0]:
                print(f"    ⚠ different energy bin counts: our={_our_d.shape[0]}, sang={_sang_d.shape[0]}")
                print(f"    → would need energy rebin to compare")
            else:
                print(f"    ⚠ cannot verify energy alignment")
        else:
            print(f"    ⚠ Sanghwan crop out of bounds: ours={_our_d.shape}, sang={_sang_d.shape}")
    _our_hdu.close()
    _sang_hdul_l.close()
    _os.unlink(_tmp)

# Compare Group A (XLIX) and Group B (X) for all 3 components
for _m in ['XLIX', 'X', 'I']:
    if (_m, 'A') in _all_tested or (_m, 'B') in _all_tested or (_m, 'C') in _all_tested:
        for _c in ['pion', 'bremss', 'ics']:
            _our_p = _our_path(_c, _m)
            _sang_p = _sang_path(_c, _m)
            if _our_p and _sang_p:
                _compare_cubes_properly(_our_p, _sang_p,
                                        f"{_c} Model {_m}")
            else:
                print(f"\n  [{_c} Model {_m}] — file missing")

print()
print("=" * 80)
print("Interpretation:")
print("  - If shapes match AND ratio ≈ 1.000 for Group A + Group B:")
print("    MapCubes are identical → problem is in gtsrcmaps step")
print("  - If shapes match but Group A ratio ≠ Group B ratio:")
print("    MapCube content differs specifically for Group B")
print("  - If shapes differ:")
print("    Our MapCube was post-processed from Sanghwan's or from Zenodo")
print("    differently → conversion step is the divergence point")
print("=" * 80)


In [ ]:
# ============================================================================
# V12d. Sanghwan's Pion/ Bremss/ ICs/ subdirectory inventory
# ----------------------------------------------------------------------------
# V12c showed that Sanghwan's Mapcubes/ has 3 subdirs (Pion, Bremss, ICs) but
# we didn't open them. This cell does, and if files are present, compares
# file sizes + HDU structure with ours for Group A vs Group B models.
# ============================================================================
import os as _os
import numpy as _np
from astropy.io import fits as _fits

SANGHWAN_MAPCUBES = '/home/sanghwan/FermiLAT/Fermi-LAT-GCE-Analysis/Fermi-LAT-Galactic-Center-Excess-analysis/Model_maps/GDE_maps/GALPROP_Mapcubes/Mapcubes'
OUR_MAPCUBES = './MapCubes'

GROUP_A = ['XLIX', 'II', 'XV']
GROUP_B = ['X', 'XLVIII', 'LIII']
GROUP_C = ['I']
ALL_TESTED = GROUP_A + GROUP_B + GROUP_C

_components = [('Pion',   'pion'),
               ('Bremss', 'bremss'),
               ('ICs',    'ics')]

print("=" * 78)
print("V12d: Sanghwan Mapcubes subdirectory inventory")
print("=" * 78)

# ---- Read README first ----
_readme = _os.path.join(SANGHWAN_MAPCUBES, 'README.md')
if _os.path.exists(_readme):
    print(f"\n  README.md contents:")
    with open(_readme) as _f:
        print("  " + _f.read().replace('\n', '\n  '))

# ---- Inventory each subdir ----
_sang_files_by_comp = {}
for _subdir_name, _comp in _components:
    _path = _os.path.join(SANGHWAN_MAPCUBES, _subdir_name)
    print(f"\n  {_subdir_name}/  ({_path}):")
    if not _os.path.isdir(_path):
        print(f"    ✗ NOT A DIRECTORY")
        continue
    try:
        _entries = sorted(_os.listdir(_path))
    except PermissionError:
        print(f"    ✗ permission denied")
        continue
    print(f"    {len(_entries)} entries")
    _sang_files_by_comp[_comp] = {}
    for _e in _entries[:40]:
        _full = _os.path.join(_path, _e)
        if _os.path.isfile(_full):
            _sz = _os.path.getsize(_full)
            _sz_str = f"{_sz/1e6:.2f}M" if _sz > 1e6 else f"{_sz/1e3:.0f}K" if _sz > 1e3 else f"{_sz}B"
            print(f"    [F] {_e}   ({_sz_str})")
            _sang_files_by_comp[_comp][_e] = _full
        else:
            print(f"    [D] {_e}/")
    if len(_entries) > 40:
        print(f"    ... and {len(_entries)-40} more")

# ---- Try to match Sanghwan files to our Roman-numeral models ----
if _sang_files_by_comp:
    print()
    print("=" * 78)
    print("Attempting to match Sanghwan files to our tested models")
    print("=" * 78)

    # Build a folder → roman mapping from NAMING_CONVENTION
    NAMING_FILE = '../GCE_TEMPLATES_FILES_v3/NAMING_CONVENTION_OF_DIFFUSE_EMISSION_MODELS.dat'
    if not _os.path.exists(NAMING_FILE):
        NAMING_FILE = './NAMING_CONVENTION_OF_DIFFUSE_EMISSION_MODELS.dat'
    _folder_to_roman = {}
    _roman_to_folder = {}
    if _os.path.exists(NAMING_FILE):
        with open(NAMING_FILE) as _f:
            for _line in _f:
                _line = _line.strip()
                if not _line or _line.startswith('#'):
                    continue
                _parts = _line.split()
                if len(_parts) >= 2:
                    _folder_to_roman[_parts[1]] = _parts[0]
                    _roman_to_folder[_parts[0]] = _parts[1]

    print()
    print(f"  For each tested model, find Sanghwan file matching:")
    print(f"  {'Group':<5} {'Model':<8} {'Folder':<8} {'pion_*.fits':<30} {'ours_size':>10} {'sang_size':>10} {'match':>8}")

    for _group_label, _models in [('A', GROUP_A), ('B', GROUP_B), ('C', GROUP_C)]:
        for _m in _models:
            _folder = _roman_to_folder.get(_m, '?')
            # Possible Sanghwan filename patterns
            _candidates = [
                f'pion_mapcube_{_folder}.fits',
                f'pion_{_folder}_mapcube.fits',
                f'pion_mapcube_model{_m}.fits',
                f'{_folder}_pion.fits',
                f'pion_{_folder}.fits',
            ]
            _pion_files = _sang_files_by_comp.get('pion', {})
            _found_sang = None
            for _c in _candidates:
                if _c in _pion_files:
                    _found_sang = _pion_files[_c]
                    break
            # Also look at partial match
            if _found_sang is None:
                for _fn in _pion_files:
                    if _folder in _fn.lower() or _m.lower() in _fn.lower():
                        _found_sang = _pion_files[_fn]
                        break

            _our = f'./MapCubes/pion_mapcube_model{_m}.fits'
            _our_sz = _os.path.getsize(_our) if _os.path.exists(_our) else 0

            if _found_sang:
                _sang_sz = _os.path.getsize(_found_sang)
                _match_str = "same" if abs(_our_sz - _sang_sz) < 100 else "DIFF"
                _bn = _os.path.basename(_found_sang)
                print(f"  {_group_label:<5} {_m:<8} {_folder:<8} {_bn:<30} "
                      f"{_our_sz/1e6:>8.2f}M {_sang_sz/1e6:>8.2f}M {_match_str:>8}")
            else:
                print(f"  {_group_label:<5} {_m:<8} {_folder:<8} {'(not found)':<30} "
                      f"{_our_sz/1e6:>8.2f}M {'-':>10} {'-':>8}")

# ---- If MapCube files DO exist, perform content comparison for 2 models ----
print()
print("=" * 78)
print("Content comparison (if possible): Group A vs Group B model")
print("=" * 78)

def _compare_cubes(our_path, sang_path, label):
    """Compare pixel-by-pixel two MapCube files."""
    if not _os.path.exists(our_path):
        print(f"    ours missing: {our_path}")
        return
    if not _os.path.exists(sang_path):
        print(f"    sang missing: {sang_path}")
        return
    _o = _fits.open(our_path)[0].data
    _s = _fits.open(sang_path)[0].data
    print(f"    {label}:")
    print(f"      ours  shape = {_o.shape}")
    print(f"      sang  shape = {_s.shape}")
    if _o.shape != _s.shape:
        print(f"      ⚠ shape mismatch")
        return
    # Total sum
    _r = _np.sum(_o) / _np.sum(_s) if _np.sum(_s) > 0 else _np.nan
    print(f"      total sum ratio (ours/sang) = {_r:.4f}")
    # Pixel median ratio (non-zero only)
    _nz = (_s > 0) & (_o > 0)
    if _nz.any():
        _rpix = _o[_nz] / _s[_nz]
        print(f"      pixel ratio: median={_np.median(_rpix):.4f}, mean={_rpix.mean():.4f}, std={_rpix.std():.4f}")
    # A few specific energy bins
    _n_e = _o.shape[0]
    for _i in [0, _n_e // 2, _n_e - 1]:
        _r_bin = _np.sum(_o[_i]) / _np.sum(_s[_i]) if _np.sum(_s[_i]) > 0 else _np.nan
        print(f"      bin {_i:>3}: ratio = {_r_bin:.4f}")

if _sang_files_by_comp.get('pion'):
    # Try Group A (XLIX) and Group B (X) for pion component
    print("\n  If the subdirs contain per-model files, compare XLIX (Group A) vs X (Group B):")

    for _comp in ['pion', 'bremss', 'ics']:
        for _m in ['XLIX', 'X']:
            _folder = _roman_to_folder.get(_m, '?')
            _our = f'./MapCubes/{_comp}_mapcube_model{_m}.fits'
            # Try finding Sanghwan's file
            _pion_map = _sang_files_by_comp.get(_comp, {})
            _sang = None
            for _candidate in [
                f'{_comp}_mapcube_{_folder}.fits',
                f'{_comp}_{_folder}_mapcube.fits',
                f'{_comp}_mapcube_model{_m}.fits',
            ]:
                if _candidate in _pion_map:
                    _sang = _pion_map[_candidate]
                    break
            if _sang is None:
                # Partial match
                for _fn in _pion_map:
                    if _folder in _fn:
                        _sang = _pion_map[_fn]
                        break
            if _sang:
                print()
                print(f"  [{_comp} / Model {_m} (folder={_folder})]")
                _compare_cubes(_our, _sang, f"{_comp} Model {_m}")
            else:
                print(f"  [{_comp} / Model {_m}] — no matching Sanghwan file found")

print()
print("=" * 78)
print("Interpretation:")
print("  - If ratios ≈ 1.000 in both Group A and Group B models:")
print("    MapCubes are identical → divergence is in gtsrcmaps step")
print("    Next: compare ./GC_analysis_sanghwan/GC_pion_model*_clean.fits")
print("  - If Group A ratio ≈ 1.000 but Group B ratio ≠ 1.000:")
print("    ⭐ MapCube content differs for Group B models")
print("    Next: investigate our convert_zenodo_to_mapcube step")
print("  - If no Sanghwan files found in Pion/Bremss/ICs subdirs:")
print("    Pivot to Option β: compare our own gtsrcmaps outputs Model X vs XLIX")
print("=" * 78)


In [ ]:
# ============================================================================
# V12e. gtsrcmaps output comparison — Group A (XLIX) vs Group B (X)
# ----------------------------------------------------------------------------
# Compares our own gtsrcmaps outputs (_clean.fits component maps) between
# representative Group A and Group B models, to see if the divergence
# observed in final fits already exists at the gtsrcmaps stage.
#
# Also re-runs the "total flux in mask" sanity check per component per bin
# to see if Group B really has a deficit at the component-map level.
# ============================================================================
import os as _os
import numpy as _np
import matplotlib.pyplot as plt
from astropy.io import fits as _fits

GROUP_A_MODEL = 'XLIX'
GROUP_B_MODEL = 'X'

# Defensive lookup for front_tag and existing variables
if 'front' not in dir() or not isinstance(front, str):
    front = '_front'

# File paths: _clean.fits (PSF-convolved), _no_convol.fits (pre-convolution)
def _comp_path(comp, model, tag='_clean'):
    return f'./GC_analysis_sanghwan/GC_{comp}_model{model}_12yr{front}{tag}.fits'

print("=" * 80)
print(f"V12e: gtsrcmaps output ratio — Group A ({GROUP_A_MODEL}) vs Group B ({GROUP_B_MODEL})")
print("=" * 80)

# Load masks and exposure for proper "flux in mask" per bin
_mask_path = './GC_analysis_sanghwan/Model/GC_mask_60x60_definitions_DR2.npy'
_disk_path = './GC_analysis_sanghwan/Model/GC_disk_mask_60x60_definitions.npy'
_exp_path  = './GC_analysis_sanghwan/GC_expcube_center_12yr{}_clean.fits'.format(front)
_cc_path   = './GC_analysis_sanghwan/GC_ccube_12yr{}_clean.fits'.format(front)

if not all(_os.path.exists(p) for p in [_mask_path, _disk_path, _exp_path, _cc_path]):
    print("  ✗ missing one of required files; cannot proceed")
    raise SystemExit

_psc_mask  = _np.load(_mask_path)[:, 100:500, 100:500]
_disk_mask = _np.load(_disk_path)[100:500, 100:500]
_exp       = _fits.open(_exp_path)[0].data[:, 100:500, 100:500]
_cc        = _fits.open(_cc_path)[0].data[:, 100:500, 100:500]
_n_e = _cc.shape[0]

# Ensure E, delta_E available
if 'E' not in dir() or not hasattr(E, 'shape') or E.shape != (14,):
    _Eb = _fits.open(_cc_path)[1].data
    E = _np.array([_np.sqrt(_b[2]*_b[1]*1e-6)*1e-3 for _b in _Eb])
    delta_E = _np.array([(_b[2]-_b[1])*1e-6 for _b in _Eb])

# Solid-angle weighting: sr/pixel × cos(b)
from astropy.wcs import WCS as _WCS
_wcs = _WCS(_fits.open(_cc_path)[0].header).dropaxis(2)
_sr_pix = _np.zeros((600, 600))
for _i in range(600):
    for _j in range(600):
        _l, _b = _wcs.wcs_pix2world(_j, _i, 0)
        _sr_pix[_i, _j] = _np.radians(0.1)**2 * _np.cos(_np.radians(_b))
_sr_roi = _sr_pix[100:500, 100:500]
_exp_sr = _exp * _sr_roi

# Compare function: for each component, per-bin total counts within mask
def _per_bin_masked_total(fits_path, full_mask):
    """Returns 14-bin array of masked total counts from a component map."""
    if not _os.path.exists(fits_path):
        return None
    _d = _fits.open(fits_path)[0].data
    _out = _np.zeros(_n_e)
    for _i in range(_n_e):
        _px = _d[_i][100:500, 100:500] if _d.ndim == 3 and _d.shape[1] == 600 else _d[_i]
        _out[_i] = _np.sum(_px * full_mask[_i])
    return _out

_full_mask = _psc_mask * _disk_mask   # (n_e, 400, 400)

# CCUBE reference (data)
_data_per_bin = _per_bin_masked_total(_cc_path, _full_mask)

# Components for each model
_components = ['pion', 'bremss', 'ics', 'GCE', 'fermi_bubble', 'isotropic']
# Non-GCE/bubble/iso are per-model; GCE/bubble/iso are model-independent
_per_model_comps = ['pion', 'bremss', 'ics']
_shared_comps = ['GCE', 'fermi_bubble', 'isotropic']

_ratios_by_comp = {}  # comp → (A, B, ratio)
for _comp in _per_model_comps:
    _path_A = _comp_path(_comp, GROUP_A_MODEL)
    _path_B = _comp_path(_comp, GROUP_B_MODEL)
    if not _os.path.exists(_path_A) or not _os.path.exists(_path_B):
        print(f"  [skip {_comp}] one of files missing")
        continue
    _A = _per_bin_masked_total(_path_A, _full_mask)
    _B = _per_bin_masked_total(_path_B, _full_mask)
    _ratios_by_comp[_comp] = (_A, _B, _B / _A)

# Print table
print(f"\n  Per-bin masked total counts: Model XLIX (Group A) vs Model X (Group B)")
print(f"  {'bin':>3} {'E[GeV]':>8} {'pion_A':>10} {'pion_B':>10} {'B/A':>7}  "
      f"{'bremss_A':>10} {'bremss_B':>10} {'B/A':>7}  "
      f"{'ics_A':>10} {'ics_B':>10} {'B/A':>7}")
for _i in range(_n_e):
    _row = f"  {_i:>3} {E[_i]:>8.3f}"
    for _c in _per_model_comps:
        if _c not in _ratios_by_comp:
            _row += f"   -          -         -   "
            continue
        _A, _B, _r = _ratios_by_comp[_c]
        _row += f"   {_A[_i]:>8.2e} {_B[_i]:>8.2e} {_r[_i]:>7.4f}  "
    print(_row)

# Bin-averaged summary
print(f"\n  Summary (mean ratio across bins, with different E ranges):")
print(f"  {'Component':<10} {'0.3-1 GeV':>12} {'1-10 GeV':>12} {'>10 GeV':>12} {'all':>12}")
_m_lo = (E >= 0.3) & (E < 1.0)
_m_mid = (E >= 1.0) & (E <= 10.0)
_m_hi = E > 10.0
for _c in _per_model_comps:
    if _c not in _ratios_by_comp:
        continue
    _A, _B, _r = _ratios_by_comp[_c]
    print(f"  {_c:<10} {_np.nanmean(_r[_m_lo]):>12.4f} "
          f"{_np.nanmean(_r[_m_mid]):>12.4f} {_np.nanmean(_r[_m_hi]):>12.4f} "
          f"{_np.nanmean(_r):>12.4f}")

# CCUBE itself has B/A = 1.0000 by construction (data is model-independent)
print()
print(f"  For sanity: CCUBE B/A (should be 1.0) = "
      f"{_np.nanmean(_data_per_bin / _data_per_bin):.4f}")

# ---------------------------------------------------------------------------
# Plot side-by-side for one energy bin (mid-E, ~1.5 GeV, bin 6)
# ---------------------------------------------------------------------------
_bin_show = 6
fig, axes = plt.subplots(len(_per_model_comps), 3, figsize=(14, 4*len(_per_model_comps)))
for _row_idx, _c in enumerate(_per_model_comps):
    if _c not in _ratios_by_comp:
        continue
    _A_cube = _fits.open(_comp_path(_c, GROUP_A_MODEL))[0].data[_bin_show]
    _B_cube = _fits.open(_comp_path(_c, GROUP_B_MODEL))[0].data[_bin_show]
    if _A_cube.ndim == 2 and _A_cube.shape == (600, 600):
        _A_roi = _A_cube[100:500, 100:500]
        _B_roi = _B_cube[100:500, 100:500]
    else:
        _A_roi = _A_cube
        _B_roi = _B_cube
    _mask_here = _full_mask[_bin_show]
    _A_masked = _np.where(_mask_here, _A_roi, _np.nan)
    _B_masked = _np.where(_mask_here, _B_roi, _np.nan)
    _ratio_map = _np.where((_mask_here > 0) & (_A_roi > 0),
                            _B_roi / _np.maximum(_A_roi, 1e-30),
                            _np.nan)
    _vmax = _np.nanmax([_A_masked, _B_masked])
    axes[_row_idx, 0].imshow(_A_masked, origin='lower', cmap='inferno', vmin=0, vmax=_vmax)
    axes[_row_idx, 0].set_title(f'{_c} — Model {GROUP_A_MODEL} (A)\nmasked total = {_ratios_by_comp[_c][0][_bin_show]:.3e}')
    axes[_row_idx, 0].grid(False)
    axes[_row_idx, 1].imshow(_B_masked, origin='lower', cmap='inferno', vmin=0, vmax=_vmax)
    axes[_row_idx, 1].set_title(f'{_c} — Model {GROUP_B_MODEL} (B)\nmasked total = {_ratios_by_comp[_c][1][_bin_show]:.3e}')
    axes[_row_idx, 1].grid(False)
    axes[_row_idx, 2].imshow(_ratio_map, origin='lower', cmap='RdBu_r', vmin=0.8, vmax=1.2)
    axes[_row_idx, 2].set_title(f'ratio B/A map  mean={_np.nanmean(_ratio_map):.3f}')
    axes[_row_idx, 2].grid(False)
fig.suptitle(f'gtsrcmaps output comparison at bin {_bin_show} (E≈{E[_bin_show]:.2f} GeV)\n'
             f'Group A ({GROUP_A_MODEL}) vs Group B ({GROUP_B_MODEL})', fontsize=13)
fig.tight_layout()
_out = f'./gtsrcmaps_A_vs_B_bin{_bin_show}.png'
fig.savefig(_out, dpi=120)
plt.show()
print(f"\n  [OK] Saved {_out}")

# ---------------------------------------------------------------------------
# Interpretation
# ---------------------------------------------------------------------------
print()
print("=" * 80)
print("INTERPRETATION")
print("=" * 80)
print()
print(f"  V12b showed: final fit Model X / Model XLIX ≈ 0.90 (Group B vs A)")
print(f"  V12d showed: MapCube X / MapCube XLIX has same Group difference → need to check")
print()
print(f"  If V12e pion/bremss/ics B/A ≈ 1.00 in 1-10 GeV:")
print(f"    → gtsrcmaps outputs show no systematic Group A vs B difference")
print(f"    → Divergence must be in FIT stage (per-bin likelihood, prior strengths,")
print(f"      or how c_coefficients couple across components)")
print(f"    → Residual-based hypothesis: Group B models have different spectral shape")
print(f"      that fits GDE/GCE differently due to degeneracy with bubble/iso")
print()
print(f"  If V12e pion/bremss/ics B/A ≠ 1.00 in 1-10 GeV:")
print(f"    → gtsrcmaps already encodes the Group difference")
print(f"    → But this is EXPECTED — different GDE models produce different fluxes!")
print(f"    → The absolute Model X flux is different from Model XLIX flux")
print(f"    → Real question: 'is our Model X/XLIX ratio the same as Sanghwan's?'")
print(f"      We'd need Sanghwan's component maps to answer, which he didn't save.")
print()
print(f"  CRITICAL TAKEAWAY:")
print(f"    Since MapCube is identical (V12d) and CCUBE is identical (V12a),")
print(f"    and we can't see Sanghwan's intermediate gtsrcmaps output,")
print(f"    the divergence must come from either:")
print(f"    (a) gtsrcmaps env/XML/fermitools version difference")
print(f"    (b) fit-stage numerical differences (likelihood, starting point,")
print(f"        convergence)")
print(f"    Neither is easy to reproduce without rerunning Sanghwan's exact code")
print(f"    in his exact environment.")


In [ ]:
# ============================================================================
# V20 v2. 12yr vs 16yr flux comparison (Sanghwan 12yr + haebarg's 16yr_data dir)
# ----------------------------------------------------------------------------
# 12yr fits: /home/sanghwan/FermiLAT/Sanghwan/   (Sanghwan only ran 12yr there)
# 16yr fits: /home/haebarg/.../GCE_16yr_data/   (already copied to user's tree)
# .dat columns confirmed: [E, flux_best, stat_unc, lo_1sig, hi_1sig]
# ============================================================================
import os as _os
import numpy as _np
import matplotlib.pyplot as plt

SANGHWAN_DIR = '/home/sanghwan/FermiLAT/Sanghwan'
HBG_16YR_DIR = '/home/haebarg/GCE-Chi-square-fitting/GCE_16yr_data'

# Models that Sanghwan analyzed at 12yr
MODELS_12YR = ['X', 'II', 'XV', 'XLVIII', 'XLIX', 'LIII', 'LXIV', 'LXIX', 'LXX', 'LXXI']

def _path_12yr(m):
    if m == 'I':
        p = f'{SANGHWAN_DIR}/GCE_model_I_12yr.dat'
    else:
        p = f'{SANGHWAN_DIR}/GCE_model_{m}_12yr_cholis.dat'
    return p if _os.path.exists(p) else None

def _path_16yr_front(m):
    p = f'{HBG_16YR_DIR}/GCE_model_{m}_front_16yr_cholis.dat'
    return p if _os.path.exists(p) else None

def _path_16yr_frontback(m):
    p = f'{HBG_16YR_DIR}/GCE_model_{m}_front_back_16yr_cholis.dat'
    return p if _os.path.exists(p) else None

# Inventory
print("=" * 78)
print("V20 v2: 12yr vs 16yr (Sanghwan 12yr + haebarg 16yr) — file inventory")
print("=" * 78)
_inventory = {}
for _m in MODELS_12YR + ['I']:
    _entry = {
        '12yr':       _path_12yr(_m),
        '16yr_front': _path_16yr_front(_m),
        '16yr_fb':    _path_16yr_frontback(_m),
    }
    _inventory[_m] = _entry
    _have = [k for k, v in _entry.items() if v]
    print(f"  Model {_m:<8}  {_have}")

_to_plot = [_m for _m in _inventory
            if _inventory[_m]['12yr'] and _inventory[_m]['16yr_front']]
print(f"\n  → {len(_to_plot)} models have both 12yr and 16yr_front")

if not _to_plot:
    print("  No suitable models. Aborting V20.")
else:
    _ncols = min(3, len(_to_plot))
    _nrows = (len(_to_plot) + _ncols - 1) // _ncols
    fig, axes = plt.subplots(_nrows, _ncols, figsize=(5.5*_ncols, 4.5*_nrows),
                             squeeze=False)
    axes_flat = axes.flat
    _ratios = {}

    for _idx, _m in enumerate(_to_plot):
        _ax = axes_flat[_idx]
        _d12 = _np.loadtxt(_inventory[_m]['12yr'])
        _d16f = _np.loadtxt(_inventory[_m]['16yr_front'])

        # Columns: [E, flux_best, stat_unc, lo_1sig, hi_1sig]
        _E = _d12[:, 0]
        _yerr_12 = [_np.maximum(_d12[:, 1] - _d12[:, 3], 0),
                    _np.maximum(_d12[:, 4] - _d12[:, 1], 0)]
        _ax.errorbar(_E, _d12[:, 1], yerr=_yerr_12,
                     marker='s', color='crimson', markersize=5, linewidth=1.5,
                     label='12yr (Sanghwan)', capsize=3)

        _E16 = _d16f[:, 0]
        _yerr_16 = [_np.maximum(_d16f[:, 1] - _d16f[:, 3], 0),
                    _np.maximum(_d16f[:, 4] - _d16f[:, 1], 0)]
        _ax.errorbar(_E16, _d16f[:, 1], yerr=_yerr_16,
                     marker='o', color='royalblue', markersize=5, linewidth=1.5,
                     label='16yr front', capsize=3, alpha=0.85)

        if _inventory[_m]['16yr_fb']:
            _d16fb = _np.loadtxt(_inventory[_m]['16yr_fb'])
            _yerr_16fb = [_np.maximum(_d16fb[:, 1] - _d16fb[:, 3], 0),
                          _np.maximum(_d16fb[:, 4] - _d16fb[:, 1], 0)]
            _ax.errorbar(_d16fb[:, 0], _d16fb[:, 1], yerr=_yerr_16fb,
                         marker='^', color='seagreen', markersize=5, linewidth=1.5,
                         label='16yr front+back', capsize=3, alpha=0.7)

        _m110_16 = (_E16 >= 1.0) & (_E16 <= 10.0)
        _f12_interp = _np.interp(_E16, _E, _d12[:, 1])
        _ratio_1_10 = _np.nanmean((_d16f[:, 1] / _f12_interp)[_m110_16])
        _ratios[_m] = _ratio_1_10

        _ax.set_xscale('log'); _ax.set_yscale('log')
        _ax.set_xlim(0.3, 100); _ax.set_ylim(1e-8, 3e-6)
        _ax.set_xlabel('E [GeV]')
        _ax.set_ylabel(r'$E^2\,dN/dE$  [GeV cm$^{-2}$ s$^{-1}$ sr$^{-1}$]')
        _ax.set_title(f'Model {_m}\n16yr/12yr (1-10 GeV) = {_ratio_1_10:.3f}',
                      fontsize=10)
        _ax.legend(fontsize=8, loc='lower left')
        _ax.grid(True, which='major', alpha=0.3)

    for _i in range(len(_to_plot), len(axes_flat)):
        axes_flat[_i].axis('off')

    fig.suptitle("12yr (Sanghwan) vs 16yr (haebarg's GCE_16yr_data) GCE flux per model",
                 fontsize=14, y=0.995)
    fig.tight_layout()
    _out = './sanghwan_12yr_vs_16yr_flux.png'
    fig.savefig(_out, dpi=120)
    plt.show()
    print(f"\n  [OK] Saved {_out}")

    print(f"\n  Summary — 16yr / 12yr ratio (1-10 GeV):")
    print(f"  {'Model':<8} {'ratio':>8} {'verdict':>15}")
    for _m, _r in _ratios.items():
        if 0.95 < _r < 1.05: _v = "✓ stable"
        elif 0.85 < _r < 1.15: _v = "~ minor change"
        else: _v = "⚠ shifted"
        print(f"  {_m:<8} {_r:>8.3f} {_v:>15}")
    _mean_r = _np.nanmean(list(_ratios.values()))
    print(f"\n  Average ratio across models: {_mean_r:.3f}")


In [ ]:
# ============================================================================
# V21 v2. 16yr 80-model envelope + 17yr expected band overlay
# ----------------------------------------------------------------------------
# Loads all of haebarg's 16yr_front fits, computes envelope, and overlays
# a "17yr expected" band assuming purely statistical scaling:
#   stat_unc(17yr) = stat_unc(16yr) × √(16/17) ≈ 0.97 × stat_unc(16yr)
# Best-fit flux unchanged.
#
# CAVEAT (must be explicit in talks): GCE is systematic-dominated, so the
# narrowing from 16yr → 17yr is small. This plot shows the statistical-only
# projection. Real 17yr will have its own systematic budget.
# ============================================================================
import os as _os
import glob as _glob
import numpy as _np
import matplotlib.pyplot as plt

HBG_16YR_DIR = '/home/haebarg/GCE-Chi-square-fitting/GCE_16yr_data'

_pattern = f'{HBG_16YR_DIR}/GCE_model_*_front_16yr_cholis.dat'
_files = sorted(_glob.glob(_pattern))
print("=" * 78)
print(f"V21 v2: 16yr envelope + 17yr expected overlay")
print("=" * 78)
print(f"  Found {len(_files)} 16yr_front .dat files in {HBG_16YR_DIR}")

if len(_files) == 0:
    print("  No files. Aborting.")
else:
    _models = []; _data = {}; _E_ref = None
    for _f in _files:
        _bn = _os.path.basename(_f)
        _m = _bn.replace('GCE_model_', '').replace('_front_16yr_cholis.dat', '')
        try:
            _d = _np.loadtxt(_f)
            if _d.ndim == 2 and _d.shape[1] >= 5:
                _data[_m] = _d
                _models.append(_m)
                if _E_ref is None: _E_ref = _d[:, 0]
        except Exception as _e:
            print(f"  [skip] {_m}: {_e}")
    print(f"  Loaded {len(_data)} models")

    # Stack flux + stat_unc (col 2)
    _all_flux = _np.array([_data[_m][:, 1] for _m in _data])
    _all_stat = _np.array([_data[_m][:, 2] for _m in _data])

    _median_16 = _np.nanmedian(_all_flux, axis=0)
    _p16_16    = _np.nanpercentile(_all_flux, 16, axis=0)
    _p84_16    = _np.nanpercentile(_all_flux, 84, axis=0)

    # 17yr expected: same best-fit (flux unchanged), stat scaled by √(16/17)
    _scale_17 = _np.sqrt(16.0 / 17.0)
    # stat-driven envelope: per-model error band scaled
    _median_17  = _median_16   # central value unchanged
    # Apply scaling to spread (16-84 percentile range as proxy for "stat envelope")
    _half_16    = (_p84_16 - _p16_16) / 2
    _half_17    = _half_16 * _scale_17
    _p16_17     = _median_17 - _half_17
    _p84_17     = _median_17 + _half_17

    # Plot
    fig, ax = plt.subplots(figsize=(11, 7.5))

    # Individual models
    for _m, _d in _data.items():
        ax.plot(_d[:, 0], _d[:, 1], '-', color='gray', alpha=0.12, linewidth=0.6)

    # 16yr envelope
    ax.fill_between(_E_ref, _p16_16, _p84_16, color='steelblue', alpha=0.35,
                    label=f'16yr 16-84% envelope ({len(_data)} models)')
    ax.plot(_E_ref, _median_16, '-', color='navy', linewidth=2.0,
            label='16yr median (haebarg data)', zorder=10)

    # 17yr expected (overlay)
    ax.fill_between(_E_ref, _p16_17, _p84_17, color='darkorange', alpha=0.25,
                    edgecolor='orange', linewidth=0,
                    label=f'17yr expected envelope (stat scaling × {_scale_17:.3f})')
    ax.plot(_E_ref, _median_17, '--', color='darkorange', linewidth=1.8,
            label='17yr expected median (= 16yr, stat only)', zorder=11)

    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlim(0.3, 100); ax.set_ylim(5e-8, 3e-6)
    ax.set_xlabel('E [GeV]', fontsize=12)
    ax.set_ylabel(r'$E^2\,dN/dE$  [GeV cm$^{-2}$ s$^{-1}$ sr$^{-1}$]', fontsize=12)
    ax.set_title("GCE flux: 16yr envelope (haebarg data) + 17yr statistical projection\n"
                 "17yr expected = same best-fit, stat_unc × √(16/17) ≈ 0.97×",
                 fontsize=12)
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(True, which='major', alpha=0.3)

    # Caveat box
    ax.text(0.02, 0.04,
            "Caveat: 17yr projection assumes statistical-only scaling.\n"
            "GCE is systematic-dominated → real narrowing may be < 3%.\n"
            "Catalog (FL16Y vs DR2) and exposure changes not modeled here.",
            transform=ax.transAxes, fontsize=8, ha='left', va='bottom',
            bbox=dict(boxstyle='round', facecolor='lemonchiffon',
                      alpha=0.85, edgecolor='gray'))

    fig.tight_layout()
    _out = './envelope_16yr_with_17yr_projection.png'
    fig.savefig(_out, dpi=140)
    plt.show()
    print(f"\n  [OK] Saved {_out}")

    _peak_idx = int(_np.argmax(_median_16))
    print(f"\n  Peak at E = {_E_ref[_peak_idx]:.2f} GeV:")
    print(f"    16yr: median={_median_16[_peak_idx]:.3e}, "
          f"16-84%=[{_p16_16[_peak_idx]:.3e}, {_p84_16[_peak_idx]:.3e}]")
    print(f"    17yr expected: median={_median_17[_peak_idx]:.3e}, "
          f"16-84%=[{_p16_17[_peak_idx]:.3e}, {_p84_17[_peak_idx]:.3e}]")
    _sys_16 = (_p84_16[_peak_idx] - _p16_16[_peak_idx]) / _median_16[_peak_idx] * 100
    _sys_17 = (_p84_17[_peak_idx] - _p16_17[_peak_idx]) / _median_17[_peak_idx] * 100
    print(f"    Relative width: 16yr={_sys_16:.1f}%, 17yr expected={_sys_17:.1f}%")
    print(f"    → Stat-only narrowing: {_sys_16 - _sys_17:.2f} percentage points")


In [ ]:
# ============================================================================
# V22 v5. bb̄ DM contour: 16yr vs 17yr expected + Fermi-LAT dSph 95% UL
# ----------------------------------------------------------------------------
# Single-channel (bb̄) version. Uses pre-existing covariance from ./Cov/.
# Fixes for v4 bugs:
#   - dSph filenames use DOT: bb_14.0yr (not bb_14_0yr — that was upload variant)
#   - Use saved Cov file, no mock regeneration
#   - 60×60 grid for smoother contours
#   - Diagnostic output to verify contour level existence
# ============================================================================
import os as _os
import warnings as _warnings
import numpy as _np
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator as _AutoMinorLocator
from scipy.interpolate import interp1d as _i1d
from scipy.interpolate import RegularGridInterpolator as _RGI

# ----------------------------- Configuration -----------------------------
HBG_16YR_DIR = '/home/haebarg/GCE-Chi-square-fitting/GCE_16yr_data'
HBG_17YR_DIR = '/home/haebarg/GCE-Chi-square-fitting/GCE_17yr_data'
COV_DIR      = '/home/haebarg/GCE-Chi-square-fitting/Cov'

# User's reference values (Cholis-style)
J_FACTOR = 3.5251837158376415e+21
SR       = 0.4288213187542626
SCALE_17_OVER_16 = _np.sqrt(16.0 / 17.0)

_PPPC4_CANDIDATES = [
    './Prompt_spectra/AtProduction_gammas.dat',
    '/home/haebarg/ipynb/AtProduction_gammas.dat',
    '/home/haebarg/PPPC4DMID/AtProduction_gammas.dat',
    './AtProduction_gammas.dat',
]
PPPC4_PATH = next((_c for _c in _PPPC4_CANDIDATES if _os.path.exists(_c)), None)

SCAN_CONFIG = {
    'model': 'X',
    'front': 'front',
    'n_bin': 14,
}

print("=" * 78)
print(f"V22 v5: bb̄ DM contour — 16yr vs 17yr + dSph UL "
      f"(Model {SCAN_CONFIG['model']}, {SCAN_CONFIG['front']}, "
      f"{SCAN_CONFIG['n_bin']} bins)")
print("=" * 78)

if PPPC4_PATH is None:
    print("  ✗ PPPC4 file not found"); raise SystemExit
print(f"  PPPC4: {PPPC4_PATH}")

# ----------------------------- Load PPPC4 (bb only, col 13) -----------------------------
print(f"  Loading PPPC4 spectra...")
_pppc = _np.loadtxt(PPPC4_PATH, skiprows=1)
_masses = _np.unique(_pppc[:, 0]); _logx = _np.unique(_pppc[:, 1])
_n_m, _n_x = len(_masses), len(_logx)
_z_bb = _pppc[:, 13].reshape(_n_m, _n_x)
_interp_bb = _RGI((_masses, _logx), _z_bb, bounds_error=False, fill_value=0.0)
print(f"  PPPC4 grid: {_n_m} masses ({_masses[0]:g}-{_masses[-1]:g} GeV)")

def extract_pppc_bb(mass):
    _pts = _np.column_stack([_np.full(len(_logx), mass), _logx])
    _dNdlogx = _interp_bb(_pts)
    _E = mass * (10**_logx)
    with _np.errstate(divide='ignore', invalid='ignore'):
        _dNdE = _dNdlogx / (_E * _np.log(10))
    return _E, _np.nan_to_num(_dNdE, nan=0.0, posinf=0.0, neginf=0.0)

# ----------------------------- Load 16yr GCE data -----------------------------
_dat_p = f'{HBG_16YR_DIR}/GCE_model_{SCAN_CONFIG["model"]}_{SCAN_CONFIG["front"]}_16yr_cholis.dat'
_g = _np.loadtxt(_dat_p)
_n_use = SCAN_CONFIG['n_bin']
emeans   = _g[:_n_use, 0]
flux_16  = _g[:_n_use, 1]
stat_16  = _g[:_n_use, 2]
print(f"\n  Loaded 16yr GCE: {_dat_p}")
print(f"  E range: [{emeans[0]:.2f}, {emeans[-1]:.2f}] GeV ({_n_use} bins)")
print(f"  flux range: [{flux_16.min():.2e}, {flux_16.max():.2e}]")
print(f"  stat (col 2) range: [{stat_16.min():.2e}, {stat_16.max():.2e}]")

# ----------------------------- Load existing covariance -----------------------------
_cov_p_16 = f'{COV_DIR}/approx_covariance_{_n_use}x{_n_use}_{SCAN_CONFIG["front"]}_model_{SCAN_CONFIG["model"]}_16yr.npy'
if not _os.path.exists(_cov_p_16):
    raise FileNotFoundError(f"Required covariance not found: {_cov_p_16}")
cov_emp_16 = _np.load(_cov_p_16)
print(f"\n  ✓ Loaded covariance: {_cov_p_16}")
print(f"    shape = {cov_emp_16.shape}")
# User's pattern: stat² + cov_emp from the loaded file
# (Note: depending on what's saved in the .npy, it may already include stat or just sys)
# Try: total = diag(stat²) + cov_emp; if singular or negative-def, fall back to cov_emp directly
try:
    cov_total_16 = _np.diag(stat_16**2) + cov_emp_16
    inv_cov_16 = _np.linalg.inv(cov_total_16)
    _cov_strategy = "stat² + cov_file"
except _np.linalg.LinAlgError:
    cov_total_16 = cov_emp_16
    inv_cov_16 = _np.linalg.inv(cov_total_16)
    _cov_strategy = "cov_file (direct)"
print(f"    covariance strategy: {_cov_strategy}")
print(f"    diag(cov_total_16) range: [{_np.diag(cov_total_16).min():.2e}, "
      f"{_np.diag(cov_total_16).max():.2e}]")
print(f"    diag(stat²) range: [{(stat_16**2).min():.2e}, {(stat_16**2).max():.2e}]")

# 17yr expected
stat_17 = stat_16 * SCALE_17_OVER_16
flux_17 = flux_16
# Sys part of covariance = cov_total - diag(stat²); apply scaling only to stat
_sys_cov = cov_total_16 - _np.diag(stat_16**2)
cov_total_17 = _np.diag(stat_17**2) + _sys_cov
inv_cov_17 = _np.linalg.inv(cov_total_17)
print(f"\n  17yr expected: stat × √(16/17) = ×{SCALE_17_OVER_16:.4f}, sys unchanged")

# ----------------------------- Load dSph data (FIX FILENAMES — use DOT) -----------------------------
def _load_dsph(prefix, year):
    """Use DOT in filename, not underscore: bb_14.0yr_30_dSphs.txt"""
    _p = f'{HBG_17YR_DIR}/{prefix}_{year}.0yr_30_dSphs.txt'
    if not _os.path.exists(_p):
        return None, _p
    _d = _np.loadtxt(_p)
    return _d[_d[:, 1] > 1.1e-28], _p

dsph_14, _p14 = _load_dsph('bb', 14)
dsph_17, _p17 = _load_dsph('bb', 17)
print(f"\n  dSph bb 14yr: {_p14}")
print(f"    {'✓ loaded ' + str(len(dsph_14)) + ' points' if dsph_14 is not None else '✗ MISSING'}")
print(f"  dSph bb 17yr: {_p17}")
print(f"    {'✓ loaded ' + str(len(dsph_17)) + ' points' if dsph_17 is not None else '✗ MISSING'}")

# ----------------------------- χ² function -----------------------------
def chi_square_bb(dm_mass, sigma_v, flux_data, inv_cov):
    _warnings.simplefilter("ignore", category=RuntimeWarning)
    _E, _dNdE = extract_pppc_bb(dm_mass)
    _interp_func = _i1d(_E, _dNdE, fill_value=0, bounds_error=False, kind='linear')
    _dNdE_at = _interp_func(emeans)
    model = (emeans**2) * _dNdE_at * (sigma_v / dm_mass**2) * J_FACTOR / SR
    delta = model - flux_data
    return delta.T @ inv_cov @ delta

# ----------------------------- Grid scan (finer than v4) -----------------------------
sigmav_range = _np.logspace(-27, -25, 60)
mass_range   = _np.logspace(_np.log10(10), _np.log10(200), 60)
DM_mass_grid, Sigmav_grid = _np.meshgrid(mass_range, sigmav_range)
print(f"\n  χ² grid: {len(mass_range)} × {len(sigmav_range)}")

print(f"  Computing χ² for 16yr...")
chi2_16 = _np.vectorize(lambda m, sv: chi_square_bb(m, sv, flux_16, inv_cov_16))(DM_mass_grid, Sigmav_grid)
print(f"  Computing χ² for 17yr expected...")
chi2_17 = _np.vectorize(lambda m, sv: chi_square_bb(m, sv, flux_17, inv_cov_17))(DM_mass_grid, Sigmav_grid)

# Diagnostic for contour level existence
_idx_16 = _np.unravel_index(_np.argmin(chi2_16), chi2_16.shape)
_idx_17 = _np.unravel_index(_np.argmin(chi2_17), chi2_17.shape)
_bm16, _bsv16, _min16 = DM_mass_grid[_idx_16], Sigmav_grid[_idx_16], chi2_16[_idx_16]
_bm17, _bsv17, _min17 = DM_mass_grid[_idx_17], Sigmav_grid[_idx_17], chi2_17[_idx_17]
_dof = _n_use - 2

print(f"\n  16yr best-fit: m_χ = {_bm16:.1f} GeV, σv = {_bsv16:.3e}, "
      f"χ² = {_min16:.2f}, χ²/dof = {_min16/_dof:.2f}")
print(f"  17yr expected: m_χ = {_bm17:.1f} GeV, σv = {_bsv17:.3e}, "
      f"χ² = {_min17:.2f}, χ²/dof = {_min17/_dof:.2f}")

# Check contour level existence
_lvl_1sig_16 = _min16 + 2.30
_lvl_2sig_16 = _min16 + 6.18
_n_pts_1s = _np.sum(chi2_16 <= _lvl_1sig_16)
_n_pts_2s = _np.sum(chi2_16 <= _lvl_2sig_16)
_n_total = chi2_16.size
print(f"\n  16yr contour diagnostic:")
print(f"    χ²_min = {_min16:.2f}, 1σ level = {_lvl_1sig_16:.2f}, 2σ level = {_lvl_2sig_16:.2f}")
print(f"    χ²_max in grid = {_np.nanmax(chi2_16):.2f}")
print(f"    points within 1σ region: {_n_pts_1s}/{_n_total}")
print(f"    points within 2σ region: {_n_pts_2s}/{_n_total}")
if _n_pts_1s < 3:
    print(f"    ⚠ very few points within 1σ — contour may not render properly")
    print(f"      grid may be too coarse near best fit, or chi² changes too steeply")

# ----------------------------- Plot (single bb panel, two dataset comparison) -----------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 6.0))

for _ax_idx, (year, _bm, _bsv, _min, _color, _chi2_grid) in enumerate([
    (16, _bm16, _bsv16, _min16, 'blue',       chi2_16),
    (17, _bm17, _bsv17, _min17, 'darkorange', chi2_17),
]):
    _ax = axes[_ax_idx]
    _levels = [_min + 2.30, _min + 6.18]
    try:
        _cs = _ax.contour(DM_mass_grid, Sigmav_grid, _chi2_grid,
                          levels=_levels, colors=_color,
                          linestyles=['--', '-'], linewidths=[1.5, 1.8])
        # Manual labels
        _ax.clabel(_cs, inline=True, fontsize=8,
                   fmt={_levels[0]: '1σ', _levels[1]: '2σ'})
    except Exception as _e:
        print(f"  ⚠ contour({year}yr) failed: {_e}")

    # Best-fit point
    _ax.plot(_bm, _bsv, 'o', color=_color, markersize=10,
             markeredgecolor='black', label='Best Fit', zorder=10)

    # dSph upper limits
    if dsph_14 is not None:
        _ax.plot(dsph_14[:, 0], dsph_14[:, 1], '-', color='black',
                 linewidth=1.6, alpha=0.85,
                 label='Fermi-LAT dSph 95% UL (14yr)')
    if dsph_17 is not None:
        _ax.plot(dsph_17[:, 0], dsph_17[:, 1], '--', color='red',
                 linewidth=1.6, alpha=0.85,
                 label='Fermi-LAT dSph 95% UL (17yr prelim)')
        # Excluded shading above 17yr UL
        _m_in = (dsph_17[:, 0] >= 10) & (dsph_17[:, 0] <= 200)
        if _m_in.any():
            _ax.fill_between(dsph_17[_m_in, 0], dsph_17[_m_in, 1], 1e-24,
                             color='red', alpha=0.07)

    _ax.axhline(3e-26, color='gray', linestyle=':', linewidth=1.2, alpha=0.7,
                label=r'thermal relic ($3\times10^{-26}$)')

    _ax.set_yscale('log'); _ax.set_xscale('linear')
    _ax.set_xlim(10, 200); _ax.set_ylim(1e-27, 1e-24)
    _ax.tick_params(which='major', direction='in', length=6, width=1.2)
    _ax.tick_params(which='minor', direction='in', length=3, width=0.8)
    _ax.xaxis.set_minor_locator(_AutoMinorLocator())
    _ax.grid(True, which='major', linestyle='--', linewidth=0.5)

    _ax.text(0.95, 0.05,
             r'68% CL ($1\sigma$ dashed), 95% CL ($2\sigma$ solid)',
             transform=_ax.transAxes, ha='right', va='bottom', fontsize=8,
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.7,
                       edgecolor='gray'))
    _ax.text(0.05, 0.05, fr'$\chi^2$/dof = {_min/_dof:.2f}',
             transform=_ax.transAxes, ha='left', va='bottom', fontsize=11)
    _ax.text(0.05, 0.95, r'$b\bar{b}$', transform=_ax.transAxes,
             ha='left', va='top', fontsize=18, fontweight='bold')
    _front_label = "Front" if SCAN_CONFIG['front'] == 'front' else "Front+Back"
    _Elimit = "50" if SCAN_CONFIG['n_bin'] == 14 else "560"
    _year_tag = (f'{year}yr (haebarg data)' if year == 16
                 else f'{year}yr expected (×{SCALE_17_OVER_16:.3f})')
    _ax.set_title(f'{_year_tag}  {_front_label} GCE < {_Elimit} GeV  (Model {SCAN_CONFIG["model"]})',
                  fontsize=11)
    _ax.set_xlabel(r'$m_\chi$ [GeV]', fontsize=12)
    _ax.set_ylabel(r'$\langle\sigma v\rangle$ [cm$^3$ s$^{-1}$]', fontsize=12)
    _ax.legend(loc='upper left', fontsize=8.5, framealpha=0.9)

fig.suptitle(f"$b\\bar{{b}}$ DM contour: 16yr vs 17yr expected (GCE) "
             f"+ Fermi-LAT dSph 95% upper limits — Model {SCAN_CONFIG['model']}",
             fontsize=13, y=1.00)
fig.tight_layout()
_out = (f'./DM_contour_bb_16yr_vs_17yr_dSph_'
        f'{SCAN_CONFIG["model"]}_{SCAN_CONFIG["front"]}_E{SCAN_CONFIG["n_bin"]}.png')
fig.savefig(_out, bbox_inches='tight', dpi=180)
plt.show()
print(f"\n  [OK] Saved {_out}")

# Tension check
if dsph_17 is not None:
    print()
    print(f"  GCE best-fit vs dSph 17yr UL tension (bb̄, Model {SCAN_CONFIG['model']}):")
    for year, _bm, _bsv in [(16, _bm16, _bsv16), (17, _bm17, _bsv17)]:
        _ul = _np.interp(_bm, dsph_17[:, 0], dsph_17[:, 1])
        _ratio = _bsv / _ul
        _verdict = "✗ EXCLUDED" if _ratio > 1 else f"✓ allowed (×{_ratio:.2f})"
        print(f"    {year}yr GCE: m={_bm:.0f} GeV, σv={_bsv:.2e}  vs  "
              f"dSph UL @ same m = {_ul:.2e}  →  {_verdict}")


In [ ]:
# ============================================================================
# V23 v3. dSph 95% UL evolution: 14yr vs 17yr (preliminary) — bb + 4b only
# ----------------------------------------------------------------------------
# Drops tau channel (17yr UL crosses 14yr in unexplained way — exclude
# from talk material until understood) and all-channels overlay (clutter).
# Two panels: bb̄ (prompt), 4b cascade (relevant to SFDM).
# ============================================================================
import os as _os
import numpy as _np
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator as _AutoMinorLocator

HBG_17YR_DIR = '/home/haebarg/GCE-Chi-square-fitting/GCE_17yr_data'

CHANNELS_DSPH = [
    ('bb',   r'$b\bar{b}$',                       'royalblue'),
    ('bbbb', r'$\chi\chi\to\phi\phi\to 4b$',     'darkgreen'),
]

def _load_dsph(prefix, year):
    """Server filename uses DOT: bb_14.0yr_30_dSphs.txt"""
    _p = f'{HBG_17YR_DIR}/{prefix}_{year}.0yr_30_dSphs.txt'
    if not _os.path.exists(_p):
        return None
    _d = _np.loadtxt(_p)
    return _d[_d[:, 1] > 1.1e-28]

print("=" * 78)
print("V23 v3: dSph 95% UL — 14yr vs 17yr preliminary (bb + 4b)")
print("=" * 78)

_data = {}
for _prefix, _label, _color in CHANNELS_DSPH:
    _d14 = _load_dsph(_prefix, 14)
    _d17 = _load_dsph(_prefix, 17)
    _data[_prefix] = (_d14, _d17, _label, _color)
    print(f"  {_prefix}: 14yr {('OK ('+str(len(_d14))+' pts)') if _d14 is not None else 'MISSING'}, "
          f"17yr {('OK ('+str(len(_d17))+' pts)') if _d17 is not None else 'MISSING'}")

if not any(_d[0] is not None for _d in _data.values()):
    print("  No dSph data found. Aborting.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

    for _i, (_prefix, _label, _color) in enumerate(CHANNELS_DSPH):
        _ax = axes[_i]
        _d14, _d17, _label, _color = _data[_prefix]
        if _d14 is None or _d17 is None:
            _ax.text(0.5, 0.5, f'{_prefix}\nNo data', transform=_ax.transAxes,
                     ha='center', va='center', fontsize=14)
            continue
        _ax.plot(_d14[:, 0], _d14[:, 1], '-', color=_color,
                 linewidth=2.0, label='14yr')
        _ax.plot(_d17[:, 0], _d17[:, 1], '--', color=_color,
                 linewidth=2.0, label='17yr (prelim)')
        _ax.fill_between(_d14[:, 0], _d14[:, 1],
                         _np.minimum(_d17[:, 1], _d14[:, 1]),
                         color=_color, alpha=0.15,
                         label='improvement region')
        _ax.axhline(3e-26, color='gray', linestyle=':',
                    linewidth=1.2, alpha=0.7, label=r'thermal relic')

        _ax.set_xscale('log'); _ax.set_yscale('log')
        _ax.set_xlim(2, 1e3); _ax.set_ylim(1e-27, 1e-23)
        _ax.set_xlabel(r'$m_\chi$ [GeV]', fontsize=12)
        _ax.set_ylabel(r'$\langle\sigma v\rangle_{95\%\,UL}$ [cm$^3$ s$^{-1}$]',
                       fontsize=12)
        _ax.set_title(_label, fontsize=13)
        _ax.tick_params(which='major', direction='in', length=6, width=1.2)
        _ax.tick_params(which='minor', direction='in', length=3, width=0.8)
        _ax.grid(True, which='major', linestyle='--', linewidth=0.5)
        _ax.legend(loc='upper left', fontsize=9.5)

    fig.suptitle("Fermi-LAT dwarf spheroidal 95% CL upper limits: "
                 "14yr vs 17yr preliminary\n"
                 "30 dSphs stacked analysis — region above lines is excluded",
                 fontsize=12, y=1.00)
    fig.tight_layout()
    _out = './dSph_UL_14yr_vs_17yr_bb_4b.png'
    fig.savefig(_out, bbox_inches='tight', dpi=160)
    plt.show()
    print(f"\n  [OK] Saved {_out}")

    print()
    print(f"  17yr / 14yr improvement (lower = stronger constraint):")
    print(f"  {'Channel':<8} {'@ m=50 GeV':>15} {'@ m=100 GeV':>15} {'@ m=200 GeV':>15}")
    for _prefix, _label, _color in CHANNELS_DSPH:
        _d14, _d17, _, _ = _data[_prefix]
        if _d14 is None or _d17 is None: continue
        _row = f"  {_prefix:<8}"
        for _m in [50, 100, 200]:
            _ul14 = _np.interp(_m, _d14[:, 0], _d14[:, 1])
            _ul17 = _np.interp(_m, _d17[:, 0], _d17[:, 1])
            _r = _ul17 / _ul14
            _row += f" {_r:>15.3f}"
        print(_row)


In [ ]:
# ============================================================================
# V24 v2. GCE flux: Cholis+2022 12yr (paper) vs haebarg 17yr expected
# ----------------------------------------------------------------------------
# Single-panel comparison for Model X showing analysis evolution:
#   Cholis 12yr (paper)  →  haebarg 16yr (current)  →  haebarg 17yr expected
#
# Cholis reference: ../GCE_TEMPLATES_FILES_v3/Figures_12_and_14_GCE_Spectra/
#                   GCE_Model{Roman}_flux_Inner40x40_masked_disk.dat
#                   columns: E, flux, lo_1sig, hi_1sig
#
# 17yr expected = haebarg 16yr best-fit, stat × √(16/17), sys unchanged
# ============================================================================
import os as _os
import numpy as _np
import matplotlib.pyplot as plt

# ----------------------------- Configuration -----------------------------
HBG_16YR_DIR = '/home/haebarg/GCE-Chi-square-fitting/GCE_16yr_data'
CHOLIS_ZEN   = '../GCE_TEMPLATES_FILES_v3/Figures_12_and_14_GCE_Spectra'
SCALE_17_OVER_16 = _np.sqrt(16.0 / 17.0)

MODEL = 'X'   # single model per user request

print("=" * 78)
print(f"V24 v2: Cholis 12yr vs haebarg 17yr expected — Model {MODEL}")
print("=" * 78)

# ----------------------------- Load Cholis 12yr -----------------------------
_cholis_p = f'{CHOLIS_ZEN}/GCE_Model{MODEL}_flux_Inner40x40_masked_disk.dat'
print(f"  Cholis 12yr file: {_cholis_p}")
if not _os.path.exists(_cholis_p):
    print(f"    ✗ NOT FOUND — please verify path")
    raise FileNotFoundError(_cholis_p)
_cd = _np.loadtxt(_cholis_p)
print(f"    ✓ shape: {_cd.shape}")
print(f"    columns interpreted: E, flux, lo_1sig, hi_1sig")
_E_c, _f_c, _lo_c, _hi_c = _cd[:, 0], _cd[:, 1], _cd[:, 2], _cd[:, 3]

# ----------------------------- Load haebarg 16yr -----------------------------
_hb_p = f'{HBG_16YR_DIR}/GCE_model_{MODEL}_front_16yr_cholis.dat'
print(f"\n  haebarg 16yr file: {_hb_p}")
if not _os.path.exists(_hb_p):
    print(f"    ✗ NOT FOUND")
    raise FileNotFoundError(_hb_p)
_hd = _np.loadtxt(_hb_p)
_E_h    = _hd[:, 0]
_f_h    = _hd[:, 1]
_stat_h = _hd[:, 2]
_lo_h   = _hd[:, 3]
_hi_h   = _hd[:, 4]
print(f"    ✓ shape: {_hd.shape}")
print(f"    columns: E, flux, stat_unc, lo_1sig, hi_1sig")

# ----------------------------- Construct 17yr expected -----------------------------
_stat_17 = _stat_h * SCALE_17_OVER_16
_f_17    = _f_h    # central value unchanged (statistical scaling assumption)
_lo_17   = _f_17 - _stat_17
_hi_17   = _f_17 + _stat_17
print(f"\n  17yr expected: stat scaled by √(16/17) = {SCALE_17_OVER_16:.4f}")

# ----------------------------- Plot single panel -----------------------------
fig, ax = plt.subplots(figsize=(10, 7))

# Cholis 12yr — black squares + shaded band
_yerr_c = [_np.maximum(_f_c - _lo_c, 0), _np.maximum(_hi_c - _f_c, 0)]
ax.errorbar(_E_c, _f_c, yerr=_yerr_c,
            marker='s', color='black', markersize=7, linewidth=1.8,
            capsize=4, label='Cholis+2022 12yr (paper)', zorder=10)
ax.fill_between(_E_c, _lo_c, _hi_c, color='black', alpha=0.10, zorder=1)


# 17yr expected — orange triangles + shaded band
_yerr_17 = [_stat_17, _stat_17]
ax.errorbar(_E_h, _f_17, yerr=_yerr_17,
            marker='^', color='darkorange', markersize=7, linewidth=2.0,
            capsize=4, alpha=0.95,
            label=fr'17yr', zorder=9)
ax.fill_between(_E_h, _lo_17, _hi_17, color='darkorange', alpha=0.18, zorder=2,
                label=r'17yr')

# Axes / styling
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlim(0.3, 36); ax.set_ylim(1e-7, 3e-6)
ax.set_xlabel('E [GeV]', fontsize=13)
ax.set_ylabel(r'$E^2\,dN/dE$  [GeV cm$^{-2}$ s$^{-1}$ sr$^{-1}$]', fontsize=13)
ax.tick_params(which='major', direction='in', length=7, width=1.3)
ax.tick_params(which='minor', direction='in', length=4, width=0.9)
ax.grid(True, which='major', linestyle='--', linewidth=0.5, alpha=0.5)
ax.legend(loc='lower left', fontsize=11, framealpha=0.92)

# Compute summary ratios in 1-10 GeV
_m_in_h = (_E_h >= 1.0) & (_E_h <= 10.0)
_f_c_at_h = _np.interp(_E_h, _E_c, _f_c)
_ratio_16_vs_c = _np.nanmean((_f_h / _f_c_at_h)[_m_in_h])
_ratio_17_vs_c = _np.nanmean((_f_17 / _f_c_at_h)[_m_in_h])


ax.set_title(f"GCE flux evolution — Model {MODEL}: Cholis 12yr → 17yr",
             fontsize=13)

fig.tight_layout()
_out = f'./cholis_12yr_vs_haebarg_17yr_Model{MODEL}.png'
fig.savefig(_out, bbox_inches='tight', dpi=180)
plt.show()
print(f"\n  [OK] Saved {_out}")

print()
print(f"  Summary (Model {MODEL}, 1-10 GeV):")
print(f"    haebarg 16yr / Cholis 12yr ratio = {_ratio_16_vs_c:.3f}")
print(f"    17yr expected / Cholis 12yr ratio = {_ratio_17_vs_c:.3f}")
print(f"    (these are equal because 17yr expected uses 16yr best-fit unchanged)")
print()
print(f"  Note for talks:")
print(f"  - Cholis (12yr) and haebarg (16yr) baselines may differ if our pipeline")
print(f"    has a systematic offset (V12b: Model X ours/cholis ≈ 0.87 in 12yr)")
print(f"  - The 17yr 'expected' inherits this offset by construction")
print(f"  - Real 17yr will need its own validation, this is statistical projection")


In [ ]:
# ============================================================================
# V25. DM contour: Cholis+2022 12yr (paper Figure 18) vs haebarg 17yr expected
# ----------------------------------------------------------------------------
# bb̄ channel only (Cholis published contour is for bb̄)
#
# Cholis Figure 18 left panel data: /home/sanghwan/FermiLAT/Sanghwan/GCE_references/
#   2112fig18left_best_fit_point.txt   — single (m, σv) point
#   2112fig18left_dashed_line.txt      — 1σ contour curve
#   2112fig18left_solid_line.txt       — 2σ contour curve
# (filename pattern from arXiv:2112.09706 Figure 18 left = bb̄ best-fit + contours)
#
# haebarg 17yr expected contour: same logic as V22 v5 (16yr × stat scale)
# ============================================================================
import os as _os
import warnings as _warnings
import numpy as _np
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator as _AutoMinorLocator
from scipy.interpolate import interp1d as _i1d
from scipy.interpolate import RegularGridInterpolator as _RGI

# ----------------------------- Configuration -----------------------------
HBG_16YR_DIR = '/home/haebarg/GCE-Chi-square-fitting/GCE_16yr_data'
COV_DIR      = '/home/haebarg/GCE-Chi-square-fitting/Cov'
SANGHWAN_REF = '/home/sanghwan/FermiLAT/Sanghwan/GCE_references'

J_FACTOR = 3.5251837158376415e+21
SR       = 0.4288213187542626
SCALE_17_OVER_16 = _np.sqrt(16.0 / 17.0)

_PPPC4_CANDIDATES = [
    './Prompt_spectra/AtProduction_gammas.dat',
    '/home/haebarg/ipynb/AtProduction_gammas.dat',
    '/home/haebarg/PPPC4DMID/AtProduction_gammas.dat',
    './AtProduction_gammas.dat',
]
PPPC4_PATH = next((_c for _c in _PPPC4_CANDIDATES if _os.path.exists(_c)), None)

MODEL = 'X'
FRONT = 'front'
N_BIN = 14

print("=" * 78)
print(f"V25: DM contour — Cholis 12yr (Fig 18) vs haebarg 17yr expected (bb̄)")
print("=" * 78)

if PPPC4_PATH is None:
    print("  ✗ PPPC4 not found"); raise SystemExit
print(f"  PPPC4: {PPPC4_PATH}")

# ----------------------------- Load Cholis Fig 18 reference -----------------------------
_cholis_best_p   = f'{SANGHWAN_REF}/2112fig18left_best_fit_point.txt'
_cholis_dashed_p = f'{SANGHWAN_REF}/2112fig18left_dashed_line.txt'
_cholis_solid_p  = f'{SANGHWAN_REF}/2112fig18left_solid_line.txt'

_have_cholis_ref = True
for _p in [_cholis_best_p, _cholis_dashed_p, _cholis_solid_p]:
    if not _os.path.exists(_p):
        print(f"  ⚠ MISSING: {_p}")
        _have_cholis_ref = False
    else:
        print(f"  ✓ {_p}")

def _safe_loadtxt(p):
    """Robust loader: tries direct load, then skips 1 row (header), then skips 2."""
    if not _os.path.exists(p):
        return None
    for _skip in (0, 1, 2):
        try:
            return _np.loadtxt(p, skiprows=_skip)
        except (ValueError, StopIteration):
            continue
    print(f"  ⚠ could not parse {p}")
    return None

# Cholis Fig 18 files have header 'x y' — _safe_loadtxt handles it.
# y values are in units of 10^-26 cm³/s — convert to absolute σv (cm³/s).
SIGMAV_UNIT_SCALE = 1e-26

cholis_best   = _safe_loadtxt(_cholis_best_p)
cholis_dashed = _safe_loadtxt(_cholis_dashed_p)
cholis_solid  = _safe_loadtxt(_cholis_solid_p)

# Apply unit scaling: y → y × 10^-26
for _arr in (cholis_best, cholis_dashed, cholis_solid):
    if _arr is not None and _arr.ndim >= 1:
        if _arr.ndim == 1:
            _arr[1] *= SIGMAV_UNIT_SCALE
        else:
            _arr[:, 1] *= SIGMAV_UNIT_SCALE
print(f"  Applied y-axis scaling: y_data × {SIGMAV_UNIT_SCALE} (σv in cm³/s)")

if cholis_best is not None:
    if cholis_best.ndim == 1:
        print(f"  Cholis best-fit point: m={cholis_best[0]:.1f} GeV, σv={cholis_best[1]:.2e}")
    else:
        print(f"  Cholis best-fit shape: {cholis_best.shape}")

# ----------------------------- Load PPPC4 bb̄ -----------------------------
_pppc = _np.loadtxt(PPPC4_PATH, skiprows=1)
_masses = _np.unique(_pppc[:, 0]); _logx = _np.unique(_pppc[:, 1])
_z_bb = _pppc[:, 13].reshape(len(_masses), len(_logx))
_interp_bb = _RGI((_masses, _logx), _z_bb, bounds_error=False, fill_value=0.0)

def extract_pppc_bb(mass):
    _pts = _np.column_stack([_np.full(len(_logx), mass), _logx])
    _dNdlogx = _interp_bb(_pts)
    _E = mass * (10**_logx)
    with _np.errstate(divide='ignore', invalid='ignore'):
        _dNdE = _dNdlogx / (_E * _np.log(10))
    return _E, _np.nan_to_num(_dNdE, nan=0.0, posinf=0.0, neginf=0.0)

# ----------------------------- Load 16yr GCE + covariance -----------------------------
_dat_p = f'{HBG_16YR_DIR}/GCE_model_{MODEL}_{FRONT}_16yr_cholis.dat'
_g = _np.loadtxt(_dat_p)
emeans  = _g[:N_BIN, 0]
flux_16 = _g[:N_BIN, 1]
stat_16 = _g[:N_BIN, 2]

_cov_p = f'{COV_DIR}/approx_covariance_{N_BIN}x{N_BIN}_{FRONT}_model_{MODEL}_16yr.npy'
cov_emp_16 = _np.load(_cov_p)
print(f"  ✓ Loaded covariance: {_cov_p}")
try:
    cov_total_16 = _np.diag(stat_16**2) + cov_emp_16
    inv_cov_16 = _np.linalg.inv(cov_total_16)
except _np.linalg.LinAlgError:
    cov_total_16 = cov_emp_16
    inv_cov_16 = _np.linalg.inv(cov_total_16)

# 17yr expected
stat_17 = stat_16 * SCALE_17_OVER_16
flux_17 = flux_16
_sys_cov = cov_total_16 - _np.diag(stat_16**2)
cov_total_17 = _np.diag(stat_17**2) + _sys_cov
inv_cov_17 = _np.linalg.inv(cov_total_17)

# ----------------------------- χ² function -----------------------------
def chi_square_bb(dm_mass, sigma_v, flux_data, inv_cov):
    _warnings.simplefilter("ignore", category=RuntimeWarning)
    _E, _dNdE = extract_pppc_bb(dm_mass)
    _interp_func = _i1d(_E, _dNdE, fill_value=0, bounds_error=False, kind='linear')
    _dNdE_at = _interp_func(emeans)
    model = (emeans**2) * _dNdE_at * (sigma_v / dm_mass**2) * J_FACTOR / SR
    delta = model - flux_data
    return delta.T @ inv_cov @ delta

# ----------------------------- Grid scan -----------------------------
sigmav_range = _np.logspace(-27, -25, 60)
mass_range   = _np.logspace(_np.log10(10), _np.log10(200), 60)
DM_mass_grid, Sigmav_grid = _np.meshgrid(mass_range, sigmav_range)

print(f"\n  Computing 17yr expected χ² grid...")
chi2_17 = _np.vectorize(lambda m, sv: chi_square_bb(m, sv, flux_17, inv_cov_17))(DM_mass_grid, Sigmav_grid)
_idx_17 = _np.unravel_index(_np.argmin(chi2_17), chi2_17.shape)
_bm17, _bsv17, _min17 = DM_mass_grid[_idx_17], Sigmav_grid[_idx_17], chi2_17[_idx_17]
_dof = N_BIN - 2
print(f"  haebarg 17yr expected best-fit: m={_bm17:.1f} GeV, σv={_bsv17:.3e}, "
      f"χ²/dof = {_min17/_dof:.2f}")

# ----------------------------- Plot single panel -----------------------------
fig, ax = plt.subplots(figsize=(10, 7.5))

# Cholis 12yr contours (from Fig 18)
if cholis_dashed is not None and cholis_dashed.shape[1] >= 2:
    ax.plot(cholis_dashed[:, 0], cholis_dashed[:, 1], '--', color='black',
            linewidth=1.8, alpha=0.85, label='Cholis 12yr 1σ (paper)', zorder=8)
if cholis_solid is not None and cholis_solid.shape[1] >= 2:
    ax.plot(cholis_solid[:, 0], cholis_solid[:, 1], '-', color='black',
            linewidth=2.0, alpha=0.85, label='Cholis 12yr 2σ (paper)', zorder=8)
if cholis_best is not None:
    if cholis_best.ndim == 1:
        ax.plot(cholis_best[0], cholis_best[1], '*', color='black', markersize=18,
                markeredgecolor='gold', label='Cholis 12yr best-fit', zorder=12)
    else:
        # If multiple points
        ax.plot(cholis_best[:, 0], cholis_best[:, 1], '*', color='black',
                markersize=14, markeredgecolor='gold',
                label='Cholis 12yr best-fit', zorder=12)

# haebarg 17yr expected contours
_levels_17 = [_min17 + 2.30, _min17 + 6.18]
try:
    ax.contour(DM_mass_grid, Sigmav_grid, chi2_17,
               levels=_levels_17, colors='darkorange',
               linestyles=['--', '-'], linewidths=[1.8, 2.2])
except Exception as _e:
    print(f"  ⚠ 17yr contour render failed: {_e}")

ax.plot(_bm17, _bsv17, 'o', color='darkorange', markersize=11,
        markeredgecolor='black',
        label=f'17yr expected best-fit\n(m={_bm17:.0f} GeV, σv={_bsv17:.2e})',
        zorder=11)

# Thermal relic
ax.axhline(3e-26, color='gray', linestyle=':', linewidth=1.3, alpha=0.7,
           label=r'thermal relic ($3\times10^{-26}$)')

# Axes / styling
ax.set_yscale('log'); ax.set_xscale('linear')
ax.set_xlim(10, 200); ax.set_ylim(1e-27, 1e-24)
ax.tick_params(which='major', direction='in', length=7, width=1.3)
ax.tick_params(which='minor', direction='in', length=4, width=0.9)
ax.xaxis.set_minor_locator(_AutoMinorLocator())
ax.grid(True, which='major', linestyle='--', linewidth=0.5, alpha=0.5)

ax.text(0.95, 0.05, '68% CL (1σ dashed), 95% CL (2σ solid)',
        transform=ax.transAxes, ha='right', va='bottom', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.85, edgecolor='gray'))
ax.text(0.05, 0.05, fr'17yr $\chi^2$/dof = {_min17/_dof:.2f}',
        transform=ax.transAxes, ha='left', va='bottom', fontsize=10)
ax.text(0.05, 0.95, r'$b\bar{b}$', transform=ax.transAxes,
        ha='left', va='top', fontsize=22, fontweight='bold')

ax.set_xlabel(r'$m_\chi$ [GeV]', fontsize=13)
ax.set_ylabel(r'$\langle\sigma v\rangle$ [cm$^3$ s$^{-1}$]', fontsize=13)
ax.set_title(f'DM contour ($b\\bar{{b}}$): Cholis+2022 12yr (Fig 18) vs '
             f'haebarg 17yr expected — Model {MODEL}',
             fontsize=12)
ax.legend(loc='upper left', fontsize=9.5, framealpha=0.92)

fig.tight_layout()
_out = f'./DM_contour_cholis_12yr_vs_haebarg_17yr_bb_Model{MODEL}.png'
fig.savefig(_out, bbox_inches='tight', dpi=180)
plt.show()
print(f"\n  [OK] Saved {_out}")

# Comparison table
print()
print(f"  Best-fit comparison (bb̄, Model {MODEL}):")
if cholis_best is not None:
    if cholis_best.ndim == 1:
        print(f"    Cholis 12yr (paper):   m = {cholis_best[0]:>7.1f} GeV, σv = {cholis_best[1]:.3e}")
    else:
        for _row in cholis_best:
            print(f"    Cholis 12yr point:     m = {_row[0]:>7.1f} GeV, σv = {_row[1]:.3e}")
print(f"    haebarg 17yr expected: m = {_bm17:>7.1f} GeV, σv = {_bsv17:.3e}")

print()
print(f"  Notes for talks:")
print(f"  - Cholis's contour is a published 12yr result (Figure 18, arXiv:2112.09706)")
print(f"  - Our 17yr expected uses haebarg 16yr best-fit, stat × √(16/17)")
print(f"  - If best-fit points differ → systematic offset between pipelines")
print(f"    (V12b documented Model X ours/cholis ≈ 0.87 in flux)")
print(f"  - Real 17yr will need its own full analysis to update this comparison")


In [ ]:
# ============================================================================
# V26 v6. Combined DM constraints — bb̄ + 4b GCE + 4b antiproton (PNG-extracted)
# ----------------------------------------------------------------------------
# bb̄ panel: GCE contour (PPPC4) + dSph (14/17yr) + antiproton (17yr)
# 4b panel: GCE contour (MG5) + dSph (14/17yr)
#
# MG5Interpolator is now defined inline (no need to run separate cell first).
# Module-level cache _MG5_DATA_CACHE keeps the heavy file-loading work to once.
# ============================================================================
import os as _os
import re as _re
import glob as _glob
import warnings as _warnings
import numpy as _np
import pandas as _pd
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator as _AutoMinorLocator
from scipy.interpolate import interp1d as _i1d
from scipy.interpolate import RegularGridInterpolator as _RGI

# ----------------------------- Configuration -----------------------------
HBG_16YR_DIR = '/home/haebarg/GCE-Chi-square-fitting/GCE_16yr_data'
HBG_17YR_DIR = '/home/haebarg/GCE-Chi-square-fitting/GCE_17yr_data'
COV_DIR      = '/home/haebarg/GCE-Chi-square-fitting/Cov'
MG5_4B_DIR   = '/home/haebarg/MG5_aMC_v3_5_12/Spectra_Data_sfdm_4b_r0.5/'

J_FACTOR = 3.5251837158376415e+21
SR       = 0.4288213187542626
SCALE_17_OVER_16 = _np.sqrt(16.0 / 17.0)

_PPPC4_CANDIDATES = [
    './Prompt_spectra/AtProduction_gammas.dat',
    '/home/haebarg/ipynb/AtProduction_gammas.dat',
    '/home/haebarg/PPPC4DMID/AtProduction_gammas.dat',
    './AtProduction_gammas.dat',
]
PPPC4_PATH = next((_c for _c in _PPPC4_CANDIDATES if _os.path.exists(_c)), None)

PBAR_PREFIX_MAP = {'bb': '2b', 'bbbb': '4b'}

MODEL = 'X'
FRONT = 'front'
N_BIN = 14

# ============================================================================
# Embedded MG5Interpolator (user's reference code, verbatim)
# ============================================================================
try:
    _MG5_DATA_CACHE
except NameError:
    _MG5_DATA_CACHE = {}

class MG5Interpolator:
    """User's reference MG5 spectrum interpolator (embedded for self-containment)."""
    def __init__(self, mass, channel='bb',
                 base_dir='/home/haebarg/MG5_aMC_v3_5_12/Spectra_Data_bb/'):
        self.target_mass = float(mass)
        self.channel = channel
        self.base_dir = base_dir
        cache_key = (base_dir, channel)
        if cache_key not in _MG5_DATA_CACHE:
            print(f"  [MG5/{channel}] data loading from {base_dir}...")
            _MG5_DATA_CACHE[cache_key] = self._build_interpolator(base_dir, channel)
        self.interp_func, self.common_energy_grid = _MG5_DATA_CACHE[cache_key]

    def _build_interpolator(self, base_dir, channel):
        pattern = _os.path.join(base_dir, f"MM_mpsi*GeV_{channel}_photon.csv")
        files = _glob.glob(pattern)
        if not files:
            raise FileNotFoundError(f"MG5 files not found: {pattern}")
        data_list = []
        for fpath in files:
            fname = _os.path.basename(fpath)
            match = _re.search(r"MM_mpsi([0-9\.]+)GeV", fname)
            if match:
                mass = float(match.group(1))
                try:
                    df = _pd.read_csv(fpath)
                    if 'Energy_GeV' in df.columns and 'dNdE' in df.columns:
                        df = df.sort_values('Energy_GeV')
                        data_list.append({
                            'mass': mass,
                            'energy': df['Energy_GeV'].values,
                            'dNdE': df['dNdE'].values
                        })
                except Exception as e:
                    print(f"  ⚠️ {fname} read fail: {e}")
        if not data_list:
            raise ValueError("No valid MG5 CSV data loaded.")
        data_list.sort(key=lambda x: x['mass'])
        dm_masses = _np.array([x['mass'] for x in data_list])
        min_e = 1e-5
        max_e = max([x['energy'][-1] for x in data_list])
        common_energy = _np.geomspace(min_e, max_e, 500)
        flux_matrix = []
        for item in data_list:
            f = _i1d(item['energy'], item['dNdE'], bounds_error=False, fill_value=0.0)
            flux_matrix.append(f(common_energy))
        flux_matrix = _np.array(flux_matrix)
        interp = _RGI(
            (dm_masses, _np.log10(common_energy)),
            flux_matrix,
            bounds_error=False, fill_value=0.0
        )
        print(f"  ✅ MG5/{channel} loaded ({len(dm_masses)} files, "
              f"mass {dm_masses.min():.1f}–{dm_masses.max():.1f} GeV)")
        return interp, common_energy

    def interpolated_table(self):
        pts = _np.column_stack([
            _np.full_like(self.common_energy_grid, self.target_mass),
            _np.log10(self.common_energy_grid)
        ])
        dNdE = self.interp_func(pts)
        mask = self.common_energy_grid <= self.target_mass
        dNdE[~mask] = 0.0
        return self.common_energy_grid, dNdE

# ----------------------------- Helpers -----------------------------
def _load_dsph(prefix, year):
    _p = f'{HBG_17YR_DIR}/{prefix}_{year}.0yr_30_dSphs.txt'
    if not _os.path.exists(_p):
        return None, _p
    _d = _np.loadtxt(_p)
    return _d[_d[:, 1] > 1.1e-28], _p

def _load_antiproton_17yr(channel_name):
    _pbar_prefix = PBAR_PREFIX_MAP.get(channel_name)
    if _pbar_prefix is None:
        return None, None
    _candidates = [
        f'{HBG_17YR_DIR}/{_pbar_prefix}_phi_avg_prior_True_data_model_cov.txt',
        f'{HBG_17YR_DIR}/{_pbar_prefix}_phi_avg_data_model_cov.txt',
        f'{HBG_17YR_DIR}/antiproton_{_pbar_prefix}_17yr.txt',
    ]
    for _p in _candidates:
        if _os.path.exists(_p):
            _d = _np.loadtxt(_p)
            return _d[_d[:, 1] > 1.1e-28], _p
    return None, _candidates[0]

# ----------------------------- Header -----------------------------
print("=" * 78)
print(f"V26 v6: Combined DM constraints (bb̄ + 4b GCE) — Model {MODEL}")
print("=" * 78)
print()

# ----------------------------- Load shared data -----------------------------
print("  Fermi dSph 95% UL:")
dsph_bb_14,   _p = _load_dsph('bb',   14); print(f"    bb̄  14yr: {'✓ ' + str(len(dsph_bb_14)) + ' pts' if dsph_bb_14 is not None else '✗'}")
dsph_bb_17,   _p = _load_dsph('bb',   17); print(f"    bb̄  17yr: {'✓ ' + str(len(dsph_bb_17)) + ' pts' if dsph_bb_17 is not None else '✗'}")
dsph_bbbb_14, _p = _load_dsph('bbbb', 14); print(f"    4b 14yr: {'✓ ' + str(len(dsph_bbbb_14)) + ' pts' if dsph_bbbb_14 is not None else '✗'}")
dsph_bbbb_17, _p = _load_dsph('bbbb', 17); print(f"    4b 17yr: {'✓ ' + str(len(dsph_bbbb_17)) + ' pts' if dsph_bbbb_17 is not None else '✗'}")

print("\n  AMS-02 antiproton 95% UL:")
ap_bb_17, _p = _load_antiproton_17yr('bb'); print(f"    bb̄  17yr: {'✓ ' + str(len(ap_bb_17)) + ' pts (' + _os.path.basename(_p) + ')' if ap_bb_17 is not None else '✗'}")
ap_bbbb_17, _p = _load_antiproton_17yr('bbbb'); print(f"    4b 17yr: {'✓ ' + str(len(ap_bbbb_17)) + ' pts (' + _os.path.basename(_p) + ', PNG-extracted, ~10% unc)' if ap_bbbb_17 is not None else '✗'}")

# Load 16yr GCE data + covariance
_dat_p = f'{HBG_16YR_DIR}/GCE_model_{MODEL}_{FRONT}_16yr_cholis.dat'
_g = _np.loadtxt(_dat_p)
emeans  = _g[:N_BIN, 0]
flux_16 = _g[:N_BIN, 1]
stat_16 = _g[:N_BIN, 2]

_cov_p = f'{COV_DIR}/approx_covariance_{N_BIN}x{N_BIN}_{FRONT}_model_{MODEL}_16yr.npy'
cov_emp_16 = _np.load(_cov_p)
try:
    cov_total_16 = _np.diag(stat_16**2) + cov_emp_16
    _ = _np.linalg.inv(cov_total_16)
except _np.linalg.LinAlgError:
    cov_total_16 = cov_emp_16

stat_17 = stat_16 * SCALE_17_OVER_16
flux_17 = flux_16
_sys_cov = cov_total_16 - _np.diag(stat_16**2)
cov_total_17 = _np.diag(stat_17**2) + _sys_cov
inv_cov_17 = _np.linalg.inv(cov_total_17)

# ----------------------------- Spectrum functions -----------------------------
# bb̄: PPPC4
if PPPC4_PATH is None:
    print(f"\n  ⚠ PPPC4 not found, bb̄ contour will be skipped")
    pppc_interp_bb = None
else:
    print(f"\n  PPPC4: {PPPC4_PATH}")
    _pppc = _np.loadtxt(PPPC4_PATH, skiprows=1)
    _masses_p = _np.unique(_pppc[:, 0]); _logx_p = _np.unique(_pppc[:, 1])
    _z_bb = _pppc[:, 13].reshape(len(_masses_p), len(_logx_p))
    pppc_interp_bb = _RGI((_masses_p, _logx_p), _z_bb, bounds_error=False, fill_value=0.0)

def spectrum_pppc_bb(mass):
    if pppc_interp_bb is None:
        return _np.array([1.0]), _np.array([0.0])
    _pts = _np.column_stack([_np.full(len(_logx_p), mass), _logx_p])
    _dNdlogx = pppc_interp_bb(_pts)
    _E = mass * (10**_logx_p)
    with _np.errstate(divide='ignore', invalid='ignore'):
        _dNdE = _dNdlogx / (_E * _np.log(10))
    return _E, _np.nan_to_num(_dNdE, nan=0.0, posinf=0.0, neginf=0.0)

def spectrum_mg5_4b(mass):
    """4b cascade spectrum from MG5 (uses embedded MG5Interpolator)."""
    try:
        interp = MG5Interpolator(mass, '4b', base_dir=MG5_4B_DIR)
        return interp.interpolated_table()
    except Exception as _e:
        # First call may fail if files missing — print once
        if not hasattr(spectrum_mg5_4b, '_warned'):
            print(f"  ⚠ MG5 4b unavailable: {_e}")
            spectrum_mg5_4b._warned = True
        return _np.array([1.0]), _np.array([0.0])

# Pre-test MG5 availability with a representative mass
_mg5_works = False
try:
    _e, _d = spectrum_mg5_4b(50.0)
    if len(_d) > 1 and _d.max() > 0:
        _mg5_works = True
        print(f"  ✓ MG5Interpolator works (4b at m=50 GeV: dN/dE max = {_d.max():.3e})")
except Exception:
    pass
if not _mg5_works:
    print(f"  ⚠ MG5 4b not functional — 4b GCE contour will be skipped")

# ----------------------------- Generalized χ² calculation -----------------------------
def chi_square(dm_mass, sigma_v, spectrum_func):
    _warnings.simplefilter("ignore", category=RuntimeWarning)
    try:
        _E, _dNdE = spectrum_func(dm_mass)
        if len(_E) < 2 or _dNdE.max() == 0:
            return 1e9
        _interp_func = _i1d(_E, _dNdE, fill_value=0, bounds_error=False, kind='linear')
        _dNdE_at = _interp_func(emeans)
        model = (emeans**2) * _dNdE_at * (sigma_v / dm_mass**2) * J_FACTOR / SR
        delta = model - flux_17
        return delta.T @ inv_cov_17 @ delta
    except Exception:
        return 1e9

def compute_chi2_grid(spectrum_func, label):
    sigmav_range = _np.logspace(-27, -25, 60)
    mass_range   = _np.logspace(_np.log10(10), _np.log10(1000), 60)
    DM_grid, SV_grid = _np.meshgrid(mass_range, sigmav_range)
    print(f"  Computing χ² grid for {label}...")
    chi2_vec = _np.vectorize(lambda m, sv: chi_square(m, sv, spectrum_func))
    chi2 = chi2_vec(DM_grid, SV_grid)
    _idx = _np.unravel_index(_np.argmin(chi2), chi2.shape)
    _bm, _bsv, _min = DM_grid[_idx], SV_grid[_idx], chi2[_idx]
    _dof = N_BIN - 2
    print(f"    {label} best-fit: m={_bm:.1f} GeV, σv={_bsv:.3e}, χ²/dof = {_min/_dof:.2f}")
    return DM_grid, SV_grid, chi2, _bm, _bsv, _min, _dof

# ----------------------------- Compute both grids -----------------------------
chi2_bb = None
if pppc_interp_bb is not None:
    chi2_bb = compute_chi2_grid(spectrum_pppc_bb, 'bb̄ (PPPC4)')

chi2_4b = None
if _mg5_works:
    chi2_4b = compute_chi2_grid(spectrum_mg5_4b, '4b (MG5)')

# ----------------------------- Plot helper -----------------------------
def _plot_channel(channel_label, channel_tex, gce_data,
                  dsph_14, dsph_17, ap_17, save_name):
    fig, _ax = plt.subplots(figsize=(10, 7.5))

    if gce_data is not None:
        DM_g, SV_g, chi2_g, bm, bsv, mn, dof = gce_data
        _levels = [mn + 2.30, mn + 6.18]
        try:
            _ax.contour(DM_g, SV_g, chi2_g, levels=_levels,
                        colors='darkorange', linestyles=['--', '-'],
                        linewidths=[1.8, 2.2])
        except Exception as _e:
            print(f"    ⚠ contour render failed for {channel_label}: {_e}")
        _ax.plot(bm, bsv, 'o', color='darkorange', markersize=11,
                 markeredgecolor='black',
                 label=f'GCE 17yr best-fit\n($m={bm:.0f}$ GeV, $\\sigma v={bsv:.1e}$)',
                 zorder=20)
        _ax.plot([], [], '--', color='darkorange', linewidth=1.8, label=r'GCE 17yr 1$\sigma$')
        _ax.plot([], [], '-',  color='darkorange', linewidth=2.2, label=r'GCE 17yr 2$\sigma$')

    if dsph_14 is not None:
        _ax.plot(dsph_14[:, 0], dsph_14[:, 1], '-', color='black',
                 linewidth=1.6, alpha=0.85, label='Fermi dSph 95% UL (14yr)')
    if dsph_17 is not None:
        _ax.plot(dsph_17[:, 0], dsph_17[:, 1], '--', color='red',
                 linewidth=1.6, alpha=0.9,
                 label='Fermi dSph 95% UL (17yr prelim)')

    if ap_17 is not None:
        _ax.plot(ap_17[:, 0], ap_17[:, 1], '-', color='magenta',
                 linewidth=1.8, alpha=0.9,
                 label=r'AMS-02 antiproton 95% UL')

    _ax.axhline(3e-26, color='gray', linestyle=':', linewidth=1.3, alpha=0.7,
                label=r'thermal relic ($3\times10^{-26}$)')

    _ax.set_yscale('log'); _ax.set_xscale('log')
    _ax.set_xlim(20, 200); _ax.set_ylim(1e-27, 5e-25)
    _ax.set_xlabel(r'$m_\chi$ [GeV]', fontsize=13)
    _ax.set_ylabel(r'$\langle\sigma v\rangle$ [cm$^3$ s$^{-1}$]', fontsize=13)
    _ax.tick_params(which='major', direction='in', length=7, width=1.3)
    _ax.tick_params(which='minor', direction='in', length=4, width=0.9)
    _ax.grid(True, which='major', linestyle='--', linewidth=0.5, alpha=0.5)

    _ax.text(0.05, 0.95, channel_tex, transform=_ax.transAxes,
             ha='left', va='top', fontsize=22, fontweight='bold')

    _ax.set_title(f'{channel_tex}: GCE 17yr + external '
                  f'(Model {MODEL})', fontsize=12)
    _ax.legend(loc='lower right', fontsize=9, framealpha=0.92)

    fig.tight_layout()
    fig.savefig(save_name, bbox_inches='tight', dpi=180)
    plt.show()
    print(f"  [OK] Saved {save_name}")

# ----------------------------- Generate per-channel plots -----------------------------
print("\n  Generating bb̄ panel...")
_plot_channel('bb', r'$b\bar{b}$',
              gce_data=chi2_bb,
              dsph_14=dsph_bb_14, dsph_17=dsph_bb_17,
              ap_17=ap_bb_17,
              save_name=f'./combined_constraints_bb_Model{MODEL}.png')

print("\n  Generating 4b panel...")
_plot_channel('bbbb', r'$\chi\chi\to\phi\phi\to 4b$',
              gce_data=chi2_4b,
              dsph_14=dsph_bbbb_14, dsph_17=dsph_bbbb_17,
              ap_17=ap_bbbb_17,
              save_name=f'./combined_constraints_4b_Model{MODEL}.png')

# ----------------------------- Tension summary -----------------------------
print()
print(f"  Tension check (GCE 17yr best-fit vs each external 95% UL):")
for ch_label, ch_data, ext_pairs in [
    (r'bb̄', chi2_bb, [('Fermi dSph 14yr', dsph_bb_14),
                       ('Fermi dSph 17yr', dsph_bb_17),
                       ('AMS-02 p̄ 17yr',   ap_bb_17)]),
    (r'4b',  chi2_4b, [('Fermi dSph 14yr', dsph_bbbb_14),
                       ('Fermi dSph 17yr', dsph_bbbb_17),
                       ('AMS-02 p̄ (slide ext)', ap_bbbb_17)]),
]:
    if ch_data is None:
        continue
    bm, bsv = ch_data[3], ch_data[4]
    print(f"\n  {ch_label} (best-fit m={bm:.0f} GeV, σv={bsv:.2e}):")
    for name, _ul_data in ext_pairs:
        if _ul_data is None: continue
        _ul = _np.interp(bm, _ul_data[:, 0], _ul_data[:, 1])
        _ratio = bsv / _ul
        _verdict = ("✗ EXCLUDED" if _ratio > 1
                    else f"✓ allowed (×{_ratio:.2f} below UL)")
        print(f"    {name:<22}: σv_UL @ m={bm:.0f} GeV = {_ul:.2e}  → {_verdict}")


In [ ]:
# ---- V27 PRE-CHECK: 실제 energy grid 확인 -----------------------------------
from astropy.io import fits
from pathlib import Path

EXPCUBE_FILE = Path("./GC_analysis_sanghwan/GC_expcube_center_12yr_front_clean.fits")
with fits.open(EXPCUBE_FILE) as hdul:
    hdul.info()
    names = [h.name for h in hdul]
    print(f"\nExtensions: {names}")

    if 'EBOUNDS' in names:
        eb = hdul['EBOUNDS'].data
        print(f"\n[EBOUNDS] {len(eb)} bins (units = keV per Fermi standard)")
        for i, row in enumerate(eb):
            emin_mev = row['E_MIN'] / 1e3
            emax_mev = row['E_MAX'] / 1e3
            ectr_gev = (emin_mev * emax_mev)**0.5 / 1e3
            print(f"  {i:2d}  {emin_mev:10.3f} - {emax_mev:10.3f} MeV  "
                  f"(geomean = {ectr_gev:.4f} GeV)")
    elif 'ENERGIES' in names:
        ec = hdul['ENERGIES'].data
        col = ec.columns.names[0]
        print(f"\n[ENERGIES] {len(ec)} centers, col = {col}")
        for i, e in enumerate(ec[col]):
            print(f"  {i:2d}  {e/1e3:.4f} GeV")
    else:
        # Primary HDU의 NAXIS3 + WCS로부터 추정
        h = hdul[0].header
        print(f"\nPrimary header (no EBOUNDS/ENERGIES):")
        for k in ('NAXIS3', 'CRVAL3', 'CDELT3', 'CRPIX3', 'CTYPE3'):
            if k in h: print(f"  {k} = {h[k]}")

In [ ]:
# ============================================================================
# V27 — Stage A: Pre-fit Component SED Verification (Model I, 12yr)
# ----------------------------------------------------------------------------
# 목적: Cholis+2022 Fig 11의 solid line (pre-fit)을 우리 파이프라인이
#       재현하는지 확인. c_param = 1, 영역 40°×40° |b|>2°.
# 검증 항목:
#   - Pi0+Bremss, ICS : gtsrcmaps/gtmodel + exposure + unit handling
#   - Bubble, Isotropic: prior 템플릿이 의도된 절대 normalization으로 들어가는지
#   - GCE NFW²        : c=1 reference scale (정량 비교 X, 시각 참고용)
# Ref: arXiv:2112.09706 Fig 11 (page 16, 4FGL-DR2)
# ============================================================================

import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from pathlib import Path
from scipy.interpolate import interp1d

# ---- 0. Configuration ------------------------------------------------------
STAGE_A_MODEL     = "I"
SANGHWAN_DIR      = Path("./GC_analysis_sanghwan")
PRIOR_DIR         = SANGHWAN_DIR / "Model"
EXPCUBE_FILE      = SANGHWAN_DIR / "GC_expcube_center_12yr_front_clean.fits"
CHOLIS_FIG11_FILE = Path("./2112_09706_FIG11_digitized.txt")  # 옵션
OUTPUT_PNG = f"./V27_StageA_prefit_Model{STAGE_A_MODEL}_12yr.png"
OUTPUT_TXT = f"./V27_StageA_prefit_Model{STAGE_A_MODEL}_12yr.txt"

# 600×600 CCUBE 중 40°×40° 영역 = [100:500, 100:500]
SLICE_Y, SLICE_X = slice(100, 500), slice(100, 500)
NY = NX = 400
B_CENTER  = 200      # b=0 행 (slice 후 grid)
DISK_HALF = 20       # ±2° / 0.1°  → |b|<2° 마스킹

print("=" * 78)
print(f" V27 Stage A — Pre-fit Verification | Model {STAGE_A_MODEL} | 12yr | 4FGL-DR2")
print(f" Region: 40°×40° centered on GC, |b|<2° masked, c_param = 1")
print(f" Reference: Cholis+2022 Fig 11 (solid lines)")
print("=" * 78)

# ---- 1. 컴포넌트 맵 로드 (NON-CONVOL) -------------------------------------
def load_compmap(comp_name, model=None):
    """Sanghwan port 컨벤션의 non-convol 맵 → 40×40 영역 slice."""
    if comp_name in ("pion", "bremss", "ics"):
        path = SANGHWAN_DIR / f"GC_{comp_name}_model{model}_12yr_front_clean_no_convol.fits"
    else:  # GCE, fermi_bubble, isotropic (모델 무관)
        path = SANGHWAN_DIR / f"GC_{comp_name}_model_12yr_front_clean_no_convol.fits"
    if not path.exists():
        raise FileNotFoundError(f"Missing component map: {path}")
    data = fits.getdata(path)
    if data.ndim == 3:
        return data[:, SLICE_Y, SLICE_X]
    return data[SLICE_Y, SLICE_X]

pion_nc   = load_compmap("pion",   STAGE_A_MODEL)
bremss_nc = load_compmap("bremss", STAGE_A_MODEL)
ics_nc    = load_compmap("ics",    STAGE_A_MODEL)
gce_nc    = load_compmap("GCE")
bub_nc    = load_compmap("fermi_bubble")
iso_nc    = load_compmap("isotropic")

n_ebin = pion_nc.shape[0]
print(f"\n[Loaded] Component maps: shape per bin = {pion_nc.shape[1:]}, n_ebin = {n_ebin}")

# ---- 2. Exposure cube × steradian-per-pixel -------------------------------
exp_raw = fits.getdata(EXPCUBE_FILE)[:, SLICE_Y, SLICE_X]     # cm²·s
steradian_per_pixel = (np.pi/180.0 * 0.1) ** 2                # 0.1°×0.1°, b≈0 근사
exp_cube = exp_raw * steradian_per_pixel                       # cm²·s·sr / pix
print(f"[Loaded] Exposure cube: shape = {exp_cube.shape}, sr/pix = {steradian_per_pixel:.4e}")
# 주의: |b|<20° 영역에서 cos(b) 변화는 b=20°에서 약 0.94 (~6%). 후속 정밀화 항목.

# ---- 3. Disk mask (|b|<2°) ------------------------------------------------
disk_mask = np.ones((NY, NX), dtype=bool)
disk_mask[B_CENTER - DISK_HALF : B_CENTER + DISK_HALF, :] = False
print(f"[Mask] |b|>2° retains {disk_mask.sum()} / {NY*NX} pixels "
      f"({100*disk_mask.sum()/(NY*NX):.1f}%)")

# ---- 4. Energy grid (Cholis Table III + 17-bin DM 확장, exact boundaries) ----
# 12yr/17yr 표준 17-bin grid:
#   bins 0-13 ↔ Cholis+2022 Table III (= Fig 11 비교 범위)
#   bins 14-16 ↔ Cholis 38-bin native 그룹 27-29, 30-32, 33-35 (DM tail)
EBIN_BOUNDARIES_MEV = np.array([
    # Cholis Table III bins 0-13
    [   274.698,    357.014],
    [   357.014,    463.995],
    [   463.995,    603.034],
    [   603.034,    783.737],
    [   783.737,   1018.59 ],
    [  1018.59,    1323.82 ],
    [  1323.82,    1720.51 ],
    [  1720.51,    2236.07 ],
    [  2236.07,    2906.12 ],
    [  2906.12,    3776.96 ],
    [  3776.96,    4908.75 ],
    [  4908.75,   10776.0  ],
    [ 10776.0,    23656.1  ],
    [ 23656.1,    51931.2  ],
    # 17-bin 확장 (DM tail, Fig 11 범위 밖)
    [ 51931.2,   114002.0  ],
    [114002.0,   250265.0  ],
    [250265.0,   549396.0  ],
])

emin_MeV = EBIN_BOUNDARIES_MEV[:, 0]
emax_MeV = EBIN_BOUNDARIES_MEV[:, 1]
E       = np.sqrt(emin_MeV * emax_MeV) / 1000.0
delta_E = (emax_MeV - emin_MeV)         / 1000.0
n_ebin_grid = len(E)

# FITS 헤더의 centers와 일관성 검증 (regression guard)
def verify_centers(fits_path, E_expected_GeV, tol=0.01):
    with fits.open(fits_path) as hdul:
        names = [h.name for h in hdul]
        if 'ENERGIES' in names:
            col = hdul['ENERGIES'].columns.names[0]
            actual_MeV = np.asarray(hdul['ENERGIES'].data[col], dtype=float)
        elif 'EBOUNDS' in names:
            eb = hdul['EBOUNDS'].data
            actual_MeV = np.sqrt(eb['E_MIN'] * eb['E_MAX']) / 1e3  # keV → MeV
        else:
            raise RuntimeError("No EBOUNDS/ENERGIES extension")
    actual_GeV = actual_MeV / 1000.0
    rel_err = np.abs(actual_GeV - E_expected_GeV) / E_expected_GeV
    if (rel_err > tol).any():
        bad = np.where(rel_err > tol)[0]
        raise ValueError(
            f"FITS energy grid mismatch in {len(bad)} bin(s) "
            f"(indices {bad.tolist()}), max rel_err = {rel_err.max():.4f}"
        )
    return rel_err.max()

max_err = verify_centers(EXPCUBE_FILE, E, tol=0.02)  # 17-bin DM 확장 bin 0.5-1% 누적 허용
assert n_ebin_grid == n_ebin, f"Grid ({n_ebin_grid}) ≠ map n_ebin ({n_ebin})"

print(f"\n[Energy grid] 17 bins from Cholis Table III + DM extension")
print(f"  Max center rel_err vs FITS: {max_err:.4e}  (tol = 0.01)")
print(f"  Fig 11 비교 범위: bins 0–13 (0.31 → 35.2 GeV)")
print(f"  DM 확장 (참고용):  bins 14–16 (77.5 → 375 GeV)\n")
print(f"  {'idx':>3} {'Emin[GeV]':>10} {'Emax[GeV]':>10} {'E_ctr[GeV]':>11} {'ΔE[GeV]':>10}")
for i in range(n_ebin_grid):
    flag = "" if i <= 13 else "  (DM ext)"
    print(f"  {i:3d} {emin_MeV[i]/1000:10.3f} {emax_MeV[i]/1000:10.3f} "
          f"{E[i]:11.4f} {delta_E[i]:10.4f}{flag}")
    
with fits.open(EXPCUBE_FILE) as hdul:
    fits_centers_GeV = np.asarray(hdul['ENERGIES'].data['Energy'], dtype=float) / 1000.0
E = fits_centers_GeV
print(f"  (E ← FITS centers, ΔE ← Cholis boundaries; SED는 두 grid가 모두 0.5% 이내 일치)")

# ---- 5. Pre-fit SED 계산 ---------------------------------------------------
def compute_sed_prefit(comp_3d):
    """E²·dΦ/dE [GeV cm⁻² s⁻¹ sr⁻¹] per bin, spatial mean over |b|>2° mask."""
    sed = np.zeros(n_ebin)
    for i in range(n_ebin):
        with np.errstate(divide='ignore', invalid='ignore'):
            ratio = np.where(exp_cube[i] > 0, comp_3d[i] / exp_cube[i], 0.0)
        sed[i] = (ratio * disk_mask).sum() / disk_mask.sum()
    return sed * E**2 / delta_E

sed_gas = compute_sed_prefit(pion_nc + bremss_nc)
sed_ics = compute_sed_prefit(ics_nc)
sed_bub = compute_sed_prefit(bub_nc)
sed_iso = compute_sed_prefit(iso_nc)
sed_gce = compute_sed_prefit(gce_nc)

# ---- 6. Prior 참조 (Bubble + Iso) -----------------------------------------
def load_prior(path, kind='quadratic'):
    arr = np.loadtxt(path)
    Ep, Fp = arr[:, 0], arr[:, 1]
    return interp1d(Ep, Fp, kind=kind, fill_value='extrapolate')(E)

try:
    bub_prior = load_prior(PRIOR_DIR / "bubble_constraints.txt")
    iso_prior = load_prior(PRIOR_DIR / "iso_constraints_full_err.txt")
    iso_prior_E2 = E**2 * iso_prior  # dN/dE → E²·dN/dE
    print(f"[Loaded] Bubble + Iso prior (interpolated → 14 bins)")
except FileNotFoundError as e:
    print(f"[WARN] Prior file missing: {e}")
    bub_prior = iso_prior = np.full(n_ebin, np.nan)

# ---- 7. Cholis Fig 11 digitized (옵션) ------------------------------------
cholis = None
if CHOLIS_FIG11_FILE.exists():
    cholis = np.loadtxt(CHOLIS_FIG11_FILE)
    print(f"[Loaded] Cholis Fig 11 digitized: shape = {cholis.shape}")
else:
    print(f"[Info] Cholis Fig 11 digitized 미생성: {CHOLIS_FIG11_FILE}")
    print( "       → WebPlotDigitizer로 추출 후 같은 경로에 저장")
    print( "       format (8 cols): E_GeV  Pi0Brem  ICS  Bub  Iso  GCE")

# ---- 8. 수치 표 출력 -------------------------------------------------------
print("\n" + "─" * 78)
print(f"  Pre-fit E²·dΦ/dE [GeV/cm²/s/sr] — Model {STAGE_A_MODEL}, 40°×40° |b|>2°")
print("─" * 78)
print(f"  {'E[GeV]':>8} {'Pi0+Brem':>11} {'ICS':>11} {'Bubble':>11} {'Iso':>11} {'GCE_c=1':>11}")
print("─" * 78)
for i in range(n_ebin):
    print(f"  {E[i]:8.3f} {sed_gas[i]:11.3e} {sed_ics[i]:11.3e} "
          f"{sed_bub[i]:11.3e} {sed_iso[i]:11.3e} {sed_gce[i]:11.3e}")

# Bubble / Iso pre-fit vs prior 진단
print("\n  [Diagnostic] Bubble & Iso pre-fit vs prior data:")
print(f"  {'E[GeV]':>8} {'Bub_pri':>11} {'Bub/pri':>9} {'Iso_pri':>11} {'Iso/pri':>9}")
for i in range(n_ebin):
    rb = sed_bub[i] / bub_prior[i] if np.isfinite(bub_prior[i]) and bub_prior[i] > 0 else np.nan
    ri = sed_iso[i] / iso_prior_E2[i] if np.isfinite(iso_prior_E2[i]) and iso_prior_E2[i] > 0 else np.nan
    print(f"  {E[i]:8.3f} {bub_prior[i]:11.3e} {rb:9.3f} {iso_prior_E2[i]:11.3e} {ri:9.3f}")
print("  * Iso/pri ≈ 1.0 ± 0.05 이 정상 (균일 템플릿).")
print("  * Bub/pri 는 에너지 무관한 상수여야 함 (= bubble filling fraction in ROI).")

# ASCII 저장
out_arr = np.column_stack([E, sed_gas, sed_ics, sed_bub, sed_iso, sed_gce,
                           bub_prior, iso_prior])
np.savetxt(OUTPUT_TXT, out_arr, fmt='%12.5e',
           header='E[GeV]  Pi0+Brem  ICS  Bubble  Iso  GCE_c=1  Bub_prior  Iso_prior')
print(f"\n[Saved] {OUTPUT_TXT}")

# ---- 9. Plot ---------------------------------------------------------------
C = {'gold': '#E09F3E', 'teal': '#1C7293', 'green': '#2D6A4F',
     'muted': '#6B7A88', 'red': '#9E2A2B', 'ink': '#1E2A38'}

fig, ax = plt.subplots(figsize=(8.5, 6.5), dpi=110)

# 우리 pre-fit
ax.loglog(E, sed_gas, 'o-', color=C['gold'],  lw=2.0, ms=6, label='Pi0+Bremss (ours)')
ax.loglog(E, sed_ics, 's-', color=C['teal'],  lw=2.0, ms=6, label='ICS (ours)')
ax.loglog(E, sed_bub, '^-', color=C['green'], lw=2.0, ms=6, label='Bubble (ours, c=1)')
ax.loglog(E, sed_iso, 'v-', color=C['muted'], lw=2.0, ms=6, label='Isotropic (ours, c=1)')
ax.loglog(E, sed_gce, 'D-', color=C['red'],   lw=2.0, ms=6, label='GCE NFW² (c=1, ref)')

# Prior reference
if np.isfinite(bub_prior).all():
    ax.loglog(E, bub_prior, '--', color=C['green'], lw=1.2, alpha=0.7,
              label='Bubble prior (1407.7905 T2)')
if np.isfinite(iso_prior_E2).all():
    ax.loglog(E, iso_prior_E2, '--', color=C['muted'], lw=1.2, alpha=0.7,
              label='Iso prior (Ackermann+2015 T3)')

# Cholis Fig 11 digitized overlay
if cholis is not None:
    Ec = cholis[:, 0]
    pairs = [('Pi0+Brem', C['gold']), ('ICS', C['teal']),
             ('Bubble',   C['green']),('Iso', C['muted']),
             ('GCE',      C['red'])]
    for j, (name, col) in enumerate(pairs, start=1):
        if cholis.shape[1] > j:
            ax.loglog(Ec, cholis[:, j], ':', color=col, lw=2.0, alpha=0.95,
                      label=f'Cholis Fig 11 {name}')

ax.set_xlabel('E [GeV]', fontsize=12)
ax.set_ylabel(r'$E^2\, d\Phi/dE\ \ [\mathrm{GeV\, cm^{-2}\, s^{-1}\, sr^{-1}}]$', fontsize=12)
ax.set_title(f'V27 Stage A — Pre-fit Components (Model {STAGE_A_MODEL}, 12yr)\n'
             '40°×40° |b|>2° integration', fontsize=12, color=C['ink'])
ax.axvspan(50, 500, color='lightgray', alpha=0.25, zorder=0)
ax.text(0.97, 0.97, 'DM tail\n(Fig 11 범위 밖)', transform=ax.transAxes,
        ha='right', va='top', fontsize=8.5, color=C['muted'])
ax.set_xlim(0.2, 500)
ax.set_ylim(1e-8, 1e-4)
ax.legend(loc='lower left', fontsize=8.5, ncol=2, framealpha=0.95)
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_PNG, dpi=130, bbox_inches='tight')
plt.show()
print(f"[Saved] {OUTPUT_PNG}")

# ---- 10. 해석 가이드 -------------------------------------------------------
print("\n" + "=" * 78)
print(" 해석 가이드:")
print(" 1) Iso/pri ≈ 1.000 ± 0.05  → isotropic prior 적용 정상")
print(" 2) Bub/pri = 에너지 무관 상수 → bubble spatial normalization 정상")
print("    (변동 크면 prior 파일 또는 bubble 템플릿 normalization 점검)")
print(" 3) Pi0+Bremss, ICS는 Cholis Fig 11 solid line의 ~5% 이내 권장 (Model I)")
print(" 4) GCE c=1 은 reference scale (정량 의미 X)")
print("=" * 78)

In [ ]:
# ============================================================================
# V28 — Stage B: Post-fit Component SED + Cholis Fig 11 overlay (Model I, 12yr)
# ----------------------------------------------------------------------------
# 목적: V27 pre-fit + MCMC 결과로 post-fit SED 5종 (Pi0+Bremss, ICS, Bub, Iso, GCE)
#       + 2σ band 계산 → 디지타이즈한 Cholis Fig 11과 직접 overlay.
# 의존: V27 변수 (E, delta_E, exp_cube, disk_mask, 6개 component map,
#               sed_gas/sed_ics/sed_bub/sed_iso/sed_gce, n_ebin=17)
#       Cholis Fig 11 digitized: 2112_09706_Fig11_ModelI_ALL.txt
#       MCMC .npz       : GCE_model_I_12yr_cholis_fit.npz
# 비교 범위: bins 0–13 (= Cholis Fig 11 에너지 범위, 0.31–35 GeV)
# ============================================================================

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ---- 0. Configuration ------------------------------------------------------
NPZ_PATH        = Path(f"./GCE_model_{STAGE_A_MODEL}_12yr_cholis_fit.npz")
CHOLIS_ALL_FILE = Path("./2112_09706_Fig11_ModelI_ALL.txt")
OUTPUT_PNG      = f"./V28_StageB_postfit_Model{STAGE_A_MODEL}_12yr.png"
OUTPUT_TXT      = f"./V28_StageB_postfit_Model{STAGE_A_MODEL}_12yr.txt"
FIG11_BIN_MAX   = 13   # bins 0..13 = Cholis Fig 11 범위 (14..16은 DM tail)

print("=" * 78)
print(f" V28 Stage B — Post-fit Overlay | Model {STAGE_A_MODEL} | 12yr | 4FGL-DR2")
print(f" Comparison vs Cholis+2022 Fig 11 (digitized)")
print(f" Bins 0–{FIG11_BIN_MAX} (0.31–35.2 GeV) for quantitative comparison")
print("=" * 78)

# ---- 1. MCMC 결과 로드 ----------------------------------------------------
if not NPZ_PATH.exists():
    raise FileNotFoundError(
        f"Missing MCMC .npz: {NPZ_PATH}\n"
        f"  → runner ≥ v3.10 (with .npz save block) 로 Model {STAGE_A_MODEL} 재실행 필요"
    )

npz = np.load(NPZ_PATH)
print(f"\n[NPZ] keys = {list(npz.keys())}")

# Parse fitted_params layout: flat (5*n,) 또는 (5, n)
def _reshape(arr, n):
    return arr.reshape(5, n) if arr.ndim == 1 else arr

fp     = _reshape(npz['fitted_params'],       n_ebin)   # best fit (median)
fp_lo  = _reshape(npz['fitted_params_lower'], n_ebin)   # 16th percentile (1σ-)
fp_hi  = _reshape(npz['fitted_params_upper'], n_ebin)   # 84th percentile (1σ+)

# Order: [c_gas, c_ics, c_gce, c_bub, c_iso] (Sanghwan/Cholis 컨벤션)
c_gas, c_ics, c_gce, c_bub, c_iso             = fp
c_gas_lo, c_ics_lo, c_gce_lo, c_bub_lo, c_iso_lo = fp_lo
c_gas_hi, c_ics_hi, c_gce_hi, c_bub_hi, c_iso_hi = fp_hi

print(f"\n[MCMC] c_param best-fit mean over bins 0–{FIG11_BIN_MAX}:")
print(f"  c_gas = {c_gas[:FIG11_BIN_MAX+1].mean():.3f}  "
      f"c_ics = {c_ics[:FIG11_BIN_MAX+1].mean():.3f}  "
      f"c_gce = {c_gce[:FIG11_BIN_MAX+1].mean():.3f}")
print(f"  c_bub = {c_bub[:FIG11_BIN_MAX+1].mean():.1f}    "
      f"c_iso = {c_iso[:FIG11_BIN_MAX+1].mean():.3f}")

# ---- 2. Post-fit SED + 2σ band -------------------------------------------
# pre-fit SED는 V27에서 이미 c=1 기준으로 계산됨. post-fit = c × pre-fit.
# 2σ ≈ 2 × (1σ deviation from best). asymmetric posterior 대응:
#   upper_2σ = best + 2 × (hi_1σ - best)
#   lower_2σ = best - 2 × (best - lo_1σ)
def post_fit_2sigma(sed_prefit, c, c_lo, c_hi):
    cen = sed_prefit * c
    up  = sed_prefit * (c + 2*(c_hi - c))
    lo  = sed_prefit * (c - 2*(c - c_lo))
    return cen, np.clip(lo, 0, None), up  # flux ≥ 0

gas_cen, gas_lo, gas_hi = post_fit_2sigma(sed_gas, c_gas, c_gas_lo, c_gas_hi)
ics_cen, ics_lo, ics_hi = post_fit_2sigma(sed_ics, c_ics, c_ics_lo, c_ics_hi)
bub_cen, bub_lo, bub_hi = post_fit_2sigma(sed_bub, c_bub, c_bub_lo, c_bub_hi)
iso_cen, iso_lo, iso_hi = post_fit_2sigma(sed_iso, c_iso, c_iso_lo, c_iso_hi)
gce_cen, gce_lo, gce_hi = post_fit_2sigma(sed_gce, c_gce, c_gce_lo, c_gce_hi)

# ---- 3. Cholis Fig 11 digitized 로드 -------------------------------------
if not CHOLIS_ALL_FILE.exists():
    raise FileNotFoundError(
        f"Missing Cholis Fig 11 digitized: {CHOLIS_ALL_FILE}\n"
        f"  → V28 helper output 파일을 작업 디렉토리에 복사 필요"
    )

cd = np.loadtxt(CHOLIS_ALL_FILE)
E_chol = cd[:, 0]
# 컬럼 순서: E, [cen,lo,hi]×5 in order Pi0+Bremss, ICS, Bub, Iso, GCE
chol = {}
for k, name in enumerate(["Pi0p", "ICS", "Bub", "Iso", "GCE"]):
    chol[name] = dict(cen=cd[:, 1+3*k], lo=cd[:, 2+3*k], hi=cd[:, 3+3*k])
print(f"[Cholis] Loaded digitized Fig 11: {len(E_chol)} sample points, "
      f"E [{E_chol.min():.2f}, {E_chol.max():.2f}] GeV")

# Interp Cholis to our 17-bin E grid (log-linear, for ratio comparison)
def chol_interp(name, key, target_E):
    e_src, f_src = E_chol, chol[name][key]
    return 10**np.interp(np.log10(target_E),
                         np.log10(e_src), np.log10(np.clip(f_src, 1e-15, None)))

chol_on_E = {}
for short, our_cen in [("Pi0p", gas_cen), ("ICS", ics_cen), ("Bub", bub_cen),
                       ("Iso", iso_cen), ("GCE", gce_cen)]:
    chol_on_E[short] = dict(
        cen=chol_interp(short, 'cen', E),
        lo =chol_interp(short, 'lo',  E),
        hi =chol_interp(short, 'hi',  E),
    )

# ---- 4. 정량 비교 표: our_post / Cholis_cen ----------------------------
print("\n" + "─" * 90)
print(f"  Post-fit ratio: ours / Cholis (Model {STAGE_A_MODEL}, bins 0–{FIG11_BIN_MAX})")
print("─" * 90)
print(f"  {'idx':>3} {'E[GeV]':>8} {'Pi0+Brem':>10} {'ICS':>10} {'Bubble':>10} "
      f"{'Iso':>10} {'GCE':>10}")
print("─" * 90)
for i in range(FIG11_BIN_MAX + 1):
    r_gas = gas_cen[i] / chol_on_E['Pi0'][i] if False else gas_cen[i] / chol_on_E['Pi0p']['cen'][i]
    r_ics = ics_cen[i] / chol_on_E['ICS']['cen'][i]
    r_bub = bub_cen[i] / chol_on_E['Bub']['cen'][i]
    r_iso = iso_cen[i] / chol_on_E['Iso']['cen'][i]
    r_gce = gce_cen[i] / chol_on_E['GCE']['cen'][i]
    print(f"  {i:3d} {E[i]:8.3f} {r_gas:10.3f} {r_ics:10.3f} {r_bub:10.3f} "
          f"{r_iso:10.3f} {r_gce:10.3f}")

# Mean / max deviation
for short, our_cen in [("Pi0p+Brem", gas_cen), ("ICS", ics_cen), ("Bub", bub_cen),
                       ("Iso", iso_cen), ("GCE", gce_cen)]:
    short_key = short.replace('+Brem', '').replace('Pi0p', 'Pi0p')
    ratios = our_cen[:FIG11_BIN_MAX+1] / chol_on_E[short_key]['cen'][:FIG11_BIN_MAX+1]
    print(f"  {short:12s}: ratio_mean = {ratios.mean():.3f},  "
          f"max|dev| = {np.max(np.abs(ratios-1)):.3f},  "
          f"std = {ratios.std():.3f}")

# ASCII 저장
out_arr = np.column_stack([
    E,
    gas_cen, gas_lo, gas_hi,
    ics_cen, ics_lo, ics_hi,
    bub_cen, bub_lo, bub_hi,
    iso_cen, iso_lo, iso_hi,
    gce_cen, gce_lo, gce_hi,
])
np.savetxt(OUTPUT_TXT, out_arr, fmt='%12.5e',
           header=('Post-fit SED + 2σ band, Model I, 12yr, 40°×40° |b|>2°\n'
                   'cols: E[GeV]  ['
                   'Pi0+Brem  ICS  Bub  Iso  GCE] × [cen, 2σ_lo, 2σ_hi]'))
print(f"\n[Saved] {OUTPUT_TXT}")

# ---- 5. Plot ---------------------------------------------------------------
C = {'gold': '#E09F3E', 'teal': '#1C7293', 'green': '#2D6A4F',
     'muted': '#6B7A88', 'red': '#9E2A2B', 'ink': '#1E2A38'}

fig, ax = plt.subplots(figsize=(10.5, 7.5), dpi=110)

# DM tail shading
ax.axvspan(50, 500, color='lightgray', alpha=0.2, zorder=0)

# Our post-fit: bands + best-fit lines
specs = [
    ('Pi0+Bremss', gas_cen, gas_lo, gas_hi, 'Pi0p', C['gold']),
    ('ICS',        ics_cen, ics_lo, ics_hi, 'ICS',  C['teal']),
    ('Bubbles',    bub_cen, bub_lo, bub_hi, 'Bub',  C['green']),
    ('Isotropic',  iso_cen, iso_lo, iso_hi, 'Iso',  C['muted']),
    ('GCE',        gce_cen, gce_lo, gce_hi, 'GCE',  C['red']),
]
for label, cen, lo, hi, short_key, col in specs:
    ax.fill_between(E, lo, hi, color=col, alpha=0.22, zorder=2)
    ax.plot(E, cen, '-', color=col, lw=2.0, marker='o', ms=5, zorder=3,
            label=f'{label} (ours post-fit)')

# Cholis Fig 11 digitized: dashed best-fit + lighter band
for label, _, _, _, short_key, col in specs:
    c = chol[short_key]
    # 디지타이즈된 sample은 dense하므로 fill로 충분
    ax.plot(E_chol, c['cen'], '--', color=col, lw=1.4, alpha=0.75, zorder=4,
            label=f'{label} (Cholis Fig 11)')
    # Cholis 2σ envelope (얇게)
    ax.fill_between(E_chol, c['lo'], c['hi'], color=col, alpha=0.10,
                    edgecolor=col, lw=0.5, zorder=1)

ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlim(0.25, 50)
ax.set_ylim(1e-8, 3e-5)
ax.set_xlabel('E [GeV]', fontsize=12)
ax.set_ylabel(r'$E^2\, d\Phi/dE\ \ [\mathrm{GeV\, cm^{-2}\, s^{-1}\, sr^{-1}}]$', fontsize=12)
ax.set_title(f'V28 Stage B — Post-fit vs Cholis Fig 11 (Model {STAGE_A_MODEL}, 12yr)\n'
             '40°×40° |b|>2° integration, solid=ours, dashed=Cholis',
             fontsize=11, color=C['ink'])
ax.legend(loc='lower left', fontsize=7.5, ncol=2, framealpha=0.95)
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_PNG, dpi=130, bbox_inches='tight')
plt.show()
print(f"[Saved] {OUTPUT_PNG}")

# ---- 6. 해석 가이드 -------------------------------------------------------
print("\n" + "=" * 78)
print(" 해석 가이드:")
print(" - Pi0+Bremss·ICS: post-fit ratio ≈ 0.9–1.1 이 정상 (Cholis 본문 '~10% within pre-fit')")
print(" - Bubbles·Iso  : prior에 강하게 묶여 있으므로 ratio가 ~1.0 ± 0.3 권장")
print(" - GCE          : 차이가 크면 fit/likelihood 또는 PSC mask 차이 시사")
print(" - 모든 성분에서 ≤20% 일치 → '12yr methodology validated'로 thesis에 기재 가능")
print("=" * 78)

In [ ]:
# ============================================================================
# V28c — Post-fit (new strict-mask) vs Cholis Fig 11 (Model I, 12yr)
# ----------------------------------------------------------------------------
# V27 변수 (E, n_ebin, sed_gas/ics/bub/iso/gce, STAGE_A_MODEL) 재사용
# ============================================================================
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

NPZ_PATH        = Path(f"./GCE_model_{STAGE_A_MODEL}_12yr_cholis_fit.npz")
CHOLIS_ALL_FILE = Path("./2112_09706_Fig11_ModelI_ALL.txt")
OUTPUT_PNG      = f"./V28c_Fig11_overlay_Model{STAGE_A_MODEL}_12yr.png"
OUTPUT_TXT      = f"./V28c_postfit_Model{STAGE_A_MODEL}_12yr.txt"
FIG11_BIN_MAX   = 13

print("=" * 78)
print(f" V28c — Cholis Fig 11 reproduction | Model {STAGE_A_MODEL} | 12yr | strict mask")
print("=" * 78)

# ---- Load MCMC -------------------------------------------------------------
npz = np.load(NPZ_PATH)
def rs(a): return a.reshape(5, n_ebin) if a.ndim == 1 else a
fp     = rs(npz['fitted_params'])
fp_lo  = rs(npz['fitted_params_lower'])
fp_hi  = rs(npz['fitted_params_upper'])
c_gas, c_ics, c_gce, c_bub, c_iso = fp

print(f"\n[c_param mean over Fig 11 range]:")
print(f"  c_gas = {c_gas[:14].mean():.3f}   c_ics = {c_ics[:14].mean():.3f}   "
      f"c_gce = {c_gce[:14].mean():.3f}")
print(f"  c_bub = {c_bub[:14].mean():.1f}     c_iso = {c_iso[:14].mean():.3f}")

# ---- Post-fit SEDs + 2σ bands ---------------------------------------------
def post_fit_2sigma(sed_prefit, c, c_lo, c_hi):
    cen = sed_prefit * c
    up  = sed_prefit * (c + 2*(c_hi - c))
    lo  = sed_prefit * (c - 2*(c - c_lo))
    return cen, np.clip(lo, 0, None), up

gas_cen, gas_lo, gas_hi = post_fit_2sigma(sed_gas, c_gas, fp_lo[0], fp_hi[0])
ics_cen, ics_lo, ics_hi = post_fit_2sigma(sed_ics, c_ics, fp_lo[1], fp_hi[1])
gce_cen, gce_lo, gce_hi = post_fit_2sigma(sed_gce, c_gce, fp_lo[2], fp_hi[2])
bub_cen, bub_lo, bub_hi = post_fit_2sigma(sed_bub, c_bub, fp_lo[3], fp_hi[3])
iso_cen, iso_lo, iso_hi = post_fit_2sigma(sed_iso, c_iso, fp_lo[4], fp_hi[4])

# ---- Cholis Fig 11 digitized ----------------------------------------------
cd = np.loadtxt(CHOLIS_ALL_FILE)
E_chol = cd[:, 0]
chol = {name: dict(cen=cd[:, 1+3*k], lo=cd[:, 2+3*k], hi=cd[:, 3+3*k])
        for k, name in enumerate(["Pi0p", "ICS", "Bub", "Iso", "GCE"])}

def chol_at(name, key, E_target):
    return 10**np.interp(np.log10(E_target), np.log10(E_chol),
                          np.log10(np.clip(chol[name][key], 1e-15, None)))

# ---- Ratio table ----------------------------------------------------------
print(f"\n  Post-fit ratio (ours / Cholis), bins 0–{FIG11_BIN_MAX}:")
print(f"  {'idx':>3} {'E[GeV]':>8} {'Pi0+Brem':>10} {'ICS':>10} {'Bub':>10} "
      f"{'Iso':>10} {'GCE':>10}")
print("─" * 72)
for i in range(FIG11_BIN_MAX+1):
    rgas = gas_cen[i] / chol_at('Pi0p','cen',E[i])
    rics = ics_cen[i] / chol_at('ICS','cen',E[i])
    rbub = bub_cen[i] / chol_at('Bub','cen',E[i])
    riso = iso_cen[i] / chol_at('Iso','cen',E[i])
    rgce = gce_cen[i] / chol_at('GCE','cen',E[i])
    print(f"  {i:3d} {E[i]:8.3f} {rgas:10.3f} {rics:10.3f} {rbub:10.3f} "
          f"{riso:10.3f} {rgce:10.3f}")

# Summary statistics
print(f"\n  Mean ratio + max deviation over bins 0–{FIG11_BIN_MAX}:")
for name, our_cen, key in [("Pi0+Brem", gas_cen, "Pi0p"), ("ICS", ics_cen, "ICS"),
                            ("Bub", bub_cen, "Bub"), ("Iso", iso_cen, "Iso"),
                            ("GCE", gce_cen, "GCE")]:
    chol_cen = chol_at(key, 'cen', E[:14])
    ratios = our_cen[:14] / chol_cen
    print(f"    {name:10s}: mean = {ratios.mean():.3f}, "
          f"max|dev| = {np.max(np.abs(ratios-1)):.3f}, std = {ratios.std():.3f}")

# ASCII 저장
np.savetxt(OUTPUT_TXT,
           np.column_stack([E, gas_cen, gas_lo, gas_hi,
                            ics_cen, ics_lo, ics_hi,
                            bub_cen, bub_lo, bub_hi,
                            iso_cen, iso_lo, iso_hi,
                            gce_cen, gce_lo, gce_hi]),
           fmt='%12.5e',
           header='V28c Model I post-fit (strict Cholis mask), 40°×40° |b|>2°\n'
                  'cols: E[GeV] [Pi0+Brem ICS Bub Iso GCE] × [cen, 2σ_lo, 2σ_hi]')
print(f"\n[Saved] {OUTPUT_TXT}")

# ---- Plot -----------------------------------------------------------------
C = {'gold':'#E09F3E','teal':'#1C7293','green':'#2D6A4F',
     'muted':'#6B7A88','red':'#9E2A2B','ink':'#1E2A38'}
fig, ax = plt.subplots(figsize=(10.5, 7.5), dpi=110)
specs = [
    ('Pi0+Bremss', gas_cen, gas_lo, gas_hi, 'Pi0p', C['gold']),
    ('ICS',        ics_cen, ics_lo, ics_hi, 'ICS',  C['teal']),
    ('Bubbles',    bub_cen, bub_lo, bub_hi, 'Bub',  C['green']),
    ('Isotropic',  iso_cen, iso_lo, iso_hi, 'Iso',  C['muted']),
    ('GCE',        gce_cen, gce_lo, gce_hi, 'GCE',  C['red']),
]
for label, cen, lo, hi, key, col in specs:
    ax.fill_between(E, lo, hi, color=col, alpha=0.22, zorder=2)
    ax.plot(E, cen, '-o', color=col, lw=2.0, ms=5, zorder=3,
            label=f'{label} (ours)')
for label, _, _, _, key, col in specs:
    ax.plot(E_chol, chol[key]['cen'], '--', color=col, lw=1.4, alpha=0.80, zorder=4,
            label=f'{label} (Cholis Fig 11)')
    ax.fill_between(E_chol, chol[key]['lo'], chol[key]['hi'],
                    color=col, alpha=0.10, zorder=1)

ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlim(0.25, 50); ax.set_ylim(1e-8, 3e-5)
ax.set_xlabel('E [GeV]', fontsize=12)
ax.set_ylabel(r'$E^2\, d\Phi/dE\ [\mathrm{GeV\, cm^{-2}\, s^{-1}\, sr^{-1}}]$', fontsize=12)
ax.set_title(f'Model {STAGE_A_MODEL}, 12yr — Reproduction of Cholis+2022 Fig 11\n'
             '40°×40° |b|>2°, solid = our post-fit, dashed = Cholis Fig 11',
             fontsize=11, color=C['ink'])
ax.legend(loc='lower left', fontsize=7.5, ncol=2, framealpha=0.95)
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_PNG, dpi=130, bbox_inches='tight')
plt.show()
print(f"[Saved] {OUTPUT_PNG}")

In [ ]:
# ============================================================================
# V28d — Bubble XML 패치 후 검증 (Model I, 12yr)
# ----------------------------------------------------------------------------
# V27, V28c와 동일 셀 logic. 차이: 새 NPZ 사용 + V28c 결과와 비교 표 추가.
# 의존: V27 변수 (E, n_ebin, sed_gas/ics/bub/iso/gce 6종), pre-existing V28c .txt
# ============================================================================
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

NPZ_NEW = Path(f"./GCE_model_{STAGE_A_MODEL}_12yr_cholis_fit.npz")  # bubble 패치 후
NPZ_OLD = Path(f"./GCE_model_{STAGE_A_MODEL}_12yr_cholis_fit.npz.bak_bubscale_buggy")
CHOLIS_ALL_FILE = Path("./2112_09706_Fig11_ModelI_ALL.txt")
OUTPUT_PNG = f"./V28d_Fig11_overlay_Model{STAGE_A_MODEL}_12yr.png"
OUTPUT_TXT = f"./V28d_postfit_Model{STAGE_A_MODEL}_12yr.txt"
FIG11_BIN_MAX = 13

print("=" * 78)
print(f" V28d — Bubble XML patched (scale=1 value=1) | Model {STAGE_A_MODEL}")
print("=" * 78)

# ---- 0. V27 pre-fit 재계산 필요 ---------------------------------------------
# Bubble component map이 바뀌었으므로 sed_bub만 재계산. 다른 5개는 V27 그대로.
from astropy.io import fits
print("\n[Recomputing sed_bub with patched Bubble component map]")
bub_nc_new = fits.getdata("./GC_analysis_sanghwan/GC_fermi_bubble_model_12yr_front_clean_no_convol.fits")[:, SLICE_Y, SLICE_X]
def compute_sed_prefit(comp_3d):
    sed = np.zeros(n_ebin)
    for i in range(n_ebin):
        with np.errstate(divide='ignore', invalid='ignore'):
            ratio = np.where(exp_cube[i] > 0, comp_3d[i] / exp_cube[i], 0.0)
        sed[i] = (ratio * disk_mask).sum() / disk_mask.sum()
    return sed * E**2 / delta_E
sed_bub_new = compute_sed_prefit(bub_nc_new)
print(f"  sed_bub @ 0.31 GeV: old = {sed_bub[0]:.3e}, new = {sed_bub_new[0]:.3e}, "
      f"ratio = {sed_bub_new[0]/sed_bub[0]:.1f}× (expected ~200×)")

# ---- 1. NPZ 로드 ------------------------------------------------------------
def load_params(path):
    npz = np.load(path)
    def rs(a): return a.reshape(5, n_ebin) if a.ndim == 1 else a
    return (rs(npz['fitted_params']),
            rs(npz['fitted_params_lower']),
            rs(npz['fitted_params_upper']),
            npz['max_likelihood'])
fp_new, fp_lo_new, fp_hi_new, mlhd_new = load_params(NPZ_NEW)
fp_old, _, _, mlhd_old = load_params(NPZ_OLD)

c_gas, c_ics, c_gce, c_bub, c_iso = fp_new

# ---- 2. c_param 비교: 옛 (bubble 버그) vs 새 (패치 후) -------------------
print(f"\n  c_param mean over Fig 11 range (bins 0–{FIG11_BIN_MAX}):")
print(f"  {'param':>7} {'OLD (bug)':>12} {'NEW (patched)':>14} {'change':>10}")
labels = ['c_gas', 'c_ics', 'c_gce', 'c_bub', 'c_iso']
for i, lab in enumerate(labels):
    old_m = fp_old[i, :FIG11_BIN_MAX+1].mean()
    new_m = fp_new[i, :FIG11_BIN_MAX+1].mean()
    fmt_old = f"{old_m:12.3f}" if i != 3 else f"{old_m:12.1f}"
    fmt_new = f"{new_m:14.3f}" if i != 3 else f"{new_m:14.1f}"
    delta = new_m - old_m
    fmt_d = f"{delta:+10.3f}" if i != 3 else f"{delta:+10.1f}"
    print(f"  {lab:>7} {fmt_old} {fmt_new} {fmt_d}")

# Swap 진폭 비교
swap_old = abs(1 - fp_old[0, :14].mean()) + abs(1 - fp_old[1, :14].mean())
swap_new = abs(1 - fp_new[0, :14].mean()) + abs(1 - fp_new[1, :14].mean())
print(f"\n  Swap amplitude |1-c_gas| + |1-c_ics|: OLD = {swap_old:.3f}, NEW = {swap_new:.3f}")
if swap_new < swap_old * 0.5:
    print(f"  ✓ Swap 크게 감소. Bubble normalization이 fit dynamic range를 망가뜨리던 원인.")
elif swap_new < swap_old * 0.85:
    print(f"  ◑ Swap 부분 감소. Bubble fix 효과 있지만 잔여 swap이 큼 (PSF/IRF 의심).")
else:
    print(f"  ○ Swap 거의 동일. 예상대로 — Bubble은 dynamic range 문제고 morphology와 무관.")

# ln L 비교
print(f"\n  ln L (bins 0–{FIG11_BIN_MAX}):")
print(f"    OLD (bug):  {mlhd_old[:14].sum():14.2f}")
print(f"    NEW:        {mlhd_new[:14].sum():14.2f}  Δ = {mlhd_new[:14].sum()-mlhd_old[:14].sum():+.1f}")
print(f"    Cholis ref: {-1875264.68:14.2f}")

# ---- 3. Post-fit SED + Cholis overlay (V28c와 동일) ----------------------
def post_fit_2sigma(sed_prefit, c, c_lo, c_hi):
    cen = sed_prefit * c
    up  = sed_prefit * (c + 2*(c_hi - c))
    lo  = sed_prefit * (c - 2*(c - c_lo))
    return cen, np.clip(lo, 0, None), up

gas_cen, gas_lo, gas_hi = post_fit_2sigma(sed_gas, c_gas, fp_lo_new[0], fp_hi_new[0])
ics_cen, ics_lo, ics_hi = post_fit_2sigma(sed_ics, c_ics, fp_lo_new[1], fp_hi_new[1])
gce_cen, gce_lo, gce_hi = post_fit_2sigma(sed_gce, c_gce, fp_lo_new[2], fp_hi_new[2])
bub_cen, bub_lo, bub_hi = post_fit_2sigma(sed_bub_new, c_bub, fp_lo_new[3], fp_hi_new[3])
iso_cen, iso_lo, iso_hi = post_fit_2sigma(sed_iso, c_iso, fp_lo_new[4], fp_hi_new[4])

cd = np.loadtxt(CHOLIS_ALL_FILE)
E_chol = cd[:, 0]
chol = {name: dict(cen=cd[:, 1+3*k], lo=cd[:, 2+3*k], hi=cd[:, 3+3*k])
        for k, name in enumerate(["Pi0p", "ICS", "Bub", "Iso", "GCE"])}
def chol_at(name, key, Et):
    return 10**np.interp(np.log10(Et), np.log10(E_chol),
                          np.log10(np.clip(chol[name][key], 1e-15, None)))

# Ratio table
print(f"\n  Post-fit ratio (ours / Cholis), bins 0–{FIG11_BIN_MAX}:")
print(f"  {'idx':>3} {'E[GeV]':>8} {'Pi0+Brem':>10} {'ICS':>10} {'Bub':>10} {'Iso':>10} {'GCE':>10}")
print("─" * 72)
for i in range(FIG11_BIN_MAX+1):
    print(f"  {i:3d} {E[i]:8.3f}  "
          f"{gas_cen[i]/chol_at('Pi0p','cen',E[i]):9.3f} "
          f"{ics_cen[i]/chol_at('ICS','cen',E[i]):9.3f} "
          f"{bub_cen[i]/chol_at('Bub','cen',E[i]):9.3f} "
          f"{iso_cen[i]/chol_at('Iso','cen',E[i]):9.3f} "
          f"{gce_cen[i]/chol_at('GCE','cen',E[i]):9.3f}")

print(f"\n  Mean ratio + max deviation, bins 0–{FIG11_BIN_MAX}:")
for name, our_cen, key in [("Pi0+Brem", gas_cen, "Pi0p"), ("ICS", ics_cen, "ICS"),
                            ("Bub", bub_cen, "Bub"), ("Iso", iso_cen, "Iso"),
                            ("GCE", gce_cen, "GCE")]:
    cc = chol_at(key, 'cen', E[:14])
    r = our_cen[:14] / cc
    print(f"    {name:10s}: mean = {r.mean():.3f}, max|dev| = {np.max(np.abs(r-1)):.3f}")

# Save
np.savetxt(OUTPUT_TXT,
           np.column_stack([E, gas_cen, gas_lo, gas_hi,
                            ics_cen, ics_lo, ics_hi,
                            bub_cen, bub_lo, bub_hi,
                            iso_cen, iso_lo, iso_hi,
                            gce_cen, gce_lo, gce_hi]),
           fmt='%12.5e',
           header='V28d Model I post-fit (bubble XML patched), 40°×40° |b|>2°')

# ---- 4. Plot --------------------------------------------------------------
C = {'gold':'#E09F3E','teal':'#1C7293','green':'#2D6A4F',
     'muted':'#6B7A88','red':'#9E2A2B'}
fig, ax = plt.subplots(figsize=(10.5, 7.5), dpi=110)
specs = [('Pi0+Bremss', gas_cen, gas_lo, gas_hi, 'Pi0p', C['gold']),
         ('ICS',        ics_cen, ics_lo, ics_hi, 'ICS',  C['teal']),
         ('Bubbles',    bub_cen, bub_lo, bub_hi, 'Bub',  C['green']),
         ('Isotropic',  iso_cen, iso_lo, iso_hi, 'Iso',  C['muted']),
         ('GCE',        gce_cen, gce_lo, gce_hi, 'GCE',  C['red'])]
for label, cen, lo, hi, key, col in specs:
    ax.fill_between(E, lo, hi, color=col, alpha=0.22, zorder=2)
    ax.plot(E, cen, '-o', color=col, lw=2.0, ms=5, zorder=3, label=f'{label} (ours)')
for label, _, _, _, key, col in specs:
    ax.plot(E_chol, chol[key]['cen'], '--', color=col, lw=1.4, alpha=0.80, zorder=4,
            label=f'{label} (Cholis)')
    #ax.fill_between(E_chol, chol[key]['lo'], chol[key]['hi'], color=col, alpha=0.10)

ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlim(0.25, 50); ax.set_ylim(1e-8, 3e-5)
ax.set_xlabel('E [GeV]', fontsize=12)
ax.set_ylabel(r'$E^2\, d\Phi/dE\ [\mathrm{GeV\, cm^{-2}\, s^{-1}\, sr^{-1}}]$', fontsize=12)
ax.set_title(f'Model {STAGE_A_MODEL}, 12yr — Bubble XML patched\n'
             '40°×40° |b|>2°, solid = ours, dashed = Cholis Fig 11', fontsize=11)
ax.legend(loc='lower left', fontsize=7.5, ncol=2, framealpha=0.95)
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_PNG, dpi=130, bbox_inches='tight')
plt.show()
print(f"\n[Saved] {OUTPUT_PNG}")

In [ ]:
# ============================================================================
# V29 — LogL comparison vs Cholis (Model I, 12yr) 
# ----------------------------------------------------------------------------
# 목적: 우리 fit이 같은 likelihood landscape에서 sub-optimal minimum에 있는지
#       (case a, 해결 가능), 진짜 다른 minimum에 있는지 (case b, 17yr 직진) 판별.
# 의존: V27/V28 변수 (E, n_ebin=17, STAGE_A_MODEL='I')
#       NPZ: GCE_model_I_12yr_cholis_fit.npz
# Reference: Cholis_ZENODO/GCE_Models_LogLikelihoods_..._GCE_vs_Background.dat
# ============================================================================

import numpy as np
from pathlib import Path

NPZ_PATH = Path(f"./GCE_model_{STAGE_A_MODEL}_12yr_cholis_fit.npz")
FIG11_BIN_MAX = 13  # bins 0..13 = Cholis Table III 14 bins

# Cholis Model I reference (from GCE_Models_LogLikelihoods file)
CHOLIS_LOGL_WITH_GCE  = -1875264.67772333
CHOLIS_LOGL_BACK_ONLY = -1877198.32328766
CHOLIS_TS_GCE         = 2 * (CHOLIS_LOGL_WITH_GCE - CHOLIS_LOGL_BACK_ONLY)

print("=" * 78)
print(f" V29 LogL Comparison | Model {STAGE_A_MODEL} | 12yr | 4FGL-DR2")
print("=" * 78)

# ---- 1. NPZ 로드 + raw max_likelihood inspection ---------------------------
npz = np.load(NPZ_PATH)
print(f"\n[NPZ] keys = {list(npz.keys())}")

mlhd = npz['max_likelihood']  # expected shape: (17,) per-bin best -2 ln L
print(f"\n[max_likelihood] shape = {mlhd.shape}, dtype = {mlhd.dtype}")
print(f"  per-bin values:")
for i in range(len(mlhd)):
    in_fig11 = "✓" if i <= FIG11_BIN_MAX else "(DM ext)"
    print(f"    bin {i:2d}  E={E[i]:7.3f} GeV  max_lhd = {mlhd[i]:14.2f}  {in_fig11}")

# ---- 2. 협약 자동 진단 -----------------------------------------------------
# Sanghwan runner의 likelihood form:
#   log_probability = poisson_ln_L - 0.5 * (chi2_bub + chi2_iso)
# poisson_ln_L itself is typically computed as:
#   ln L = sum [obs * log(expected) - expected]  → negative O(10^5) per bin
# But if stored as `lhd = 2*sum(expected - obs*log(expected))` = -2 ln L, 
# values are POSITIVE O(10^5) per bin.
sum_first14 = mlhd[:FIG11_BIN_MAX+1].sum()
sum_all17   = mlhd.sum()

print(f"\n[Sum over Cholis 14-bin range (bins 0–{FIG11_BIN_MAX})]: {sum_first14:.2f}")
print(f"[Sum over all 17 bins]:                               {sum_all17:.2f}")

# Auto-detect sign convention
if sum_first14 > 0:
    # Likely stored as -2 ln L (positive, smaller = better)
    our_ln_L = -sum_first14 / 2.0
    convention = "stored as -2 ln L (Cash-like, minimized)"
else:
    # Likely stored as ln L directly (negative, larger = better)
    our_ln_L = sum_first14
    convention = "stored as ln L (maximized directly)"

print(f"\n[Convention detected]: {convention}")
print(f"[Our ln L (14-bin)]:    {our_ln_L:.2f}")
print(f"[Cholis ln L (14-bin)]: {CHOLIS_LOGL_WITH_GCE:.2f}")
print(f"[Difference (us-them)]: {our_ln_L - CHOLIS_LOGL_WITH_GCE:+.2f}")

# ---- 3. 해석 가이드 --------------------------------------------------------
delta = our_ln_L - CHOLIS_LOGL_WITH_GCE
print("\n" + "=" * 78)
print(" 해석:")
if abs(delta) < 500:
    print(f"  Δ = {delta:+.1f}: 두 fit이 거의 같은 likelihood에 있음 (~0.03% level)")
    print(f"  → 같은 minimum 근처. V28의 c_gas/c_ics swap은 진짜 likelihood landscape 차이")
    print(f"     혹은 두 동등 minimum 사이의 선택 — fit이 sub-optimal한 것은 아님.")
    print(f"  → 더 깊이 좁히려면 spatial template (gtsrcmaps) 출력 직접 비교 필요.")
elif delta < -2000:
    print(f"  Δ = {delta:+.1f}: 우리 fit이 명확히 더 나쁨 ({100*delta/CHOLIS_LOGL_WITH_GCE:+.2f}%)")
    print(f"  → Sub-optimal minimum 가능성. Walker initial point 변경 또는")
    print(f"     burn-in/chain length 늘려서 재실행하면 c_gas/c_ics가 ~1.0으로")
    print(f"     수렴할 가능성. V28 hero figure 회복 가능.")
elif delta > 2000:
    print(f"  Δ = {delta:+.1f}: 우리 fit이 명확히 더 좋음 ({100*delta/CHOLIS_LOGL_WITH_GCE:+.2f}%)")
    print(f"  → 통상적으로 비현실적. PSC mask가 덜 마스킹되어 더 많은 자유도가")
    print(f"     fit에 들어왔거나, chi² prior가 더 약하게 적용되었을 가능성.")
    print(f"     이 경우 더 'fit이 잘 됐다'는 의미가 아니라 setup이 다른 것.")
else:
    print(f"  Δ = {delta:+.1f}: 차이가 중간 정도. 정확히 판별하기 위해서는")
    print(f"     prior chi² penalty 포함 여부도 정렬 필요.")
print("=" * 78)

# ---- 4. 추가 진단: 우리 GCE TS도 계산 (background-only 결과 있으면) -----
# Sanghwan 결과 .dat에는 background-only가 없으므로 우리도 추정만 가능.
# Sanghwan 본인 슬라이드: GCE significance Λ = 3779.66 (17 dof)
print(f"\n[Reference] Cholis Model I GCE TS = 2 × ΔLogL = {CHOLIS_TS_GCE:.1f}")
print(f"            Sanghwan DMPNP2026 GCE Λ ≈ 3779.66 (17 yr, 17 dof, slide 29)")
print(f"            우리 TS는 background-only fit 재실행 필요 (별도 작업)")

In [ ]:
# ============================================================================
# V30 — LogL 직접 비교: 우리 vs Sanghwan (동일 코드 convention)
# ----------------------------------------------------------------------------
# 목적: V29의 Δ = -155,673이 fit quality 차이인지 convention 차이인지 판별.
#       Sanghwan과 같은 코드를 쓰므로 convention은 자동 정렬됨.
# ============================================================================

import numpy as np
from pathlib import Path

NPZ_PATH = Path(f"./GCE_model_{STAGE_A_MODEL}_12yr_cholis_fit.npz")

# Sanghwan likelihood_value 후보 경로 (메모리 #15 + PORT_v2 SUMMARY 기준)
SANG_LHD_CANDIDATES = [
    Path("/home/sanghwan/FermiLAT/Sanghwan/GCE_model_I_12yr_cholis_likelihood_value"),
    Path("/home/sanghwan/FermiLAT/Sanghwan/GCE_model_I_12yr_likelihood_value"),
    Path("/home/sanghwan/FermiLAT/Sanghwan/GCE_model_I_12yr_cholis.dat"),  # may contain
    Path("/home/sanghwan/FermiLAT/Sanghwan/GCE_model_I_12yr.dat"),
]

print("=" * 78)
print(f" V30 LogL Comparison | Model {STAGE_A_MODEL} | ours vs Sanghwan (same code)")
print("=" * 78)

# 우리 max_likelihood
npz = np.load(NPZ_PATH)
ours = npz['max_likelihood']
print(f"\n[Ours] 17-bin max_likelihood:")
print(f"  bins 0–13 sum = {ours[:14].sum():.2f}")
print(f"  per-bin       = {ours[:14]}")

# Sanghwan 파일 탐색
print(f"\n[Sanghwan] Searching for likelihood file...")
sang_lhd = None
for path in SANG_LHD_CANDIDATES:
    if path.exists():
        print(f"  found: {path}")
        try:
            data = np.loadtxt(path)
            print(f"    shape = {data.shape}, sample = {data.flat[:3]}")
            sang_lhd = data
            break
        except Exception as e:
            print(f"    load error: {e}")
    else:
        print(f"  not found: {path}")

if sang_lhd is None:
    print("\n[Action needed] Sanghwan likelihood 파일을 직접 찾아주세요:")
    print("  ls /home/sanghwan/FermiLAT/Sanghwan/ | grep -i likelihood")
    print("  ls /home/sanghwan/FermiLAT/Sanghwan/ | grep model_I")
else:
    # 형태 정리: per-bin 14개 array로
    if sang_lhd.ndim > 1:
        # multi-column .dat이면 어느 컬럼인지 확인 필요
        print(f"\n  Multi-column data. Columns: {sang_lhd.shape[1]}")
        print(f"  First row: {sang_lhd[0]}")
        # GCE_model_*.dat 컬럼: E, flux, std, lower, upper — likelihood는 별도 파일
    elif sang_lhd.size == 14:
        print(f"\n[Comparison] per-bin lnL (ours bins 0–13 vs Sanghwan):")
        print(f"  {'bin':>3} {'E[GeV]':>8} {'ours':>14} {'sang':>14} {'Δ(o-s)':>12}")
        for i in range(14):
            print(f"  {i:3d} {E[i]:8.3f} {ours[i]:14.2f} {sang_lhd[i]:14.2f} "
                  f"{ours[i]-sang_lhd[i]:+12.2f}")
        print(f"\n  Sum bins 0–13: ours={ours[:14].sum():.2f}, "
              f"sang={sang_lhd.sum():.2f}, Δ={ours[:14].sum()-sang_lhd.sum():+.2f}")

In [ ]:
# V30 부록 — 전체 per-bin 비교 + sum
sang = np.loadtxt("/home/sanghwan/FermiLAT/Sanghwan/GCE_model_I_12yr_likelihood_value")
print(f"\n[Full per-bin lnL comparison]")
print(f"  {'bin':>3} {'E[GeV]':>8} {'ours':>14} {'sang':>14} {'Δ(o-s)':>12}")
for i in range(17):
    flag = "" if i <= 13 else "  (DM ext)"
    print(f"  {i:3d} {E[i]:8.3f} {ours[i]:14.2f} {sang[i]:14.2f} "
          f"{ours[i]-sang[i]:+12.2f}{flag}")
print(f"\n  Sum bins 0–13:  ours = {ours[:14].sum():14.2f},  "
      f"sang = {sang[:14].sum():14.2f},  Δ = {ours[:14].sum()-sang[:14].sum():+.2f}")
print(f"  Sum all 17:     ours = {ours.sum():14.2f},  "
      f"sang = {sang.sum():14.2f},  Δ = {ours.sum()-sang.sum():+.2f}")

In [ ]:
# Mask cube 검증 - 2분
import numpy as np
mask = np.load('./GC_analysis_sanghwan/Model/GC_mask_60x60_definitions_DR2.npy')
print(f"shape: {mask.shape}, dtype: {mask.dtype}")
print(f"unique pixel values: {np.unique(mask)}")  # 0과 1만 있어야 정상 (pixel mask는 binary)

# bin별 마스킹 분율 - Cholis Table III와 비교
for i in range(mask.shape[0]):
    masked = (mask[i] == 0).sum()
    total = mask[i].size
    print(f"  bin {i:2d}: masked = {100*masked/total:.1f}% of pixels")

In [ ]:
# Mask 검증 v2 — Cholis Table III와 apples-to-apples 비교
import numpy as np

psc_mask  = np.load('./GC_analysis_sanghwan/Model/GC_mask_60x60_definitions_DR2.npy')
disk_mask = np.load('./GC_analysis_sanghwan/Model/GC_disk_mask_60x60_definitions.npy')

# 40×40 영역 (runner와 동일하게 [100:500, 100:500])
psc_inner  = psc_mask[:, 100:500, 100:500]      # (17, 400, 400)
disk_inner = disk_mask[100:500, 100:500]         # (400, 400)
full_mask  = psc_inner * disk_inner              # (17, 400, 400), runner의 self.full_mask와 동일

# Cholis Table III (disk+PSC mask, 40×40 영역)
CHOLIS_MASKED = {
    0: 71.8, 1: 62.9, 2: 52.2,  3: 38.5,  4: 29.2,
    5: 23.4, 6: 19.0, 7: 16.3,  8: 13.0,  9: 12.9,
   10: 11.6,11: 11.5,12: 10.3, 13: 10.3,
}

print(f"  {'bin':>3} {'E[GeV]':>8} {'ours_masked':>13} {'Cholis':>10} {'Δ(o-c)':>10} {'ratio':>8}")
print("─" * 70)
for i in range(17):
    masked = (full_mask[i] == 0).sum()
    total = full_mask[i].size
    pct = 100 * masked / total
    cpct = CHOLIS_MASKED.get(i, np.nan)
    if i <= 13:
        delta = pct - cpct
        ratio = pct / cpct
        flag = "" if abs(delta) < 2 else (" ⚠" if abs(delta) < 5 else " ⚠⚠")
        print(f"  {i:3d} {E[i]:8.3f} {pct:12.1f}% {cpct:9.1f}% {delta:+9.1f}% {ratio:7.3f}{flag}")
    else:
        print(f"  {i:3d} {E[i]:8.3f} {pct:12.1f}% {'--':>10} (DM ext)")

In [ ]:
# V32 - Pre-fit absolute comparison vs Cholis Fig 11 solid line
# (V27 .txt + V28c digitized Cholis 파일 사용)

import numpy as np
pre = np.loadtxt("./V27_StageA_prefit_ModelI_12yr.txt")
chol = np.loadtxt("./2112_09706_Fig11_ModelI_ALL.txt")
E_us = pre[:14, 0]
sed_us = {'gas': pre[:14, 1], 'ics': pre[:14, 2],
          'bub': pre[:14, 3], 'iso': pre[:14, 4], 'gce': pre[:14, 5]}

E_c = chol[:, 0]
def interp_log(E_target, E_src, F_src):
    return 10**np.interp(np.log10(E_target), np.log10(E_src),
                          np.log10(np.clip(F_src, 1e-15, None)))

# Cholis solid line은 Fig 11에서 dashed의 lower edge × something이 아니라
# 별도로 우리가 그릴 때 우리 pre-fit과 직접 비교 가능
# 우선 dashed line (post-fit center) 기준으로 우리 pre-fit과 ratio 봐 본다
# c_gas ≈ 0.41이라는 건 우리 pre-fit이 Cholis post-fit의 ~2.5배라는 의미

print(f"  {'idx':>3} {'E':>8} {'comp':>5} {'ours_pre':>12} {'chol_post':>12} "
      f"{'ratio':>8} {'c_param':>9}")
for i in range(14):
    for comp, key, cparam_idx in [('gas','Pi0p',0), ('ics','ICS',1),
                                    ('bub','Bub',3), ('iso','Iso',4),
                                    ('gce','GCE',2)]:
        chol_post = interp_log(E_us[i], E_c, chol[:, 1+3*cparam_idx])
        ratio = sed_us[comp][i] / chol_post
        c_expected = 1.0 / ratio  # fit이 c=ratio_inverse 정도로 흘러갈 것
        print(f"  {i:3d} {E_us[i]:8.3f} {comp:>5} {sed_us[comp][i]:12.3e} "
              f"{chol_post:12.3e} {ratio:8.3f} (c→{c_expected:.3f})")

In [ ]:
from astropy.io import fits
import numpy as np

OUR  = './GC_analysis_sanghwan/Allsky_ltcube_12yr_front_clean.fits'
SANG = '/home/sanghwan/FermiLAT/Fermi-LAT-GCE/Analysis_data/12yr/evtype_front_evclass_clean/Allsky_ltcube.fits'

for path, label in [(OUR, 'OUR'), (SANG, 'SANG')]:
    h = fits.open(path)
    print(f"\n=== {label} ===")
    for i in [1, 2]:  # EXPOSURE, WEIGHTED_EXPOSURE
        d = h[i].data
        # Show column names
        print(f"  HDU {i} ({h[i].name}): columns = {d.columns.names}")
        # First row sample
        for col in d.columns.names:
            arr = d[col]
            print(f"    {col}: shape={arr.shape}, sum={np.asarray(arr).sum():.6e}, "
                  f"min={np.asarray(arr).min():.4e}, max={np.asarray(arr).max():.4e}")
    h.close()

# Direct numerical comparison
print("\n=== EXPOSURE table direct comparison ===")
ho = fits.open(OUR); hs = fits.open(SANG)
for i in [1, 2]:
    print(f"\nHDU {i} ({ho[i].name}):")
    for col in ho[i].data.columns.names:
        a1 = np.asarray(ho[i].data[col])
        a2 = np.asarray(hs[i].data[col])
        if a1.shape != a2.shape:
            print(f"  {col}: shape mismatch {a1.shape} vs {a2.shape}")
            continue
        if a1.dtype.kind not in 'fiu':
            continue
        diff = np.abs(a1.astype(float) - a2.astype(float))
        rel = diff.mean() / max(abs(a1.astype(float).mean()), 1e-30)
        print(f"  {col}: max|diff|={diff.max():.4e}, mean rel diff={rel:.4e}, "
              f"identical={np.array_equal(a1, a2)}")
ho.close(); hs.close()

In [ ]:
# ============================================================================
# V33 — GDE component template audit (no Sanghwan reference, Cholis direct)
# ----------------------------------------------------------------------------
# 목적: V27 pre-fit SED 5종을 Cholis가 사용한 input과 1:1로 분리 비교.
#       어느 성분에서 어떤 종류의 mismatch (normalization / shape)가
#       생기는지 정량 분리.
# 의존: V27 변수 (E, n_ebin, delta_E, exp_cube, disk_mask,
#                sed_gas, sed_ics, sed_bub, sed_iso, sed_gce, STAGE_A_MODEL)
# ============================================================================
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.interpolate import interp1d

CHOLIS_ALL_FILE = Path("./2112_09706_Fig11_ModelI_ALL.txt")
PRIOR_DIR       = Path("./GC_analysis_sanghwan/Model")  # bubble/iso prior txt 위치
OUTPUT_PNG      = f"./V33_component_audit_Model{STAGE_A_MODEL}_12yr.png"

# Cholis post-fit dashed lines (digitized)
cd = np.loadtxt(CHOLIS_ALL_FILE)
E_chol = cd[:, 0]
chol = {n: cd[:, 1+3*k] for k, n in enumerate(["Pi0p","ICS","Bub","Iso","GCE"])}
def chol_at(name, Et):
    return 10**np.interp(np.log10(Et), np.log10(E_chol),
                         np.log10(np.clip(chol[name], 1e-15, None)))

# Bubble / Iso prior input (Cholis가 사용한 외부 SED) — col 0, 1만 사용 (V27 방식)
_b = np.loadtxt(PRIOR_DIR/"bubble_constraints.txt")
bub_prior = interp1d(_b[:,0], _b[:,1],
                     kind='quadratic', fill_value='extrapolate')(E)  # E²dF/dE
_i = np.loadtxt(PRIOR_DIR/"iso_constraints_full_err.txt")
iso_prior_dNdE = interp1d(_i[:,0], _i[:,1],
                          kind='quadratic', fill_value='extrapolate')(E)
iso_prior = E**2 * iso_prior_dNdE                                    # E²dF/dE

BMAX = 13  # Fig 11 비교 범위

# Cholis는 본문 명시: 저에너지에서 c_ics≈0.5, c_gas≈1
# Pi0+Brem / ICS는 "pre-fit ≈ post-fit (~10%)" by Cholis text
# → pre-fit Pi0+Brem ≈ Cholis dashed,  pre-fit ICS ≈ Cholis dashed / c_ics
print("="*86)
print(f" V33 GDE component audit — Model {STAGE_A_MODEL}, 12yr, 40°×40° |b|>2°")
print("="*86)
print(f"\n[A] Pi0+Bremss : ours pre-fit (c=1)  vs  Cholis post-fit dashed")
print(f"    (Cholis 본문: c_gas ≈ 1, 즉 pre/post-fit 거의 같음)")
print(f"\n[B] ICS        : ours pre-fit (c=1)  vs  Cholis post-fit dashed")
print(f"    (Cholis 본문: c_ics ≈ 0.5 at low E → pre-fit이 dashed의 ~2배여야 정상)")
print(f"\n[C] Bubble     : ours pre-fit  vs  Cholis bubble prior input (1407.7905 T2)")
print(f"    (filling fraction 상수 = spatial 정상, shape 변동 = template spectrum bug)")
print(f"\n[D] Isotropic  : ours pre-fit  vs  Ackermann+2015 T3 input")
print(f"    (모든 E에서 ours/prior ≈ 1.0 ± 5%여야 정상)")
print(f"\n[E] GCE NFW²   : ours pre-fit (c=1) — reference scale only\n")

# === [A]/[B] Pi0+Brem, ICS : ours vs Cholis dashed ====================
print("-"*86)
print(f"  {'idx':>3} {'E[GeV]':>8} | {'gas_ours':>10} {'gas_chol':>10} {'r_gas':>7} "
      f"| {'ics_ours':>10} {'ics_chol':>10} {'r_ics':>7}")
print("-"*86)
for i in range(BMAX+1):
    g_o = sed_gas[i]; g_c = chol_at('Pi0p', E[i])
    i_o = sed_ics[i]; i_c = chol_at('ICS',  E[i])
    print(f"  {i:3d} {E[i]:8.3f} | {g_o:10.3e} {g_c:10.3e} {g_o/g_c:7.3f} "
          f"| {i_o:10.3e} {i_c:10.3e} {i_o/i_c:7.3f}")

# === [C] Bubble : pre-fit vs bubble prior input =======================
print("\n  Bubble: ours pre-fit / bubble_prior(input)  →  shape audit")
print("-"*60)
print(f"  {'idx':>3} {'E[GeV]':>8} {'bub_ours':>10} {'bub_pri':>10} {'ratio':>8}")
ratios_bub = []
for i in range(BMAX+1):
    r = sed_bub[i] / bub_prior[i]
    ratios_bub.append(r)
    print(f"  {i:3d} {E[i]:8.3f} {sed_bub[i]:10.3e} {bub_prior[i]:10.3e} {r:8.4f}")
ratios_bub = np.asarray(ratios_bub)
print(f"\n  → ratio mean = {ratios_bub.mean():.4f}, std/mean = "
      f"{ratios_bub.std()/ratios_bub.mean():.4f}")
print("  ✓ 정상 조건: std/mean < 0.05 (energy-independent constant = filling fraction)")
if ratios_bub.std()/ratios_bub.mean() > 0.1:
    print("  ✗ 큰 변동: bubble template의 spectral shape가 input과 다름 → 생성 단계 점검")

# === [D] Iso : pre-fit vs iso prior ===================================
print("\n  Isotropic: ours pre-fit / iso_prior(input)  →  spatial + spectrum audit")
print("-"*60)
print(f"  {'idx':>3} {'E[GeV]':>8} {'iso_ours':>10} {'iso_pri':>10} {'ratio':>8}")
ratios_iso = []
for i in range(BMAX+1):
    r = sed_iso[i] / iso_prior[i]
    ratios_iso.append(r)
    print(f"  {i:3d} {E[i]:8.3f} {sed_iso[i]:10.3e} {iso_prior[i]:10.3e} {r:8.4f}")
ratios_iso = np.asarray(ratios_iso)
print(f"\n  → ratio mean = {ratios_iso.mean():.4f}, std/mean = "
      f"{ratios_iso.std()/ratios_iso.mean():.4f}")
print("  ✓ 정상 조건: 모든 ratio ∈ [0.95, 1.05] (균일 템플릿이므로 spatial mean ≈ 1)")
flags_iso = np.where((ratios_iso < 0.85) | (ratios_iso > 1.15))[0]
if len(flags_iso) > 0:
    print(f"  ✗ Ratio가 1.0에서 크게 벗어난 bin: {flags_iso.tolist()}")
    print("    → isotropic input 텍스트(iso_P8R3_CLEAN_V3_v1.txt 또는 Ackermann T3)의")
    print("      energy grid 보간/단위가 component map과 다른지 점검")

# === Summary table ====================================================
print("\n" + "="*86)
print(" SUMMARY (Stage 0 진단)")
print("="*86)
diag_summary = [
    ("Pi0+Brem", "ours(c=1) / Cholis dashed", sed_gas[:BMAX+1] / chol_at('Pi0p',E[:BMAX+1]),
     "(c_gas ≈ 1 가정 → 0.9-1.1)"),
    ("ICS",      "ours(c=1) / Cholis dashed", sed_ics[:BMAX+1] / chol_at('ICS', E[:BMAX+1]),
     "(c_ics ≈ 0.5 at low E → low-E에서 ~2)"),
    ("Bubble",   "ours / bubble_prior",       ratios_bub,
     "(filling fraction 상수, 변동 < 5%)"),
    ("Isotropic","ours / iso_prior",          ratios_iso,
     "(균일 → 모든 bin에서 1.0 ± 5%)"),
]
print(f"  {'comp':<10} {'ratio (mean)':>14} {'std/mean':>10} {'min':>8} {'max':>8}  expected")
for name, _, r, expect in diag_summary:
    print(f"  {name:<10} {r.mean():>14.3f} {r.std()/r.mean():>10.3f} "
          f"{r.min():>8.3f} {r.max():>8.3f}  {expect}")

# === Plot =============================================================
C = {'gold':'#E09F3E','teal':'#1C7293','green':'#2D6A4F',
     'muted':'#6B7A88','red':'#9E2A2B'}
fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=110)

ax = axes[0]
ax.loglog(E[:BMAX+1], sed_gas[:BMAX+1], 'o-', color=C['gold'],  lw=2, label='Pi0+Brem (ours, c=1)')
ax.loglog(E[:BMAX+1], sed_ics[:BMAX+1], 's-', color=C['teal'],  lw=2, label='ICS (ours, c=1)')
ax.loglog(E[:BMAX+1], sed_bub[:BMAX+1], '^-', color=C['green'], lw=2, label='Bubble (ours, c=1)')
ax.loglog(E[:BMAX+1], sed_iso[:BMAX+1], 'v-', color=C['muted'], lw=2, label='Iso (ours, c=1)')
ax.loglog(E[:BMAX+1], bub_prior[:BMAX+1], '--', color=C['green'], lw=1.3,
          alpha=0.8, label='Bubble prior input')
ax.loglog(E[:BMAX+1], iso_prior[:BMAX+1], '--', color=C['muted'], lw=1.3,
          alpha=0.8, label='Iso prior input')
for k, col in zip(['Pi0p','ICS','Bub','Iso','GCE'],
                  [C['gold'],C['teal'],C['green'],C['muted'],C['red']]):
    ax.loglog(E_chol, chol[k], ':', color=col, lw=1.5, alpha=0.7)
ax.set_xlabel('E [GeV]'); ax.set_ylabel(r'$E^2 d\Phi/dE$')
ax.set_title('Pre-fit components (ours) vs prior input vs Cholis dashed')
ax.legend(fontsize=8, ncol=2); ax.grid(alpha=0.3); ax.set_ylim(1e-8, 3e-5)

ax = axes[1]
ax.axhline(1.0, color='k', lw=0.8, ls='-', alpha=0.5)
ax.axhspan(0.95, 1.05, color='green', alpha=0.1, label='±5% (iso 목표)')
ax.semilogx(E[:BMAX+1], sed_gas[:BMAX+1]/chol_at('Pi0p',E[:BMAX+1]),
            'o-', color=C['gold'], lw=2, label='gas: ours/Cholis dashed')
ax.semilogx(E[:BMAX+1], sed_ics[:BMAX+1]/chol_at('ICS', E[:BMAX+1]),
            's-', color=C['teal'], lw=2, label='ICS: ours/Cholis dashed')
ax.semilogx(E[:BMAX+1], ratios_bub, '^-', color=C['green'], lw=2,
            label='Bub: ours/prior_input')
ax.semilogx(E[:BMAX+1], ratios_iso, 'v-', color=C['muted'], lw=2,
            label='Iso: ours/prior_input')
ax.set_xlabel('E [GeV]'); ax.set_ylabel('ratio')
ax.set_title('Diagnostic ratios')
ax.legend(fontsize=8); ax.grid(alpha=0.3); ax.set_ylim(0, 5)
plt.tight_layout()
plt.savefig(OUTPUT_PNG, dpi=130, bbox_inches='tight')
plt.show()
print(f"\n[Saved] {OUTPUT_PNG}")

In [ ]:
# ============================================================================
# V34 — Fit-stage diagnostic suite (post-V33, post-V28d)
# ----------------------------------------------------------------------------
# 목적: V33가 template OK를 보였음에도 fit 결과 c_gas=0.4 / c_ics=1.84 /
#       c_iso bin 7,8,10에서 0 hit이 나온 원인을 분리.
# 검증 가설 5종 (1) c_iso boundary hit, (2) c_gas↔c_ics degeneracy,
#               (3) bin별 fit quality, (4) PSC mask gap, (5) prior strength
# 의존: V27 변수 (E, n_ebin, delta_E, exp_cube, disk_mask, SLICE_Y, SLICE_X,
#                 ics_nc, sed_*, STAGE_A_MODEL)
# ============================================================================
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

NPZ_PATH      = Path(f"./GCE_model_{STAGE_A_MODEL}_12yr_cholis_fit.npz")
FIT_MASK_PATH = Path("./GC_analysis_sanghwan/Model/GC_mask_60x60_definitions_DR2.npy")
PRIOR_DIR     = Path("./GC_analysis_sanghwan/Model")
OUTPUT_PNG    = f"./V34_fit_diagnostic_Model{STAGE_A_MODEL}_12yr.png"
BMAX          = 13

# ---- Load NPZ -------------------------------------------------------------
npz = np.load(NPZ_PATH)
def rs(a): return a.reshape(5, n_ebin) if a.ndim == 1 else a
fp    = rs(npz['fitted_params'])
fp_lo = rs(npz['fitted_params_lower'])
fp_hi = rs(npz['fitted_params_upper'])
mlhd  = npz['max_likelihood']
labels = ['c_gas', 'c_ics', 'c_gce', 'c_bub', 'c_iso']

print("="*108)
print(f" V34 — Fit-stage diagnostic | Model {STAGE_A_MODEL}, 12yr")
print("="*108)

# ===== [1] Bin-by-bin fitted c_params with (16th, 84th) percentile bounds =====
print("\n[1] Fitted c_param per bin — cen (16th, 84th), '*' = c < 0.05 (boundary hit)")
print("-"*108)
hdr = f"  {'idx':>3} {'E[GeV]':>8} "
for lab in labels: hdr += f" {lab:>17} "
print(hdr); print("-"*108)
zero_hits = {lab: [] for lab in labels}
for i in range(BMAX+1):
    line = f"  {i:3d} {E[i]:8.3f} "
    for j, lab in enumerate(labels):
        cen, lo, hi = fp[j,i], fp_lo[j,i], fp_hi[j,i]
        flag = "*" if cen < 0.05 else " "
        if cen < 0.05: zero_hits[lab].append(i)
        line += f"{cen:5.2f}({lo:5.2f},{hi:5.2f}){flag}"
    print(line)
print("\n  Boundary hits (c < 0.05):")
for lab, bins_hit in zero_hits.items():
    if bins_hit: print(f"    {lab}: bins {bins_hit}")

# ===== [2] c_param cross-bin correlation =====
print("\n[2] Cross-bin Pearson correlation of c_params (bins 0–13):")
fp_b = fp[:, :BMAX+1]
corr = np.corrcoef(fp_b)
print(f"  {'':>10}", end=""); [print(f" {lab:>9}", end="") for lab in labels]; print()
for i, lab in enumerate(labels):
    print(f"  {lab:>10}", end="")
    for j in range(5): print(f" {corr[i,j]:>9.3f}", end="")
    print()
print("  → 강한 음의 상관 (≲ -0.7) = degenerate fit 신호")

# ===== [3] -2 ln L per bin =====
print(f"\n[3] -2 ln L per bin:")
print(f"  {'idx':>3} {'E[GeV]':>8} {'-2lnL':>14} {'cumulative':>14}")
print("-"*45)
neg2lnL = -2 * mlhd[:BMAX+1]
for i in range(BMAX+1):
    print(f"  {i:3d} {E[i]:8.3f} {neg2lnL[i]:14.1f} {neg2lnL[:i+1].sum():14.1f}")
print(f"  {'TOTAL':>20}  {neg2lnL.sum():14.1f}")
print(f"  {'Cholis ref':>20}  {3750529.36:14.1f}  Δ = {neg2lnL.sum() - 3750529.36:+.0f}")

# ===== [4] PSC mask gap inside 40×40 |b|>2° window (robust, shape-defensive) =====
print(f"\n[4] PSC mask 효과 (40×40 |b|>2° 안에서 ICS pre-fit이 어떻게 변하는가):")

# --- shape 정규화: disk_mask, exp_cube, ics_nc 모두 (n_ebin?, 400, 400) 보장 ---
def to_400_2d(m, name):
    m = np.asarray(m)
    if m.shape == (400, 400):
        return m.astype(bool)
    if m.shape == (600, 600):
        print(f"  [normalize] {name}: (600,600) → slice → (400,400)")
        return m[SLICE_Y, SLICE_X].astype(bool)
    raise ValueError(f"{name} shape {m.shape} unsupported (need 400² or 600²)")

def to_400_3d(c, name):
    c = np.asarray(c)
    if c.shape[1:] == (400, 400):
        return c
    if c.shape[1:] == (600, 600):
        print(f"  [normalize] {name}: (.,600,600) → slice → (.,400,400)")
        return c[:, SLICE_Y, SLICE_X]
    raise ValueError(f"{name} shape {c.shape} unsupported")

db_400      = to_400_2d(disk_mask, "disk_mask")
exp_400     = to_400_3d(exp_cube,  "exp_cube")
ics_nc_400  = to_400_3d(ics_nc,    "ics_nc")
print(f"  ✓ Normalized: disk_mask={db_400.shape}, exp_cube={exp_400.shape}, "
      f"ics_nc={ics_nc_400.shape}")

if FIT_MASK_PATH.exists():
    fmf = np.load(FIT_MASK_PATH)
    print(f"  Raw fit mask: shape = {fmf.shape}, dtype = {fmf.dtype}")

    if fmf.ndim == 2:
        m2d = to_400_2d(fmf, "fit_mask")
        fit_mask_40 = np.broadcast_to(m2d, (n_ebin, 400, 400))
        print(f"  → 2D mask, energy-independent → broadcast to {n_ebin} bins")

    elif fmf.ndim == 3:
        matching = [ax for ax, sz in enumerate(fmf.shape) if sz == n_ebin]
        if len(matching) == 1:
            eax = matching[0]
            if eax != 0:
                fmf = np.moveaxis(fmf, eax, 0)
            assert fmf.shape[1:] in [(600, 600), (400, 400)], \
                f"Spatial axes {fmf.shape[1:]} not 600² or 400²"
            fit_mask_40 = to_400_3d(fmf.astype(bool), "fit_mask")
            print(f"  → 3D mask, energy axis = {eax}, size {n_ebin}")
        elif len(matching) == 0:
            spatial = [ax for ax, sz in enumerate(fmf.shape) if sz in (400, 600)]
            if len(spatial) == 2:
                stk_ax = [ax for ax in range(3) if ax not in spatial][0]
                print(f"  → 3D mask, no energy axis. Stacking axis = {stk_ax} "
                      f"(size {fmf.shape[stk_ax]}) → AND-reduce, broadcast")
                fmf2d = np.all(fmf.astype(bool), axis=stk_ax)
                m2d = to_400_2d(fmf2d, "fit_mask_reduced")
                fit_mask_40 = np.broadcast_to(m2d, (n_ebin, 400, 400))
            else:
                raise ValueError(f"Cannot parse mask shape {fmf.shape}")
        else:
            raise ValueError(f"Ambiguous mask shape {fmf.shape}: "
                             f"multiple axes match n_ebin={n_ebin}")
    else:
        raise ValueError(f"Unsupported mask ndim={fmf.ndim}")

    assert fit_mask_40.shape == (n_ebin, 400, 400), \
        f"Final fit_mask {fit_mask_40.shape} ≠ ({n_ebin}, 400, 400)"
    print(f"  ✓ Final: fit_mask_40 = {fit_mask_40.shape} bool")
    print(f"  ✓ Bin 0 PSC retention: "
          f"{100*fit_mask_40[0].sum()/fit_mask_40[0].size:.1f}% of 400×400 pixels")

    print(f"\n  {'idx':>3} {'E[GeV]':>8} {'disk only':>11} {'PSC+disk':>11} "
          f"{'PSC kills':>11} | {'ICS_disk':>11} {'ICS_PSC':>11} {'ratio':>7}")
    print("-"*92)
    for i in range(BMAX+1):
        m_fit = np.logical_and(fit_mask_40[i], db_400)
        with np.errstate(divide='ignore', invalid='ignore'):
            rr = np.where(exp_400[i] > 0, ics_nc_400[i] / exp_400[i], 0.0)
        ics_d = (rr * db_400).sum()  / max(db_400.sum(), 1) * E[i]**2 / delta_E[i]
        ics_f = (rr * m_fit).sum()   / max(m_fit.sum(), 1)  * E[i]**2 / delta_E[i]
        disk_pct = 100 * db_400.sum()  / db_400.size
        fit_pct  = 100 * m_fit.sum()   / db_400.size
        print(f"  {i:3d} {E[i]:8.3f} {disk_pct:10.1f}% {fit_pct:10.1f}% "
              f"{disk_pct-fit_pct:10.1f}% | {ics_d:11.3e} {ics_f:11.3e} "
              f"{ics_f/ics_d:7.3f}")
    print("\n  ★ ratio < 0.85 이면: ICS-rich한 inner-galaxy 픽셀이 PSC mask로 빠짐 →")
    print("    fit에서 보는 ICS가 audit보다 작아서 c_ics를 키워 보상하려 함")
else:
    print(f"  ⚠ Fit-mask file not found: {FIT_MASK_PATH}")
    fit_mask_40 = None

# ===== [5] Bubble/Iso prior strength =====
print("\n[5] Prior file structure: fractional error per bin (frac_err 클수록 prior 약함)")
for name, fname in [("Bubble", "bubble_constraints.txt"),
                     ("Iso",    "iso_constraints_full_err.txt")]:
    pth = PRIOR_DIR / fname
    if not pth.exists():
        print(f"  ⚠ Not found: {pth}"); continue
    arr = np.loadtxt(pth)
    print(f"\n  {name}: {fname}, shape = {arr.shape}")
    print(f"  cols = {[f'col{k}' for k in range(arr.shape[1])]}")
    print(f"  First 3 rows:")
    for row in arr[:3]: print(f"    {row}")
    if arr.shape[1] >= 4:
        Ep, val, e_lo, e_hi = arr[:,0], arr[:,1], arr[:,2], arr[:,3]
        ok = val > 0
        f_lo = np.where(ok, e_lo/np.abs(val), np.nan)
        f_hi = np.where(ok, e_hi/np.abs(val), np.nan)
        print(f"  {'E[GeV]':>10} {'value':>11} {'err_lo':>11} {'err_hi':>11} "
              f"{'frac_lo':>9} {'frac_hi':>9}")
        for k in range(min(10, len(arr))):
            print(f"  {Ep[k]:10.3f} {val[k]:11.3e} {e_lo[k]:11.3e} {e_hi[k]:11.3e} "
                  f"{f_lo[k]:9.3f} {f_hi[k]:9.3f}")
        print(f"  → mean frac_err: lo = {np.nanmean(f_lo):.3f}, hi = {np.nanmean(f_hi):.3f}")
        print(f"    (>1.0 → prior 매우 약함, c가 자유롭게 움직임)")

# ===== Visualization: 4-panel summary =====
fig, axes = plt.subplots(2, 2, figsize=(14, 10), dpi=110)
C = {'gas':'#E09F3E','ics':'#1C7293','gce':'#9E2A2B',
     'bub':'#2D6A4F','iso':'#6B7A88'}
cols = [C['gas'], C['ics'], C['gce'], C['bub'], C['iso']]

# (a) c_params with 1σ band  (boundary-hit safe)
ax = axes[0,0]
for j, lab in enumerate(labels):
    cen = fp[j, :BMAX+1].astype(float).copy()
    lo  = fp_lo[j, :BMAX+1].astype(float)
    hi  = fp_hi[j, :BMAX+1].astype(float)
    # cen이 [lo, hi] 밖에 있으면 (boundary hit 또는 multimodal) → 양 percentile로 강제 정렬
    cen_plot = np.clip(cen, np.minimum(lo, hi), np.maximum(lo, hi))
    cen_plot = np.clip(cen_plot, 1e-3, None)  # log scale 보호
    el = np.maximum(cen_plot - np.minimum(lo, hi), 0)
    eh = np.maximum(np.maximum(lo, hi) - cen_plot, 0)
    ax.errorbar(E[:BMAX+1], cen_plot, yerr=[el, eh],
                fmt='o-', color=cols[j], capsize=2, lw=1.5, ms=5, label=lab)
    # boundary-hit bin 표시: cen이 plot 영역 바깥으로 갔던 곳
    hit = np.where(cen != cen_plot)[0]
    if len(hit) > 0:
        ax.scatter(E[hit], cen_plot[hit], marker='x', s=60,
                   color=cols[j], zorder=5, linewidths=2)
ax.axhline(1.0, color='k', lw=0.6, ls=':')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('E [GeV]'); ax.set_ylabel('c_param')
ax.set_title('(a) Fitted c_param ± 1σ — ×: boundary hit')
ax.legend(fontsize=8, ncol=2); ax.grid(alpha=0.3)
ax.set_ylim(1e-3, 50)

# (b) c_gas vs c_ics scatter (with bin numbers)
ax = axes[0,1]
sc = ax.scatter(fp[0,:BMAX+1], fp[1,:BMAX+1], c=np.log10(E[:BMAX+1]),
                cmap='viridis', s=120, edgecolor='k', zorder=3)
for i in range(BMAX+1):
    ax.annotate(f'{i}', (fp[0,i], fp[1,i]), fontsize=7,
                ha='center', va='center', color='white', weight='bold')
ax.axhline(1.0, color='gray', lw=0.5, ls='--')
ax.axvline(1.0, color='gray', lw=0.5, ls='--')
ax.set_xlabel('c_gas'); ax.set_ylabel('c_ics')
ax.set_title(f'(b) c_gas vs c_ics (Pearson r = {corr[0,1]:.3f})')
plt.colorbar(sc, ax=ax, label='log10 E[GeV]')
ax.grid(alpha=0.3)

# (c) -2lnL per bin
ax = axes[1,0]
ax.semilogx(E[:BMAX+1], neg2lnL, 'o-', color='k', ms=6, lw=1.5)
ax.axhline(neg2lnL.mean(), color='red', lw=0.8, ls='--',
           label=f'mean = {neg2lnL.mean():.0f}')
ax.set_xlabel('E [GeV]'); ax.set_ylabel('-2 ln L per bin')
ax.set_title(f'(c) Fit quality per bin (Σ = {neg2lnL.sum():.0f}, Cholis ≈ 3.75M)')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# (d) c_iso/c_gce/c_bub: who absorbs flux when iso → 0?
ax = axes[1,1]
ax.semilogx(E[:BMAX+1], fp[4,:BMAX+1], 'o-', color=C['iso'], label='c_iso', ms=7, lw=1.8)
ax.semilogx(E[:BMAX+1], fp[2,:BMAX+1], 's-', color=C['gce'], label='c_gce', ms=7, lw=1.8)
ax.semilogx(E[:BMAX+1], fp[3,:BMAX+1], '^-', color=C['bub'], label='c_bub', ms=7, lw=1.8,
            alpha=0.7)
ax.axhline(0, color='red', lw=1, ls='--', alpha=0.5)
ax.axhline(1, color='gray', lw=0.5, ls=':')
# annotate iso boundary hits
for ib in zero_hits['c_iso']:
    ax.axvspan(E[ib]*0.92, E[ib]*1.08, color='red', alpha=0.08)
ax.set_xlabel('E [GeV]'); ax.set_ylabel('c_param')
ax.set_title('(d) Iso/GCE/Bub — red bands = c_iso boundary hits')
ax.legend(fontsize=9); ax.grid(alpha=0.3)
ax.set_ylim(-0.2, max(2.5, fp[[2,3,4],:BMAX+1].max()*1.1))

plt.tight_layout()
plt.savefig(OUTPUT_PNG, dpi=130, bbox_inches='tight')
plt.show()
print(f"\n[Saved] {OUTPUT_PNG}")

# ===== Decision-ready summary =====
print("\n" + "="*108)
print(" 다음 결정에 필요한 5가지 신호 — 출력값으로 채워서 보고:")
print("="*108)
print(f"  S1. c_iso boundary hit bins  = {zero_hits['c_iso']}")
print(f"      → 비어있으면 fit이 iso prior 안에서 안정, 채워져 있으면 prior 압력 부족 의심")
print(f"  S2. r(c_gas, c_ics)          = {corr[0,1]:+.3f}")
print(f"      → ≲ -0.7이면 두 성분 degenerate, fit이 둘을 구분 못 함")
print(f"  S3. Δ(-2 ln L) vs Cholis     = {neg2lnL.sum() - 3750529.36:+.0f}")
print(f"      → 양수 = 우리 fit이 더 나쁜 곳에 있음 (= local min 또는 wrong template)")
print(f"  S4. PSC mask가 ICS에 미치는 영향: 위 [4] 표의 'ratio' 컬럼 평균을 확인")
print(f"      → 0.85 미만 평균이면 audit-fit mask gap이 ICS 30% 차이의 주범")
print(f"  S5. Bubble/Iso prior frac_err: 위 [5] 표 평균을 확인")
print(f"      → frac_err > 0.5 이면 prior 너무 약함, fit이 c를 자유롭게 흔듦")

In [ ]:
# ============================================================================
# V35 — Fit machinery audit: conv/no_conv consistency, total-count match,
#       likelihood landscape, fit-region SED
# ----------------------------------------------------------------------------
# 목적: V33 audit (non-conv + disk-only)와 V34 fit (conv + full mask) 차이가
#       어디서 오는지 4개 직접 측정으로 분리.
#   (1) conv vs no_conv: PSF normalization 보존 검증
#   (2) Data total vs Σ(c=1 templates) total: absolute normalization 검증
#   (3) likelihood at c=1 vs c=fit vs c=Cholis-like: fit이 정말 minimum인가
#   (4) ICS pre-fit SED를 fit region(conv + full_mask)에서 재측정
# 의존: V27 변수 (E, n_ebin, delta_E, SLICE_Y, SLICE_X, STAGE_A_MODEL)
# ============================================================================
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from pathlib import Path
from scipy.special import gammaln

SANG = Path("./GC_analysis_sanghwan")
NPZ_PATH = Path(f"./GCE_model_{STAGE_A_MODEL}_12yr_cholis_fit.npz")
BMAX = 13
sr_per_pix = (np.pi/180.0 * 0.1) ** 2

# ---- 0. Setup probes: 모든 변수 shape/dtype 확인 후 시작 ------------------
print("="*100)
print(f" V35 — Fit machinery audit | Model {STAGE_A_MODEL}, 12yr")
print("="*100)

def to_400_2d(m, name):
    m = np.asarray(m)
    if m.shape == (400, 400): return m.astype(bool)
    if m.shape == (600, 600):
        print(f"  [norm] {name}: (600,600) → slice → (400,400)")
        return m[SLICE_Y, SLICE_X].astype(bool)
    raise ValueError(f"{name}: shape {m.shape}")

def to_400_3d(c, name):
    c = np.asarray(c)
    if c.shape[1:] == (400, 400): return c
    if c.shape[1:] == (600, 600):
        print(f"  [norm] {name}: (.,600,600) → (.,400,400)")
        return c[:, SLICE_Y, SLICE_X]
    raise ValueError(f"{name}: shape {c.shape}")

# Load fit-region masks (runner와 정확히 같은 경로 + 슬라이싱)
psc_raw  = np.load(SANG / "Model/GC_mask_60x60_definitions_DR2.npy")
disk_raw = np.load(SANG / "Model/GC_disk_mask_60x60_definitions.npy")
print(f"\n[masks] psc raw {psc_raw.shape} {psc_raw.dtype}, "
      f"disk raw {disk_raw.shape} {disk_raw.dtype}")
psc_400  = to_400_3d(psc_raw.astype(bool),  "psc")
disk_400 = to_400_2d(disk_raw, "disk")
full_400 = psc_400 & disk_400[None, :, :]  # broadcast → (n_ebin, 400, 400)
print(f"[masks] psc_400 {psc_400.shape}, disk_400 {disk_400.shape}, "
      f"full_400 {full_400.shape}")

# Load convolved + non-convolved templates for Model I
front = "_front"  # runner naming convention; STAGE_A_MODEL=='I'이면 front 사용
M = STAGE_A_MODEL
def load_pair(stem, model_in_name=True):
    base = f"GC_{stem}_model{M}_12yr{front}_clean" if model_in_name \
           else f"GC_{stem}_model_12yr{front}_clean"
    c  = fits.getdata(SANG / f"{base}.fits")
    nc = fits.getdata(SANG / f"{base}_no_convol.fits")
    return c, nc

pion_c, pion_nc = load_pair("pion")
brem_c, brem_nc = load_pair("bremss")
ics_c,  ics_nc  = load_pair("ics")
gce_c,  gce_nc  = load_pair("GCE",          model_in_name=False)
bub_c,  bub_nc  = load_pair("fermi_bubble", model_in_name=False)
iso_c,  iso_nc  = load_pair("isotropic",    model_in_name=False)
print(f"\n[templates] loaded 6 pairs (conv + no_conv), each shape {pion_c.shape}")

# Crop all to 400×400
pion_c, pion_nc = pion_c[:,SLICE_Y,SLICE_X], pion_nc[:,SLICE_Y,SLICE_X]
brem_c, brem_nc = brem_c[:,SLICE_Y,SLICE_X], brem_nc[:,SLICE_Y,SLICE_X]
ics_c,  ics_nc  = ics_c[:,SLICE_Y,SLICE_X],  ics_nc[:,SLICE_Y,SLICE_X]
gce_c,  gce_nc  = gce_c[:,SLICE_Y,SLICE_X],  gce_nc[:,SLICE_Y,SLICE_X]
bub_c,  bub_nc  = bub_c[:,SLICE_Y,SLICE_X],  bub_nc[:,SLICE_Y,SLICE_X]
iso_c,  iso_nc  = iso_c[:,SLICE_Y,SLICE_X],  iso_nc[:,SLICE_Y,SLICE_X]
pb_c  = pion_c + brem_c
pb_nc = pion_nc + brem_nc

# CCUBE (data) + expcube
ccube  = fits.getdata(SANG / f"GC_ccube_12yr{front}_clean.fits")[:, SLICE_Y, SLICE_X]
expraw = fits.getdata(SANG / f"GC_expcube_center_12yr{front}_clean.fits")
exp_full = expraw[:, SLICE_Y, SLICE_X] * sr_per_pix
n_e = ccube.shape[0]
print(f"[data] ccube {ccube.shape}, exp_cube {exp_full.shape}, n_e = {n_e}")

# Load fit results
npz = np.load(NPZ_PATH)
fp = npz['fitted_params'].reshape(5, n_e) if npz['fitted_params'].ndim==1 \
     else npz['fitted_params']

# ============ Section 1: conv vs no_conv per-bin total flux ============
print("\n" + "="*100)
print(" (1) Convolved vs Non-convolved: per-bin total flux ratio")
print("     (PSF는 normalization을 보존해야 함 → ratio ≈ 1.0 ± 1%)")
print("-"*100)
print(f"  {'idx':>3} {'E[GeV]':>7} | {'gas conv/nc':>12} {'ICS conv/nc':>12} "
      f"{'GCE conv/nc':>12} {'Bub conv/nc':>12} {'Iso conv/nc':>12}")
print("-"*100)
for i in range(BMAX+1):
    rgas = pb_c[i].sum()  / pb_nc[i].sum()
    rics = ics_c[i].sum() / ics_nc[i].sum()
    rgce = gce_c[i].sum() / gce_nc[i].sum()
    rbub = bub_c[i].sum() / bub_nc[i].sum()
    riso = iso_c[i].sum() / iso_nc[i].sum()
    print(f"  {i:3d} {E[i]:7.3f} | {rgas:12.4f} {rics:12.4f} {rgce:12.4f} "
          f"{rbub:12.4f} {riso:12.4f}")
print("  ★ ratio ≠ 1.0 ± 0.02 인 성분 → gtsrcmaps PSF normalization 문제")

# ============ Section 2: Data total vs Σ(c=1 conv templates) total ============
print("\n" + "="*100)
print(" (2) Data vs Σ(c=1 templates) — PSC+disk mask 적용, 400×400 fit-region")
print("     (c=1로 잘 맞으면 fit이 c=1을 찾아야 함. 안 맞으면 absolute scale 문제)")
print("-"*100)
print(f"  {'idx':>3} {'E[GeV]':>7} | {'data':>10} {'sum(c=1)':>10} {'ratio':>8} "
      f"| {'sum(c=fit)':>11} {'ratio':>8}")
print("-"*100)
for i in range(BMAX+1):
    m  = full_400[i]
    d_tot = ccube[i][m].sum()
    sum1  = (pb_c[i] + ics_c[i] + gce_c[i] + iso_c[i] + bub_c[i])[m].sum()
    sumf  = (fp[0,i]*pb_c[i] + fp[1,i]*ics_c[i] + fp[2,i]*gce_c[i]
             + fp[4,i]*iso_c[i] + fp[3,i]*bub_c[i])[m].sum()
    print(f"  {i:3d} {E[i]:7.3f} | {d_tot:10.0f} {sum1:10.0f} {sum1/d_tot:8.3f} "
          f"| {sumf:11.0f} {sumf/d_tot:8.3f}")
print("  ★ c=1 ratio ≠ 1 → fit이 그 차이를 c로 흡수해야 하는 게 정상")
print("  ★ c=fit ratio ≈ 1 → fit이 total count는 맞춤 (spatial structure는 별개)")

# ============ Section 3: Likelihood landscape ============
print("\n" + "="*100)
print(" (3) Likelihood value: c=1 vs c=fit vs c=Cholis-like (1, 0.5, 1, 1, 1)")
print("     (fit value가 가장 작아야 정상. c=1이 더 작으면 MCMC가 local min에 갇힘)")
print("-"*100)

# bubble/iso prior (V27과 동일 보간)
from scipy.interpolate import interp1d
_bp = np.loadtxt(SANG / "Model/bubble_constraints.txt")
_ip = np.loadtxt(SANG / "Model/iso_constraints_full_err.txt")
bub_flux  = interp1d(_bp[:,0], _bp[:,1], kind='quadratic',
                     fill_value='extrapolate')(E)
bub_elo   = interp1d(_bp[:,0], _bp[:,2], kind='quadratic',
                     fill_value='extrapolate')(E)
bub_ehi   = interp1d(_bp[:,0], _bp[:,3], kind='quadratic',
                     fill_value='extrapolate')(E)
iso_dnde  = interp1d(_ip[:,0], _ip[:,1], kind='quadratic',
                     fill_value='extrapolate')(E)
iso_elo   = interp1d(_ip[:,0], _ip[:,2], kind='quadratic',
                     fill_value='extrapolate')(E) * E**2
iso_ehi   = interp1d(_ip[:,0], _ip[:,3], kind='quadratic',
                     fill_value='extrapolate')(E) * E**2
iso_flux  = iso_dnde * E**2

def eval_likelihood(c_vec, i):
    pb_p, ics_p, gce_p, bub_p, iso_p = c_vec
    m = full_400[i]
    exp_p = (pb_p*pb_c[i] + ics_p*ics_c[i] + gce_p*gce_c[i]
             + iso_p*iso_c[i] + bub_p*bub_c[i])[m]
    obs   = ccube[i][m]
    if (exp_p <= 0).any(): return np.inf
    lhd = 2*(exp_p - obs*np.log(exp_p) + gammaln(obs.astype(int)+1)).sum()
    # bubble chi^2 (E^2 dF/dE form)
    bub_sed = (E[i]**2) * (full_400[i] * bub_nc[i] / exp_full[i]).sum() \
              / m.sum() * bub_p / delta_E[i]
    err_b = max(bub_ehi[i], bub_elo[i]) if bub_sed==bub_flux[i] else \
            (bub_ehi[i] if bub_sed > bub_flux[i] else bub_elo[i])
    chi2_b = ((bub_sed - bub_flux[i]) / err_b)**2
    iso_sed = (E[i]**2) * (full_400[i] * iso_nc[i] / exp_full[i]).sum() \
              / m.sum() * iso_p / delta_E[i]
    err_i = max(iso_ehi[i], iso_elo[i]) if iso_sed==iso_flux[i] else \
            (iso_elo[i] if iso_sed > iso_flux[i] else iso_ehi[i])
    chi2_i = ((iso_sed - iso_flux[i]) / err_i)**2
    return lhd + chi2_b + chi2_i

c1     = np.ones(5)
cChol  = np.array([1.0, 0.5, 1.0, 1.0, 1.0])  # 본문 "c_ics ≈ 0.5 at low E"
print(f"  {'idx':>3} {'E[GeV]':>7} | {'L(c=1)':>14} {'L(c=fit)':>14} "
      f"{'L(c=Chol)':>14} | {'L1 - Lfit':>14}")
print("-"*100)
for i in range(BMAX+1):
    L1   = eval_likelihood(c1,           i)
    Lf   = eval_likelihood(fp[:, i],     i)
    Lch  = eval_likelihood(cChol,        i)
    print(f"  {i:3d} {E[i]:7.3f} | {L1:14.1f} {Lf:14.1f} {Lch:14.1f} | "
          f"{L1 - Lf:+14.1f}")
print("  ★ L(c=1) > L(c=fit) 이면 fit이 정말 minimum에 있음 (자체적으로 일관)")
print("  ★ L(c=Chol) ≈ L(c=fit) 이면 Cholis convention도 동일하게 좋음 (degenerate landscape)")

# ============ Section 4: ICS audit re-do in fit region ============
print("\n" + "="*100)
print(" (4) ICS pre-fit SED in fit-region (conv + PSC+disk mask) vs Cholis dashed")
print("     (V33은 no_conv + disk-only로 1.30 측정. 여기는 fit이 실제로 보는 영역)")
print("-"*100)
cd = np.loadtxt("./2112_09706_Fig11_ModelI_ALL.txt")
def chol_at(name, Et, k):
    return 10**np.interp(np.log10(Et), np.log10(cd[:,0]),
                         np.log10(np.clip(cd[:,1+3*k], 1e-15, None)))
print(f"  {'idx':>3} {'E[GeV]':>7} | {'ICS_conv_fit':>13} {'ICS_chol':>11} "
      f"{'ratio':>8} | {'gas_conv_fit':>13} {'gas_chol':>11} {'ratio':>8}")
print("-"*100)
for i in range(BMAX+1):
    m = full_400[i]
    # PSF-convolved photon counts → flux per pixel
    with np.errstate(divide='ignore', invalid='ignore'):
        ics_flux  = np.where(exp_full[i] > 0, ics_c[i] / exp_full[i], 0.0)
        gas_flux  = np.where(exp_full[i] > 0, pb_c[i]  / exp_full[i], 0.0)
    ics_sed = ics_flux[m].sum() / m.sum() * E[i]**2 / delta_E[i]
    gas_sed = gas_flux[m].sum() / m.sum() * E[i]**2 / delta_E[i]
    ics_ch = chol_at('ICS', E[i], 1)
    gas_ch = chol_at('Pi0p', E[i], 0)
    print(f"  {i:3d} {E[i]:7.3f} | {ics_sed:13.3e} {ics_ch:11.3e} "
          f"{ics_sed/ics_ch:8.3f} | {gas_sed:13.3e} {gas_ch:11.3e} "
          f"{gas_sed/gas_ch:8.3f}")
print("  ★ V33의 1.30 vs 여기 ratio 비교 → conv+full_mask가 차이를 얼마나 흡수하는가")

print("\n" + "="*100)
print(" 다음 결정 분기:")
print("="*100)
print(" • Section 1 ratio ≠ 1 → PSF normalization 깨짐, gtsrcmaps 재실행")
print(" • Section 2 c=fit ratio ≈ 1, c=1 ratio ≠ 1 → fit은 일관됨, 절대 scale OK")
print(" • Section 3 L(c=1) >> L(c=fit) → MCMC 정상 작동, c=0.4 등이 진짜 minimum")
print(" • Section 4의 ICS ratio가 V33 1.30보다 작아지면 → conv+mask가 ICS를 줄임,")
print("   즉 우리 templates의 spatial structure 자체가 Cholis와 다름")

In [ ]:
# ============================================================================
# V36 — Cholis Zenodo Exposure cube vs 우리 exposure cube 직접 비교
# ----------------------------------------------------------------------------
# 목적: gtsrcmaps의 input 중 하나인 exposure cube가 Cholis가 fit에 사용한
#       그 자체와 일치하는지 검증. V35에서 남은 단일 가설인 "templates의
#       spatial structure 차이"의 시작점.
# 데이터:
#   ours    : ./GC_analysis_sanghwan/GC_expcube_center_12yr_front_clean.fits
#             (gtexpcube2 출력, cm² s units)
#   Cholis  : /home/sanghwan/FermiLAT/Sanghwan/Templates/
#             ExposureDEnergyMapsInnerGalaxy_01degBins_60x60_E_50-814009_MeV_Cartesian.fits
#             (Zenodo, cm² s sr / GeV units, 38 energy bins, 0.1°)
# ============================================================================
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from pathlib import Path

OURS_EXP   = Path("./GC_analysis_sanghwan/GC_expcube_center_12yr_front_clean.fits")
CHOLIS_EXP = Path("../GCE_TEMPLATES_FILES_v3/GALACTIC_DIFFUSE_EMISSION_MAPS_0p25deg/"
                  "ExposureDEnergyMapsInnerGalaxy_01degBins_60x60_E_50-814009_MeV_Cartesian.fits")
OUTPUT_PNG = "./V36_exposure_audit.png"
sr_per_pix = (np.pi/180.0 * 0.1) ** 2

print("="*92)
print(" V36 — Exposure cube audit: ours vs Cholis Zenodo")
print("="*92)

# ---- 0. File existence + structure probe ----------------------------------
for label, p in [("ours", OURS_EXP), ("Cholis", CHOLIS_EXP)]:
    if not p.exists():
        print(f"  ✗ {label}: NOT FOUND at {p}")
    else:
        with fits.open(p) as h:
            print(f"  ✓ {label}: {p}")
            print(f"    HDUs: {[hdu.name for hdu in h]}")
            for hdu in h:
                if hdu.data is not None:
                    print(f"      [{hdu.name}] shape={hdu.data.shape}, "
                          f"dtype={hdu.data.dtype}")

assert OURS_EXP.exists() and CHOLIS_EXP.exists(), "필수 파일 누락"

# ---- 1. 데이터 로드 ----------------------------------------------------------
with fits.open(OURS_EXP) as h:
    ours_raw = h[0].data
    # ENERGIES 또는 EBOUNDS 확인
    ours_E_MeV = None
    for hdu in h:
        if hdu.name == 'ENERGIES':
            ours_E_MeV = np.asarray(hdu.data[hdu.columns.names[0]], dtype=float)
            break
        elif hdu.name == 'EBOUNDS':
            eb = hdu.data
            ours_E_MeV = np.sqrt(eb['E_MIN'] * eb['E_MAX']) / 1e3
            break

with fits.open(CHOLIS_EXP) as h:
    chol_raw = h[0].data
    # Cholis는 38 bin grid을 사용. README의 binning 적용 (centers)
    chol_E_MeV = np.array([
        50.0, 64.98283, 84.4553638962, 109.762971093, 142.654169817,
        185.40143332, 240.958196464, 313.162910358, 407.004243322,
        528.965751061, 687.473829541, 893.47989989, 1161.21704886,
        1509.18340158, 1961.42016848, 2549.17266733, 3313.04908164,
        4305.82610508, 5596.09531592, 7273.00221156, 9452.40532607,
        12284.8809679, 15966.1266302, 20750.4818513, 26968.5006912,
        35049.7899155, 45552.6907923, 59202.8552359, 76943.3815462,
        99999.9736528, 129965.625758, 168910.683289, 219525.884347,
        285308.264463, 370802.768944, 481916.265956, 626325.655697,
        814008.272176])
    chol_dE_MeV = np.array([  # README의 Emin/Emax에서 계산
        57.0013-43.8587, 74.082-57.0013, 96.2812-74.082, 125.133-96.2812,
        162.629-125.133, 211.362-162.629, 274.698-211.362, 357.014-274.698,
        463.995-357.014, 603.034-463.995, 783.737-603.034, 1018.59-783.737,
        1323.82-1018.59, 1720.51-1323.82, 2236.07-1720.51, 2906.12-2236.07,
        3776.96-2906.12, 4908.75-3776.96, 6379.69-4908.75, 8291.4-6379.69,
        10776.0-8291.4, 14005.1-10776.0, 18201.8-14005.1, 23656.1-18201.8,
        30744.8-23656.1, 39957.6-30744.8, 51931.2-39957.6, 67492.7-51931.2,
        87717.4-67492.7, 114002.0-87717.4, 148164.0-114002.0,
        192562.0-148164.0, 250265.0-192562.0, 325258.0-250265.0,
        422724.0-325258.0, 549396.0-422724.0, 714027.0-549396.0,
        927989.0-714027.0])
    chol_dE_GeV = chol_dE_MeV / 1000.0

print(f"\n[grids] ours: {len(ours_E_MeV) if ours_E_MeV is not None else '?'} bins")
print(f"        Cholis: {len(chol_E_MeV)} bins (Zenodo 38-bin native)")
print(f"        ours raw shape: {ours_raw.shape}, Cholis raw shape: {chol_raw.shape}")

# ---- 2. 단위 정렬 -----------------------------------------------------------
#   ours: cm² s (per pixel, gtexpcube2 default)
#   Cholis: cm² s sr / GeV (per pixel, divided by bin width)
# 비교를 위해 둘 다 "cm² s" (per pixel, energy 적분 안 함) 단위로 통일.
# Cholis × sr_per_pix^(-1)? 아니다, exposure는 sr-independent.
# 정확히: Cholis cube * dE = cm² s sr (per pixel and bin)
#         ours cube       = cm² s (per pixel and energy point, sr 빠짐)
# Cholis는 per GeV이므로 dE 곱하면 "bin-integrated cm² s sr per pixel"
# 한편 ours는 effective area × livetime이라 단위가 cm² s 이고 sr는 픽셀이 sr_per_pix를 차지
# 즉 ours에 sr_per_pix를 곱하면 Cholis와 같은 의미 단위가 됨

chol_int = chol_raw * chol_dE_GeV[:, None, None]   # cm² s sr (per pixel, per bin)
ours_int = ours_raw * sr_per_pix                    # cm² s sr (per pixel, energy point)
print(f"\n[units] ours × sr_per_pix → {ours_int.shape} (cm²·s·sr per pixel)")
print(f"        Cholis × dE        → {chol_int.shape} (cm²·s·sr per pixel per bin)")

# ---- 3. Energy grid 매핑 (17 bin ↔ 38 bin) ---------------------------------
# 우리 grid: Cholis Table III 14 bin + DM ext 3 bin = 17 bin
# Cholis Zenodo 38 bin → 14 bin grouping (README):
#   our 0-10 ↔ chol 7-17, 1:1
#   our 11   ↔ chol 18+19+20  (geom mean)
#   our 12   ↔ chol 21+22+23
#   our 13   ↔ chol 24+25+26
# DM extension은 14 이상이라 우선 0-13만 비교
BIN_MAP = {  # (our_idx, [chol_idx_list])
    0: [7], 1: [8], 2: [9], 3: [10], 4: [11], 5: [12], 6: [13],
    7: [14], 8: [15], 9: [16], 10: [17],
    11: [18, 19, 20], 12: [21, 22, 23], 13: [24, 25, 26],
}

print(f"\n[bin map] 14 bin (ours 0-13) ↔ Cholis 38-bin (idx 7-26)")
print(f"  {'our':>3} {'E_our[GeV]':>11} | {'chol idx':>14} {'E_chol[GeV]':>12} "
      f"{'rel_err':>9}")
for io, chi_list in BIN_MAP.items():
    Eo = ours_E_MeV[io] / 1000 if ours_E_MeV is not None else np.nan
    Ec_centers = chol_E_MeV[chi_list] / 1000
    Ec_geom = np.exp(np.mean(np.log(Ec_centers)))
    print(f"  {io:3d} {Eo:11.3f} | {str(chi_list):>14} {Ec_geom:12.3f} "
          f"{(Eo-Ec_geom)/Ec_geom*100:+9.2f}%")

# ---- 4. Bin-grouped Cholis exposure → 14 bin -------------------------------
# Cholis 38-bin을 group합으로 합쳐서 우리와 동일 14-bin 만들기
chol_grouped = np.zeros((14,) + chol_int.shape[1:])
for io, chi_list in BIN_MAP.items():
    chol_grouped[io] = chol_int[chi_list].sum(axis=0)  # cm² s sr per pixel

# 우리 exposure 17 bin 중 0-13만 사용
ours_14 = ours_int[:14]
print(f"\n[reduced] ours_14 {ours_14.shape}, chol_grouped {chol_grouped.shape}")

# ---- 5. Per-bin spatial mean ratio (전체 영역 + inner ±10°) ----------------
NY, NX = chol_grouped.shape[1:]
cy, cx = NY//2, NX//2
inner_slice = (slice(cy-100, cy+100), slice(cx-100, cx+100))  # ±10° in 0.1° pix

print(f"\n  {'idx':>3} {'E[GeV]':>10} | {'ours_mean':>11} {'chol_mean':>11} "
      f"{'ratio(all)':>11} | {'ours_inner':>11} {'chol_inner':>11} {'ratio(±10°)':>12}")
print("-"*100)
for i in range(14):
    om_all = ours_14[i].mean()
    cm_all = chol_grouped[i].mean()
    om_in  = ours_14[i][inner_slice].mean()
    cm_in  = chol_grouped[i][inner_slice].mean()
    r_all  = om_all / cm_all if cm_all > 0 else np.nan
    r_in   = om_in  / cm_in  if cm_in  > 0 else np.nan
    Eo = ours_E_MeV[i] / 1000 if ours_E_MeV is not None else np.nan
    print(f"  {i:3d} {Eo:10.3f} | {om_all:11.3e} {cm_all:11.3e} {r_all:11.4f} "
          f"| {om_in:11.3e} {cm_in:11.3e} {r_in:12.4f}")

# ---- 6. Spatial ratio map at 3 representative bins ------------------------
fig, axes = plt.subplots(2, 3, figsize=(15, 9), dpi=110)
bins_to_plot = [0, 5, 11]   # 0.31 GeV, 1.16 GeV, 7.3 GeV
for j, ib in enumerate(bins_to_plot):
    ratio = np.where(chol_grouped[ib] > 0, ours_14[ib] / chol_grouped[ib], np.nan)
    ax = axes[0, j]
    im = ax.imshow(ratio, origin='lower', cmap='RdBu_r',
                   vmin=0.9, vmax=1.1, extent=[-30, 30, -30, 30])
    ax.set_title(f'bin {ib}: ratio (ours/Cholis) — E≈{ours_E_MeV[ib]/1000:.2f} GeV'
                 if ours_E_MeV is not None else f'bin {ib}')
    ax.set_xlabel('l [deg]'); ax.set_ylabel('b [deg]')
    plt.colorbar(im, ax=ax)
    
    # Latitude profile at l=0
    ax = axes[1, j]
    b_axis = np.linspace(-30, 30, NY)
    ax.plot(b_axis, ours_14[ib][:, cx], '-', color='C0', lw=2, label='ours')
    ax.plot(b_axis, chol_grouped[ib][:, cx], '--', color='C1', lw=2, label='Cholis')
    ax.set_xlabel('b [deg]'); ax.set_ylabel('exposure (cm²·s·sr/pix)')
    ax.set_title(f'bin {ib}: latitude profile @ l=0')
    ax.legend(); ax.grid(alpha=0.3)
    
plt.tight_layout()
plt.savefig(OUTPUT_PNG, dpi=130, bbox_inches='tight')
plt.show()
print(f"\n[Saved] {OUTPUT_PNG}")

print("\n" + "="*92)
print(" 다음 결정 분기:")
print("="*92)
print(" • ratio 모든 bin/영역에서 1.00 ± 0.01 → exposure는 OK, B (flux maps)로 진행")
print(" • ratio가 systematic하게 다름 (예: 1.05, 0.95) → exposure 차이 = gtexpcube2 설정 차이")
print("   - IRF version (P8R3_CLEAN_V3 vs others), zmax cut, livetime cube 입력")
print(" • ratio가 spatial gradient → 우리 LTCUBE 또는 zmax cut이 다른 분포 생성")
print(" • inner-galaxy ratio ≠ outer ratio → ROI-dependent systematic")

In [ ]:
# ============================================================================
# V37 — Cholis Zenodo exposure file vs 우리 exposure header 비교
# 목적: evtype/IRF/zmax/data_range/version 등 generation parameter 확인
# ============================================================================
from astropy.io import fits
from pathlib import Path

CHOLIS_EXP = Path("../GCE_TEMPLATES_FILES_v3/GALACTIC_DIFFUSE_EMISSION_MAPS_0p25deg/"
                  "ExposureDEnergyMapsInnerGalaxy_01degBins_60x60_E_50-814009_MeV_Cartesian.fits")
OURS_EXP   = Path("./GC_analysis_sanghwan/GC_expcube_center_12yr_front_clean.fits")

KEYS_OF_INTEREST = [
    'TELESCOP', 'INSTRUME', 'CREATOR', 'DATE', 'DATE-OBS', 'DATE-END',
    'TSTART', 'TSTOP', 'MJDREFI', 'MJDREFF',
    'PASS_VER', 'IRF', 'IRFVER',
    'EVCLASS', 'EVCLSMIN', 'EVCLSMAX', 'EVTYPE', 'EVT_TYPE',
    'ZMAX', 'ZMIN', 'THMAX', 'THMIN', 'COSTHMIN', 'COSTHMAX',
    'NSIDE', 'PIXTYPE', 'BUNIT',
    'GTI_NAME', 'LTNAME', 'EDISP', 'EDISP_BINS',
    'ROCK_MAX', 'ABS_ROCK_ANGLE',
    'CHECKSUM', 'DATASUM',
    'COMMENT', 'HISTORY',
]

def dump_header(path, label):
    print("\n" + "="*84)
    print(f" {label}: {path.name}")
    print("="*84)
    if not path.exists():
        print(f"  ✗ NOT FOUND: {path}")
        return
    with fits.open(path) as h:
        for hi, hdu in enumerate(h):
            print(f"\n  --- HDU {hi}: {hdu.name} ({type(hdu).__name__}) ---")
            print(f"      data: shape={hdu.data.shape if hdu.data is not None else None}")
            print(f"      header has {len(hdu.header)} cards")
            # 관심 키워드만 출력
            for k in KEYS_OF_INTEREST:
                if k in hdu.header:
                    val = hdu.header[k]
                    if isinstance(val, str) and len(val) > 70:
                        val = val[:70] + '...'
                    print(f"        {k:>20} = {val!r}")
            # COMMENT, HISTORY는 별도로 (보통 multi-card)
            for k in ['COMMENT', 'HISTORY']:
                if k in hdu.header:
                    cards = hdu.header[k]
                    cards_list = list(cards) if hasattr(cards, '__iter__') else [cards]
                    for j, c in enumerate(cards_list[:8]):
                        print(f"        {k:>20}[{j}] = {str(c)[:70]}")
                    if len(cards_list) > 8:
                        print(f"        {k:>20} ... ({len(cards_list)} total)")
            # 그 외 알아두면 좋은 키들 전부 (size 5 이하로 제한)
            other_keys = [k for k in hdu.header.keys()
                          if k not in KEYS_OF_INTEREST
                          and k not in ('COMMENT', 'HISTORY', '')
                          and not k.startswith(('NAXIS', 'CTYPE', 'CRPIX', 'CRVAL',
                                                'CDELT', 'CUNIT', 'TTYPE', 'TFORM',
                                                'TUNIT', 'PC', 'CD'))]
            if other_keys:
                print(f"      [other non-WCS keys]")
                for k in other_keys[:20]:
                    val = hdu.header[k]
                    if isinstance(val, str) and len(val) > 50:
                        val = val[:50] + '...'
                    print(f"        {k:>20} = {val!r}")

dump_header(CHOLIS_EXP, "Cholis Zenodo")
dump_header(OURS_EXP,   "Ours")

In [ ]:
# ============================================================================
# V37 — Cholis Zenodo Pi0/Brem/ICS flux map vs 우리 MapCube (Model I)
# 단위: Cholis = E²·dΦ/dE [GeV cm⁻² s⁻¹ sr⁻¹], 우리 = dΦ/dE [MeV⁻¹ cm⁻² s⁻¹ sr⁻¹]
# Grid: 둘 다 0.25° pixels, 240×240, 38 native energy bins (Zenodo binning)
# Model I → Cholis code 'bs' (NAMING_CONVENTION)
# ============================================================================
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from pathlib import Path

CHOL_DIR  = Path("../GCE_TEMPLATES_FILES_v3/GALACTIC_DIFFUSE_EMISSION_MAPS_0p25deg")
OURS_DIR  = Path("./MapCubes")
M_PAPER   = "I"
M_CHOL    = "bs"
OUTPUT_PNG= f"./V37_flux_map_audit_Model{M_PAPER}.png"

print("="*84)
print(f" V37 — Flux map audit | Cholis '{M_CHOL}' (paper Model {M_PAPER}) vs ours")
print("="*84)

# ---- 0. File probe ---------------------------------------------------------
chol_files = {
    'pi0':    CHOL_DIR / f"pi0_{M_CHOL}_Map_flux_E_50-814008_MeV_InnerGalaxy_60x60.fits",
    'bremss': CHOL_DIR / f"bremss_{M_CHOL}_Map_flux_E_50-814008_MeV_InnerGalaxy_60x60.fits",
    'ICS':    CHOL_DIR / f"ICS_{M_CHOL}_Map_flux_E_50-814008_MeV_InnerGalaxy_60x60.fits",
}
print("\n[Cholis Zenodo files]")
for label, p in chol_files.items():
    if not p.exists():
        print(f"  ✗ {label}: MISSING {p}")
    else:
        with fits.open(p) as h:
            print(f"  ✓ {label}: shape={h[0].data.shape}, dtype={h[0].data.dtype}, "
                  f"BUNIT={h[0].header.get('BUNIT','?')}, "
                  f"CDELT1={h[0].header.get('CDELT1','?')}")

# ---- Find our MapCube files for Model I ------------------------------------
print(f"\n[Our MapCubes/ contents — looking for Model {M_PAPER}]")
if not OURS_DIR.exists():
    print(f"  ✗ Directory not found: {OURS_DIR}")
else:
    all_files = sorted(OURS_DIR.glob("*.fits"))
    print(f"  Total .fits files: {len(all_files)}")
    candidates = {}
    for comp in ['pi0', 'pion', 'bremss', 'ics', 'ICS']:
        comp_files = [f for f in all_files
                      if comp in f.name.lower() and
                      (f"_{M_PAPER}_" in f.name or f"_{M_PAPER}." in f.name or
                       f"model{M_PAPER}_" in f.name or f.name.endswith(f"_{M_PAPER}.fits"))]
        for f in comp_files:
            print(f"    [{comp}] {f.name}")
        if comp_files and comp.lower() in ('pi0', 'pion'):
            candidates['pi0'] = comp_files[0]
        elif comp_files and comp.lower() == 'bremss':
            candidates['bremss'] = comp_files[0]
        elif comp_files and comp.lower() in ('ics',):
            candidates['ICS'] = comp_files[0]
    # Fallback: show first 8 files if no obvious match
    if not candidates:
        print("  ⚠ No automatic match. First 10 files:")
        for f in all_files[:10]:
            print(f"    {f.name}")
        print("  → 파일명 알려주시면 ours_files dict에 수동 지정 가능")
    ours_files = candidates

# Manual override: V8.1 MapCubes/ naming convention
ours_files = {
    'pi0':    OURS_DIR / f"pi0_mapcube_model{M_PAPER}.fits",
    'bremss': OURS_DIR / f"bremss_mapcube_model{M_PAPER}.fits",
    'ICS':    OURS_DIR / f"ics_mapcube_model{M_PAPER}.fits",
}

assert ours_files and all(p.exists() for p in ours_files.values()), \
    "우리 MapCube 파일을 찾지 못함 — 위 출력 보고 수동 지정"

# ---- 1. Load + unit conversion ---------------------------------------------
def load_chol(p):
    """Cholis Zenodo: E²·dΦ/dE [GeV cm⁻² s⁻¹ sr⁻¹]"""
    with fits.open(p) as h:
        data = h[0].data.astype(np.float64)        # (38, 240, 240)
        # ENERGIES extension에서 E grid 추출
        E_MeV = None
        for hdu in h:
            if hdu.name == 'ENERGIES':
                E_MeV = np.asarray(hdu.data[hdu.columns.names[0]], dtype=float)
                break
    return data, E_MeV

def load_ours(p):
    """우리 MapCube: dΦ/dE [MeV⁻¹ cm⁻² s⁻¹ sr⁻¹] (V8.1 SUMMARY)"""
    with fits.open(p) as h:
        data = h[0].data.astype(np.float64)
        E_MeV = None
        for hdu in h:
            if hdu.name == 'ENERGIES':
                E_MeV = np.asarray(hdu.data[hdu.columns.names[0]], dtype=float)
                break
    return data, E_MeV

chol_data = {}
ours_data = {}
for k in ['pi0', 'bremss', 'ICS']:
    chol_data[k], E_chol_MeV = load_chol(chol_files[k])
    ours_data[k], E_ours_MeV = load_ours(ours_files[k])
print(f"\n[Loaded]")
print(f"  Cholis E grid: {len(E_chol_MeV)} bins, "
      f"first={E_chol_MeV[0]:.3f} MeV, last={E_chol_MeV[-1]:.1f} MeV")
print(f"  Ours   E grid: {len(E_ours_MeV)} bins, "
      f"first={E_ours_MeV[0]:.3f} MeV, last={E_ours_MeV[-1]:.1f} MeV")
print(f"  Cholis shape: {chol_data['pi0'].shape}, ours shape: {ours_data['pi0'].shape}")

# Energy grid 검증
if len(E_chol_MeV) == len(E_ours_MeV):
    rel = np.abs(E_chol_MeV - E_ours_MeV) / E_chol_MeV
    print(f"  Energy grid relative diff: max={rel.max():.4f}, mean={rel.mean():.4f}")

# 단위 변환: 우리 dΦ/dE [MeV⁻¹] × E²[MeV²] × 1e-3 [GeV/MeV] = GeV cm⁻²s⁻¹sr⁻¹
# Cholis는 이미 GeV cm⁻²s⁻¹sr⁻¹ 단위
E_MeV = E_chol_MeV  # 사용
ours_E2 = {}
for k in ['pi0', 'bremss', 'ICS']:
    ours_E2[k] = ours_data[k] * E_MeV[:, None, None]**2 * 1e-3  # → GeV cm⁻²s⁻¹sr⁻¹

# Spatial assertion
assert chol_data['pi0'].shape == ours_E2['pi0'].shape, \
    f"Shape mismatch: {chol_data['pi0'].shape} vs {ours_E2['pi0'].shape}"

# ---- 2. Per-bin ratio: full 60×60 mean + inner ±10° mean -------------------
NY, NX = chol_data['pi0'].shape[1:]
cy, cx = NY//2, NX//2
inner = (slice(cy-40, cy+40), slice(cx-40, cx+40))  # ±10° in 0.25° pix = 40 pix

print("\n[Per-bin ratio: ours / Cholis] (after unit conversion to GeV cm⁻²s⁻¹sr⁻¹)")
print(f"{'idx':>3} {'E[GeV]':>8} | {'pi0 ratio':>10} {'brem ratio':>10} {'ICS ratio':>10}"
      f" | {'pi0 inner':>10} {'brem inner':>10} {'ICS inner':>10}")
print("-"*100)
# Show bins 7-26 (14-bin grid의 0-13에 해당)
for i in range(38):
    if i < 5 or i > 30: continue  # 의미 있는 에너지 영역만
    rats_all = {}; rats_in = {}
    for k in ['pi0', 'bremss', 'ICS']:
        c_all = chol_data[k][i].mean()
        o_all = ours_E2[k][i].mean()
        c_in  = chol_data[k][i][inner].mean()
        o_in  = ours_E2[k][i][inner].mean()
        rats_all[k] = o_all/c_all if c_all > 0 else np.nan
        rats_in[k]  = o_in/c_in   if c_in > 0  else np.nan
    print(f"{i:3d} {E_MeV[i]/1000:8.3f} | {rats_all['pi0']:10.4f} {rats_all['bremss']:10.4f} "
          f"{rats_all['ICS']:10.4f} | {rats_in['pi0']:10.4f} {rats_in['bremss']:10.4f} "
          f"{rats_in['ICS']:10.4f}")

print("  ★ ratio ≈ 1.000 ± 0.01 → 우리 MapCube = Cholis Zenodo (단순 단위 변환)")
print("  ★ ratio ≠ 1 → MapCube 생성 단계에 문제 (보간/binning/단위)")
print("  ★ inner != all → spatial-dependent 차이, 특정 영역에 systematic")

# ---- 3. Spatial ratio map @ 3 bins + latitude profile ---------------------
fig, axes = plt.subplots(3, 3, figsize=(15, 13), dpi=110)
bins_to_plot = [7, 12, 17]  # ~0.31, ~1.16, ~4.3 GeV
for j, ib in enumerate(bins_to_plot):
    for r, k in enumerate(['pi0', 'bremss', 'ICS']):
        ax = axes[r, j]
        with np.errstate(divide='ignore', invalid='ignore'):
            ratio = np.where(chol_data[k][ib] > 0,
                             ours_E2[k][ib] / chol_data[k][ib], np.nan)
        # auto-scale vmin/vmax to actual range
        valid = ratio[np.isfinite(ratio)]
        if len(valid) > 0:
            vmin, vmax = np.percentile(valid, [5, 95])
        else:
            vmin, vmax = 0.9, 1.1
        im = ax.imshow(ratio, origin='lower', cmap='RdBu_r',
                       vmin=vmin, vmax=vmax, extent=[-30, 30, -30, 30])
        ax.set_title(f'{k}: ratio bin {ib} (E≈{E_MeV[ib]/1000:.2f} GeV)\n'
                     f'vmin={vmin:.3f}, vmax={vmax:.3f}')
        ax.set_xlabel('l [deg]'); ax.set_ylabel('b [deg]')
        plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.savefig(OUTPUT_PNG, dpi=130, bbox_inches='tight')
plt.show()
print(f"\n[Saved] {OUTPUT_PNG}")

print("\n" + "="*84)
print(" 분기:")
print("="*84)
print(" • 모든 ratio ≈ 1.000 → MapCube 깨끗. c_gas 비대칭 원인은 gtsrcmaps 단계")
print("                       (V36의 exposure 60% 차이 또는 PSF convolution)")
print(" • ratio ≠ 1 (균일) → MapCube 단위 변환 또는 normalization 오류")
print(" • ratio가 spatial gradient → 우리 보간/resampling이 spatial structure 변형")
print(" • inner≠all → inner galaxy에서 fit이 보는 영역의 정보가 왜곡됨")

In [ ]:
# ============================================================================
# V38 — Exposure cube environment audit
# 목적: 우리 exposure 60% 부족분 추적. ScienceTools version, gtexpcube2 par,
#       LTCUBE info, exposure full header. Cholis(P8v27h5b5c8) 와 비교 정보 수집.
# ============================================================================
import subprocess
import os
from pathlib import Path
from astropy.io import fits

print("="*84)
print(" V38 — Exposure environment audit")
print("="*84)

# ---- 1. ScienceTools version ----------------------------------------------
print("\n[1] ScienceTools version")
for cmd in ['fermitools-info', 'gtvcut --version', 'gtbin --version',
            'gtexpcube2 --version']:
    try:
        res = subprocess.run(cmd.split(), capture_output=True, text=True, timeout=10)
        out = (res.stdout + res.stderr).strip()
        if out:
            print(f"  $ {cmd}")
            for line in out.split('\n')[:5]:
                print(f"    {line}")
    except Exception as e:
        print(f"  {cmd}: {type(e).__name__}: {str(e)[:60]}")

# fermitools-info가 가장 결정적
try:
    import fermitools
    print(f"  fermitools module: {fermitools.__version__ if hasattr(fermitools, '__version__') else 'no __version__'}")
except ImportError:
    pass

# conda env 정보
try:
    res = subprocess.run(['conda', 'list', 'fermitools'],
                         capture_output=True, text=True, timeout=10)
    print(f"  conda list fermitools:")
    for line in res.stdout.split('\n'):
        if 'fermitools' in line.lower() or 'fermi' in line.lower():
            print(f"    {line}")
except Exception as e:
    print(f"  conda not available: {e}")

# ---- 2. gtexpcube2.par (현재 working par file) -----------------------------
print("\n[2] gtexpcube2.par contents")
par_locations = [
    Path("./gtexpcube2.par"),
    Path(os.environ.get('PFILES', '').split(';')[0] if os.environ.get('PFILES') else '/dev/null') / "gtexpcube2.par",
    Path.home() / ".pfiles/gtexpcube2.par",
]
for pp in par_locations:
    if pp.exists():
        print(f"  Found: {pp}")
        with open(pp) as f:
            for line in f:
                if not line.strip().startswith('#') and ',' in line:
                    print(f"    {line.rstrip()}")
        break
else:
    print(f"  ✗ gtexpcube2.par not found in expected locations")
    print(f"    PFILES env: {os.environ.get('PFILES', '(unset)')}")

# ---- 3. LTCUBE info -------------------------------------------------------
print("\n[3] LTCUBE info")
ltcube_candidates = [
    Path("./GC_analysis_sanghwan/Allsky_ltcube_12yr_front_clean.fits"),
    Path("./GCE_12yr_ltcube.fits"),
]
for lt in ltcube_candidates:
    if lt.exists():
        print(f"  ✓ {lt}")
        with fits.open(lt) as h:
            for hi, hdu in enumerate(h):
                shape = hdu.data.shape if hdu.data is not None else None
                print(f"    HDU {hi} {hdu.name}: shape={shape}")
                for k in ['DATE-OBS', 'DATE-END', 'TSTART', 'TSTOP',
                          'TELAPSE', 'ONTIME', 'LIVETIME',
                          'NDSKEYS', 'EVCLASS', 'EVTYPE',
                          'COSTHMIN', 'COSTHMAX', 'PHIBINS',
                          'DSTYP1', 'DSVAL1', 'DSTYP2', 'DSVAL2',
                          'DSTYP3', 'DSVAL3', 'DSTYP4', 'DSVAL4']:
                    if k in hdu.header:
                        print(f"      {k} = {hdu.header[k]}")
        break
else:
    print(f"  ✗ No LTCUBE found")

# ---- 4. 우리 exposure cube full PRIMARY header dump -----------------------
print("\n[4] Our exposure cube — full PRIMARY header")
EXP = Path("./GC_analysis_sanghwan/GC_expcube_center_12yr_front_clean.fits")
if EXP.exists():
    with fits.open(EXP) as h:
        cards = h[0].header.cards
        # WCS 카드는 일단 제외하고 보기 (이미 V36에서 확인)
        skip_prefixes = ('NAXIS', 'CTYPE', 'CRPIX', 'CRVAL', 'CDELT',
                         'CUNIT', 'PC', 'CD', 'EXTEND', 'SIMPLE', 'BITPIX')
        for card in cards:
            if any(card.keyword.startswith(p) for p in skip_prefixes):
                continue
            if card.keyword in ('', 'COMMENT', 'HISTORY'):
                if card.keyword == 'HISTORY':
                    val = str(card.value)[:75]
                    print(f"  HISTORY: {val}")
                continue
            print(f"  {card.keyword:<15} = {card.value!r}")
else:
    print(f"  ✗ Not found: {EXP}")

# ---- 5. bin_definitions 확인 (38 vs 14 vs 17 bin 검증) ---------------------
print("\n[5] bin_definitions used for our gtexpcube2")
bdef_candidates = [
    Path("./GC_analysis_sanghwan/bin_definitions.fits"),
    Path("./GC_analysis_sanghwan/bin_definitions.txt"),
    Path("./bin_definitions.fits"),
]
for bp in bdef_candidates:
    if bp.exists():
        print(f"  ✓ {bp}")
        if bp.suffix == '.txt':
            with open(bp) as f:
                content = f.read()
            print(f"    {content[:500]}")
        else:
            with fits.open(bp) as h:
                for hi, hdu in enumerate(h):
                    if hdu.data is not None and len(hdu.data) < 50:
                        print(f"    HDU {hi} ({hdu.name}): {len(hdu.data)} rows")
                        if hi > 0:
                            print(f"    First 5 rows: {hdu.data[:5]}")
                            print(f"    Last 3 rows:  {hdu.data[-3:]}")
        break
else:
    print(f"  ✗ No bin_definitions found")

print("\n" + "="*84)
print(" 비교 포인트 (Cholis paper 본문 기준):")
print("="*84)
print(" • Cholis ScienceTools: P8v27h5b5c8")
print(" • Cholis evtype = 1 (FRONT), evclass = 256 (CLEAN), zmax = 100°")
print(" • Cholis filters: DATA_QUAL==1, LAT_CONFIG==1, ABS(ROCK_ANGLE) < 52")
print(" • Cholis bin: 14 bins (0.275 - 51.9 GeV), CCW Table III")
print(" • 우리는 ScienceTools 무엇인지 확인 필요. 60% ratio가 P8v27h5b5c8 →")
print("   다른 fermitools 버전에 의한 IRF 차이일 가능성 있음")

In [ ]:
# ============================================================================
# V39 — gtsrcmaps spatial structure audit
# 목적: V35 Sec 1이 보인 total-flux conservation (~1.0) 위에서, PSF가 만드는
#       spatial redistribution을 영역별/component별로 측정. c_gas의 잔여 비대칭
#       (exposure scaling으로 설명 안 되는 부분)이 PSF 작동 방식의 차이에서
#       오는지 확인.
# ============================================================================
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from pathlib import Path

SANG = Path("./GC_analysis_sanghwan")
OUTPUT_PNG = "./V39_psf_spatial_audit.png"

# 비교할 3개 모델 (V12d의 Group A/B/C 대표)
MODELS = {
    'I'    : 'C (c_gas=0.40, worst)',
    'X'    : 'B (intermediate)',
    'XLIX' : 'A (best agreement)',
}
front = "_front"

# Slice: 600×600 → 400×400 fit region
SLICE_Y, SLICE_X = slice(100, 500), slice(100, 500)
NY = NX = 400
cy, cx = NY//2, NX//2

# 영역 정의 (0.1° pixel → ±5°=50pix, ±15°=150pix)
regions = {
    'inner ±5°':   (slice(cy-50, cy+50),   slice(cx-50, cx+50)),
    'mid 5-15°':   None,    # ring 1, 아래에서 mask로 생성
    'outer 15-20°':None,    # ring 2
    'all 40×40':   (slice(None), slice(None)),
}
# Ring mask 생성
yy, xx = np.ogrid[:NY, :NX]
r = np.sqrt((yy - cy)**2 + (xx - cx)**2)  # in 0.1° pixels
ring_5_15  = (r >= 50)  & (r < 150)
ring_15_20 = (r >= 150) & (r < 200)

print("="*100)
print(" V39 — gtsrcmaps spatial structure audit | 3 models, 5 components")
print("="*100)

def load_pair(model, stem, model_in_name=True):
    """conv + no_conv 쌍을 400×400으로 slice해서 반환."""
    base = f"GC_{stem}_model{model}_12yr{front}_clean" if model_in_name \
           else f"GC_{stem}_model_12yr{front}_clean"
    c  = fits.getdata(SANG / f"{base}.fits")[:, SLICE_Y, SLICE_X]
    nc = fits.getdata(SANG / f"{base}_no_convol.fits")[:, SLICE_Y, SLICE_X]
    return c, nc

# bin 4 (≈0.89 GeV, GCE peak 근처) 와 bin 11 (≈7.3 GeV) 두 곳에서 측정
ENERGY_BINS_TO_AUDIT = [4, 11]  # peak, high

# ---- 1. Region-wise conv/no_conv ratio per (model, component, energy bin) ----
print("\n[1] Region-wise conv/no_conv ratio (PSF redistribution by region)")
print("    Total ≈ 1.0 (V35 confirmed). Region-by-region 차이가 PSF spread의 크기.")
print("-"*100)
results = {}  # (model, comp, bin, region) → ratio
for model in MODELS:
    print(f"\n  Model {model} ({MODELS[model]}):")
    pion_c, pion_nc     = load_pair(model, "pion")
    brem_c, brem_nc     = load_pair(model, "bremss")
    ics_c,  ics_nc      = load_pair(model, "ics")
    gce_c,  gce_nc      = load_pair(model, "GCE",          model_in_name=False)
    bub_c,  bub_nc      = load_pair(model, "fermi_bubble", model_in_name=False)
    pb_c, pb_nc = pion_c + brem_c, pion_nc + brem_nc
    components = {
        'Pi0+Brem': (pb_c,  pb_nc),
        'ICS':      (ics_c, ics_nc),
        'GCE':      (gce_c, gce_nc),
        'Bubble':   (bub_c, bub_nc),
    }
    for ib in ENERGY_BINS_TO_AUDIT:
        print(f"    bin {ib} ({'~0.89' if ib==4 else '~7.3'} GeV):")
        print(f"      {'component':<10} {'all':>10} {'inner±5°':>10} "
              f"{'ring 5-15°':>11} {'ring 15-20°':>12}")
        for cname, (c_map, nc_map) in components.items():
            r_all  = c_map[ib].sum() / nc_map[ib].sum()
            r_in   = c_map[ib][cy-50:cy+50, cx-50:cx+50].sum() / \
                     nc_map[ib][cy-50:cy+50, cx-50:cx+50].sum()
            r_mid  = c_map[ib][ring_5_15].sum() / nc_map[ib][ring_5_15].sum()
            r_out  = c_map[ib][ring_15_20].sum() / nc_map[ib][ring_15_20].sum()
            print(f"      {cname:<10} {r_all:10.4f} {r_in:10.4f} {r_mid:11.4f} "
                  f"{r_out:12.4f}")
            results[(model, cname, ib, 'all')]   = r_all
            results[(model, cname, ib, 'inner')] = r_in
            results[(model, cname, ib, 'mid')]   = r_mid
            results[(model, cname, ib, 'outer')] = r_out

print("\n  ★ 모든 component에서 inner < 1, outer > 1 → PSF가 inner에서 outer로 photon spread")
print("  ★ Pi0+Brem vs ICS의 inner ratio 차이 → disk-concentrated vs spread component의 비대칭")
print("  ★ Model별 동일 component ratio 차이 → IRF/PSF 일관성, 다르면 systematic")

# ---- 2. Latitude profile (l=0 단면) at GCE-peak bin 4 ---------------------
print("\n[2] Latitude profile at l=0, bin 4 (~0.89 GeV)")
b_axis = np.arange(NY) * 0.1 - 20.0  # 0.1° pixel → ±20°

fig, axes = plt.subplots(2, 4, figsize=(20, 10), dpi=110)

for col, model in enumerate(MODELS):
    pion_c, pion_nc = load_pair(model, "pion")
    brem_c, brem_nc = load_pair(model, "bremss")
    ics_c,  ics_nc  = load_pair(model, "ics")
    pb_c, pb_nc = pion_c + brem_c, pion_nc + brem_nc

    # Profile @ l=0 (cx column)
    ib = 4
    ax = axes[0, col]
    ax.semilogy(b_axis, pb_nc[ib, :, cx], '--', color='C1', lw=1.2,
                label='Pi0+Brem (no_conv)', alpha=0.7)
    ax.semilogy(b_axis, pb_c[ib, :, cx],  '-',  color='C1', lw=2,
                label='Pi0+Brem (conv)')
    ax.semilogy(b_axis, ics_nc[ib, :, cx], '--', color='C0', lw=1.2,
                label='ICS (no_conv)', alpha=0.7)
    ax.semilogy(b_axis, ics_c[ib, :, cx],  '-',  color='C0', lw=2,
                label='ICS (conv)')
    ax.set_xlabel('b [deg]'); ax.set_ylabel('counts/pix')
    ax.set_title(f'Model {model} — lat profile @ l=0, bin 4')
    ax.legend(fontsize=8, loc='lower center'); ax.grid(alpha=0.3)
    ax.set_xlim(-10, 10)

    # conv/no_conv ratio profile (PSF redistribution map)
    ax = axes[1, col]
    with np.errstate(divide='ignore', invalid='ignore'):
        ratio_pb  = np.where(pb_nc[ib, :, cx] > 0,
                              pb_c[ib, :, cx] / pb_nc[ib, :, cx], np.nan)
        ratio_ics = np.where(ics_nc[ib, :, cx] > 0,
                              ics_c[ib, :, cx] / ics_nc[ib, :, cx], np.nan)
    ax.plot(b_axis, ratio_pb,  '-', color='C1', lw=2, label='Pi0+Brem conv/no_conv')
    ax.plot(b_axis, ratio_ics, '-', color='C0', lw=2, label='ICS conv/no_conv')
    ax.axhline(1.0, color='k', lw=0.5, ls=':')
    ax.set_xlabel('b [deg]'); ax.set_ylabel('conv / no_conv')
    ax.set_title(f'Model {model} — PSF redistribution @ l=0, bin 4')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    ax.set_xlim(-10, 10); ax.set_ylim(0, 2.5)

# 4번째 column: 영역별 ratio comparison across models
ax = axes[0, 3]
x = np.arange(4)
width = 0.25
region_labels = ['all', 'inner', 'mid', 'outer']
for j, model in enumerate(MODELS):
    pb_ratios = [results[(model, 'Pi0+Brem', 4, r)] for r in region_labels]
    ax.bar(x + j*width - width, pb_ratios, width, label=f'Model {model}')
ax.set_xticks(x); ax.set_xticklabels(region_labels)
ax.axhline(1.0, color='k', lw=0.5, ls=':')
ax.set_ylabel('conv/no_conv ratio')
ax.set_title('Pi0+Brem ratio by region, bin 4')
ax.legend(fontsize=8); ax.grid(alpha=0.3, axis='y')

ax = axes[1, 3]
for j, model in enumerate(MODELS):
    ics_ratios = [results[(model, 'ICS', 4, r)] for r in region_labels]
    ax.bar(x + j*width - width, ics_ratios, width, label=f'Model {model}')
ax.set_xticks(x); ax.set_xticklabels(region_labels)
ax.axhline(1.0, color='k', lw=0.5, ls=':')
ax.set_ylabel('conv/no_conv ratio')
ax.set_title('ICS ratio by region, bin 4')
ax.legend(fontsize=8); ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(OUTPUT_PNG, dpi=130, bbox_inches='tight')
plt.show()
print(f"\n[Saved] {OUTPUT_PNG}")

# ---- 3. Summary table: cross-model + cross-component patterns ------------
print("\n" + "="*100)
print(" SUMMARY — bin 4 (~0.89 GeV) inner±5° conv/no_conv ratio")
print("="*100)
print(f"  {'Model':<8} {'Group':<25} {'Pi0+Brem':>10} {'ICS':>10} "
      f"{'GCE':>10} {'Bubble':>10}")
for model in MODELS:
    row = f"  {model:<8} {MODELS[model]:<25}"
    for cname in ['Pi0+Brem', 'ICS', 'GCE', 'Bubble']:
        row += f" {results[(model, cname, 4, 'inner')]:>10.4f}"
    print(row)

print("\n  분기 해석:")
print("  • 모델별 ratio 동일 (e.g. ICS inner 모두 0.7) → PSF가 model 무관, IRF 일관성 OK")
print("  • 모델별 ratio 다름 → IRF/PSF가 model에 따라 다르게 작동, 추가 진단 필요")
print("  • Pi0+Brem inner << ICS inner → PSF가 sharp Pi0+Brem을 더 많이 spread,")
print("    fit이 inner counts 부족을 c_gas로 보상하려고 작아짐 (c_gas=0.40 자연 설명)")
print("  • Pi0+Brem inner ≈ ICS inner → c_gas 비대칭은 PSF가 아닌 다른 원인")

In [ ]:
# ============================================================================
# V40 — Cholis 실제 c_param 추출 + 우리 c와 정량 비교
# 목적: Fig 11 digitized post-fit / Cholis Zenodo pre-fit = c_Cholis 측정.
#       그 후 c_ours / c_Cholis ratio를 component별로 비교 → exposure 1.67x
#       scaling으로 모든 차이가 설명되는지 검증.
# 영역: 40×40 |b|>2° (Fig 11 정의), PSC mask 미적용 (paper 본문)
# 의존: V27 변수 (E, n_ebin, delta_E, SLICE_Y, SLICE_X, NY, NX, B_CENTER,
#                 DISK_HALF, disk_mask, STAGE_A_MODEL, sed_gce 등)
#      NPZ_PATH (우리 fit 결과)
# ============================================================================
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from pathlib import Path
from scipy.interpolate import interp1d

ZENODO_DIR = Path("../GCE_TEMPLATES_FILES_v3/GALACTIC_DIFFUSE_EMISSION_MAPS_0p25deg")
CHOL_FIG11 = Path("./2112_09706_Fig11_ModelI_ALL.txt")
PRIOR_DIR  = Path("./GC_analysis_sanghwan/Model")
NPZ_PATH   = Path(f"./GCE_model_{STAGE_A_MODEL}_12yr_cholis_fit.npz")
OUTPUT_PNG = f"./V40_c_extraction_Model{STAGE_A_MODEL}.png"
M_CHOL     = "bs"          # Model I → 'bs' (NAMING_CONVENTION)
BMAX       = 13

# Zenodo 38-bin → 14-bin grouping (README의 정의)
BIN_MAP = {
    0: [7], 1: [8], 2: [9], 3: [10], 4: [11], 5: [12], 6: [13],
    7: [14], 8: [15], 9: [16], 10: [17],
    11: [18, 19, 20], 12: [21, 22, 23], 13: [24, 25, 26],
}

print("="*100)
print(f" V40 — Cholis 실제 c_param 추출 | Model {STAGE_A_MODEL}, 40×40° |b|>2°")
print("="*100)

# ============ Step 1: Cholis Zenodo Pi0/Brem/ICS pre-fit ====================
# Zenodo: 0.25° pixel, 240×240, 38 bin, units = GeV cm⁻² s⁻¹ sr⁻¹ (E²·dΦ/dE)
zen = {}
zen_E_MeV = None
for k in ['pi0', 'bremss', 'ICS']:
    p = ZENODO_DIR / f"{k}_{M_CHOL}_Map_flux_E_50-814008_MeV_InnerGalaxy_60x60.fits"
    if not p.exists():
        raise FileNotFoundError(p)
    with fits.open(p) as h:
        zen[k] = h[0].data.astype(np.float64)  # (38, 240, 240)
        if zen_E_MeV is None:
            for hdu in h:
                if hdu.name == 'ENERGIES':
                    zen_E_MeV = np.asarray(hdu.data[hdu.columns.names[0]], dtype=float)
                    break
zen['Pi0+Brem'] = zen['pi0'] + zen['bremss']
print(f"\n[Zenodo loaded] shape={zen['pi0'].shape}, "
      f"E range = {zen_E_MeV[0]/1000:.3f}-{zen_E_MeV[-1]/1000:.1f} GeV")

# 40×40° |b|>2° in 0.25° pixel: 160×160 area, |b|<2° = ±8 pix from center
NY_Z, NX_Z = 240, 240
cy_Z, cx_Z = NY_Z//2, NX_Z//2
# 40×40 → ±80 pix from center; |b|<2° → ±8 pix
y_slice = slice(cy_Z - 80, cy_Z + 80)
x_slice = slice(cx_Z - 80, cx_Z + 80)
b_pix = np.arange(160) - 80    # b in 0.25° pixels from center
disk_mask_zen = np.abs(b_pix) >= 8  # |b|>=2°  (160 rows)
print(f"  40×40 slice = {y_slice.start}:{y_slice.stop}, "
      f"|b|>2° retains {disk_mask_zen.sum()}/{len(b_pix)} rows")

def zen_sed_in_region(data_3d):
    """40×40 |b|>2° 영역 spatial mean per energy bin (38 bins)"""
    sed = np.zeros(38)
    for i in range(38):
        region = data_3d[i][y_slice, x_slice]   # (160, 160)
        sed[i] = region[disk_mask_zen, :].mean()
    return sed

sed_zen = {k: zen_sed_in_region(zen[k]) for k in ['Pi0+Brem', 'ICS', 'pi0', 'bremss']}

# 38-bin → 14-bin grouping (Zenodo bin width로 가중 평균)
zen_dE_MeV = np.array([   # README의 Emax-Emin from bin boundaries
    13.1426, 17.0807, 22.1992, 28.8518, 37.4960, 48.7330, 63.3360,
    82.3160, 106.981, 139.039, 180.703, 234.853, 305.230, 396.690,
    515.560, 670.050, 870.840, 1131.79, 1470.94, 1911.71, 2484.60,
    3228.88, 4196.70, 5454.30, 7088.70, 9212.80, 11973.6, 15561.5,
    20224.7, 26284.6, 34162.0, 44398.0, 57703.0, 74993.0, 97466.0,
    126672.0, 164631.0, 213962.0])
zen_E_GeV = zen_E_MeV / 1000

def zen_to_14bin(sed_38):
    """38-bin → 14-bin: bin-width-weighted average (E²dF/dE 보존)"""
    out = np.zeros(14)
    for i14, ilist in BIN_MAP.items():
        w = zen_dE_MeV[ilist]
        out[i14] = (sed_38[ilist] * w).sum() / w.sum()
    return out

# Pre-fit (c=1) in 14-bin Cholis grid (Pi0+Brem, ICS만 Zenodo로 직접)
pref_gas_chol = zen_to_14bin(sed_zen['Pi0+Brem'])
pref_ics_chol = zen_to_14bin(sed_zen['ICS'])
print(f"\n[Cholis pre-fit (c=1) at 14-bin Table III grid]")
print(f"  bin 5 (~1.16 GeV): Pi0+Brem = {pref_gas_chol[5]:.3e}, "
      f"ICS = {pref_ics_chol[5]:.3e}")

# ============ Step 2: Bubble/Iso/GCE pre-fit ================================
# Bubble: external prior file (1407.7905)
_b = np.loadtxt(PRIOR_DIR / "bubble_constraints.txt")
bub_prior_E = interp1d(_b[:,0], _b[:,1], kind='quadratic',
                       fill_value='extrapolate')(E[:14])
# Iso: Ackermann+2015 prior, file is dN/dE, → E²·dN/dE
_i = np.loadtxt(PRIOR_DIR / "iso_constraints_full_err.txt")
iso_dnde = interp1d(_i[:,0], _i[:,1], kind='quadratic',
                    fill_value='extrapolate')(E[:14])
iso_prior_E = E[:14]**2 * iso_dnde
# GCE: V27의 sed_gce (NFW² template, c=1)는 우리 자체 계산. Cholis convention
# (γ=1.2, r_s=20kpc, ρ_solar=0.4)이 우리 V9 검증과 일치 → 같은 값 사용 가능
gce_prior   = sed_gce[:14]   # V27 변수
print(f"\n[Bubble/Iso/GCE pre-fit at 14-bin]")
print(f"  bin 5: Bub = {bub_prior_E[5]:.3e}, Iso = {iso_prior_E[5]:.3e}, "
      f"GCE = {gce_prior[5]:.3e}")

# ============ Step 3: Cholis post-fit (Fig 11 digitized) ====================
cd = np.loadtxt(CHOL_FIG11)
E_chol = cd[:, 0]
chol_post = {n: cd[:, 1+3*k] for k, n in enumerate(["Pi0p", "ICS", "Bub", "Iso", "GCE"])}
def chol_at(name, E_target):
    return 10**np.interp(np.log10(E_target), np.log10(E_chol),
                         np.log10(np.clip(chol_post[name], 1e-15, None)))
postf_gas_chol = chol_at('Pi0p', E[:14])
postf_ics_chol = chol_at('ICS',  E[:14])
postf_bub_chol = chol_at('Bub',  E[:14])
postf_iso_chol = chol_at('Iso',  E[:14])
postf_gce_chol = chol_at('GCE',  E[:14])

# ============ Step 4: c_Cholis = post / pre ================================
c_gas_chol = postf_gas_chol / pref_gas_chol
c_ics_chol = postf_ics_chol / pref_ics_chol
c_bub_chol = postf_bub_chol / bub_prior_E
c_iso_chol = postf_iso_chol / iso_prior_E
c_gce_chol = postf_gce_chol / gce_prior

# ============ Step 5: 우리 c (NPZ에서) =====================================
npz = np.load(NPZ_PATH)
fp = npz['fitted_params'].reshape(5, n_ebin) if npz['fitted_params'].ndim == 1 \
     else npz['fitted_params']
c_gas_ours = fp[0, :14]
c_ics_ours = fp[1, :14]
c_gce_ours = fp[2, :14]
c_bub_ours = fp[3, :14]
c_iso_ours = fp[4, :14]

# ============ Step 6: Ratio table ==========================================
print("\n" + "="*100)
print(" c_param 정량 비교: 우리 vs Cholis (Fig 11 + Zenodo로 추출)")
print("="*100)
print(f"  {'idx':>3} {'E[GeV]':>7} | "
      f"{'c_gas_o':>8} {'c_gas_C':>8} {'r_gas':>7} | "
      f"{'c_ics_o':>8} {'c_ics_C':>8} {'r_ics':>7} | "
      f"{'c_gce_o':>8} {'c_gce_C':>8} {'r_gce':>7}")
print("-"*100)
for i in range(14):
    r_gas = c_gas_ours[i] / c_gas_chol[i]
    r_ics = c_ics_ours[i] / c_ics_chol[i]
    r_gce = c_gce_ours[i] / c_gce_chol[i]
    print(f"  {i:3d} {E[i]:7.3f} | "
          f"{c_gas_ours[i]:8.3f} {c_gas_chol[i]:8.3f} {r_gas:7.3f} | "
          f"{c_ics_ours[i]:8.3f} {c_ics_chol[i]:8.3f} {r_ics:7.3f} | "
          f"{c_gce_ours[i]:8.3f} {c_gce_chol[i]:8.3f} {r_gce:7.3f}")

print(f"\n  {'idx':>3} {'E[GeV]':>7} | "
      f"{'c_bub_o':>8} {'c_bub_C':>8} {'r_bub':>7} | "
      f"{'c_iso_o':>8} {'c_iso_C':>8} {'r_iso':>7}")
print("-"*60)
for i in range(14):
    r_bub = c_bub_ours[i] / c_bub_chol[i]
    r_iso = c_iso_ours[i] / c_iso_chol[i]
    print(f"  {i:3d} {E[i]:7.3f} | "
          f"{c_bub_ours[i]:8.3f} {c_bub_chol[i]:8.3f} {r_bub:7.3f} | "
          f"{c_iso_ours[i]:8.3f} {c_iso_chol[i]:8.3f} {r_iso:7.3f}")

# ============ Step 7: Summary statistics ===================================
print("\n" + "="*100)
print(" SUMMARY — c_ours / c_Cholis ratio (균일한 값이면 exposure 단일 원인)")
print("="*100)
for name, our, chol in [('gas', c_gas_ours, c_gas_chol),
                          ('ics', c_ics_ours, c_ics_chol),
                          ('gce', c_gce_ours, c_gce_chol),
                          ('bub', c_bub_ours, c_bub_chol),
                          ('iso', c_iso_ours, c_iso_chol)]:
    ratio = our / chol
    # 음수/0 제거
    valid = np.isfinite(ratio) & (chol > 0.01) & (our > 0.01)
    if valid.sum() > 0:
        r = ratio[valid]
        print(f"  c_{name}: mean = {r.mean():7.3f}, median = {np.median(r):7.3f}, "
              f"std = {r.std():.3f}, min/max = {r.min():.3f}/{r.max():.3f}")

print("\n  ★ 모든 5개 component의 mean ratio가 ~1.67 (exposure 1.67x 기대값)이면")
print("    → exposure 단일 원인 확정. fermitools 재실행으로 해결 가능.")
print("  ★ component별 mean ratio가 다르면 → 추가 원인 존재")

# ============ Step 8: Plot ==================================================
fig, axes = plt.subplots(2, 3, figsize=(16, 9), dpi=110)
specs = [('Pi0+Brem', c_gas_ours, c_gas_chol, 'C1'),
         ('ICS',      c_ics_ours, c_ics_chol, 'C0'),
         ('GCE',      c_gce_ours, c_gce_chol, 'C3'),
         ('Bubble',   c_bub_ours, c_bub_chol, 'C2'),
         ('Iso',      c_iso_ours, c_iso_chol, 'C7')]
for ax, (name, ours, chol, col) in zip(axes.flat[:5], specs):
    ax.semilogx(E[:14], ours, 'o-', color=col, lw=2, ms=6, label=f'c_{name} ours')
    ax.semilogx(E[:14], chol, 's--', color=col, lw=2, ms=6, alpha=0.5,
                label=f'c_{name} Cholis')
    ax.axhline(1.0, color='k', lw=0.5, ls=':')
    ax.set_xlabel('E [GeV]'); ax.set_ylabel('c_param')
    ax.set_title(f'{name}: ours vs Cholis')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Last panel: ratio
ax = axes[1, 2]
for name, ours, chol, col in specs:
    with np.errstate(divide='ignore', invalid='ignore'):
        ratio = np.where((chol > 0.01) & (ours > 0.01), ours/chol, np.nan)
    ax.semilogx(E[:14], ratio, 'o-', color=col, lw=1.8, label=name)
ax.axhline(1.67, color='red', lw=1.2, ls='--', alpha=0.7,
           label='exposure 1.67×')
ax.axhline(1.0, color='k', lw=0.5, ls=':')
ax.set_xlabel('E [GeV]'); ax.set_ylabel('c_ours / c_Cholis')
ax.set_title('Ratio — 1.67이 균일하면 exposure 단일 원인')
ax.legend(fontsize=8); ax.grid(alpha=0.3); ax.set_ylim(0, 5)

plt.tight_layout()
plt.savefig(OUTPUT_PNG, dpi=130, bbox_inches='tight')
plt.show()
print(f"\n[Saved] {OUTPUT_PNG}")

In [ ]:
# ============================================================================
# V42 — Spatial profile audit at gtsrcmaps output level
# 목적:
#  (1) gtsrcmaps의 0.25°→0.1° resampling이 spatial structure를 보존하는지
#  (2) PSF가 component별로 다른 spatial 효과를 주는지
#  (3) ICS/Pi0+Brem inner concentration ratio 측정
#       → Zenodo input vs 우리 conv가 다르면 c_ratio swap의 spatial 기원 확정
# 영역: Model I, bin 4 (0.784-1.02 GeV, GCE peak 근처)
# ============================================================================
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from pathlib import Path

SANG = Path("./GC_analysis_sanghwan")
ZEN  = Path("../GCE_TEMPLATES_FILES_v3/GALACTIC_DIFFUSE_EMISSION_MAPS_0p25deg")
M, M_CHOL = "I", "bs"

# Bin 4 (우리 grid) ↔ bin 11 (Zenodo 38-grid) — 동일 에너지 점
IB_OURS, IB_ZEN = 4, 11
E_CTR_MeV = 893.479899                  # geometric mean of bin 4
DE_MeV    = 1018.59 - 783.737           # 234.853 MeV
SR_PIX    = (0.1 * np.pi / 180)**2      # 0.1° pixel → sr

OUTPUT_PNG = "./V42_spatial_audit.png"

print("="*84)
print(f" V42 — Spatial profile audit | Model {M}, bin {IB_OURS} (~0.89 GeV)")
print("="*84)

# ---- Step 1: Variable probe & load ---------------------------------------
print("\n[1] Loading inputs")
# Zenodo flux maps (0.25° pixel, GeV cm⁻²s⁻¹sr⁻¹ = E²·dN/dE)
zen = {}
for stem, label in [('pi0', 'pi0'), ('bremss', 'bremss'), ('ICS', 'ICS')]:
    p = ZEN / f"{stem}_{M_CHOL}_Map_flux_E_50-814008_MeV_InnerGalaxy_60x60.fits"
    assert p.exists(), f"missing: {p}"
    zen[label] = fits.getdata(p)[IB_ZEN].astype(np.float64)
zen['Pi0+Brem'] = zen['pi0'] + zen['bremss']
print(f"  Zenodo shape: {zen['ICS'].shape}, dtype: {zen['ICS'].dtype}")

# 우리 conv & no_conv srcmap (0.1° pixel, counts/pix/bin)
conv = {}; ncv = {}
for stem, label in [('pion', 'pi0'), ('bremss', 'bremss'), ('ics', 'ICS')]:
    pc  = SANG / f"GC_{stem}_model{M}_12yr_front_clean.fits"
    pnc = SANG / f"GC_{stem}_model{M}_12yr_front_clean_no_convol.fits"
    assert pc.exists() and pnc.exists(), f"missing: {pc} or {pnc}"
    conv[label] = fits.getdata(pc)[IB_OURS].astype(np.float64)
    ncv[label]  = fits.getdata(pnc)[IB_OURS].astype(np.float64)
conv['Pi0+Brem'] = conv['pi0'] + conv['bremss']
ncv['Pi0+Brem']  = ncv['pi0']  + ncv['bremss']
print(f"  Our conv shape: {conv['ICS'].shape}, no_conv shape: {ncv['ICS'].shape}")

# Exposure cube
exp_cube = fits.getdata(SANG / "GC_expcube_center_12yr_front_clean.fits"
                       )[IB_OURS].astype(np.float64)
print(f"  Exposure shape: {exp_cube.shape}, mean: {exp_cube.mean():.3e} cm²·s/pix")

# ---- Step 2: counts → flux 변환 (우리 → Zenodo와 같은 단위) ---------------
def counts_to_E2dNdE(counts_map):
    """counts/pixel/bin → E²·dN/dE in GeV cm⁻²s⁻¹sr⁻¹
       flux [per MeV cm² s sr] = counts / (exp × dE_MeV × sr_per_pix)
       E²·dN/dE [GeV·cm⁻²s⁻¹sr⁻¹] = E_MeV² × flux × 1e-6  (MeV²→GeV²)
    """
    with np.errstate(divide='ignore', invalid='ignore'):
        flux_perMeV = counts_map / (exp_cube * DE_MeV * SR_PIX)
    return (E_CTR_MeV**2) * flux_perMeV * 1e-6

our_input = {k: counts_to_E2dNdE(ncv[k])  for k in ['ICS', 'Pi0+Brem']}
our_conv  = {k: counts_to_E2dNdE(conv[k]) for k in ['ICS', 'Pi0+Brem']}

# Sanity: 우리 reconstructed input의 magnitude vs Zenodo (어느 정도 일치해야 함)
print(f"\n[Magnitude sanity check] 60×60° mean E²·dN/dE [GeV cm⁻²s⁻¹sr⁻¹]")
print(f"  {'map':<28} {'ICS':>12} {'Pi0+Brem':>12}")
print(f"  {'Zenodo input (0.25°)':<28} {zen['ICS'].mean():>12.3e} "
      f"{zen['Pi0+Brem'].mean():>12.3e}")
print(f"  {'Our recon (no_conv / exp)':<28} {our_input['ICS'].mean():>12.3e} "
      f"{our_input['Pi0+Brem'].mean():>12.3e}")
print(f"  {'Our conv / exp':<28} {our_conv['ICS'].mean():>12.3e} "
      f"{our_conv['Pi0+Brem'].mean():>12.3e}")
print(f"  ★ Zenodo vs Our recon ratio ≈ 1 → resampling은 단순 spatial 재배치")
print(f"  ★ 다르면 → resampling이 magnitude bias 도입 (V36 exposure 60% 차이 반영됨)")

# ---- Step 3: Same-grid 비교를 위해 Zenodo 0.25° → 0.1° upsample ----------
def upsample_25_to_01(arr_240):
    """240×240 (0.25°) → 600×600 (0.1°) nearest-neighbor. factor 2.5."""
    factor = 2.5
    y_idx = np.minimum((np.arange(600) / factor).astype(int), arr_240.shape[0]-1)
    x_idx = np.minimum((np.arange(600) / factor).astype(int), arr_240.shape[1]-1)
    return arr_240[np.ix_(y_idx, x_idx)]

zen_up = {k: upsample_25_to_01(zen[k]) for k in ['ICS', 'Pi0+Brem']}

# Slice to 400×400 fit region (= [100:500])
zen_400  = {k: zen_up[k][100:500, 100:500]    for k in ['ICS', 'Pi0+Brem']}
inp_400  = {k: our_input[k][100:500, 100:500] for k in ['ICS', 'Pi0+Brem']}
conv_400 = {k: our_conv[k][100:500, 100:500]  for k in ['ICS', 'Pi0+Brem']}

# ---- Step 4: Concentration index + ICS/PB ratio --------------------------
cy, cx = 200, 200
yy, xx = np.ogrid[:400, :400]
r_deg  = np.sqrt((yy - cy)**2 + (xx - cx)**2) * 0.1
in_msk  = (r_deg <= 2)              # inner ±2°
out_msk = (r_deg >= 15) & (r_deg <= 20)  # outer ring

def conc(arr):
    """inner ±2° mean / outer 15-20° mean"""
    a_in  = arr[in_msk]
    a_out = arr[out_msk]
    return a_in.mean() / a_out.mean() if a_out.mean() > 0 else np.nan

print(f"\n[Concentration index] inner ±2° / outer 15-20°")
print(f"  {'map':<32} {'ICS':>10} {'Pi0+Brem':>10} {'ICS/PB':>10}")
print("-"*70)
for label, dct in [('Zenodo (resampled NN)',       zen_400),
                    ('Our input recon (no_conv)',   inp_400),
                    ('Our conv srcmap',             conv_400)]:
    c_ics = conc(dct['ICS'])
    c_pb  = conc(dct['Pi0+Brem'])
    print(f"  {label:<32} {c_ics:>10.3f} {c_pb:>10.3f} {c_ics/c_pb:>10.3f}")

print(f"\n  ★ ICS/PB ratio가 세 row 모두 비슷 → spatial structure 보존")
print(f"  ★ Zenodo vs Our conv에서 ratio 다름 → fit이 보는 spatial structure가")
print(f"    Cholis input과 다름 → c_ratio swap의 spatial 기원")

# ---- Step 5: Plot — 3행 × 3열 그리드 -------------------------------------
fig, axes = plt.subplots(3, 3, figsize=(16, 13), dpi=110)
b_axis = (np.arange(400) - cy) * 0.1
l_axis = (np.arange(400) - cx) * 0.1
panels = [('Zenodo input (resampled)', zen_400),
          ('Our input recon',           inp_400),
          ('Our conv srcmap',           conv_400)]

# Row 1: latitude profile @ l=0
for col, (title, dct) in enumerate(panels):
    ax = axes[0, col]
    for comp, color in [('ICS', 'C0'), ('Pi0+Brem', 'C1')]:
        prof = dct[comp][:, cx-5:cx+5].mean(axis=1)
        ax.semilogy(b_axis, prof, '-', color=color, lw=2, label=comp)
    ax.set_title(f'{title} — lat profile @ l=0')
    ax.set_xlabel('b [deg]'); ax.set_ylabel('E²·dN/dE')
    ax.legend(fontsize=9); ax.grid(alpha=0.3); ax.set_xlim(-15, 15)

# Row 2: longitude profile @ b=0
for col, (title, dct) in enumerate(panels):
    ax = axes[1, col]
    for comp, color in [('ICS', 'C0'), ('Pi0+Brem', 'C1')]:
        prof = dct[comp][cy-5:cy+5, :].mean(axis=0)
        ax.semilogy(l_axis, prof, '-', color=color, lw=2, label=comp)
    ax.set_title(f'{title} — lon profile @ b=0')
    ax.set_xlabel('l [deg]'); ax.set_ylabel('E²·dN/dE')
    ax.legend(fontsize=9); ax.grid(alpha=0.3); ax.set_xlim(-15, 15)

# Row 3: ICS/Pi0+Brem ratio profile (각 row 마지막에 세 map overlay)
ax_b = axes[2, 0]
ax_l = axes[2, 1]
ax_2d = axes[2, 2]
for title, dct, color in [('Zenodo',          zen_400,  'C3'),
                           ('Our input recon', inp_400,  'C0'),
                           ('Our conv',        conv_400, 'C2')]:
    ratio_b = dct['ICS'][:, cx-5:cx+5].mean(axis=1) / \
              dct['Pi0+Brem'][:, cx-5:cx+5].mean(axis=1)
    ratio_l = dct['ICS'][cy-5:cy+5, :].mean(axis=0) / \
              dct['Pi0+Brem'][cy-5:cy+5, :].mean(axis=0)
    ax_b.plot(b_axis, ratio_b, '-', color=color, lw=2, label=title)
    ax_l.plot(l_axis, ratio_l, '-', color=color, lw=2, label=title)

ax_b.set_title('ICS / Pi0+Brem  vs  b @ l=0')
ax_b.set_xlabel('b [deg]'); ax_b.set_ylabel('ICS / Pi0+Brem')
ax_b.legend(fontsize=9); ax_b.grid(alpha=0.3)
ax_b.set_xlim(-15, 15); ax_b.set_yscale('log')

ax_l.set_title('ICS / Pi0+Brem  vs  l @ b=0')
ax_l.set_xlabel('l [deg]'); ax_l.set_ylabel('ICS / Pi0+Brem')
ax_l.legend(fontsize=9); ax_l.grid(alpha=0.3)
ax_l.set_xlim(-15, 15); ax_l.set_yscale('log')

# Last panel: 2D ICS/PB ratio for Our conv (가장 결정적 시각)
with np.errstate(divide='ignore', invalid='ignore'):
    ratio2d_zen  = np.where(zen_400['Pi0+Brem']  > 0, zen_400['ICS']/zen_400['Pi0+Brem'],   np.nan)
    ratio2d_conv = np.where(conv_400['Pi0+Brem'] > 0, conv_400['ICS']/conv_400['Pi0+Brem'], np.nan)
diff = np.log10(ratio2d_conv / ratio2d_zen)
im = ax_2d.imshow(diff, origin='lower', cmap='RdBu_r', vmin=-0.3, vmax=0.3,
                   extent=[-20, 20, -20, 20])
ax_2d.set_title('log₁₀(conv ratio / Zenodo ratio)\n→ where spatial swap occurs')
ax_2d.set_xlabel('l [deg]'); ax_2d.set_ylabel('b [deg]')
plt.colorbar(im, ax=ax_2d, label='log₁₀ shift in ICS/PB')

plt.tight_layout()
plt.savefig(OUTPUT_PNG, dpi=130, bbox_inches='tight')
plt.show()

print(f"\n[Saved] {OUTPUT_PNG}")
print("\n" + "="*84)
print(" 분기 해석 가이드:")
print("="*84)
print(" Concentration index 표에서:")
print(" • Zenodo의 ICS/PB ratio와 우리 conv의 ICS/PB ratio가 거의 같음 →")
print("     spatial structure는 보존됨. c_swap 원인은 다른 곳 (mask, fit 구현)")
print(" • Zenodo와 우리 conv의 ICS/PB ratio가 크게 다름 →")
print("     gtsrcmaps의 resampling+PSF 단계에서 ICS와 Pi0+Brem이 비대칭 처리됨")
print("     → c_swap의 spatial 기원 확정")
print(" 2D map (마지막 panel):")
print(" • 모든 픽셀이 빨강/파랑 균일 → 단순 magnitude 차이")
print(" • inner+disk 영역만 빨강/파랑 → spatial-dependent swap")

In [ ]:
# ============================================================================
# V43 — Mask position spatial audit
# 목적: mask retention 비율은 Table III와 일치 (V34 확인됨). 하지만 mask hole의
#       spatial 위치가 component-specific (ICS-rich vs Pi0+Brem-rich)으로
#       비대칭인지 확인. c_swap의 mask 기원 가설 검증.
# 의존: V27 변수 (E, n_ebin, SLICE_Y, SLICE_X), STAGE_A_MODEL
# ============================================================================
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from pathlib import Path

SANG = Path("./GC_analysis_sanghwan")
M = STAGE_A_MODEL   # "I"
OUTPUT_PNG = f"./V43_mask_audit_Model{M}.png"

print("="*100)
print(f" V43 — Mask position spatial audit | Model {M}")
print("="*100)

# ---- 1. Mask + template load --------------------------------------------
psc_mask  = np.load(SANG / "Model/GC_mask_60x60_definitions_DR2.npy")     # (17, 600, 600)
disk_mask_full = np.load(SANG / "Model/GC_disk_mask_60x60_definitions.npy")  # (600, 600)

psc_400  = psc_mask[:, SLICE_Y, SLICE_X].astype(np.float32)   # (17, 400, 400)
disk_400 = disk_mask_full[SLICE_Y, SLICE_X].astype(np.float32) # (400, 400)
full_mask = psc_400 * disk_400[None, :, :]                     # (17, 400, 400)
print(f"\n[Loaded] psc_400: {psc_400.shape}, disk_400: {disk_400.shape}, "
      f"full_mask: {full_mask.shape}")

# Templates (no_conv, 우리 fit이 보는 그대로)
templates = {}
for stem, label in [('pion', 'Pi0'), ('bremss', 'Brem'), ('ics', 'ICS')]:
    p = SANG / f"GC_{stem}_model{M}_12yr_front_clean_no_convol.fits"
    templates[label] = fits.getdata(p)[:, SLICE_Y, SLICE_X].astype(np.float64)
for stem, label in [('GCE_model', 'GCE'), ('fermi_bubble_model', 'Bubble')]:
    p = SANG / f"GC_{stem}_12yr_front_clean_no_convol.fits"
    templates[label] = fits.getdata(p)[:, SLICE_Y, SLICE_X].astype(np.float64)
templates['Pi0+Brem'] = templates['Pi0'] + templates['Brem']
print(f"[Templates loaded] 5 components, shape = {templates['ICS'].shape}")

# ---- 2. Component-weighted mask retention per bin ------------------------
NY = NX = 400
cy, cx = NY//2, NX//2

print("\n[Component-weighted mask retention per bin]")
print(f"  {'bin':>3} {'E[GeV]':>8} | {'PB ret':>8} {'ICS ret':>8} {'GCE ret':>8} "
      f"{'Bub ret':>8} | {'ICS/PB':>8} {'GCE/PB':>8}")
print("-"*90)
ret_by_bin = {comp: np.zeros(14) for comp in ['Pi0+Brem', 'ICS', 'GCE', 'Bubble']}
for i in range(14):
    row = f"  {i:>3d} {E[i]:>8.3f} |"
    for comp in ['Pi0+Brem', 'ICS', 'GCE', 'Bubble']:
        num = (templates[comp][i] * full_mask[i]).sum()
        den = templates[comp][i].sum()
        ret_by_bin[comp][i] = num / den if den > 0 else np.nan
        row += f" {ret_by_bin[comp][i]:>8.4f}"
    row += " |"
    row += f" {ret_by_bin['ICS'][i]/ret_by_bin['Pi0+Brem'][i]:>8.4f}"
    row += f" {ret_by_bin['GCE'][i]/ret_by_bin['Pi0+Brem'][i]:>8.4f}"
    print(row)

# Mean across bins
print("\n[Mean retention across 14 bins]")
for comp in ['Pi0+Brem', 'ICS', 'GCE', 'Bubble']:
    r = ret_by_bin[comp]
    print(f"  {comp:<10}: mean={r.mean():.4f}, std={r.std():.4f}, "
          f"min={r.min():.4f}, max={r.max():.4f}")
ratio_ics_pb = ret_by_bin['ICS'] / ret_by_bin['Pi0+Brem']
ratio_gce_pb = ret_by_bin['GCE'] / ret_by_bin['Pi0+Brem']
print(f"\n  ICS/PB retention ratio: mean={ratio_ics_pb.mean():.4f}, "
      f"std={ratio_ics_pb.std():.4f}")
print(f"  GCE/PB retention ratio: mean={ratio_gce_pb.mean():.4f}, "
      f"std={ratio_gce_pb.std():.4f}")

# ---- 3. Radial profile of mask hole density -----------------------------
yy, xx = np.ogrid[:NY, :NX]
r_deg = np.sqrt((yy - cy)**2 + (xx - cx)**2) * 0.1

regions = {
    'inner ±2°':   r_deg <= 2,
    'inner 2-5°':  (r_deg > 2) & (r_deg <= 5),
    'mid 5-10°':   (r_deg > 5) & (r_deg <= 10),
    'mid 10-15°':  (r_deg > 10) & (r_deg <= 15),
    'outer 15-20°':(r_deg > 15) & (r_deg <= 20),
}

print("\n[PSC mask hole density by region (bin 4, ~0.89 GeV)]")
print(f"  {'region':<14} {'hole frac':>10} {'PB retain':>10} {'ICS retain':>10} "
      f"{'ICS/PB':>8}")
print("-"*60)
for label, mask_region in regions.items():
    psc_in = psc_400[4][mask_region]
    hole_frac = 1 - psc_in.mean()
    # Component retention in this region only
    pb_num = (templates['Pi0+Brem'][4] * full_mask[4] * mask_region.astype(np.float32)).sum()
    pb_den = (templates['Pi0+Brem'][4] * mask_region.astype(np.float32)).sum()
    ics_num = (templates['ICS'][4] * full_mask[4] * mask_region.astype(np.float32)).sum()
    ics_den = (templates['ICS'][4] * mask_region.astype(np.float32)).sum()
    pb_ret  = pb_num / pb_den if pb_den > 0 else np.nan
    ics_ret = ics_num / ics_den if ics_den > 0 else np.nan
    print(f"  {label:<14} {hole_frac:>10.4f} {pb_ret:>10.4f} {ics_ret:>10.4f} "
          f"{ics_ret/pb_ret if pb_ret > 0 else np.nan:>8.4f}")

print("\n  ★ inner ±2° hole frac이 paper Table III와 일치 (V34 확인됨)")
print("  ★ ICS/PB ratio가 1.0에서 멀어지면 component-specific mask 비대칭")

# ---- 4. Visualization ----------------------------------------------------
fig, axes = plt.subplots(2, 3, figsize=(16, 10), dpi=110)
extent = [-20, 20, -20, 20]
ib = 4  # bin 4 (~0.89 GeV)

# Top row: template × mask (log scale, after masking)
for col, comp in enumerate(['Pi0+Brem', 'ICS']):
    ax = axes[0, col]
    masked = templates[comp][ib] * full_mask[ib]
    im = ax.imshow(np.log10(masked + 1e-3), origin='lower', cmap='viridis',
                   extent=extent)
    ax.set_title(f'{comp} × full_mask (bin {ib})')
    ax.set_xlabel('l [deg]'); ax.set_ylabel('b [deg]')
    plt.colorbar(im, ax=ax, label='log10(counts)')
    # Circle markers at radial boundaries
    for r in [2, 5, 10, 15]:
        theta = np.linspace(0, 2*np.pi, 100)
        ax.plot(r*np.cos(theta), r*np.sin(theta), 'r--', lw=0.5, alpha=0.5)

# Top row right: log(ICS / Pi0+Brem) after masking
ax = axes[0, 2]
with np.errstate(divide='ignore', invalid='ignore'):
    num = templates['ICS'][ib] * full_mask[ib]
    den = templates['Pi0+Brem'][ib] * full_mask[ib]
    ratio_map = np.where(den > 1e-6, num / den, np.nan)
    ratio_map_log = np.log10(ratio_map)
im = ax.imshow(ratio_map_log, origin='lower', cmap='RdBu_r', vmin=-0.5, vmax=0.5,
                extent=extent)
ax.set_title(f'log₁₀(ICS / Pi0+Brem), after mask (bin {ib})')
ax.set_xlabel('l [deg]'); ax.set_ylabel('b [deg]')
plt.colorbar(im, ax=ax)

# Bottom row 1: PSC mask itself (bin 4)
ax = axes[1, 0]
im = ax.imshow(psc_400[ib], origin='lower', cmap='gray', extent=extent, vmin=0, vmax=1)
ax.set_title(f'PSC mask (bin {ib}, 1=kept, 0=masked)')
ax.set_xlabel('l [deg]'); ax.set_ylabel('b [deg]')
plt.colorbar(im, ax=ax)
for r in [2, 5, 10]:
    theta = np.linspace(0, 2*np.pi, 100)
    ax.plot(r*np.cos(theta), r*np.sin(theta), 'r--', lw=0.5)

# Bottom row 2: full mask (PSC × disk)
ax = axes[1, 1]
im = ax.imshow(full_mask[ib], origin='lower', cmap='gray', extent=extent, vmin=0, vmax=1)
ax.set_title(f'Full mask = PSC × disk (bin {ib})')
ax.set_xlabel('l [deg]'); ax.set_ylabel('b [deg]')
plt.colorbar(im, ax=ax)

# Bottom row 3: ICS/PB retention ratio vs bin
ax = axes[1, 2]
ax.plot(E[:14], ratio_ics_pb, 'o-', color='C0', lw=2, label='ICS/PB retention')
ax.plot(E[:14], ratio_gce_pb, 's-', color='C3', lw=2, label='GCE/PB retention')
ax.axhline(1.0, color='k', lw=0.5, ls=':')
ax.set_xscale('log')
ax.set_xlabel('E [GeV]'); ax.set_ylabel('retention ratio')
ax.set_title('Component-weighted retention ratio per bin')
ax.legend(); ax.grid(alpha=0.3)
ax.set_ylim(0.8, 1.2)

plt.tight_layout()
plt.savefig(OUTPUT_PNG, dpi=130, bbox_inches='tight')
plt.show()
print(f"\n[Saved] {OUTPUT_PNG}")

# ---- 5. 분기 해석 ---------------------------------------------------------
print("\n" + "="*100)
print(" 분기 해석:")
print("="*100)
print(" • ICS/PB ratio ≈ 1.000 (mean, std < 0.005) → mask는 component-blind,")
print("   c_swap의 mask 기원 가설 폐기")
print(" • ICS/PB ratio < 1 (ICS가 더 많이 가려짐) → fit에서 ICS counts 부족")
print("   → c_ics 키워 보상 → 우리 c_ics=2.40 패턴과 일치 → mask가 원인 후보")
print(" • ICS/PB ratio > 1 (Pi0+Brem이 더 많이 가려짐) → c_gas 키워야 정상,")
print("   우리는 c_gas=0.43으로 작음 → 반대 방향, mask 원인 아님")
print(" • inner ±2°에 mask hole 밀집은 Table III TS>49 strict cutoff과 일치")
print("   (paper convention)")

In [ ]:
# ============================================================================
# V44 — Paper-provided mask vs 우리 mask 직접 비교
# 목적: 2112.09706가 공개한 official mask (via Zhong+2024 / github)와
#       우리가 만든 mask의 pixel-level 차이 정량 분석.
#       paper mask로 ICS/PB retention ratio가 1.0에 가까워지면 →
#       우리 mask 구현 문제가 c_swap의 원인임을 확정.
# 의존: V27 변수 (E, SLICE_Y, SLICE_X), V43 templates
# ============================================================================
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from pathlib import Path

SANG = Path("./GC_analysis_sanghwan")
PAPER_MASK = Path("./gce_mask-main/mask/mask_4FGL-DR2_14_Ebin_20x20window_normal.npy")
M = STAGE_A_MODEL
OUTPUT_PNG = f"./V44_mask_compare_Model{M}.png"

print("="*100)
print(" V44 — Paper-provided mask vs 우리 mask")
print("="*100)

# ---- Step 1: Load both masks + shape/dtype probe ------------------------
assert PAPER_MASK.exists(), f"Paper mask not found: {PAPER_MASK}"
paper_mask = np.load(PAPER_MASK)
print(f"\n[Paper mask]")
print(f"  Path: {PAPER_MASK}")
print(f"  Shape: {paper_mask.shape}, dtype: {paper_mask.dtype}")
print(f"  min: {paper_mask.min()}, max: {paper_mask.max()}")
print(f"  unique values: {np.unique(paper_mask)}")
print(f"  bin-wise mask retention (fraction of 1s):")
for i in range(paper_mask.shape[0]):
    print(f"    bin {i:2d}: {paper_mask[i].mean():.4f}")

# Our mask
our_psc = np.load(SANG / "Model/GC_mask_60x60_definitions_DR2.npy")  # (17, 600, 600)
our_disk = np.load(SANG / "Model/GC_disk_mask_60x60_definitions.npy")  # (600, 600)
our_psc_400  = our_psc[:14, SLICE_Y, SLICE_X].astype(np.float32)
our_disk_400 = our_disk[SLICE_Y, SLICE_X].astype(np.float32)
our_full     = our_psc_400 * our_disk_400[None, :, :]
print(f"\n[Our mask]")
print(f"  PSC original: {our_psc.shape}")
print(f"  PSC sliced [0:14, 100:500, 100:500]: {our_psc_400.shape}")
print(f"  Full (PSC × disk): {our_full.shape}")

# Convention check
assert paper_mask.shape == our_full.shape, \
    f"Shape mismatch: paper {paper_mask.shape} vs ours {our_full.shape}"

# ---- Step 2: Pixel-level XOR + retention 비교 ---------------------------
print("\n" + "="*100)
print(" [2] Pixel-level retention comparison")
print("="*100)
print(f"  {'bin':>3} {'E[GeV]':>8} | {'paper ret':>10} {'ours ret':>10} {'diff':>8} "
      f"{'XOR frac':>10} {'paper-only':>11} {'ours-only':>10}")
print("-"*100)
for i in range(14):
    p_ret  = paper_mask[i].mean()
    o_ret  = our_full[i].mean()
    diff   = o_ret - p_ret
    # XOR: 둘 중 한쪽만 unmasked (=1)
    xor_frac     = np.logical_xor(paper_mask[i] > 0.5, our_full[i] > 0.5).mean()
    paper_only   = ((paper_mask[i] > 0.5) & (our_full[i] < 0.5)).mean()  # paper keeps, ours masks
    ours_only    = ((our_full[i] > 0.5) & (paper_mask[i] < 0.5)).mean()  # ours keeps, paper masks
    print(f"  {i:>3d} {E[i]:>8.3f} | {p_ret:>10.4f} {o_ret:>10.4f} {diff:>+8.4f} "
          f"{xor_frac:>10.4f} {paper_only:>11.4f} {ours_only:>10.4f}")

# ---- Step 3: Component-weighted retention with PAPER mask ---------------
# templates는 V43에서 load 했을 것
print("\n" + "="*100)
print(" [3] Component-weighted retention USING PAPER MASK (key diagnostic)")
print("="*100)
print(f"  {'bin':>3} {'E[GeV]':>8} | {'PB ret':>8} {'ICS ret':>8} {'ICS/PB':>8}")
print("-"*60)
ret_paper = {comp: np.zeros(14) for comp in ['Pi0+Brem', 'ICS', 'GCE', 'Bubble']}
for i in range(14):
    row = f"  {i:>3d} {E[i]:>8.3f} |"
    for comp in ['Pi0+Brem', 'ICS', 'GCE', 'Bubble']:
        num = (templates[comp][i] * paper_mask[i]).sum()
        den = templates[comp][i].sum()
        ret_paper[comp][i] = num / den if den > 0 else np.nan
    row += f" {ret_paper['Pi0+Brem'][i]:>8.4f} {ret_paper['ICS'][i]:>8.4f}"
    row += f" {ret_paper['ICS'][i]/ret_paper['Pi0+Brem'][i]:>8.4f}"
    print(row)

ratio_paper = ret_paper['ICS'] / ret_paper['Pi0+Brem']
print(f"\n  PAPER mask  ICS/PB retention ratio: mean={ratio_paper.mean():.4f}, "
      f"std={ratio_paper.std():.4f}")
print(f"  OURS mask   ICS/PB retention ratio: mean=1.1252, std=0.0427  (V43에서)")

if abs(ratio_paper.mean() - 1.0) < 0.02:
    print("\n  ★ Paper mask로는 ICS/PB ≈ 1.0 → 우리 mask 구현에 systematic 문제")
    print("  ★ paper mask로 fit 재실행하면 c_swap 해결 가능성 큼")
elif abs(ratio_paper.mean() - 1.1252) < 0.02:
    print("\n  ★ Paper mask도 같은 비대칭 → mask가 아닌 다른 원인")
    print("  ★ template의 spatial structure에 본질적 ICS/PB 비대칭 존재")
else:
    print(f"\n  ★ 중간 수준 — 우리 mask와 paper mask 모두 일부 기여")

# ---- Step 4: Visualization ----------------------------------------------
fig, axes = plt.subplots(3, 3, figsize=(16, 14), dpi=110)
extent = [-20, 20, -20, 20]
ib = 4

# Row 1: paper mask, our mask, XOR
ax = axes[0, 0]
ax.imshow(paper_mask[ib], origin='lower', cmap='gray', extent=extent, vmin=0, vmax=1)
ax.set_title(f'Paper mask (bin {ib}, ~0.89 GeV)\n'
             f'retention = {paper_mask[ib].mean():.3f}')
ax.set_xlabel('l [deg]'); ax.set_ylabel('b [deg]')

ax = axes[0, 1]
ax.imshow(our_full[ib], origin='lower', cmap='gray', extent=extent, vmin=0, vmax=1)
ax.set_title(f'Our mask (bin {ib})\nretention = {our_full[ib].mean():.3f}')
ax.set_xlabel('l [deg]'); ax.set_ylabel('b [deg]')

ax = axes[0, 2]
xor_map = paper_mask[ib].astype(int) - our_full[ib].astype(int)  # +1=paper only, -1=ours only
im = ax.imshow(xor_map, origin='lower', cmap='RdBu_r', extent=extent, vmin=-1, vmax=1)
ax.set_title(f'XOR: red=paper only kept, blue=ours only kept\n'
             f'XOR frac = {np.logical_xor(paper_mask[ib]>0.5, our_full[ib]>0.5).mean():.3f}')
ax.set_xlabel('l [deg]'); ax.set_ylabel('b [deg]')
plt.colorbar(im, ax=ax)

# Row 2: bin별 retention 비교
ax = axes[1, 0]
ax.plot(E[:14], [paper_mask[i].mean() for i in range(14)], 'o-', color='C0', lw=2,
        label='Paper mask')
ax.plot(E[:14], [our_full[i].mean() for i in range(14)], 's-', color='C3', lw=2,
        label='Our mask')
ax.set_xscale('log'); ax.set_xlabel('E [GeV]'); ax.set_ylabel('retention fraction')
ax.set_title('Mask retention per bin')
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1, 1]
ax.plot(E[:14], ratio_paper, 'o-', color='C0', lw=2, label='Paper mask: ICS/PB')
# V43 결과 (우리 mask)
our_ratio_v43 = np.array([1.022, 1.053, 1.086, 1.110, 1.122, 1.128, 1.136,
                          1.140, 1.148, 1.153, 1.157, 1.160, 1.167, 1.173])
ax.plot(E[:14], our_ratio_v43, 's-', color='C3', lw=2, label='Our mask: ICS/PB')
ax.axhline(1.0, color='k', lw=0.5, ls=':')
ax.set_xscale('log'); ax.set_xlabel('E [GeV]'); ax.set_ylabel('ICS/PB retention')
ax.set_title('Component asymmetry comparison')
ax.legend(); ax.grid(alpha=0.3); ax.set_ylim(0.9, 1.25)

ax = axes[1, 2]
xor_per_bin = [np.logical_xor(paper_mask[i]>0.5, our_full[i]>0.5).mean()
               for i in range(14)]
ax.plot(E[:14], xor_per_bin, 'o-', color='C2', lw=2)
ax.set_xscale('log'); ax.set_xlabel('E [GeV]'); ax.set_ylabel('XOR fraction')
ax.set_title('Pixel-level disagreement per bin')
ax.grid(alpha=0.3)

# Row 3: 차이 영역에서 ICS/PB 우세 — 어디가 다른지의 component 의미
for col, comp in enumerate(['Pi0+Brem', 'ICS']):
    ax = axes[2, col]
    # In XOR region, which component dominates
    paper_only_mask = (paper_mask[ib] > 0.5) & (our_full[ib] < 0.5)
    ours_only_mask  = (our_full[ib] > 0.5) & (paper_mask[ib] < 0.5)
    pb_in_paper = (templates[comp][ib] * paper_only_mask).sum()
    pb_in_ours  = (templates[comp][ib] * ours_only_mask).sum()
    print(f"\n  bin {ib} {comp}:")
    print(f"    {comp} sum in 'paper-only-kept' pixels: {pb_in_paper:.3e}")
    print(f"    {comp} sum in 'ours-only-kept' pixels: {pb_in_ours:.3e}")
    # Spatial map of disagreement weighted by this component
    weighted = templates[comp][ib] * (paper_mask[ib] - our_full[ib].astype(int))
    im = ax.imshow(weighted, origin='lower', cmap='RdBu_r', extent=extent,
                   vmin=-np.percentile(np.abs(weighted), 95),
                   vmax=np.percentile(np.abs(weighted), 95))
    ax.set_title(f'{comp} flux × (paper mask - ours mask)\n'
                 f'red=more in paper, blue=more in ours')
    ax.set_xlabel('l [deg]'); ax.set_ylabel('b [deg]')
    plt.colorbar(im, ax=ax)

# Last panel: summary text
ax = axes[2, 2]
ax.axis('off')
summary = (f"V44 핵심 결과:\n\n"
           f"Paper mask ICS/PB:\n"
           f"  mean={ratio_paper.mean():.4f}\n"
           f"  std ={ratio_paper.std():.4f}\n\n"
           f"Our mask ICS/PB (V43):\n"
           f"  mean=1.1252\n"
           f"  std =0.0427\n\n"
           f"XOR fraction range:\n"
           f"  min={min(xor_per_bin):.4f}\n"
           f"  max={max(xor_per_bin):.4f}\n\n"
           f"분기:\n"
           f"  paper ratio ≈ 1.0 → 우리 mask 문제\n"
           f"  paper ratio ≈ 1.12 → mask가 원인 아님")
ax.text(0.05, 0.95, summary, transform=ax.transAxes, fontsize=11,
        verticalalignment='top', family='monospace')

plt.tight_layout()
plt.savefig(OUTPUT_PNG, dpi=130, bbox_inches='tight')
plt.show()
print(f"\n[Saved] {OUTPUT_PNG}")

In [ ]:
# ============================================================================
# V45 — Mask orientation correctness check
# 목적: 우리 mask가 우리 ccube의 점 source를 실제로 가리는지 직접 확인.
#       paper mask + flipped, paper mask + as-is, our mask를 각각 ccube 위에 overlay.
#       hot pixel과 hole이 정렬되는 mask가 올바른 orientation.
# 의존: V44 변수 (paper_mask, our_full, templates)
# ============================================================================
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from pathlib import Path

SANG = Path("./GC_analysis_sanghwan")
PAPER_MASK = Path("./gce_mask-main/mask/mask_4FGL-DR2_14_Ebin_20x20window_normal.npy")
OUTPUT_PNG = "./V45_mask_orientation.png"
IB = 4   # bin 4 (~0.89 GeV)

print("="*100)
print(" V45 — Mask orientation correctness check")
print("="*100)

# ---- Step 1: Load CCUBE (no mask applied) + 우리 mask + paper mask -------
ccube_full = fits.getdata(SANG / "GC_ccube_12yr_front_clean.fits")[IB].astype(np.float64)
ccube = ccube_full[SLICE_Y, SLICE_X]   # 400×400
print(f"\n[CCUBE bin {IB}] shape: {ccube.shape}, max: {ccube.max():.0f}, "
      f"mean: {ccube.mean():.1f}")

paper_mask = np.load(PAPER_MASK)
our_full = our_psc_400 * our_disk_400[None, :, :]   # V44에서 만들어진

# Three variants of paper mask for orientation test
mask_paper_asis    = paper_mask[IB]                  # 그대로
mask_paper_flipL   = paper_mask[IB][:, ::-1]         # longitude (axis=1 of 2D) flip
mask_paper_flipB   = paper_mask[IB][::-1, :]         # latitude flip (드물지만 sanity)
mask_ours          = our_full[IB]

# ---- Step 2: Find bright spots in CCUBE (= 점 source candidates) ---------
# Use disk_mask to remove |b|<2° (galactic disk emission), then top N pixels
ccube_offdisk = ccube * our_disk_400   # zero out |b|<2°
# Top 30 brightest off-disk pixels
flat = ccube_offdisk.flatten()
top_n = 30
top_idx = np.argpartition(flat, -top_n)[-top_n:]
top_idx = top_idx[np.argsort(-flat[top_idx])]   # sort descending
top_y, top_x = np.unravel_index(top_idx, ccube.shape)

# Convert to (l, b) degrees: pixel (200, 200) = (0, 0)
top_b = (top_y - 200) * 0.1
top_l = (top_x - 200) * 0.1

print(f"\n[Top {top_n} bright off-disk pixels in CCUBE bin {IB}]")
print(f"  (these should be point sources, mostly catalog 4FGL-DR2 sources)")
print(f"  {'rank':>4} {'(l, b) [deg]':>20} {'counts':>10} | "
      f"{'in ours hole?':>14} {'in paper asis?':>15} {'in paper flipL?':>16}")
print("-"*100)
hits_ours = 0
hits_asis = 0
hits_flipL = 0
for rank in range(top_n):
    py, px = top_y[rank], top_x[rank]
    cnt = ccube[py, px]
    in_ours  = mask_ours[py, px]      < 0.5   # mask=0 means masked (hole)
    in_asis  = mask_paper_asis[py, px] < 0.5
    in_flipL = mask_paper_flipL[py, px] < 0.5
    hits_ours  += int(in_ours)
    hits_asis  += int(in_asis)
    hits_flipL += int(in_flipL)
    print(f"  {rank+1:>4d} ({top_l[rank]:>+5.1f}, {top_b[rank]:>+5.1f})"
          f"{'':>5} {cnt:>10.0f} | "
          f"{'YES' if in_ours else 'no':>14} "
          f"{'YES' if in_asis else 'no':>15} "
          f"{'YES' if in_flipL else 'no':>16}")

print(f"\n  Hot pixel coverage (of top {top_n}):")
print(f"    Our mask:           {hits_ours}/{top_n} = {hits_ours/top_n:.1%}")
print(f"    Paper mask as-is:   {hits_asis}/{top_n} = {hits_asis/top_n:.1%}")
print(f"    Paper mask flipped: {hits_flipL}/{top_n} = {hits_flipL/top_n:.1%}")

print("\n  ★ 가장 높은 coverage = 우리 ccube와 정확히 정렬된 mask")
print("  ★ 우리 mask vs paper flipped 중 우세한 것이 우리 WCS convention과 일치")

# ---- Step 3: XOR 정량 비교 (paper mask flip vs 우리 mask) ---------------
print("\n" + "="*100)
print(" Mirror image 검증")
print("="*100)
for label, m in [('paper_asis  vs ours', mask_paper_asis),
                  ('paper_flipL vs ours', mask_paper_flipL),
                  ('paper_flipB vs ours', mask_paper_flipB)]:
    xor = np.logical_xor(m > 0.5, mask_ours > 0.5).mean()
    print(f"  {label}: XOR fraction = {xor:.4f}")

print("\n  ★ XOR ≈ 0 → 두 mask가 사실상 동일 (또는 거의 동일)")
print("  ★ 우리 mask가 paper_flipL과 가장 XOR 작으면 우리 mask = paper × flip(axis=2)")
print("    → 메모리 #2 기록과 일치")

# ---- Step 4: Visualization ----------------------------------------------
fig, axes = plt.subplots(2, 3, figsize=(17, 10), dpi=120)
extent = [-20, 20, -20, 20]

# Top row: CCUBE + each mask boundary overlay
for col, (title, m) in enumerate([
    ('Ours mask', mask_ours),
    ('Paper mask (as-is)', mask_paper_asis),
    ('Paper mask (flip axis=1)', mask_paper_flipL),
]):
    ax = axes[0, col]
    # CCUBE log scale
    im = ax.imshow(np.log10(ccube + 1), origin='lower', cmap='gray_r',
                   extent=extent, vmin=0, vmax=np.log10(ccube.max()))
    # Mask hole overlay (alpha=0.4 red where masked)
    hole = np.where(m < 0.5, 1, np.nan)
    ax.imshow(hole, origin='lower', cmap='Reds', alpha=0.35, extent=extent)
    # Top 10 hot pixel markers
    ax.scatter(top_l[:10], top_b[:10], facecolors='none', edgecolors='cyan',
                s=60, lw=1.5)
    ax.set_title(f'{title}\nbin {IB} (~0.89 GeV), red=mask hole, cyan=top10 hot pix')
    ax.set_xlabel('l [deg]'); ax.set_ylabel('b [deg]')

# Bottom row: 우리 mask vs paper variants의 XOR 시각화
for col, (title, m) in enumerate([
    ('Ours XOR paper_asis', mask_paper_asis),
    ('Ours XOR paper_flipL', mask_paper_flipL),
    ('Ours XOR paper_flipB', mask_paper_flipB),
]):
    ax = axes[1, col]
    xor_map = np.logical_xor(m > 0.5, mask_ours > 0.5).astype(float)
    im = ax.imshow(xor_map, origin='lower', cmap='Greens', extent=extent, vmin=0, vmax=1)
    ax.set_title(f'{title}\nfrac = {xor_map.mean():.4f}')
    ax.set_xlabel('l [deg]'); ax.set_ylabel('b [deg]')
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.savefig(OUTPUT_PNG, dpi=130, bbox_inches='tight')
plt.show()
print(f"\n[Saved] {OUTPUT_PNG}")

print("\n" + "="*100)
print(" 분기 해석:")
print("="*100)
print("  • '우리 mask' coverage가 가장 높음 → 우리 mask 정상, paper와 다른 WCS")
print("    → mask orientation 정상, c_swap은 mask가 아닌 다른 원인")
print("  • 'paper as-is' coverage가 가장 높음 → 우리 mask 잘못된 orientation!")
print("    → mask 생성 코드에 좌표 bug, fix 후 fit 재실행 필요")
print("  • 'paper flipL'와 '우리' XOR ≈ 0 → 둘이 사실상 동일 mask (메모리 #2 확인)")

In [ ]:
# ============================================================
# V46: Bubble/Iso FileFunction file vs paper 정합성 점검
# (V41은 χ² constraint file만 점검 — 이 셀은 XML의 FileFunction 파일을 점검)
# ============================================================
import os, subprocess, numpy as np, matplotlib.pyplot as plt

WORK_DIR = os.getcwd()  # 작업 dir에서 실행 가정
print(f"[CWD] {WORK_DIR}")

# ---------- (1) 관련 파일 inventory ----------
files_bub = [
    'fermi_bubble_spectrum.txt',
    'fermi_bubble_spectrum.txt.orig',
    'fermi_bubbles.txt',
    'bubble_constraints.txt',
]
files_iso = [
    'isotropic_spectrum_ff.txt',
    'isotropic_spectrum_ff.txt.orig',
    'iso_1410_3696_modelA.txt',
    'iso_constraints_full_err.txt',
]

def head(path, n=8):
    if not os.path.exists(path):
        print(f"  [MISSING] {path}")
        return
    sz = os.path.getsize(path)
    print(f"  --- {path} (size {sz} B) ---")
    with open(path) as f:
        for i, ln in enumerate(f):
            if i >= n: break
            print(f"    {ln.rstrip()}")

print("\n[BUBBLE family]")
for f in files_bub: head(f)
print("\n[ISO family]")
for f in files_iso: head(f)

# ---------- (2) XML이 실제 가리키는 file 경로 grep ----------
print("\n" + "="*64)
print("XML / py 파일 안의 file= 패턴 (bubble / iso / spectrum)")
print("="*64)
search_dirs = ['./XML_models', './GC_analysis_sanghwan/Model', '.']
for d in search_dirs:
    if not os.path.isdir(d): continue
    try:
        out = subprocess.run(
            ['grep', '-h', '-r', '-I', '--include=*.xml', '--include=*.py',
             '-E', r'file\s*=.*\.(txt|fits)', d],
            capture_output=True, text=True, timeout=10
        ).stdout
    except Exception as e:
        print(f"  grep err {d}: {e}"); continue
    hits = [ln.strip() for ln in out.splitlines()
            if any(k in ln.lower() for k in ('bubble','iso','spectrum'))]
    print(f"\n[{d}]  ({len(hits)} hits)")
    for ln in hits[:20]:
        print(f"  {ln}")

# ---------- (3) load + numerical summary ----------
def load_2col(path, verbose=True):
    """robust 2-col loader.
    여러 (delimiter, comments, skiprows) 조합을 순차 시도하고,
    실패하면 genfromtxt로 fallback. 어떤 옵션으로 로드됐는지 verbose 출력."""
    if not os.path.exists(path):
        return None
    attempts = [
        # (delimiter, comments, skiprows)  — whitespace + #/% comment 부터
        (None, ('#', '%'), 0),
        (None, ('#', '%'), 1),
        (',',  ('#', '%'), 0),
        (',',  ('#', '%'), 1),
        (None, '#',        1),
        (',',  '#',        1),
    ]
    for delim, com, sk in attempts:
        try:
            a = np.loadtxt(path, comments=com, delimiter=delim, skiprows=sk)
            if a.size > 0:
                if verbose:
                    print(f"    [load] {os.path.basename(path)}: "
                          f"delim={delim!r} skiprows={sk}")
                return a
        except Exception:
            continue
    # genfromtxt fallback (잘못된 라인은 skip)
    for kw in [{'delimiter': ',', 'skip_header': 1},
               {'skip_header': 1},
               {'delimiter': ','}]:
        try:
            a = np.genfromtxt(path, invalid_raise=False, **kw)
            if a is not None and a.size > 0 and not np.all(np.isnan(a)):
                # 모든 행에서 NaN만 있는 열 제거하지는 않음 — diagnostic만
                if verbose:
                    print(f"    [load] {os.path.basename(path)}: "
                          f"genfromtxt {kw}")
                return a
        except Exception:
            continue
    if verbose:
        print(f"    [load] {os.path.basename(path)}: ALL FAIL")
    return None

print("\n" + "="*64)
print("shape / column range / value range")
print("="*64)
loaded = {}
for f in files_bub + files_iso:
    a = load_2col(f)
    loaded[f] = a
    if a is None:
        print(f"  {f}: MISSING"); continue
    if a.ndim == 1:
        print(f"  {f}: 1D len={len(a)}  range={a.min():.3e}..{a.max():.3e}")
        continue
    cols = a.shape[1]
    print(f"  {f}: shape={a.shape}  E[{a[:,0].min():.3e},{a[:,0].max():.3e}] "
          f"v[{a[:,1].min():.3e},{a[:,1].max():.3e}]" + (f" ncol={cols}" if cols>2 else ""))

# ---------- (4) .orig vs current diff ----------
print("\n" + "="*64)
print(".orig vs current numerical diff")
print("="*64)
for base in ['fermi_bubble_spectrum.txt', 'isotropic_spectrum_ff.txt']:
    cur, org = loaded.get(base), loaded.get(base + '.orig')
    if cur is None or org is None:
        print(f"  {base}: skip (missing)"); continue
    if cur.shape != org.shape:
        print(f"  {base}: SHAPE diff  cur={cur.shape}  orig={org.shape}")
        continue
    d = cur - org
    print(f"  {base}: max|Δ|={np.abs(d).max():.3e}  "
          f"E identical? {np.allclose(cur[:,0], org[:,0])}")
    nz = org[:,1] != 0
    if nz.any():
        r = cur[nz,1] / org[nz,1]
        print(f"      value ratio: min={r.min():.4f}  median={np.median(r):.4f}  "
              f"max={r.max():.4f}")

# ---------- (5) paper formula reconstruction ----------
# Bubble: 1407.7905 Eq.13 log-parabola
#   dN/dE = I * (E/10GeV)^{-α - β log10(E/1GeV)}    [photons / cm² / s / sr / GeV]
#   α = 1.77, β = 0.063
#   I (normalization at 10 GeV) — paper Fig 18 baseline 값에서 추정:
#   E²dN/dE at 1 GeV ≈ 4–5 ×10⁻⁷ GeV/cm²/s/sr (Fig 18 우측 baseline)
# Iso: 1410.3696 Model A — paper Table III: PLE
#   dN/dE = K * (E/E0)^{-γ} * exp(-E/Ecut)
#   K = 0.95e-7 cm⁻² s⁻¹ sr⁻¹ GeV⁻¹ at E0=0.1 GeV, γ=2.32, Ecut=278.7 GeV
def bubble_LP_paper(E_GeV, I_10GeV=4.74e-7, alpha=1.77, beta=0.063):
    # dN/dE in 1/(GeV cm² s sr)
    return I_10GeV * (E_GeV/10.0)**(-alpha - beta*np.log10(E_GeV/1.0))
def iso_PLE_paper(E_GeV, K=0.95e-7, gamma=2.32, Ecut=278.7):
    return K * (E_GeV/0.1)**(-gamma) * np.exp(-E_GeV/Ecut)

# ---------- (6) plot: 같은 E²dN/dE [GeV/cm²/s/sr] 단위로 overlay ----------
# 파일의 E 단위가 MeV인지 GeV인지 자동 판단: max E > 1e3 이면 MeV
def to_E_GeV(arr):
    if arr is None or arr.ndim != 2: return None, None
    E_raw = arr[:,0]
    if E_raw.max() > 1e3:   # MeV 가정
        E_GeV = E_raw / 1000.0
        unit_label = 'MeV'
    else:
        E_GeV = E_raw
        unit_label = 'GeV'
    return E_GeV, unit_label

# FileFunction 단위: 보통 dN/dE [1/MeV/cm²/s/sr] (LAT 표준). 
# E²dN/dE [GeV/cm²/s/sr] = (E_GeV*1000)² * value / 1000  if value is per MeV
# = (E_GeV*1000)² * value * 1e-3      [GeV/cm²/s/sr]
# 그러나 일부 prior file은 이미 E²dN/dE 형태. heuristic: value < 1e-3 면 dN/dE per MeV
def to_e2dNdE(E_GeV, v):
    # 시도1: assume v is dN/dE [1/MeV/cm²/s/sr]
    cand_per_MeV = (E_GeV*1000.0)**2 * v * 1e-3
    # 시도2: assume v is already E²dN/dE [GeV/cm²/s/sr]
    cand_e2 = v
    # 시도3: assume v is dN/dE [1/GeV/...]
    cand_per_GeV = E_GeV**2 * v
    return {'as_dNdE_perMeV': cand_per_MeV,
            'as_E2dNdE_GeV':  cand_e2,
            'as_dNdE_perGeV': cand_per_GeV}

E_grid = np.geomspace(0.1, 500, 200)
paper_bub = E_grid**2 * bubble_LP_paper(E_grid)   # GeV/cm²/s/sr
paper_iso = E_grid**2 * iso_PLE_paper(E_grid)

fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True)
for col, fname in enumerate(['fermi_bubble_spectrum.txt',
                              'fermi_bubble_spectrum.txt.orig',
                              'bubble_constraints.txt']):
    ax = axes[0, col]
    a = loaded.get(fname)
    if a is None or a.ndim != 2:
        ax.set_title(f'{fname}\n(missing)'); continue
    E_GeV, u = to_E_GeV(a)
    cand = to_e2dNdE(E_GeV, a[:,1])
    for lab, y in cand.items():
        ax.loglog(E_GeV, np.abs(y), 'o-', ms=3, alpha=.7, label=lab)
    ax.loglog(E_grid, paper_bub, 'k--', lw=2, label='paper LP (Eq.13)')
    ax.set_title(f'{fname}\n(file E unit: {u})')
    ax.set_xlabel('E [GeV]'); ax.set_ylabel(r'E$^2$dN/dE [GeV/cm²/s/sr]')
    ax.set_ylim(1e-9, 1e-4); ax.legend(fontsize=7); ax.grid(alpha=.3)

for col, fname in enumerate(['isotropic_spectrum_ff.txt',
                              'isotropic_spectrum_ff.txt.orig',
                              'iso_constraints_full_err.txt']):
    ax = axes[1, col]
    a = loaded.get(fname)
    if a is None or a.ndim != 2:
        ax.set_title(f'{fname}\n(missing)'); continue
    E_GeV, u = to_E_GeV(a)
    cand = to_e2dNdE(E_GeV, a[:,1])
    for lab, y in cand.items():
        ax.loglog(E_GeV, np.abs(y), 'o-', ms=3, alpha=.7, label=lab)
    ax.loglog(E_grid, paper_iso, 'k--', lw=2, label='paper PLE Model A')
    ax.set_title(f'{fname}\n(file E unit: {u})')
    ax.set_xlabel('E [GeV]'); ax.set_ylabel(r'E$^2$dN/dE [GeV/cm²/s/sr]')
    ax.set_ylim(1e-9, 1e-4); ax.legend(fontsize=7); ax.grid(alpha=.3)

plt.suptitle('V46: FileFunction file vs paper formula\n'
             '3 candidate unit interpretations per file. '
             '검은 dashed가 paper. 정합되는 interpretation의 점들이 dashed 위에 와야 함.',
             fontsize=11)
plt.tight_layout()
plt.savefig('V46_bubble_iso_filefunction_audit.png', dpi=100, bbox_inches='tight')
plt.show()
print("\nSaved: V46_bubble_iso_filefunction_audit.png")

In [ ]:
# ============================================================
# V47: Likelihood chi² audit — runner v3.x의 chi²_bub / chi²_iso 형태 확인
# Paper Cholis+2022 Eq.13 및 Sanghwan 원본과 line-by-line 비교용
# ============================================================
import os, re, textwrap

# ---------- runner 후보 자동 탐색 ----------
candidates = [
    './run_main_loop_subprocess.py',
    './run_main_loop.py',
]
runner_path = None
for c in candidates:
    if os.path.exists(c):
        runner_path = c
        break
if runner_path is None:
    raise FileNotFoundError(f"runner not found in {candidates}")
print(f"[runner] {runner_path}")

with open(runner_path) as f:
    src = f.read()
lines = src.split('\n')

# ---------- (1) RUNNER_VERSION ----------
m = re.search(r'RUNNER_VERSION\s*=\s*[\'"]([^\'"]+)[\'"]', src)
print(f"[version] RUNNER_VERSION = {m.group(1) if m else 'NOT FOUND'}")

# ---------- (2) likelihood_constrained 함수 전체 추출 ----------
def_lines = [(i, ln) for i, ln in enumerate(lines)
             if re.match(r'\s*def\s+likelihood_constrained', ln)]
if not def_lines:
    print("[FATAL] likelihood_constrained 함수를 못 찾음")
else:
    for start_i, def_ln in def_lines:
        head_indent = len(def_ln) - len(def_ln.lstrip())
        body = []
        for j in range(start_i + 1, len(lines)):
            ln = lines[j]
            if ln.strip() == '':
                body.append((j + 1, ln))
                continue
            cur_indent = len(ln) - len(ln.lstrip())
            if cur_indent <= head_indent:
                break
            body.append((j + 1, ln))
        print(f"\n{'='*72}\nlikelihood_constrained @ line {start_i+1}  "
              f"(body {len(body)} lines)\n{'='*72}")
        print(f"{start_i+1:4d}  {def_ln}")
        for ln_no, ln in body:
            print(f"{ln_no:4d}  {ln}")

# ---------- (3) chi² 관련 라인 grep (전 파일) ----------
print(f"\n{'='*72}\n[grep] chi² 관련 토큰\n{'='*72}")
tokens = ['chi2_bub', 'chi2_bubble', 'chi2_iso', 'chi2_isotropic',
          'upper_err', 'lower_err', 'upper_error', 'lower_error',
          'bubble_flux_data', 'isotropic_flux_data',
          'bubble_constraints', 'iso_constraints',
          'bubble_sed', 'iso_sed', 'iso_flux']
seen = set()
for i, ln in enumerate(lines):
    if any(t in ln for t in tokens):
        if i in seen:
            continue
        seen.add(i)
        print(f"{i+1:4d}  {ln.rstrip()}")

# ---------- (4) reference form 출력 ----------
print(f"\n{'='*72}\nPAPER Eq.13 (Cholis+2022)\n{'='*72}")
print(textwrap.dedent("""\
  -2 ln L_j = 2 Σ_p [C_jp - D_jp·ln(C_jp) + ln(D_jp!)]
              + χ²_Bub,j  +  χ²_Iso,j

  Poisson 항: per-bin pixel sum, expected C = Σ c_m · template_m
  χ² 항: paper 본문에 정확한 form 명시 없음 (sym vs asym)
"""))

print(f"{'='*72}\nSanghwan 원본 (참조)\n{'='*72}")
print(textwrap.dedent("""\
  # bubble (ASYMMETRIC error)
  if flux_data[i] < bubble_sed[i]:    # model > data
      chi2 = ((bubble_sed[i] - flux_data[i]) / upper_error_data[i])**2
  if flux_data[i] > bubble_sed[i]:    # model < data
      chi2 = ((bubble_sed[i] - flux_data[i]) / lower_error_data[i])**2
  if flux_data[i] == bubble_sed[i]:
      chi2 = ((bubble_sed[i] - flux_data[i]) / larger_error)**2

  # iso (SYMMETRIC — single error column)
  chi2_iso = ((iso_flux[i] - iso_constraints_flux[i])**2
              / iso_constraints_flux_err[i]**2)
"""))

# ---------- (5) constraint file column 매핑 점검 ----------
print(f"{'='*72}\n[grep] constraint file 로드 + column 인덱싱\n{'='*72}")
for i, ln in enumerate(lines):
    if any(t in ln for t in ['bubble_constraints.txt', 'iso_constraints_full_err.txt',
                              'fermi_bubble_spectrum.txt', 'isotropic_spectrum_ff.txt']):
        # 주변 context 3줄
        lo = max(0, i - 1)
        hi = min(len(lines), i + 3)
        print(f"--- line {i+1} context ---")
        for k in range(lo, hi):
            mark = '>>>' if k == i else '   '
            print(f"{mark} {k+1:4d}  {lines[k].rstrip()}")
        print()

# ---------- (6) chi²의 단위 일관성 빠른 확인 ----------
# constraint 파일은 dN/dE [1/GeV/cm²/s/sr] (V46에서 확인)
# bubble_sed, iso_sed는 NON-CONVOL map / exp_cube / mask-avg → 무슨 단위?
print(f"{'='*72}\n[grep] bubble_sed / iso_flux 계산 라인 (단위 점검)\n{'='*72}")
for i, ln in enumerate(lines):
    if ('bubble_sed' in ln or 'iso_flux' in ln or 'iso_sed' in ln) and '=' in ln:
        lo = max(0, i - 1)
        hi = min(len(lines), i + 4)
        print(f"--- line {i+1} context ---")
        for k in range(lo, hi):
            mark = '>>>' if k == i else '   '
            print(f"{mark} {k+1:4d}  {lines[k].rstrip()}")
        print()

In [ ]:
# ============================================================
# V48a: runner v3.15 — chi² external constraint toggle patch
# SKIP_CHI2_EXTERNAL=1 환경변수일 때 chi²_bub + chi²_iso 항 skip
# PATCH_MARKER 기반 idempotent (재실행 안전)
# ============================================================
import os, re

runner = './run_main_loop_subprocess.py'
PATCH_MARKER = 'PATCH_V48_NO_CHI2'

with open(runner) as f:
    src = f.read()

if PATCH_MARKER in src:
    print(f"[skip] {runner} already patched ({PATCH_MARKER})")
else:
    target = '            return np.sum(lhd) + chi2_bubble + chi2_isotropic'
    if target not in src:
        raise RuntimeError(
            f"target line not found in {runner}. "
            "runner version이 v3.15에서 변경되었을 가능성. "
            "수동 확인 필요.")
    replacement = (
        f'            # {PATCH_MARKER} begin\n'
        f'            if os.environ.get(\'SKIP_CHI2_EXTERNAL\', \'0\') == \'1\':\n'
        f'                return np.sum(lhd)\n'
        f'            # {PATCH_MARKER} end\n'
        f'            return np.sum(lhd) + chi2_bubble + chi2_isotropic'
    )
    # 한 번만 치환 (가장 첫 매치)
    src_patched = src.replace(target, replacement, 1)
    # 안전 점검: import os가 이미 있는지
    if 'import os' not in src_patched:
        raise RuntimeError("'import os' not found at top — abort, manual fix needed")
    with open(runner, 'w') as f:
        f.write(src_patched)
    print(f"[patched] {runner}: {PATCH_MARKER} 추가됨")
    print(f"          SKIP_CHI2_EXTERNAL=1 환경변수로 chi² 항 toggle")

In [ ]:
# V48b-precheck: runner의 [SKIP MODEL] 분기 조건 확인
import re
runner = './run_main_loop_subprocess.py'
with open(runner) as f:
    lines = f.read().split('\n')

print("[grep] SKIP / already exists / needs_run")
for i, ln in enumerate(lines):
    if any(k in ln for k in ['SKIP MODEL', 'already exists',
                              'needs_run', 'skip_or_run',
                              os.path.basename('GCE_model_')]):
        lo, hi = max(0, i-1), min(len(lines), i+4)
        print(f"--- line {i+1} ---")
        for k in range(lo, hi):
            mark = '>>>' if k == i else '   '
            print(f"{mark} {k+1:4d}  {lines[k].rstrip()}")
        print()

In [ ]:
# ============================================================
# V48b-rev: Model I 단일 실행 (no chi² external) — copy2 → move 변경
# 백업으로 원본 이동 → runner 실행 → 결과 rename → finally로 원본 복원
# ============================================================
import os, subprocess, shutil

MODEL = 'I'
runner = './run_main_loop_subprocess.py'

# parallel state 안전 점검 (이미 정리했지만 한 번 더)
if os.path.exists('./parallel_run_state.json'):
    print("[WARN] parallel_run_state.json 아직 존재. 확인 후 진행")
else:
    print("[ok] no parallel state file")

# 1) 원본 파일들을 백업 위치로 '이동' (원본 자리 비우기)
patterns = [
    f'./GCE_model_{MODEL}_12yr_cholis.dat',
    f'./GCE_model_{MODEL}_12yr_cholis_likelihood_value',
    f'./GCE_model_{MODEL}_12yr_cholis_fit.npz',
]
backups = []
for p in patterns:
    if os.path.exists(p):
        bak = p + '.preB_test'
        # 만약 이전 시도의 백업이 남아있으면 그것을 우선 보존 (덮어쓰지 않음)
        if os.path.exists(bak):
            print(f"  [keep] existing backup: {bak}")
            # 원본은 그냥 삭제 (백업이 이미 진본)
            os.remove(p)
            backups.append((p, bak))
        else:
            shutil.move(p, bak)
            backups.append((p, bak))
            print(f"  moved: {p} → {bak}")
    else:
        print(f"  [absent] {p} (skip backup)")

# 원본 자리가 진짜로 비었는지 확인
for p in patterns:
    if os.path.exists(p):
        raise RuntimeError(f"원본 자리가 안 비었음: {p}")
print("  [verified] all original slots are empty")

# 2) noChi2 실행
print(f"\n[run] {runner} {MODEL} with SKIP_CHI2_EXTERNAL=1")
env = os.environ.copy()
env['SKIP_CHI2_EXTERNAL'] = '1'
cmd_args = ['python', runner, MODEL]

try:
    r = subprocess.run(cmd_args, env=env, capture_output=True,
                       text=True, timeout=7200)
    print("--- STDOUT (last 4000 chars) ---")
    print(r.stdout[-4000:])
    if r.returncode != 0:
        print("--- STDERR (last 2000 chars) ---")
        print(r.stderr[-2000:])
        print(f"[FAIL] returncode={r.returncode}")
    else:
        print(f"\n[ok] runner exit 0")
    
    # 3) 결과를 noChi2 이름으로 rename
    saved_any = False
    for p in patterns:
        if os.path.exists(p):
            new = p.replace('cholis', 'cholis_noChi2')
            if os.path.exists(new):
                os.remove(new)
                print(f"  [overwrite] removed existing: {new}")
            shutil.move(p, new)
            print(f"  saved: {p} → {new}")
            saved_any = True
        else:
            print(f"  [missing] {p} — runner가 결과를 안 만듦")
    
    if not saved_any:
        print("[FATAL] runner가 어떤 결과도 안 만들었음. log 위 stdout 확인.")
        
finally:
    # 4) 원본 복원 (백업 → 원본 자리)
    for orig, bak in backups:
        if os.path.exists(bak):
            shutil.move(bak, orig)
            print(f"  restored: {bak} → {orig}")
    print("[done] 원본 결과 복원 완료")

In [ ]:
# ============================================================
# V48c: with vs no chi² external — per-bin c_param 비교
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

MODEL = 'I'
npz_with    = np.load(f'./GCE_model_{MODEL}_12yr_cholis_fit.npz')
npz_no_chi2 = np.load(f'./GCE_model_{MODEL}_12yr_cholis_noChi2_fit.npz')

print(f"[with chi²]  keys: {list(npz_with.keys())}")
print(f"[no  chi²]   keys: {list(npz_no_chi2.keys())}")
print(f"[with chi²]  fitted_params shape: {npz_with['fitted_params'].shape}")
print(f"[no  chi²]   fitted_params shape: {npz_no_chi2['fitted_params'].shape}")

fp_w = npz_with['fitted_params']
fp_n = npz_no_chi2['fitted_params']
lo_w = npz_with['fitted_params_lower']
up_w = npz_with['fitted_params_upper']
lo_n = npz_no_chi2['fitted_params_lower']
up_n = npz_no_chi2['fitted_params_upper']

# 1D flat array (size 5*n_bins) 또는 2D 모두 처리
def _to_2d(arr):
    """Sanghwan convention: arr[c*n : (c+1)*n] = component c의 n개 bin
    → reshape(5, n_bins). 이미 2D면 (5, n_bins) 방향으로 정렬."""
    if arr.ndim == 1:
        if arr.size % 5 != 0:
            raise RuntimeError(f"flat array size {arr.size} not divisible by 5")
        return arr.reshape(5, arr.size // 5)
    if arr.ndim == 2:
        if arr.shape[0] == 5:
            return arr
        if arr.shape[1] == 5:
            return arr.T
    raise RuntimeError(f"unexpected shape: {arr.shape}")

fp_w = _to_2d(fp_w);   fp_n = _to_2d(fp_n)
lo_w = _to_2d(lo_w);   up_w = _to_2d(up_w)
lo_n = _to_2d(lo_n);   up_n = _to_2d(up_n)
n_bins = fp_w.shape[1]
print(f"  → 5 params × {n_bins} bins (after reshape)")

# E grid
if 'E' in npz_with.files:
    E = npz_with['E']
    if len(E) != n_bins:
        print(f"  [warn] E length {len(E)} ≠ n_bins {n_bins}, using bin index")
        E = np.arange(n_bins)
else:
    E = np.arange(n_bins)
    print(f"  [warn] E not in npz, using bin index")

labels = ['c_pb (gas)', 'c_ics', 'c_gce', 'c_bub', 'c_iso']
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
for i, lab in enumerate(labels):
    ax = axes[i]
    err_w = np.vstack([np.maximum(fp_w[i]-lo_w[i], 0),
                        np.maximum(up_w[i]-fp_w[i], 0)])
    err_n = np.vstack([np.maximum(fp_n[i]-lo_n[i], 0),
                        np.maximum(up_n[i]-fp_n[i], 0)])
    ax.errorbar(E, fp_w[i], yerr=err_w, fmt='o-', label='with chi²',
                capsize=3, alpha=0.85)
    ax.errorbar(E, fp_n[i], yerr=err_n, fmt='s--', label='no chi²',
                capsize=3, alpha=0.85)
    if np.all(np.asarray(E) > 0):
        ax.set_xscale('log')
    ax.set_title(lab)
    ax.set_xlabel('E [GeV]' if 'E' in npz_with.files else 'bin')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
axes[0].set_ylabel('c (best-fit)')
plt.suptitle(f'V48c: Model {MODEL} per-bin c_param — with vs without external chi²',
              fontsize=11)
plt.tight_layout()
plt.savefig(f'V48c_compare_chi2_off_model{MODEL}.png',
             dpi=100, bbox_inches='tight')
plt.show()

# 정량 비교
print(f"\n{'='*70}")
print(f"per-component ratio (no_chi2 / with_chi2)")
print(f"{'='*70}")
for i, lab in enumerate(labels):
    r = fp_n[i] / np.where(fp_w[i] != 0, fp_w[i], np.nan)
    r_clean = r[np.isfinite(r)]
    print(f"  {lab:14s}: mean={np.nanmean(r):.3f}  median={np.nanmedian(r):.3f}  "
          f"std={np.nanstd(r):.3f}  range=[{np.nanmin(r):.3f}, {np.nanmax(r):.3f}]")

# 분석 범위 bin 0-13에서만 별도 정량
print(f"\n[bin 0-13만 (Cholis 14-bin 범위)]")
for i, lab in enumerate(labels):
    r = fp_n[i, :14] / np.where(fp_w[i, :14] != 0, fp_w[i, :14], np.nan)
    print(f"  {lab:14s}: mean={np.nanmean(r):.3f}  median={np.nanmedian(r):.3f}")

In [ ]:
# ============================================================
# V49: Fermi_Bubbles_template.fits 직접 점검
# - shape / WCS / 값 분포 / ellipse 모양 / orientation
# ============================================================
import os, numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS

tpath = './Fermi_Bubbles_template.fits'
print(f"[file] {tpath}  exists={os.path.exists(tpath)}  "
      f"size={os.path.getsize(tpath)} B")

with fits.open(tpath) as hdul:
    print("\n[HDU list]")
    for i, h in enumerate(hdul):
        print(f"  HDU{i}: {h.__class__.__name__}  shape="
              f"{h.data.shape if h.data is not None else None}  "
              f"dtype={h.data.dtype if h.data is not None else None}")
    data = hdul[0].data
    hdr  = hdul[0].header

# 1) WCS / header
print("\n[Primary HDU header — WCS related]")
for k in ['NAXIS','NAXIS1','NAXIS2','NAXIS3','CTYPE1','CTYPE2','CTYPE3',
          'CRVAL1','CRVAL2','CRVAL3','CRPIX1','CRPIX2','CRPIX3',
          'CDELT1','CDELT2','CDELT3','CUNIT1','CUNIT2','CUNIT3',
          'BUNIT','EQUINOX']:
    if k in hdr:
        print(f"  {k:8s} = {hdr[k]}")

# 2) 값 분포 통계
print(f"\n[data stats]  shape={data.shape}  dtype={data.dtype}")
print(f"  min  = {np.nanmin(data):.6g}")
print(f"  max  = {np.nanmax(data):.6g}")
print(f"  mean = {np.nanmean(data):.6g}")
print(f"  nonzero pixels: {np.sum(data != 0)} / {data.size}  "
      f"({100*np.sum(data!=0)/data.size:.2f}%)")
print(f"  unique values count: {len(np.unique(data))}")
# 분포 histogram for binary check
uvals, ucnts = np.unique(data, return_counts=True)
if len(uvals) <= 5:
    print(f"  unique values: {dict(zip(uvals, ucnts))}")
else:
    print(f"  unique values (top 5 by count):")
    idx = np.argsort(-ucnts)[:5]
    for i in idx:
        print(f"    {uvals[i]:.6g}  count={ucnts[i]}")
    # spatial-flat 점검: 값이 0과 1만 있는가?
    nz_vals = data[data != 0]
    print(f"  nonzero values: min={nz_vals.min():.4g}  "
          f"max={nz_vals.max():.4g}  mean={nz_vals.mean():.4g}  "
          f"std={nz_vals.std():.4g}")

# 3) 2D 평면 추출 (3D면 첫 plane)
if data.ndim == 3:
    img = data[0]
elif data.ndim == 2:
    img = data
else:
    raise RuntimeError(f"unexpected ndim: {data.ndim}")

# 4) WCS로 l, b 격자 계산 + ellipse extent 추정
try:
    w = WCS(hdr).celestial
    ny, nx = img.shape
    yy, xx = np.indices(img.shape)
    l_deg, b_deg = w.wcs_pix2world(xx, yy, 0)
    # paper convention: longitude는 ±180° 표현
    l_deg = np.where(l_deg > 180, l_deg - 360, l_deg)
    
    nz_mask = img != 0
    if nz_mask.any():
        print(f"\n[non-zero pixel sky extent]")
        print(f"  l range: [{l_deg[nz_mask].min():.2f}, {l_deg[nz_mask].max():.2f}] deg")
        print(f"  b range: [{b_deg[nz_mask].min():.2f}, {b_deg[nz_mask].max():.2f}] deg")
        print(f"  Paper 1407.7905 Fig 3: l ~ ±15°, b ~ ±50°")
except Exception as e:
    print(f"  [WCS fail] {e}")
    l_deg, b_deg = None, None

# 5) 시각화 — 3 panel
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# (a) full template
im0 = axes[0].imshow(img, origin='lower', cmap='inferno')
axes[0].set_title(f'full template\nshape={img.shape}')
plt.colorbar(im0, ax=axes[0], fraction=0.04)

# (b) ROI 영역 확대 — 600×600 이면 [100:500, 100:500] (Sanghwan 컨벤션)
# 다른 크기면 중앙 60×60 deg에 해당하는 영역
if img.shape == (600, 600):
    sub = img[100:500, 100:500]
    axes[1].set_title('analysis ROI [100:500, 100:500]\n(60°×60° fit region)')
else:
    # 일반적으로 중앙 60°
    cy, cx = img.shape[0]//2, img.shape[1]//2
    sub = img[cy-200:cy+200, cx-200:cx+200] if img.shape[0]>400 else img
    axes[1].set_title(f'central crop\nshape={sub.shape}')
im1 = axes[1].imshow(sub, origin='lower', cmap='inferno')
plt.colorbar(im1, ax=axes[1], fraction=0.04)

# (c) flipped 비교 — V44 mask가 flip(axis=1) 필요했던 것 같이 template도 영향 받는지
img_flip = np.flip(img, axis=1)
im2 = axes[2].imshow(img_flip, origin='lower', cmap='inferno')
axes[2].set_title('flip(axis=1)\n(WCS CDELT1<0 convention 점검)')
plt.colorbar(im2, ax=axes[2], fraction=0.04)

plt.suptitle('V49: Fermi_Bubbles_template.fits 점검\n'
             'Paper convention: ellipse 영역 1, 외부 0 (spatial-flat)',
             fontsize=11)
plt.tight_layout()
plt.savefig('V49_bubble_template_audit.png', dpi=100, bbox_inches='tight')
plt.show()

# 6) GC 중심에서 b=0 cut, l=0 cut 프로파일 — ellipse 확인
if l_deg is not None and b_deg is not None:
    fig2, axes2 = plt.subplots(1, 2, figsize=(12, 4))
    # b ≈ 0 line (galactic plane)
    b_zero_idx = np.argmin(np.abs(b_deg[:, img.shape[1]//2]))
    profile_l = img[b_zero_idx, :]
    axes2[0].plot(l_deg[b_zero_idx, :], profile_l)
    axes2[0].set_xlabel('Galactic longitude l [deg]')
    axes2[0].set_ylabel('template value')
    axes2[0].set_title(f'cut along b ≈ 0  (row {b_zero_idx})')
    axes2[0].axvline(0, color='r', alpha=0.3)
    axes2[0].grid(alpha=0.3)
    
    # l ≈ 0 line
    l_zero_idx = np.argmin(np.abs(l_deg[img.shape[0]//2, :]))
    profile_b = img[:, l_zero_idx]
    axes2[1].plot(b_deg[:, l_zero_idx], profile_b)
    axes2[1].set_xlabel('Galactic latitude b [deg]')
    axes2[1].set_ylabel('template value')
    axes2[1].set_title(f'cut along l ≈ 0  (col {l_zero_idx})')
    axes2[1].axvline(0, color='r', alpha=0.3)
    axes2[1].grid(alpha=0.3)
    plt.suptitle('V49: profile cuts — paper ellipse: l~±15°, b~±50°', fontsize=11)
    plt.tight_layout()
    plt.savefig('V49_bubble_template_profiles.png', dpi=100, bbox_inches='tight')
    plt.show()

In [ ]:
# ============================================================
# V49b: Fermi_Bubbles_template 원본 추적 + Sanghwan/Cholis 비교
# ============================================================
import os, glob, subprocess, numpy as np
from astropy.io import fits

# 1) 우리 작업 dir 안의 bubble template 후보 모두 찾기
print("=" * 70)
print("작업 dir 안의 bubble template 후보")
print("=" * 70)
patterns = ['Fermi_Bubbles*', '*bubble*template*', '*Bubbles*template*']
for pat in patterns:
    for f in sorted(glob.glob(pat)):
        if not os.path.isfile(f):
            continue
        sz = os.path.getsize(f)
        print(f"  {f}  ({sz} B)")

# 2) Sanghwan dir 안의 bubble 관련 fits
print("\n" + "=" * 70)
print("Sanghwan Templates/ 디렉토리 검색")
print("=" * 70)
sanghwan_template_dirs = [
    '/home/sanghwan/FermiLAT/Sanghwan/Templates',
    '/home/sanghwan/FermiLAT/Fermi-LAT-GCE',
]
for d in sanghwan_template_dirs:
    if not os.path.isdir(d):
        print(f"  [absent] {d}")
        continue
    print(f"\n[{d}]")
    try:
        r = subprocess.run(['find', d, '-maxdepth', '4', '-iname',
                            '*bubble*', '-type', 'f'],
                           capture_output=True, text=True, timeout=30)
        for ln in r.stdout.strip().split('\n'):
            if ln:
                sz = os.path.getsize(ln) if os.path.exists(ln) else 0
                print(f"  {ln}  ({sz} B)")
    except Exception as e:
        print(f"  [err] {e}")

# 3) Cholis Zenodo bubble 관련
print("\n" + "=" * 70)
print("Cholis Zenodo bubble 관련")
print("=" * 70)
zenodo = '../GCE_TEMPLATES_FILES_v3'
if os.path.isdir(zenodo):
    r = subprocess.run(['find', zenodo, '-maxdepth', '4', '-iname',
                        '*bubble*', '-type', 'f'],
                       capture_output=True, text=True, timeout=30)
    for ln in r.stdout.strip().split('\n'):
        if ln:
            sz = os.path.getsize(ln) if os.path.exists(ln) else 0
            print(f"  {ln}  ({sz} B)")
else:
    print(f"  [absent] {zenodo}")

# 4) 우리 fits header의 추가 정보 (HISTORY/COMMENT 안전 접근)
print("\n" + "=" * 70)
print("우리 Fermi_Bubbles_template.fits header 추가 정보")
print("=" * 70)
with fits.open('./Fermi_Bubbles_template.fits') as hdul:
    h = hdul[0].header
    # HISTORY / COMMENT 안전 접근
    for kw in ['HISTORY', 'COMMENT']:
        if kw in h:
            cards = h[kw]
            print(f"  [{kw}] {len(cards)} cards:")
            for c in cards:
                print(f"    {kw} {c}")
        else:
            print(f"  [{kw}] absent")
    # 시간/생성자 메타
    for kw in ['DATE', 'CREATOR', 'ORIGIN', 'TELESCOP', 'INSTRUME']:
        if kw in h:
            print(f"  {kw} = {h[kw]}")
    # 전체 keyword 목록 (header 카드 종류 파악)
    print(f"\n  [all header keywords ({len(h)} cards)]")
    seen = set()
    for k in h.keys():
        if k and k not in seen:
            seen.add(k)
            print(f"    {k}")

In [ ]:
# ============================================================
# V49c: 우리 vs Sanghwan bubble template 직접 비교
# + Sanghwan creation notebook의 핵심 cell 발췌
# ============================================================
import os, json, numpy as np
from astropy.io import fits
import matplotlib.pyplot as plt

our_path = './Fermi_Bubbles_template.fits'
sang_path = '/home/sanghwan/FermiLAT/Fermi-LAT-GCE/templates/Fermi_bubble.fits'
sang_nb  = '/home/sanghwan/FermiLAT/Fermi-LAT-GCE/Fermi_bubble_template_creation.ipynb'

# 1) Sanghwan template 정보
print("=" * 70)
print("Sanghwan Fermi_bubble.fits")
print("=" * 70)
with fits.open(sang_path) as hdul:
    for i, h in enumerate(hdul):
        print(f"  HDU{i}: shape="
              f"{h.data.shape if h.data is not None else None}  "
              f"dtype={h.data.dtype if h.data is not None else None}")
    sdata = hdul[0].data
    shdr  = hdul[0].header
print(f"\n[WCS keys]")
for k in ['NAXIS','NAXIS1','NAXIS2','CTYPE1','CTYPE2','CRVAL1','CRVAL2',
          'CRPIX1','CRPIX2','CDELT1','CDELT2','BUNIT']:
    if k in shdr:
        print(f"  {k:8s} = {shdr[k]}")
print(f"\n[Sanghwan header HISTORY / COMMENT]")
try:
    for c in shdr['HISTORY']: print(f"  HISTORY {c}")
except KeyError: print("  HISTORY absent")
try:
    for c in shdr['COMMENT']: print(f"  COMMENT {c}")
except KeyError: print("  COMMENT absent")

print(f"\n[Sanghwan data stats]")
print(f"  min={np.nanmin(sdata):.6g}  max={np.nanmax(sdata):.6g}")
print(f"  nonzero pixels: {np.sum(sdata != 0)} / {sdata.size}")
suvals, sucnts = np.unique(sdata, return_counts=True)
if len(suvals) <= 5:
    print(f"  unique values: {dict(zip(suvals, sucnts))}")
else:
    idx = np.argsort(-sucnts)[:5]
    print(f"  top-5 unique values:")
    for i in idx:
        print(f"    {suvals[i]:.6g}  count={sucnts[i]}")

# 2) 우리 vs Sanghwan 비교 plot
with fits.open(our_path) as hdul:
    odata = hdul[0].data
    ohdr  = hdul[0].header

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# (a) 우리 full
im00 = axes[0,0].imshow(odata, origin='lower', cmap='inferno')
axes[0,0].set_title(f'OURS  shape={odata.shape}\n'
                     f'max={odata.max():.3f}  nz_frac={(odata!=0).mean():.3f}')
plt.colorbar(im00, ax=axes[0,0], fraction=0.04)

# (b) Sanghwan full
im01 = axes[0,1].imshow(sdata, origin='lower', cmap='inferno')
axes[0,1].set_title(f'SANGHWAN  shape={sdata.shape}\n'
                     f'max={sdata.max():.3f}  nz_frac={(sdata!=0).mean():.3f}')
plt.colorbar(im01, ax=axes[0,1], fraction=0.04)

# (c) shape이 같으면 직접 diff
if odata.shape == sdata.shape:
    diff = odata - sdata
    vmax = max(abs(diff.min()), abs(diff.max()))
    im10 = axes[1,0].imshow(diff, origin='lower', cmap='RdBu_r',
                              vmin=-vmax, vmax=vmax)
    axes[1,0].set_title(f'OURS - SANGHWAN\nmax|Δ|={np.abs(diff).max():.4g}')
    plt.colorbar(im10, ax=axes[1,0], fraction=0.04)
    
    nz_both = (odata != 0) & (sdata != 0)
    if nz_both.any():
        ratio = odata[nz_both] / sdata[nz_both]
        print(f"\n[OURS / SANGHWAN ratio in common non-zero region]")
        print(f"  median = {np.median(ratio):.4f}")
        print(f"  mean   = {np.mean(ratio):.4f}")
        print(f"  range  = [{ratio.min():.4f}, {ratio.max():.4f}]")
else:
    axes[1,0].text(0.5, 0.5, f'shape mismatch\nours={odata.shape}\n'
                              f'sang={sdata.shape}', ha='center', va='center',
                   transform=axes[1,0].transAxes, fontsize=12)
    axes[1,0].axis('off')

# (d) 양쪽 binary mask overlay (nonzero region 위치)
axes[1,1].imshow(odata != 0, origin='lower', cmap='Blues', alpha=0.5)
if odata.shape == sdata.shape:
    axes[1,1].imshow(sdata != 0, origin='lower', cmap='Reds', alpha=0.4)
    axes[1,1].set_title('OURS (blue) vs SANGHWAN (red)\nnonzero overlap')
else:
    axes[1,1].set_title('OURS nonzero (shape mismatch with sang)')

plt.suptitle('V49c: bubble template — 우리 vs Sanghwan 직접 비교', fontsize=12)
plt.tight_layout()
plt.savefig('V49c_bubble_template_compare.png', dpi=100, bbox_inches='tight')
plt.show()

# 3) Sanghwan notebook의 bubble 생성 cell 핵심 추출
print("\n" + "=" * 70)
print("Sanghwan Fermi_bubble_template_creation.ipynb 핵심 cells")
print("=" * 70)
with open(sang_nb) as f:
    nb = json.load(f)

for ci, cell in enumerate(nb['cells']):
    if cell['cell_type'] != 'code':
        continue
    src_text = ''.join(cell['source'])
    # 핵심 키워드 cell만 출력
    keywords = ['ellipse', 'bubble', 'fits.writeto', 'np.zeros', 
                'CRVAL', 'CDELT', 'mask', '*= ', 'template[']
    if any(k in src_text for k in keywords) and len(src_text) > 50:
        print(f"\n--- Cell {ci} (code, {len(src_text.splitlines())} lines) ---")
        # 너무 길면 잘라서
        if len(src_text) > 2000:
            print(src_text[:2000])
            print(f"  ... ({len(src_text)} chars total, truncated)")
        else:
            print(src_text)

In [ ]:
# ============================================================
# V49d: Sanghwan main fit이 실제로 쓰는 bubble template 추적
# ============================================================
import os, subprocess, glob

# 1) Sanghwan main fit notebook과 디렉토리에서 bubble fits 참조 grep
sang_dirs = [
    '/home/sanghwan/FermiLAT/Sanghwan',
    '/home/sanghwan/FermiLAT/Fermi-LAT-GCE',
]

print("=" * 70)
print("Sanghwan main 노트북/스크립트 안에서 bubble fits 참조")
print("=" * 70)
for d in sang_dirs:
    if not os.path.isdir(d):
        print(f"  [absent] {d}")
        continue
    print(f"\n[{d}]")
    try:
        r = subprocess.run(
            ['grep', '-r', '-h', '-I',
             '--include=*.ipynb', '--include=*.py',
             '-E', r'(Fermi_bubble|fermi_bubble).*\.fits',
             d],
            capture_output=True, text=True, timeout=30)
        # 중복 제거
        lines = set(ln.strip() for ln in r.stdout.split('\n') if '.fits' in ln)
        for ln in sorted(lines)[:20]:
            print(f"  {ln}")
        if len(lines) > 20:
            print(f"  ... ({len(lines)} total unique lines)")
    except Exception as e:
        print(f"  [err] {e}")

# 2) Sanghwan 디렉토리 내 모든 bubble 관련 fits 파일 inventory
print("\n" + "=" * 70)
print("Sanghwan 디렉토리 안의 bubble fits 파일들")
print("=" * 70)
for d in sang_dirs:
    if not os.path.isdir(d):
        continue
    print(f"\n[{d}]")
    try:
        r = subprocess.run(
            ['find', d, '-maxdepth', '6', '-iname', '*bubble*.fits',
             '-type', 'f'],
            capture_output=True, text=True, timeout=60)
        for ln in sorted(r.stdout.strip().split('\n')):
            if ln:
                sz = os.path.getsize(ln)
                # shape 빠르게 확인
                try:
                    from astropy.io import fits
                    with fits.open(ln) as hdul:
                        sh = hdul[0].data.shape if hdul[0].data is not None else 'no-data'
                        cdelt = hdul[0].header.get('CDELT1', '?')
                except Exception:
                    sh, cdelt = '?', '?'
                print(f"  {ln}")
                print(f"    size={sz} B  shape={sh}  CDELT1={cdelt}")
    except Exception as e:
        print(f"  [err] {e}")

# 3) 우리 노트북·runner가 Fermi_Bubbles_template.fits를 어디서 만들거나 받았는지
print("\n" + "=" * 70)
print("우리 디렉토리에서 Fermi_Bubbles_template.fits 만든 곳 추적")
print("=" * 70)
r = subprocess.run(
    ['grep', '-r', '-l', '-I', '--include=*.ipynb', '--include=*.py',
     'Fermi_Bubbles_template', '.'],
    capture_output=True, text=True, timeout=30)
files_using = set(r.stdout.strip().split('\n'))
for f in sorted(files_using):
    if f and os.path.isfile(f):
        print(f"  {f}")

# writeto 또는 create 패턴 찾기
print("\n[create/writeto 패턴 검색]")
r = subprocess.run(
    ['grep', '-r', '-h', '-I', '--include=*.ipynb', '--include=*.py',
     '-E', r'(writeto|fits\.PrimaryHDU).*[Bb]ubble',
     '.'],
    capture_output=True, text=True, timeout=30)
for ln in set(r.stdout.split('\n')):
    if ln.strip():
        print(f"  {ln.strip()}")

In [ ]:
# ============================================================
# V49e: 우리 vs Sanghwan의 bubble component map 직접 비교
# (gtmodel 출력 = c_bub=1로 그린 bubble의 expected counts/flux map)
# ============================================================
import os, numpy as np
from astropy.io import fits
import matplotlib.pyplot as plt

ours = './GC_analysis_sanghwan/GC_fermi_bubble_model_12yr_front_clean.fits'
sang = '/home/sanghwan/FermiLAT/Sanghwan/GC_analysis/GC_fermi_bubble_model_12yr_front_clean.fits'

for p in [ours, sang]:
    print(f"{p}: exists={os.path.exists(p)}  "
          f"size={os.path.getsize(p) if os.path.exists(p) else 'N/A'}")

with fits.open(ours) as h:
    odata = h[0].data
    ohdr  = h[0].header
with fits.open(sang) as h:
    sdata = h[0].data
    shdr  = h[0].header

print(f"\n[shape]  ours={odata.shape}  sang={sdata.shape}")
print(f"[dtype]  ours={odata.dtype}  sang={sdata.dtype}")

# 14-bin 분석 범위만 비교 (bin 0-13)
n_bins = min(odata.shape[0], sdata.shape[0], 14)
print(f"\n비교 범위: bin 0-{n_bins-1} (Cholis 14-bin)")

# 각 bin마다 통계
print(f"\n{'bin':>3} {'ours_total':>14} {'sang_total':>14} {'ratio':>8} {'max|Δ|':>12} {'medratio':>10}")
for b in range(n_bins):
    od = odata[b, 100:500, 100:500]
    sd = sdata[b, 100:500, 100:500]
    ot, st = od.sum(), sd.sum()
    ratio = ot / st if st != 0 else np.nan
    diff_max = np.abs(od - sd).max()
    nz = sd != 0
    medr = np.median(od[nz]/sd[nz]) if nz.any() else np.nan
    print(f"{b:>3} {ot:>14.4e} {st:>14.4e} {ratio:>8.4f} {diff_max:>12.4e} {medr:>10.4f}")

# 비교 plot: 첫 분석 bin
b_show = 0
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
od_b = odata[b_show, 100:500, 100:500]
sd_b = sdata[b_show, 100:500, 100:500]

im0 = axes[0].imshow(od_b, origin='lower', cmap='inferno')
axes[0].set_title(f'OURS bin {b_show}\nsum={od_b.sum():.3e}')
plt.colorbar(im0, ax=axes[0], fraction=0.04)

im1 = axes[1].imshow(sd_b, origin='lower', cmap='inferno')
axes[1].set_title(f'SANGHWAN bin {b_show}\nsum={sd_b.sum():.3e}')
plt.colorbar(im1, ax=axes[1], fraction=0.04)

diff = od_b - sd_b
vmax = max(abs(diff.min()), abs(diff.max()))
im2 = axes[2].imshow(diff, origin='lower', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
axes[2].set_title(f'OURS - SANG\nmax|Δ|={np.abs(diff).max():.3e}')
plt.colorbar(im2, ax=axes[2], fraction=0.04)

plt.suptitle(f'V49e: bubble component map (gtmodel 출력) 직접 비교 — bin {b_show}',
              fontsize=11)
plt.tight_layout()
plt.savefig('V49e_bubble_component_compare.png', dpi=100, bbox_inches='tight')
plt.show()

# spatial 분포: nonzero pixel 위치가 같은가
nz_both = (od_b != 0) & (sd_b != 0)
nz_ours_only = (od_b != 0) & (sd_b == 0)
nz_sang_only = (od_b == 0) & (sd_b != 0)
print(f"\n[bin {b_show} nonzero overlap]")
print(f"  common nonzero: {nz_both.sum()} pixels")
print(f"  ours only:      {nz_ours_only.sum()} pixels")
print(f"  sang only:      {nz_sang_only.sum()} pixels")

In [ ]:
# V49f-fix: Sanghwan .dat 결과 파일의 실제 위치 확인
import subprocess
r = subprocess.run(
    ['find', '/home/sanghwan/FermiLAT/Sanghwan',
     '-maxdepth', '6',
     '(', '-name', 'GCE_model_*_12yr*.dat', '-o',
          '-name', '*likelihood_value*', ')',
     '-type', 'f'],
    capture_output=True, text=True, timeout=60)
print(r.stdout)
print(f"--- found {len(r.stdout.strip().splitlines())} files ---")

In [ ]:
# ============================================================
# V49g: 우리 vs Sanghwan vs Cholis paper — Model I, X, XLIX의 GCE flux 직접 비교
# Sanghwan은 c_param raw 저장 안 함, but .dat의 GCE flux로 우회 비교 가능
# ============================================================
import os, numpy as np, matplotlib.pyplot as plt

# 우리: ./GCE_model_{M}_12yr_cholis.dat
# Sanghwan: /home/sanghwan/FermiLAT/Sanghwan/GCE_model_{M}_12yr_cholis.dat 
#           단 model I만 .dat에 _cholis 없음 (메모리 #15)
# Cholis Zenodo: ../GCE_TEMPLATES_FILES_v3/Figures_12_and_14_GCE_Spectra/
#                GCE_Model{M}_flux_Inner40x40_masked_disk.dat

SANG_DIR = '/home/sanghwan/FermiLAT/Sanghwan'
CHOLIS_DIR = '../GCE_TEMPLATES_FILES_v3/Figures_12_and_14_GCE_Spectra'

models = ['I', 'X', 'XLIX']

def load_dat(path):
    if not os.path.exists(path):
        return None
    try:
        a = np.loadtxt(path, comments=('#',))
    except Exception:
        a = np.loadtxt(path)
    return a

fig, axes = plt.subplots(1, len(models), figsize=(5*len(models), 5))
if len(models) == 1:
    axes = [axes]

for ax, M in zip(axes, models):
    # 우리
    ours = load_dat(f'./GCE_model_{M}_12yr_cholis.dat')
    # Sanghwan (I는 _cholis 없음)
    if M == 'I':
        sang = load_dat(f'{SANG_DIR}/GCE_model_I_12yr.dat')
    else:
        sang = load_dat(f'{SANG_DIR}/GCE_model_{M}_12yr_cholis.dat')
    # Cholis Zenodo
    cholis = load_dat(f'{CHOLIS_DIR}/GCE_Model{M}_flux_Inner40x40_masked_disk.dat')
    
    if ours is not None:
        E_o = ours[:, 0]
        flux_o = ours[:, 1]
        if ours.shape[1] >= 5:
        # column 3, 4가 lower/upper "bound"인지 "err"인지 자동 판별
        # bound면 (flux - col3) > 0이어야 함
            cand_err_lo = flux_o - ours[:, 3]
            cand_err_hi = ours[:, 4] - flux_o
            if (cand_err_lo >= 0).all() and (cand_err_hi >= 0).all():
                err_lo, err_hi = cand_err_lo, cand_err_hi
                mode = 'bound'
            else:
                # col 3, 4가 err 자체일 가능성
                err_lo = np.abs(ours[:, 3])
                err_hi = np.abs(ours[:, 4])
                mode = 'err'
            print(f"  [ours mode] {mode}  "
                  f"col3 range [{ours[:,3].min():.3e}, {ours[:,3].max():.3e}]  "
                  f"col4 range [{ours[:,4].min():.3e}, {ours[:,4].max():.3e}]")
            ax.errorbar(E_o, flux_o, yerr=[err_lo, err_hi],
                        fmt='o-', label='OURS', capsize=3, alpha=0.85, color='C0')
        else:
            ax.plot(E_o, ours[:, 1], 'o-', label='OURS', color='C0')
        print(f"[Model {M} ours] E range {E_o[0]:.3f}-{E_o[-1]:.3f}, "
              f"n_bins={len(E_o)}")
    else:
        print(f"[Model {M} ours] MISSING")
    
    if sang is not None:
        E_s = sang[:, 0]
        flux_s = sang[:, 1]
        if sang.shape[1] >= 5:
            cand_err_lo = flux_s - sang[:, 3]
            cand_err_hi = sang[:, 4] - flux_s
            if (cand_err_lo >= 0).all() and (cand_err_hi >= 0).all():
                err_lo, err_hi = cand_err_lo, cand_err_hi
                mode = 'bound'
            else:
                err_lo = np.abs(sang[:, 3])
                err_hi = np.abs(sang[:, 4])
                mode = 'err'
            print(f"  [sang mode] {mode}  "
                  f"col3 range [{sang[:,3].min():.3e}, {sang[:,3].max():.3e}]  "
                  f"col4 range [{sang[:,4].min():.3e}, {sang[:,4].max():.3e}]")
            ax.errorbar(E_s, flux_s, yerr=[err_lo, err_hi],
                        fmt='s--', label='SANG', capsize=3, alpha=0.85, color='C1')
        else:
            ax.plot(E_s, sang[:, 1], 's--', label='SANG', color='C1')
        print(f"[Model {M} sang] E range {E_s[0]:.3f}-{E_s[-1]:.3f}, "
              f"n_bins={len(E_s)}, n_cols={sang.shape[1]}")
    else:
        print(f"[Model {M} sang] MISSING")
    
    if cholis is not None:
        E_c = cholis[:, 0]
        flux_c = cholis[:, 1]
        if cholis.shape[1] >= 4:
            err_lo = flux_c - cholis[:, 2]
            err_hi = cholis[:, 3] - flux_c
            ax.errorbar(E_c, flux_c, yerr=[np.maximum(err_lo, 0),
                                            np.maximum(err_hi, 0)],
                        fmt='^:', label='CHOLIS', capsize=3, alpha=0.85,
                        color='C3')
        else:
            ax.plot(E_c, flux_c, '^:', label='CHOLIS', color='C3')
        print(f"[Model {M} cholis] E range {E_c[0]:.3f}-{E_c[-1]:.3f}, "
              f"n_bins={len(E_c)}, n_cols={cholis.shape[1]}")
    
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel('E [GeV]'); ax.set_ylabel(r'E$^2$dN/dE [GeV/cm²/s/sr]')
    ax.set_title(f'Model {M}')
    ax.grid(alpha=0.3); ax.legend(fontsize=9)

plt.suptitle('V49g: GCE flux — 우리 / Sanghwan / Cholis 직접 비교 (12yr)',
              fontsize=12)
plt.tight_layout()
plt.savefig('V49g_gce_flux_three_way.png', dpi=100, bbox_inches='tight')
plt.show()

# 정량 비교: 1-10 GeV 평균 ratio
print(f"\n{'='*70}")
print(f"1-10 GeV 평균 ratio (각 vs Cholis)")
print(f"{'='*70}")
for M in models:
    ours = load_dat(f'./GCE_model_{M}_12yr_cholis.dat')
    if M == 'I':
        sang = load_dat(f'{SANG_DIR}/GCE_model_I_12yr.dat')
    else:
        sang = load_dat(f'{SANG_DIR}/GCE_model_{M}_12yr_cholis.dat')
    cholis = load_dat(f'{CHOLIS_DIR}/GCE_Model{M}_flux_Inner40x40_masked_disk.dat')
    
    if ours is None or cholis is None:
        continue
    # 1-10 GeV mask
    def in_range(E, v):
        m = (E >= 1) & (E <= 10)
        return np.mean(v[m]) if m.any() else np.nan
    
    flux_c_at_oursE = np.interp(ours[:,0], cholis[:,0], cholis[:,1])
    r_o = in_range(ours[:,0], ours[:,1] / flux_c_at_oursE)
    
    if sang is not None:
        flux_c_at_sangE = np.interp(sang[:,0], cholis[:,0], cholis[:,1])
        r_s = in_range(sang[:,0], sang[:,1] / flux_c_at_sangE)
    else:
        r_s = np.nan
    
    print(f"  Model {M}:  ours/cholis={r_o:.3f}   sang/cholis={r_s:.3f}")